In [0]:
select max(ingestion_date) from com_raw.kom_medical_events

In [0]:
CREATE OR REPLACE TEMPORARY VIEW global_runtime_dates AS
SELECT 
    /* ELAPRASE */
    DATE('2020-08-01') AS elaprase_dx_start_date,
    DATE('2023-08-01') AS elaprase_tx_start_date,

    /* AVLAYAH */
    -- DATE('2022-08-01') AS avlayah_dx_start_date,
    DATE('2026-03-01') AS avlayah_tx_start_date;

In [0]:
CREATE OR REPLACE TEMP VIEW runtime_parameters AS

SELECT
    (SELECT MAX(service_date) FROM com_edp_prd.com_raw.kom_medical_events) AS max_medical_date,

    (SELECT MAX(fill_date) FROM com_edp_prd.com_raw.kom_pharmacy_events) AS max_pharmacy_date,

    LAST_DAY(
        ADD_MONTHS(
            LEAST(
                (SELECT MAX(service_date) FROM com_edp_prd.com_raw.kom_medical_events),
                (SELECT MAX(fill_date) FROM com_edp_prd.com_raw.kom_pharmacy_events)
            ), -1
        )
    ) AS end_date,

    CURRENT_DATE() AS run_date;

    SELECT * FROM runtime_parameters;

In [0]:
-- CREATE OR REPLACE TEMP VIEW patient_hcp_visit_summary AS
-- WITH MPSII_Diagnoses_Specified AS (

--     SELECT DISTINCT
--         PATIENT_ID,
--         SERVICE_DATE AS DX_DATE
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE DIAGNOSIS_CODES LIKE '%E761%'

--     UNION

--     SELECT DISTINCT
--         PATIENT_ID,
--         FILL_DATE AS DX_DATE
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE DIAGNOSIS_CODE = 'E761'
--       AND TRANSACTION_STATUS = 'PAID'
-- ),

-- Patients_2Dx_Specified AS (

--     SELECT PATIENT_ID
--     FROM MPSII_Diagnoses_Specified
--     GROUP BY PATIENT_ID
--     HAVING COUNT(DISTINCT DX_DATE) >= 2
-- ),

-- /* ============================================================================
--    MPSII Treatment - Elaprase Ever (NEW COHORT DEFINITION)
--    ========================================================================== */
-- MPSII_Treatment_Elaprase_Ever AS (

--     SELECT DISTINCT PATIENT_ID
--     FROM (

--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE NDC11 IN ('54092070001','540920700')

--         UNION ALL

--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_pharmacy_events
--         WHERE NDC11 IN ('54092070001','540920700')
--           AND TRANSACTION_RESULT = 'PAID'

--         UNION ALL

--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE PROCEDURE_CODE = 'J1743'

--     ) t
-- ),

-- /* ============================================================================
--    Base MPS II Universe (NEW COHORT DEFINITION)
--    ========================================================================== */
-- Base_MPSII_Universe AS (

--     SELECT PATIENT_ID
--     FROM Patients_2Dx_Specified

--     UNION

--     SELECT PATIENT_ID
--     FROM MPSII_Treatment_Elaprase_Ever
-- ),

-- /* ============================================================================
--    Recent Treatment Activity (NEW COHORT DEFINITION - using global_runtime_dates)
--    ========================================================================== */
-- Recent_Treatment_2Y AS (

--     SELECT DISTINCT PATIENT_ID
--     FROM (

--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE NDC11 IN ('54092070001','540920700')
--           AND SERVICE_DATE >= (SELECT elaprase_tx_start_date FROM global_runtime_dates)

--         UNION ALL

--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_pharmacy_events
--         WHERE NDC11 IN ('54092070001','540920700')
--           AND TRANSACTION_RESULT = 'PAID'
--           AND FILL_DATE >= (SELECT elaprase_tx_start_date FROM global_runtime_dates)

--         UNION ALL

--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE PROCEDURE_CODE = 'J1743'
--           AND SERVICE_DATE >= (SELECT elaprase_tx_start_date FROM global_runtime_dates)

--         UNION ALL

--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
--           AND SERVICE_DATE >= (SELECT elaprase_tx_start_date FROM global_runtime_dates)

--     ) t
-- ),

-- /* ============================================================================
--    FINAL ELIGIBLE PATIENTS (NEW COHORT DEFINITION)
--    ========================================================================== */
-- eligible_patients AS (

--     SELECT DISTINCT b.PATIENT_ID
--     FROM Base_MPSII_Universe b
--     INNER JOIN Recent_Treatment_2Y r
--         ON b.PATIENT_ID = r.PATIENT_ID
-- ),

-- /* ============================================================================
--    2) PROVIDER FILTER ("COHORT 3 LEARNINGS")
--    ========================================================================== */
-- cohort_3_learnings AS (
--   SELECT DISTINCT npi
--   FROM com_raw.kom_providers
--   WHERE provider_type = 'INDIVIDUAL'
--     AND (
--       PRIMARY_SPECIALTY NOT IN (
--         'Anesthesiologist Assistant','Anesthesiology','Dentist','Dietitian, Registered',
--         'Emergency Medical Technician, Basic','Emergency Medicine','General Acute Care Hospital',
--         'Nurse Anesthetist, Certified Registered','Obstetrics & Gynecology','Pathology',
--         'Radiology','Urology'
--       )
--       OR SECONDARY_SPECIALTY IN (
--         'Child & Adolescent Psychiatry','Psychiatry','Adolescent Medicine','Developmental - Behavioral Pediatrics',
--         'Neonatal-Perinatal Medicine','Nutrition, Pediatric','Oncology, Pediatrics','Pediatric Cardiology',
--         'Pediatric Critical Care Medicine','Pediatric Dermatology','Pediatric Emergency Medicine',
--         'Pediatric Endocrinology','Pediatric Gastroenterology','Pediatric Hematology-Oncology',
--         'Pediatric Infectious Diseases','Pediatric Nephrology','Pediatric Ophthalmology and Strabismus Specialist',
--         'Pediatric Orthopaedic Surgery','Pediatric Otolaryngology','Pediatric Pulmonology','Pediatric Radiology',
--         'Pediatric Rehabilitation Medicine','Pediatric Rheumatology','Pediatric Surgery','Pediatrics',
--         'Clinical Biochemical Genetics','Clinical Genetics (M.D.)','Clinical Molecular Genetics',
--         'Ph.D. Medical Genetics','Neurodevelopmental Disabilities','Neurology',
--         'Neurology with Special Qualifications in Child Neurology','Neuroradiology'
--       )
--     )
-- ),

-- /* ============================================================================
--    3) CLAIMS UNIVERSES (DX + TX) WITH NPI ATTRIBUTION
--    ========================================================================== */

-- all_dx_claims_5yr AS (
--     SELECT DISTINCT *
--     FROM (
--       SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
--         AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--       UNION
--       SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
--       FROM com_edp_prd.com_raw.kom_pharmacy_events
--       WHERE DIAGNOSIS_CODE IN ('E761','E763')
--         AND TRANSACTION_STATUS = 'PAID'
--         AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     )
--     WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
-- ),

-- all_tx_claims_5yr AS (
--     SELECT DISTINCT *
--     FROM (
--       SELECT DISTINCT
--           PATIENT_ID,
--           COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
--           SERVICE_DATE AS FILL_DATE,
--           NDC11 AS TX_CODE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE NDC11 IN ('54092070001','540920700')
--         AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--       UNION

--       SELECT DISTINCT
--           PATIENT_ID,
--           PRESCRIBER_NPI AS NPI,
--           FILL_DATE,
--           NDC11 AS TX_CODE
--       FROM com_edp_prd.com_raw.kom_pharmacy_events
--       WHERE NDC11 IN ('54092070001','540920700')
--         AND TRANSACTION_RESULT = 'PAID'
--         AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--       UNION

--       SELECT DISTINCT
--           PATIENT_ID,
--           RENDERING_NPI AS NPI,
--           SERVICE_DATE AS FILL_DATE,
--           PROCEDURE_CODE AS TX_CODE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
--                                '38206','38230','38232','38240','38241','38242','38243','38250')
--         AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     )
--     WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
-- ),

-- all_tx_claims_alltime AS (
--     SELECT DISTINCT *
--     FROM (
--       SELECT DISTINCT
--           PATIENT_ID,
--           COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
--           SERVICE_DATE AS FILL_DATE,
--           NDC11 AS TX_CODE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE NDC11 IN ('54092070001','540920700')
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--       UNION

--       SELECT DISTINCT
--           PATIENT_ID,
--           PRESCRIBER_NPI AS NPI,
--           FILL_DATE,
--           NDC11 AS TX_CODE
--       FROM com_edp_prd.com_raw.kom_pharmacy_events
--       WHERE NDC11 IN ('54092070001','540920700')
--         AND TRANSACTION_RESULT = 'PAID'
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--       UNION

--       SELECT DISTINCT
--           PATIENT_ID,
--           RENDERING_NPI AS NPI,
--           SERVICE_DATE AS FILL_DATE,
--           PROCEDURE_CODE AS TX_CODE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
--                                '38206','38230','38232','38240','38241','38242','38243','38250')
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     )
--     WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
-- ),

-- all_claims_5yr AS (
--     SELECT DISTINCT
--         PATIENT_ID,
--         NPI,
--         FILL_DATE,
--         CAST(NULL AS STRING) AS TX_CODE
--     FROM all_dx_claims_5yr
--     UNION
--     SELECT DISTINCT
--         PATIENT_ID,
--         NPI,
--         FILL_DATE,
--         TX_CODE
--     FROM all_tx_claims_5yr
-- ),

-- all_claims_3yr AS (
--     SELECT DISTINCT *
--     FROM (
--       SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
--         AND SERVICE_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--       UNION
--       SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
--       FROM com_edp_prd.com_raw.kom_pharmacy_events
--       WHERE DIAGNOSIS_CODE IN ('E761','E763')
--         AND TRANSACTION_STATUS = 'PAID'
--         AND FILL_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--       UNION
--       SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE NDC11 IN ('54092070001','540920700')
--         AND SERVICE_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--       UNION
--       SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
--       FROM com_edp_prd.com_raw.kom_pharmacy_events
--       WHERE NDC11 IN ('54092070001','540920700')
--         AND TRANSACTION_RESULT = 'PAID'
--         AND FILL_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--       UNION
--       SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI, SERVICE_DATE AS FILL_DATE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
--                                '38206','38230','38232','38240','38241','38242','38243','38250')
--         AND SERVICE_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     )
--     WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
-- ),

-- avlayah_ndc_events AS (
--     SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE NDC11 = '8497600101'
--       AND SERVICE_DATE >= '2026-03-01'
--       AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--     UNION

--     SELECT DISTINCT PATIENT_ID, FILL_DATE
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE NDC11 = '8497600101'
--       AND TRANSACTION_RESULT = 'PAID'
--       AND FILL_DATE >= '2026-03-01'
--       AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
-- ),

-- /* ============================================================================
--    4) FIRST DX / FIRST TX HCP ATTRIBUTION (5Y WINDOW)
--    ========================================================================== */

-- first_dx_hcp_ranked AS (
--     SELECT PATIENT_ID, NPI, MIN(FILL_DATE) AS first_dx_date,
--            ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY MIN(FILL_DATE) ASC, NPI) AS rn
--     FROM all_dx_claims_5yr
--     WHERE NPI IS NOT NULL
--     GROUP BY PATIENT_ID, NPI
-- ),
-- first_dx_hcp AS (
--     SELECT PATIENT_ID, NPI AS first_dx_hcp, first_dx_date
--     FROM first_dx_hcp_ranked
--     WHERE rn = 1
-- ),

-- provider_dim AS (
--     SELECT
--         npi,
--         CONCAT(FIRST_NAME, ' ', LAST_NAME) AS provider_name,
--         primary_specialty
--     FROM com_raw.kom_providers
--     WHERE provider_type = 'INDIVIDUAL'
-- ),

-- first_dx_hcp_5yr_stats AS (
--     SELECT fdh.PATIENT_ID, fdh.first_dx_hcp,
--            COUNT(DISTINCT ac.FILL_DATE) AS first_dx_all_visit_count_5yr,
--            MAX(ac.FILL_DATE) AS first_dx_last_visit_5yr
--     FROM first_dx_hcp fdh
--     LEFT JOIN all_claims_5yr ac
--       ON fdh.PATIENT_ID = ac.PATIENT_ID AND fdh.first_dx_hcp = ac.NPI
--     GROUP BY fdh.PATIENT_ID, fdh.first_dx_hcp
-- ),

-- first_tx_hcp_ranked AS (
--     SELECT PATIENT_ID, NPI, MIN(FILL_DATE) AS first_tx_date,
--            ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY MIN(FILL_DATE) ASC, NPI) AS rn
--     FROM all_tx_claims_5yr
--     WHERE NPI IS NOT NULL
--     GROUP BY PATIENT_ID, NPI
-- ),
-- first_tx_hcp AS (
--     SELECT PATIENT_ID, NPI AS first_tx_hcp, first_tx_date
--     FROM first_tx_hcp_ranked
--     WHERE rn = 1
-- ),

-- first_tx_hcp_5yr_stats AS (
--     SELECT fth.PATIENT_ID, fth.first_tx_hcp,
--            COUNT(DISTINCT ac.FILL_DATE) AS first_tx_all_visit_count_5yr,
--            MAX(ac.FILL_DATE) AS first_tx_last_visit_5yr
--     FROM first_tx_hcp fth
--     LEFT JOIN all_claims_5yr ac
--       ON fth.PATIENT_ID = ac.PATIENT_ID AND fth.first_tx_hcp = ac.NPI
--     GROUP BY fth.PATIENT_ID, fth.first_tx_hcp
-- ),

-- first_tx_hcp_5yr_tx_only AS (
--     SELECT fth.PATIENT_ID, fth.first_tx_hcp,
--            COUNT(DISTINCT tx.FILL_DATE) AS first_tx_treatment_visit_count_5yr
--     FROM first_tx_hcp fth
--     LEFT JOIN all_tx_claims_5yr tx
--       ON fth.PATIENT_ID = tx.PATIENT_ID AND fth.first_tx_hcp = tx.NPI
--     GROUP BY fth.PATIENT_ID, fth.first_tx_hcp
-- ),

-- /* ============================================================================
--    5) MOST-SEEN HCP RANKING (TOP 5) USING 3Y ACTIVITY
--    ========================================================================== */

-- most_seen_3yr_ranking AS (
--     SELECT PATIENT_ID,
--            NPI,
--            COUNT(DISTINCT FILL_DATE) AS visit_count_3yr,
--            MAX(FILL_DATE) AS last_visit_3yr,
--            ROW_NUMBER() OVER (
--              PARTITION BY PATIENT_ID
--              ORDER BY COUNT(DISTINCT FILL_DATE) DESC,
--                       MAX(FILL_DATE) DESC,
--                       NPI ASC
--            ) AS rank
--     FROM all_claims_3yr
--     WHERE NPI IS NOT NULL
--     GROUP BY PATIENT_ID, NPI
-- ),

-- most_seen_combined_stats AS (
--     SELECT
--         ms3.PATIENT_ID,
--         ms3.NPI,
--         ms3.rank,
--         ms3.visit_count_3yr,
--         ms3.last_visit_3yr,
--         COUNT(DISTINCT ac5.FILL_DATE) AS visit_count_5yr,
--         MAX(ac5.FILL_DATE)          AS last_visit_5yr
--     FROM most_seen_3yr_ranking ms3
--     LEFT JOIN all_claims_5yr ac5
--       ON ms3.PATIENT_ID = ac5.PATIENT_ID AND ms3.NPI = ac5.NPI
--     WHERE ms3.rank <= 5
--     GROUP BY ms3.PATIENT_ID, ms3.NPI, ms3.rank, ms3.visit_count_3yr, ms3.last_visit_3yr
-- ),

-- /* ============================================================================
--    6) HISTORICAL (ALL-TIME) FIRST DX / FIRST TX DATES (PATIENT LEVEL)
--    ========================================================================== */

-- historical_first_dx AS (
--     SELECT PATIENT_ID, MIN(FILL_DATE) AS incidence_date
--     FROM (
--         SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE DIAGNOSIS_CODES LIKE '%E761%'
--           AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID, FILL_DATE
--         FROM com_edp_prd.com_raw.kom_pharmacy_events
--         WHERE DIAGNOSIS_CODE = 'E761'
--           AND TRANSACTION_STATUS = 'PAID'
--           AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE DIAGNOSIS_CODES LIKE '%E763%'
--           AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID, FILL_DATE
--         FROM com_edp_prd.com_raw.kom_pharmacy_events
--         WHERE DIAGNOSIS_CODE = 'E763'
--           AND TRANSACTION_STATUS = 'PAID'
--           AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     ) all_dx
--     GROUP BY PATIENT_ID
-- ),

-- historical_first_tx AS (
--     SELECT PATIENT_ID, MIN(FILL_DATE) AS first_incidence_treatment_date
--     FROM (
--         SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE NDC11 IN ('54092070001','540920700')
--           AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID, FILL_DATE
--         FROM com_edp_prd.com_raw.kom_pharmacy_events
--         WHERE NDC11 IN ('54092070001','540920700')
--           AND TRANSACTION_RESULT = 'PAID'
--           AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
--                                  '38206','38230','38232','38240','38241','38242','38243','38250')
--           AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     ) all_tx
--     GROUP BY PATIENT_ID
-- ),

-- /* ============================================================================
--    7) LATEST HCP ATTRIBUTION (5Y WINDOW)
--    ========================================================================== */

-- all_claims_5yr_specialty_removed AS (
--     SELECT DISTINCT *
--     FROM (
--       SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
--         AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--       UNION
--       SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
--       FROM com_edp_prd.com_raw.kom_pharmacy_events
--       WHERE DIAGNOSIS_CODE IN ('E761','E763')
--         AND TRANSACTION_STATUS = 'PAID'
--         AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--       UNION
--       SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE NDC11 IN ('54092070001','540920700')
--         AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--       UNION
--       SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
--       FROM com_edp_prd.com_raw.kom_pharmacy_events
--       WHERE NDC11 IN ('54092070001','540920700')
--         AND TRANSACTION_RESULT = 'PAID'
--         AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--       UNION
--       SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI, SERVICE_DATE AS FILL_DATE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
--                                '38206','38230','38232','38240','38241','38242','38243','38250')
--         AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     )
-- ),

-- latest_claim_hcp_ranked as (
--   select patient_id, npi, fill_date, row_number() over(partition by patient_id order by fill_date desc, npi asc) as rn
--   from all_claims_5yr
-- ),

-- latest_claim_hcp as (
--   select patient_id, npi as latest_claim_hcp_npi, fill_date as latest_claim_date from latest_claim_hcp_ranked where rn = 1
-- ),

-- latest_claim_hcp_visit_count_5yr as (
--   select a.patient_id, a.latest_claim_hcp_npi, count(distinct fill_date) as latest_claim_hcp_visit_count_5yr
--   from latest_claim_hcp as a
--   left join all_claims_5yr as b on a.patient_id = b.patient_id and a.latest_claim_hcp_npi = b.npi
--   group by 1,2
-- ),

-- latest_claim_hcp_final as (
--   select a.patient_id, a.latest_claim_hcp_npi, a.latest_claim_date, b.latest_claim_hcp_visit_count_5yr
--   from latest_claim_hcp as a
--   left join latest_claim_hcp_visit_count_5yr as b on a.patient_id = b.patient_id
-- ),

-- all_tx_claims_5yr_specialty_removed AS (
--     SELECT DISTINCT *
--     FROM (
--       SELECT DISTINCT
--           PATIENT_ID,
--           COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
--           SERVICE_DATE AS FILL_DATE,
--           NDC11 AS TX_CODE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE NDC11 IN ('54092070001','540920700')
--         AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--       UNION

--       SELECT DISTINCT
--           PATIENT_ID,
--           PRESCRIBER_NPI AS NPI,
--           FILL_DATE,
--           NDC11 AS TX_CODE
--       FROM com_edp_prd.com_raw.kom_pharmacy_events
--       WHERE NDC11 IN ('54092070001','540920700')
--         AND TRANSACTION_RESULT = 'PAID'
--         AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--       UNION

--       SELECT DISTINCT
--           PATIENT_ID,
--           RENDERING_NPI AS NPI,
--           SERVICE_DATE AS FILL_DATE,
--           PROCEDURE_CODE AS TX_CODE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
--                                '38206','38230','38232','38240','38241','38242','38243','38250')
--         AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     )
-- ),

-- most_recent_tx_hcp_ranked as (
--   select patient_id, npi, fill_date, row_number() over(partition by patient_id order by fill_date desc, npi asc) as rn
--   from all_tx_claims_5yr
-- ),

-- most_recent_tx_hcp as (
--   select patient_id, npi as latest_treatment_hcp_npi, fill_date as latest_treatment_date
--   from most_recent_tx_hcp_ranked where rn = 1
-- ),

-- latest_treatment_hcp_visit_count as (
--   select a.patient_id, a.latest_treatment_hcp_npi, count(distinct fill_date) as latest_treatment_hcp_visit_count_5yr
--   from most_recent_tx_hcp as a
--   left join all_tx_claims_5yr as b on a.patient_id = b.patient_id and a.latest_treatment_hcp_npi = b.npi
--   group by 1, 2
-- ),

-- most_recent_tx_hcp_final as (
--   select a.patient_id, a.latest_treatment_hcp_npi, a.latest_treatment_date, b.latest_treatment_hcp_visit_count_5yr
--   from most_recent_tx_hcp as a
--   left join latest_treatment_hcp_visit_count as b on a.patient_id = b.patient_id
-- ),

-- /* ============================================================================
--    9) LATEST TX TYPE
--    Derived directly from recent 2Y treatment claims:
--      - Elaprase NDC, pharmacy NDC, or J1743 -> 'ELAPRASE'
--      - Other infusion/transplant procedure codes -> 'OTHER_TX'
--    Self-contained so every eligible patient gets a non-null label.
--    ========================================================================== */

-- recent_2yr_treatment_claims AS (

--     SELECT
--         PATIENT_ID,
--         SERVICE_DATE AS TX_DATE,
--         'ELAPRASE' AS TX_TYPE
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE NDC11 IN ('54092070001','540920700')
--       AND SERVICE_DATE >= (SELECT elaprase_tx_start_date FROM global_runtime_dates)
--       AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--     UNION ALL

--     SELECT
--         PATIENT_ID,
--         FILL_DATE AS TX_DATE,
--         'ELAPRASE' AS TX_TYPE
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE NDC11 IN ('54092070001','540920700')
--       AND TRANSACTION_RESULT = 'PAID'
--       AND FILL_DATE >= (SELECT elaprase_tx_start_date FROM global_runtime_dates)
--       AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--     UNION ALL

--     SELECT
--         PATIENT_ID,
--         SERVICE_DATE AS TX_DATE,
--         'ELAPRASE' AS TX_TYPE
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE PROCEDURE_CODE = 'J1743'
--       AND SERVICE_DATE >= (SELECT elaprase_tx_start_date FROM global_runtime_dates)
--       AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--     UNION ALL

--     SELECT
--         PATIENT_ID,
--         SERVICE_DATE AS TX_DATE,
--         'OTHER_TX' AS TX_TYPE
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','S9357','S9379',
--                              '38206','38230','38232','38240','38241','38242','38243','38250')
--       AND SERVICE_DATE >= (SELECT elaprase_tx_start_date FROM global_runtime_dates)
--       AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
-- ),

-- latest_mpsii_treatment_type AS (
--     SELECT PATIENT_ID, latest_mpsii_tx_type
--     FROM (
--         SELECT
--             PATIENT_ID,
--             TX_TYPE AS latest_mpsii_tx_type,
--             TX_DATE,
--             ROW_NUMBER() OVER (
--                 PARTITION BY PATIENT_ID
--                 ORDER BY TX_DATE DESC
--             ) AS rn
--         FROM recent_2yr_treatment_claims
--     ) t
--     WHERE rn = 1
-- ),

-- /* ============================================================================
--    9b) LATEST TREATMENT TYPE (NEW COLUMN)
--    Business rules:
--      - Avlayah presence: NDC 8497600101 ONLY (no proxy J-codes)
--      - Avlayah is sticky against Other ERT: once a patient has Avlayah,
--        any subsequent Other ERT proc does NOT override -- they stay AVLAYAH
--      - Avlayah is NOT sticky against Elaprase: if the patient has Elaprase
--        (NDC or J1743) AFTER their last Avlayah, show ELAPRASE
--      - Patients without Avlayah: latest by date wins between Elaprase
--        (NDC + J1743) and Other ERT (other procedure codes)
--    ========================================================================== */

-- elaprase_latest_event AS (
--     SELECT
--         PATIENT_ID,
--         MAX(FILL_DATE) AS latest_elaprase_date
--     FROM all_tx_claims_5yr
--     WHERE TX_CODE IN ('54092070001','540920700','J1743')
--     GROUP BY PATIENT_ID
-- ),

-- other_ert_latest_event AS (
--     SELECT
--         PATIENT_ID,
--         MAX(FILL_DATE) AS latest_other_ert_date
--     FROM all_tx_claims_5yr
--     WHERE TX_CODE IN ('99601','99602','96365','96366','S9357','S9379',
--                       '38206','38230','38232','38240','38241','38242','38243','38250')
--     GROUP BY PATIENT_ID
-- ),

-- avlayah_latest_event AS (
--     SELECT
--         PATIENT_ID,
--         MAX(FILL_DATE) AS latest_avlayah_date
--     FROM avlayah_ndc_events
--     GROUP BY PATIENT_ID
-- ),

-- latest_treatment_type AS (
--     SELECT
--         ep.PATIENT_ID,
--         CASE
--             WHEN av.latest_avlayah_date IS NOT NULL
--              AND el.latest_elaprase_date IS NOT NULL
--              AND el.latest_elaprase_date > av.latest_avlayah_date
--                 THEN 'ELAPRASE'

--             WHEN av.latest_avlayah_date IS NOT NULL
--                 THEN 'AVLAYAH'

--             WHEN el.latest_elaprase_date IS NOT NULL
--              AND (
--                     oe.latest_other_ert_date IS NULL
--                  OR el.latest_elaprase_date >= oe.latest_other_ert_date
--                  )
--                 THEN 'ELAPRASE'

--             WHEN oe.latest_other_ert_date IS NOT NULL
--                 THEN 'OTHER_ERT'

--             ELSE lm.latest_mpsii_tx_type
--         END AS latest_treatment_type

--     FROM eligible_patients ep

--     LEFT JOIN avlayah_latest_event av
--         ON ep.PATIENT_ID = av.PATIENT_ID

--     LEFT JOIN elaprase_latest_event el
--         ON ep.PATIENT_ID = el.PATIENT_ID

--     LEFT JOIN other_ert_latest_event oe
--         ON ep.PATIENT_ID = oe.PATIENT_ID

--     LEFT JOIN latest_mpsii_treatment_type lm
--         ON ep.PATIENT_ID = lm.PATIENT_ID
-- ),

-- /* ============================================================================
--    10) FIRST TX AFTER DIAGNOSIS (ALL-TIME TX, BUT MUST BE AFTER DX)
--    ========================================================================== */

-- first_tx_after_diagnosis AS (
--   SELECT
--     tx.patient_id,
--     MIN(tx.fill_date) AS first_tx_after_diagnosis
--   FROM all_tx_claims_alltime tx
--   INNER JOIN historical_first_dx dx
--     ON tx.patient_id = dx.patient_id
--   WHERE tx.fill_date >= dx.incidence_date
--   GROUP BY tx.patient_id
-- ),

-- /* ============================================================================
--    11) ELAPRASE FILL COUNTS (WINDOWED)
--    ========================================================================== */

-- elaprase_fills AS (
--   SELECT
--     patient_id,
--     COUNT(DISTINCT fill_date) AS elaprase_fills
--   FROM all_tx_claims_5yr
--   WHERE fill_date BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--     AND tx_code IN ('54092070001','540920700','J1743')
--   GROUP BY patient_id
-- ),

-- /* ============================================================================
--    12) AVLAYAH STATUS LOGIC
--    FIXED: avlayah_status_base now reads from avlayah_ndc_events (NDC-only
--    confirmed Avlayah events from raw tables) for has_avlayah and
--    first_avlayah_date. has_elaprase keeps original definition (any prior
--    Elaprase/J1743/Other ERT proc) so that Switched fires on ANY conversion
--    to Avlayah NDC, per business rule.
--    ========================================================================== */

-- avlayah_status_base AS (
--   SELECT
--     ep.patient_id,
--     av.first_avlayah_date,
--     CASE WHEN av.first_avlayah_date IS NOT NULL THEN 1 ELSE 0 END AS has_avlayah,
--     MAX(
--       CASE
--         WHEN tx.tx_code IN (
--           '99601','99602','96365','96366','J1743','S9357','S9379',
--           '38206','38230','38232','38240','38241','38242','38243','38250',
--           '54092070001','540920700'
--         )
--         THEN 1 ELSE 0
--       END
--     ) AS has_elaprase
--   FROM eligible_patients ep
--   LEFT JOIN (
--     SELECT patient_id, MIN(fill_date) AS first_avlayah_date
--     FROM avlayah_ndc_events
--     GROUP BY patient_id
--   ) av ON ep.patient_id = av.patient_id
--   LEFT JOIN all_tx_claims_5yr tx ON ep.patient_id = tx.patient_id
--   GROUP BY ep.patient_id, av.first_avlayah_date
-- ),

-- avlayah_status AS (
--   SELECT
--     patient_id,
--     first_avlayah_date,
--     has_avlayah,
--     has_elaprase,
--     CASE
--       WHEN has_avlayah = 1 AND has_elaprase = 0 THEN 'New'
--       WHEN has_avlayah = 1 AND has_elaprase = 1 THEN 'Switched'
--       WHEN has_avlayah = 0 AND has_elaprase = 1 THEN 'Non-Avlayah'
--       ELSE NULL
--     END AS avlayah_status
--   FROM avlayah_status_base
-- ),

-- /* ============================================================================
--    13) PATIENT DIMENSIONS
--    ========================================================================== */

-- patient_demographics AS (
--     SELECT *
--     FROM (
--         SELECT DISTINCT PATIENT_ID, PATIENT_YOB, PATIENT_GENDER,
--                ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY PATIENT_YOB ASC) AS rn
--         FROM com_edp_prd.com_raw.kom_patient_demographics
--     )
--     WHERE rn = 1
-- ),
-- patient_geography AS (
--     SELECT patient_id, patient_state
--     FROM (
--         SELECT
--             PATIENT_ID,
--             patient_state,
--             ROW_NUMBER() OVER (
--                 PARTITION BY PATIENT_ID
--                 ORDER BY
--                     CASE WHEN VALID_TO_DATE > CURRENT_DATE() THEN 1 ELSE 2 END,
--                     VALID_TO_DATE DESC
--             ) AS rn
--         FROM COM_EDP_PRD.COM_RAW.KOM_PATIENT_GEOGRAPHY
--     )
--     WHERE rn = 1
-- ),

-- /* ============================================================================
--    14) PIVOT TOP-5 MOST-SEEN HCPs INTO WIDE FORMAT
--    ========================================================================== */
-- most_seen_pivot AS (
--     SELECT
--         PATIENT_ID,

--         MAX(CASE WHEN rank = 1 THEN NPI END)              AS most_seen_hcp1_3yr_ranked,
--         MAX(CASE WHEN rank = 1 THEN visit_count_5yr END)  AS most_seen_hcp1_visit_count_5yr,
--         MAX(CASE WHEN rank = 1 THEN last_visit_5yr END)   AS most_seen_hcp1_last_visit_5yr,

--         MAX(CASE WHEN rank = 2 THEN NPI END)              AS most_seen_hcp2_3yr_ranked,
--         MAX(CASE WHEN rank = 2 THEN visit_count_5yr END)  AS most_seen_hcp2_visit_count_5yr,
--         MAX(CASE WHEN rank = 2 THEN last_visit_5yr END)   AS most_seen_hcp2_last_visit_5yr,

--         MAX(CASE WHEN rank = 3 THEN NPI END)              AS most_seen_hcp3_3yr_ranked,
--         MAX(CASE WHEN rank = 3 THEN visit_count_5yr END)  AS most_seen_hcp3_visit_count_5yr,
--         MAX(CASE WHEN rank = 3 THEN last_visit_5yr END)   AS most_seen_hcp3_last_visit_5yr,

--         MAX(CASE WHEN rank = 4 THEN NPI END)              AS most_seen_hcp4_3yr_ranked,
--         MAX(CASE WHEN rank = 4 THEN visit_count_5yr END)  AS most_seen_hcp4_visit_count_5yr,
--         MAX(CASE WHEN rank = 4 THEN last_visit_5yr END)   AS most_seen_hcp4_last_visit_5yr,

--         MAX(CASE WHEN rank = 5 THEN NPI END)              AS most_seen_hcp5_3yr_ranked,
--         MAX(CASE WHEN rank = 5 THEN visit_count_5yr END)  AS most_seen_hcp5_visit_count_5yr,
--         MAX(CASE WHEN rank = 5 THEN last_visit_5yr END)   AS most_seen_hcp5_last_visit_5yr

--     FROM most_seen_combined_stats
--     GROUP BY PATIENT_ID
-- )

-- /* ============================================================================
--    FINAL SELECT
--    ========================================================================== */
-- SELECT
--     ep.PATIENT_ID,
--     pd.PATIENT_YOB,
--     YEAR(CURRENT_DATE) - YEAR(pd.PATIENT_YOB) AS PATIENT_AGE,
--     pd.PATIENT_GENDER,
--     pg.patient_state,

--     hfdx.incidence_date,
--     hftx.first_incidence_treatment_date,

--     lch.latest_claim_date AS latest_claim_date,

--     lch.latest_claim_hcp_npi,
--     pdlch.provider_name     AS latest_claim_hcp_name,
--     pdlch.primary_specialty AS latest_claim_hcp_specialty,
--     COALESCE(lch.latest_claim_hcp_visit_count_5yr, 0) AS latest_claim_hcp_visit_count,

--     ref1.hco_name AS latest_claim_hcp_hco_name,

--     mrt.latest_treatment_date AS latest_treatment_date,

--     lmt.latest_mpsii_tx_type,
--     ltt.latest_treatment_type,

--     fta.first_tx_after_diagnosis,

--     ROUND(MONTHS_BETWEEN(fta.first_tx_after_diagnosis, hfdx.incidence_date), 0)
--       AS time_dx_to_first_tx_in_months,

--     ROUND(MONTHS_BETWEEN(mrt.latest_treatment_date, fta.first_tx_after_diagnosis), 0) AS treatment_period_months,

--     COALESCE(ef.elaprase_fills, 0) AS elaprase_fills,
--     avs.avlayah_status,

--     CASE
--       WHEN avs.avlayah_status = 'Switched'
--       THEN avs.first_avlayah_date
--     END AS avlayah_switch_date,
--     mrt.latest_treatment_hcp_npi,
--     pdtch.provider_name     AS latest_treatment_hcp_name,
--     pdtch.primary_specialty AS latest_treatment_hcp_specialty,
--     COALESCE(mrt.latest_treatment_hcp_visit_count_5yr, 0) AS latest_treatment_hcp_visit_count,

--     ref2.hco_name AS latest_treatment_hcp_hco_name,

--     fdh.first_dx_hcp AS first_dx_hcp_5yr,
--     COALESCE(fdhs.first_dx_all_visit_count_5yr, 0) AS first_dx_all_visit_count_5yr,
--     fdhs.first_dx_last_visit_5yr AS first_dx_last_visit_5yr,

--     fth.first_tx_hcp AS first_tx_hcp_5yr,
--     COALESCE(fths.first_tx_all_visit_count_5yr, 0) AS first_tx_all_visit_count_5yr,
--     COALESCE(fthtx.first_tx_treatment_visit_count_5yr, 0) AS first_tx_treatment_visit_count_5yr,
--     fths.first_tx_last_visit_5yr AS first_tx_last_visit_5yr,

--     msp.most_seen_hcp1_3yr_ranked,
--     COALESCE(msp.most_seen_hcp1_visit_count_5yr, 0) AS most_seen_hcp1_visit_count_5yr,
--     msp.most_seen_hcp1_last_visit_5yr,

--     msp.most_seen_hcp2_3yr_ranked,
--     COALESCE(msp.most_seen_hcp2_visit_count_5yr, 0) AS most_seen_hcp2_visit_count_5yr,
--     msp.most_seen_hcp2_last_visit_5yr,

--     msp.most_seen_hcp3_3yr_ranked,
--     COALESCE(msp.most_seen_hcp3_visit_count_5yr, 0) AS most_seen_hcp3_visit_count_5yr,
--     msp.most_seen_hcp3_last_visit_5yr,

--     msp.most_seen_hcp4_3yr_ranked,
--     COALESCE(msp.most_seen_hcp4_visit_count_5yr, 0) AS most_seen_hcp4_visit_count_5yr,
--     msp.most_seen_hcp4_last_visit_5yr,

--     msp.most_seen_hcp5_3yr_ranked,
--     COALESCE(msp.most_seen_hcp5_visit_count_5yr, 0) AS most_seen_hcp5_visit_count_5yr,
--     msp.most_seen_hcp5_last_visit_5yr

-- FROM eligible_patients ep
-- LEFT JOIN patient_demographics pd
--     ON ep.PATIENT_ID = pd.PATIENT_ID
-- LEFT JOIN patient_geography pg
--     ON ep.PATIENT_ID = pg.PATIENT_ID

-- LEFT JOIN historical_first_dx hfdx
--     ON ep.PATIENT_ID = hfdx.PATIENT_ID
-- LEFT JOIN historical_first_tx hftx
--     ON ep.PATIENT_ID = hftx.PATIENT_ID

-- LEFT JOIN first_tx_after_diagnosis fta
--     ON ep.PATIENT_ID = fta.PATIENT_ID

-- LEFT JOIN elaprase_fills ef
--     ON ep.PATIENT_ID = ef.PATIENT_ID

-- LEFT JOIN latest_mpsii_treatment_type lmt
--     ON ep.PATIENT_ID = lmt.PATIENT_ID

-- LEFT JOIN latest_treatment_type ltt
--     ON ep.PATIENT_ID = ltt.PATIENT_ID

-- LEFT JOIN latest_claim_hcp_final lch
--     ON ep.PATIENT_ID = lch.PATIENT_ID
-- LEFT JOIN provider_dim pdlch
--     ON lch.latest_claim_hcp_npi = pdlch.npi
-- LEFT JOIN cmpa_insights_internal_schema.reference_file ref1
--     ON lch.latest_claim_hcp_npi = ref1.hcp_npi

-- LEFT JOIN most_recent_tx_hcp_final mrt
--     ON ep.PATIENT_ID = mrt.PATIENT_ID
-- LEFT JOIN provider_dim pdtch
--     ON mrt.latest_treatment_hcp_npi = pdtch.npi
-- LEFT JOIN cmpa_insights_internal_schema.reference_file ref2
--     ON mrt.latest_treatment_hcp_npi = ref2.hcp_npi

-- LEFT JOIN first_dx_hcp fdh
--     ON ep.PATIENT_ID = fdh.PATIENT_ID
-- LEFT JOIN first_dx_hcp_5yr_stats fdhs
--     ON ep.PATIENT_ID = fdhs.PATIENT_ID
-- LEFT JOIN first_tx_hcp fth
--     ON ep.PATIENT_ID = fth.PATIENT_ID
-- LEFT JOIN first_tx_hcp_5yr_stats fths
--     ON ep.PATIENT_ID = fths.PATIENT_ID
-- LEFT JOIN first_tx_hcp_5yr_tx_only fthtx
--     ON ep.PATIENT_ID = fthtx.PATIENT_ID

-- LEFT JOIN most_seen_pivot msp
--     ON ep.PATIENT_ID = msp.PATIENT_ID
-- LEFT JOIN avlayah_status avs
--   ON ep.PATIENT_ID = avs.patient_id
-- ORDER BY ep.PATIENT_ID;

-- -- Materialize the temp view into the persistent base table.
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_base AS
-- SELECT DISTINCT * FROM patient_hcp_visit_summary;


-- -- =============================================================================
-- -- CELL 2: mpsii_tx_claims (UNCHANGED)
-- -- =============================================================================
-- CREATE OR REPLACE TEMP VIEW mpsii_tx_claims AS
-- WITH MPSII_Diagnoses_Specified AS (
--     SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE DIAGNOSIS_CODES LIKE '%E761%'
--       AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--     UNION
--     SELECT DISTINCT PATIENT_ID, FILL_DATE
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE DIAGNOSIS_CODE = 'E761'
--       AND TRANSACTION_RESULT = 'PAID'
--       AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
-- ),

-- Patients_2Dx_Specified AS (
--     SELECT PATIENT_ID
--     FROM MPSII_Diagnoses_Specified
--     GROUP BY PATIENT_ID
--     HAVING COUNT(DISTINCT FILL_DATE) >= 2
-- ),

-- MPSII_Diagnoses_Unspecified AS (
--     SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE DIAGNOSIS_CODES LIKE '%E763%'
--       AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--     UNION
--     SELECT DISTINCT PATIENT_ID, FILL_DATE
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE DIAGNOSIS_CODE = 'E763'
--       AND TRANSACTION_RESULT = 'PAID'
--       AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
-- ),

-- Patients_2Dx_Unspecified AS (
--     SELECT PATIENT_ID
--     FROM MPSII_Diagnoses_Unspecified
--     GROUP BY PATIENT_ID
--     HAVING COUNT(DISTINCT FILL_DATE) >= 2
-- ),

-- MPSII_Treatment_All AS (
--     SELECT DISTINCT PATIENT_ID FROM (
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE NDC11 IN ('54092070001','540920700')
--           AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_pharmacy_events
--         WHERE NDC11 IN ('54092070001','540920700')
--           AND TRANSACTION_RESULT = 'PAID'
--           AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
--                                  '38206','38230','38232','38240','38241','38242','38243','38250')
--           AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--     ) t
-- ),

-- MPSII_Treatment_Elaprase_Only AS (
--     SELECT DISTINCT PATIENT_ID FROM (
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE NDC11 IN ('54092070001','540920700')
--           AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_pharmacy_events
--         WHERE NDC11 IN ('54092070001','540920700')
--           AND TRANSACTION_RESULT = 'PAID'
--           AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE PROCEDURE_CODE = 'J1743'
--           AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--     ) t
-- ),

-- Patients_2Dx_Specified_With_Treatment AS (
--     SELECT DISTINCT p.PATIENT_ID
--     FROM Patients_2Dx_Specified p
--     INNER JOIN MPSII_Treatment_All t USING (PATIENT_ID)
-- ),

-- Patients_Incremental_Unspecified AS (
--     SELECT DISTINCT p.PATIENT_ID
--     FROM Patients_2Dx_Unspecified p
--     INNER JOIN MPSII_Treatment_Elaprase_Only t USING (PATIENT_ID)
--     WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
-- ),

-- eligible_patients AS (
--     SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment
--     UNION
--     SELECT PATIENT_ID FROM Patients_Incremental_Unspecified
-- ),

-- cohort_3_learnings AS (
--   SELECT DISTINCT npi
--   FROM com_raw.kom_providers
--   WHERE provider_type = 'INDIVIDUAL'
--     AND (
--       PRIMARY_SPECIALTY NOT IN (
--         'Anesthesiologist Assistant','Anesthesiology','Dentist','Dietitian, Registered',
--         'Emergency Medical Technician, Basic','Emergency Medicine','General Acute Care Hospital',
--         'Nurse Anesthetist, Certified Registered','Obstetrics & Gynecology','Pathology',
--         'Radiology','Urology'
--       )
--       OR SECONDARY_SPECIALTY IN (
--         'Child & Adolescent Psychiatry','Psychiatry','Adolescent Medicine','Developmental - Behavioral Pediatrics',
--         'Neonatal-Perinatal Medicine','Nutrition, Pediatric','Oncology, Pediatrics','Pediatric Cardiology',
--         'Pediatric Critical Care Medicine','Pediatric Dermatology','Pediatric Emergency Medicine',
--         'Pediatric Endocrinology','Pediatric Gastroenterology','Pediatric Hematology-Oncology',
--         'Pediatric Infectious Diseases','Pediatric Nephrology','Pediatric Ophthalmology and Strabismus Specialist',
--         'Pediatric Orthopaedic Surgery','Pediatric Otolaryngology','Pediatric Pulmonology','Pediatric Radiology',
--         'Pediatric Rehabilitation Medicine','Pediatric Rheumatology','Pediatric Surgery','Pediatrics',
--         'Clinical Biochemical Genetics','Clinical Genetics (M.D.)','Clinical Molecular Genetics',
--         'Ph.D. Medical Genetics','Neurodevelopmental Disabilities','Neurology',
--         'Neurology with Special Qualifications in Child Neurology','Neuroradiology'
--       )
--     )
-- ),

-- all_tx_claims_2yr AS (
--     SELECT DISTINCT *
--     FROM (
--       SELECT DISTINCT
--         PATIENT_ID,
--         COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
--         SERVICE_DATE AS FILL_DATE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE NDC11 IN ('54092070001','540920700')
--         AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--       UNION

--       SELECT DISTINCT
--         PATIENT_ID,
--         PRESCRIBER_NPI AS NPI,
--         FILL_DATE
--       FROM com_edp_prd.com_raw.kom_pharmacy_events
--       WHERE NDC11 IN ('54092070001','540920700')
--         AND TRANSACTION_RESULT = 'PAID'
--         AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--       UNION

--       SELECT DISTINCT
--         PATIENT_ID,
--         RENDERING_NPI AS NPI,
--         SERVICE_DATE AS FILL_DATE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
--                                '38206','38230','38232','38240','38241','38242','38243','38250')
--         AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     )
--     WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
-- )

-- SELECT * FROM all_tx_claims_2yr;


-- -- =============================================================================
-- -- CELL 3: most_recently_treated_hcp (UNCHANGED)
-- -- =============================================================================
-- CREATE OR REPLACE TEMPORARY VIEW most_recently_treated_hcp AS
-- WITH tx_claims AS (
--     SELECT DISTINCT *
--     FROM mpsii_tx_claims
-- ),

-- MPSII_Diagnoses_Specified AS (
--     SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE DIAGNOSIS_CODES LIKE '%E761%'
--       AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--     UNION
--     SELECT DISTINCT PATIENT_ID, FILL_DATE
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE DIAGNOSIS_CODE = 'E761'
--       AND TRANSACTION_RESULT = 'PAID'
--       AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
-- ),

-- Patients_2Dx_Specified AS (
--     SELECT PATIENT_ID
--     FROM MPSII_Diagnoses_Specified
--     GROUP BY PATIENT_ID
--     HAVING COUNT(DISTINCT FILL_DATE) >= 2
-- ),

-- MPSII_Diagnoses_Unspecified AS (
--     SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE DIAGNOSIS_CODES LIKE '%E763%'
--       AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--     UNION
--     SELECT DISTINCT PATIENT_ID, FILL_DATE
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE DIAGNOSIS_CODE = 'E763'
--       AND TRANSACTION_RESULT = 'PAID'
--       AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
-- ),

-- Patients_2Dx_Unspecified AS (
--     SELECT PATIENT_ID
--     FROM MPSII_Diagnoses_Unspecified
--     GROUP BY PATIENT_ID
--     HAVING COUNT(DISTINCT FILL_DATE) >= 2
-- ),

-- MPSII_Treatment_All AS (
--     SELECT DISTINCT PATIENT_ID FROM (
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE NDC11 IN ('54092070001','540920700')
--           AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_pharmacy_events
--         WHERE NDC11 IN ('54092070001','540920700')
--           AND TRANSACTION_RESULT = 'PAID'
--           AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
--                                  '38206','38230','38232','38240','38241','38242','38243','38250')
--           AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--     ) t
-- ),

-- MPSII_Treatment_Elaprase_Only AS (
--     SELECT DISTINCT PATIENT_ID FROM (
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE NDC11 IN ('54092070001','540920700')
--           AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_pharmacy_events
--         WHERE NDC11 IN ('54092070001','540920700')
--           AND TRANSACTION_RESULT = 'PAID'
--           AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE PROCEDURE_CODE = 'J1743'
--           AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--     ) t
-- ),

-- Patients_2Dx_Specified_With_Treatment AS (
--     SELECT DISTINCT p.PATIENT_ID
--     FROM Patients_2Dx_Specified p
--     INNER JOIN MPSII_Treatment_All t USING (PATIENT_ID)
-- ),

-- Patients_Incremental_Unspecified AS (
--     SELECT DISTINCT p.PATIENT_ID
--     FROM Patients_2Dx_Unspecified p
--     INNER JOIN MPSII_Treatment_Elaprase_Only t USING (PATIENT_ID)
--     WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
-- ),

-- eligible_patients AS (
--     SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment
--     UNION
--     SELECT PATIENT_ID FROM Patients_Incremental_Unspecified
-- ),

-- cohort_3_learnings AS (
--   SELECT DISTINCT npi
--   FROM com_raw.kom_providers
--   WHERE provider_type = 'INDIVIDUAL'
--     AND (
--       PRIMARY_SPECIALTY NOT IN (
--         'Anesthesiologist Assistant','Anesthesiology','Dentist','Dietitian, Registered',
--         'Emergency Medical Technician, Basic','Emergency Medicine','General Acute Care Hospital',
--         'Nurse Anesthetist, Certified Registered','Obstetrics & Gynecology','Pathology',
--         'Radiology','Urology'
--       )
--       OR SECONDARY_SPECIALTY IN (
--         'Child & Adolescent Psychiatry','Psychiatry','Adolescent Medicine','Developmental - Behavioral Pediatrics',
--         'Neonatal-Perinatal Medicine','Nutrition, Pediatric','Oncology, Pediatrics','Pediatric Cardiology',
--         'Pediatric Critical Care Medicine','Pediatric Dermatology','Pediatric Emergency Medicine',
--         'Pediatric Endocrinology','Pediatric Gastroenterology','Pediatric Hematology-Oncology',
--         'Pediatric Infectious Diseases','Pediatric Nephrology','Pediatric Ophthalmology and Strabismus Specialist',
--         'Pediatric Orthopaedic Surgery','Pediatric Otolaryngology','Pediatric Pulmonology','Pediatric Radiology',
--         'Pediatric Rehabilitation Medicine','Pediatric Rheumatology','Pediatric Surgery','Pediatrics',
--         'Clinical Biochemical Genetics','Clinical Genetics (M.D.)','Clinical Molecular Genetics',
--         'Ph.D. Medical Genetics','Neurodevelopmental Disabilities','Neurology',
--         'Neurology with Special Qualifications in Child Neurology','Neuroradiology'
--       )
--     )
-- ),

-- all_dx_claims_5yr AS (
--     SELECT DISTINCT *
--     FROM (
--       SELECT DISTINCT
--         PATIENT_ID,
--         COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
--         SERVICE_DATE AS FILL_DATE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
--         AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--       UNION
--       SELECT DISTINCT
--         PATIENT_ID,
--         PRESCRIBER_NPI AS NPI,
--         FILL_DATE
--       FROM com_edp_prd.com_raw.kom_pharmacy_events
--       WHERE DIAGNOSIS_CODE IN ('E761','E763')
--         AND TRANSACTION_RESULT = 'PAID'
--         AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     )
--     WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
-- ),

-- all_tx_claims_5yr AS (
--     SELECT DISTINCT *
--     FROM (
--       SELECT DISTINCT
--         PATIENT_ID,
--         COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
--         SERVICE_DATE AS FILL_DATE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE NDC11 IN ('54092070001','540920700')
--         AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--       UNION
--       SELECT DISTINCT
--         PATIENT_ID,
--         PRESCRIBER_NPI AS NPI,
--         FILL_DATE
--       FROM com_edp_prd.com_raw.kom_pharmacy_events
--       WHERE NDC11 IN ('54092070001','540920700')
--         AND TRANSACTION_RESULT = 'PAID'
--         AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--       UNION
--       SELECT DISTINCT
--         PATIENT_ID,
--         RENDERING_NPI AS NPI,
--         SERVICE_DATE AS FILL_DATE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
--                                '38206','38230','38232','38240','38241','38242','38243','38250')
--         AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     )
--     WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
-- ),

-- all_claims AS (
--     SELECT * FROM all_dx_claims_5yr
--     UNION
--     SELECT * FROM all_tx_claims_5yr
-- ),

-- latest_treating_hcp AS (
--     SELECT patient_id, npi
--     FROM (
--         SELECT
--             patient_id,
--             npi,
--             fill_date,
--             ROW_NUMBER() OVER (
--                 PARTITION BY patient_id
--                 ORDER BY
--                     CASE WHEN npi IS NOT NULL THEN 1 ELSE 2 END,
--                     fill_date DESC,
--                     npi DESC
--             ) AS rn
--         FROM tx_claims
--     ) t
--     WHERE rn = 1
--       AND npi IS NOT NULL
-- ),

-- visit_counts AS (
--     SELECT
--         patient_id,
--         npi,
--         COUNT(DISTINCT fill_date) AS visit_counts
--     FROM all_claims
--     WHERE npi IS NOT NULL
--     GROUP BY patient_id, npi
-- ),

-- last_visit_date AS (
--     SELECT
--         lth.patient_id,
--         lth.npi,
--         MAX(ac.fill_date) AS last_visit_date
--     FROM latest_treating_hcp lth
--     LEFT JOIN all_claims ac
--       ON lth.patient_id = ac.patient_id
--      AND lth.npi = ac.npi
--     GROUP BY lth.patient_id, lth.npi
-- ),

-- latest_treating_hcp_with_visits AS (
--     SELECT
--         a.patient_id,
--         a.npi AS most_recently_treated_hcp,
--         b.visit_counts AS no_of_visits,
--         c.last_visit_date AS last_visit_date_5yr
--     FROM latest_treating_hcp AS a
--     LEFT JOIN visit_counts AS b
--         ON a.patient_id = b.patient_id
--        AND a.npi        = b.npi
--     LEFT JOIN last_visit_date AS c
--         ON a.patient_id = c.patient_id
--        AND a.npi        = c.npi
-- ),

-- hcp_with_other_info AS (
--     SELECT
--         a.*,
--         CONCAT(b.FIRST_NAME, ' ', b.LAST_NAME) AS hcp_name,
--         b.PRIMARY_SPECIALTY AS hcp_specialty,
--         c.hco_name,
--         c.territory_id,
--         c.territory,
--         c.region_id,
--         c.region
--     FROM latest_treating_hcp_with_visits AS a

--     LEFT JOIN com_edp_prd.com_raw.kom_providers AS b
--         ON a.most_recently_treated_hcp = b.NPI
--        AND b.PROVIDER_TYPE = 'INDIVIDUAL'

--     LEFT JOIN (
--   SELECT
--     a.hcp_npi,
--     a.hco_name,
--     a.territory,
--     a.region,
--     b.territory_id,
--     c.region_id,
--     a.hcp_primary_specialty AS hcp_specialty
--   FROM cmpa_insights_internal_schema.reference_file a
--   LEFT JOIN (
--       SELECT DISTINCT territory_id, territory_name
--       FROM cmpa_insights_internal_schema.zip_to_territory_mapping
--   ) b
--     ON a.territory = b.territory_name
--   LEFT JOIN (
--       SELECT DISTINCT region_id, region_name
--       FROM cmpa_insights_internal_schema.zip_to_territory_mapping
--   ) c
--     ON a.region = c.region_name
-- ) c
-- ON a.most_recently_treated_hcp = c.hcp_npi)

-- SELECT
--     patient_id,
--     most_recently_treated_hcp AS most_recently_treated_hcp_2yr,
--     hcp_name AS most_recently_treated_hcp_name_2yr,

--     no_of_visits AS most_recently_treated_hcp_2yr_no_of_visits_5yr,
--     last_visit_date_5yr AS most_recent_tx_hcp_2yr_last_visit_5yr,

--     hcp_specialty AS most_recently_treated_hcp_specialty_2yr,
--     hco_name AS most_recently_treated_hcp_hco_name,
--     territory_id AS most_recently_treated_hcp_territory_id_2yr,
--     territory AS most_recently_treated_hcp_territory_2yr,
--     region_id AS most_recently_treated_hcp_region_id_2yr,
--     region AS most_recently_treated_hcp_region_2yr
--     FROM hcp_with_other_info;


-- -- =============================================================================
-- -- CELL 4: Build all_dx_claims, all_tx_claims views (UNCHANGED)
-- -- =============================================================================
-- CREATE OR REPLACE TEMPORARY VIEW all_dx_claims AS

-- SELECT DISTINCT
--     PATIENT_ID,
--     COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
--     SERVICE_DATE AS FILL_DATE,
--     'DX' AS CLAIM_TYPE
-- FROM com_edp_prd.com_raw.kom_medical_events
-- WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
--   AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)

-- UNION

-- SELECT DISTINCT
--     PATIENT_ID,
--     PRESCRIBER_NPI AS NPI,
--     FILL_DATE,
--     'DX' AS CLAIM_TYPE
-- FROM com_edp_prd.com_raw.kom_pharmacy_events
-- WHERE DIAGNOSIS_CODE IN ('E761', 'E763')
--   AND TRANSACTION_RESULT = 'PAID'
--   AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters);


-- CREATE OR REPLACE TEMPORARY VIEW all_tx_claims AS

-- SELECT DISTINCT
--     PATIENT_ID,
--     COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
--     SERVICE_DATE AS FILL_DATE,
--     'TX' AS CLAIM_TYPE,
--     NDC11 AS CODE
-- FROM com_edp_prd.com_raw.kom_medical_events
-- WHERE NDC11 IN ('54092070001', '540920700')
--   AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)

-- UNION

-- SELECT DISTINCT
--     PATIENT_ID,
--     RENDERING_NPI AS NPI,
--     SERVICE_DATE AS FILL_DATE,
--     'TX' AS CLAIM_TYPE,
--     PROCEDURE_CODE AS CODE
-- FROM com_edp_prd.com_raw.kom_medical_events
-- WHERE PROCEDURE_CODE IN ('99601', '99602', '96365', '96366', 'J1743',
--                          'S9357', 'S9379', '38206', '38230', '38232',
--                          '38240', '38241', '38242', '38243', '38250')
--   AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)

-- UNION

-- SELECT DISTINCT
--     PATIENT_ID,
--     PRESCRIBER_NPI AS NPI,
--     FILL_DATE,
--     'TX' AS CLAIM_TYPE,
--     NDC11 AS CODE
-- FROM com_edp_prd.com_raw.kom_pharmacy_events
-- WHERE NDC11 IN ('54092070001', '540920700')
--   AND TRANSACTION_RESULT = 'PAID'
--   AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters);


-- CREATE OR REPLACE TEMPORARY VIEW tx_claims_2yr AS
-- SELECT DISTINCT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE, CODE
-- FROM all_tx_claims
-- WHERE FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters);


-- CREATE OR REPLACE TEMPORARY VIEW e761_patients_2dx AS
-- SELECT PATIENT_ID
-- FROM (
--     SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE DIAGNOSIS_CODES LIKE '%E761%'
--       AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--     UNION
--     SELECT DISTINCT PATIENT_ID, FILL_DATE
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE DIAGNOSIS_CODE = 'E761'
--       AND TRANSACTION_RESULT = 'PAID'
--       AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
-- )
-- GROUP BY PATIENT_ID
-- HAVING COUNT(DISTINCT FILL_DATE) >= 2;

-- CREATE OR REPLACE TEMPORARY VIEW specified_patients AS
-- SELECT DISTINCT e.PATIENT_ID
-- FROM e761_patients_2dx e
-- INNER JOIN tx_claims_2yr t ON e.PATIENT_ID = t.PATIENT_ID;

-- CREATE OR REPLACE TEMPORARY VIEW e763_patients_2dx AS
-- SELECT PATIENT_ID
-- FROM (
--     SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE DIAGNOSIS_CODES LIKE '%E763%'
--       AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--     UNION
--     SELECT DISTINCT PATIENT_ID, FILL_DATE
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE DIAGNOSIS_CODE = 'E763'
--       AND TRANSACTION_RESULT = 'PAID'
--       AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
-- )
-- GROUP BY PATIENT_ID
-- HAVING COUNT(DISTINCT FILL_DATE) >= 2;

-- CREATE OR REPLACE TEMPORARY VIEW elaprase_tx_2yr AS
-- SELECT DISTINCT PATIENT_ID
-- FROM tx_claims_2yr
-- WHERE CODE IN ('54092070001', '540920700', 'J1743');

-- CREATE OR REPLACE TEMPORARY VIEW incremental_patients AS
-- SELECT DISTINCT e.PATIENT_ID
-- FROM e763_patients_2dx e
-- INNER JOIN elaprase_tx_2yr t ON e.PATIENT_ID = t.PATIENT_ID
-- WHERE e.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM specified_patients);

-- CREATE OR REPLACE TEMPORARY VIEW eligible_patients AS
-- SELECT PATIENT_ID FROM specified_patients
-- UNION
-- SELECT PATIENT_ID FROM incremental_patients;


-- CREATE OR REPLACE TEMPORARY VIEW cohort_3_learnings AS
-- SELECT DISTINCT npi
-- FROM com_raw.kom_providers
-- WHERE provider_type = 'INDIVIDUAL'
--   AND (
--     PRIMARY_SPECIALTY NOT IN (
--       'Anesthesiologist Assistant','Anesthesiology','Dentist','Dietitian, Registered',
--       'Emergency Medical Technician, Basic','Emergency Medicine','General Acute Care Hospital',
--       'Nurse Anesthetist, Certified Registered','Obstetrics & Gynecology','Pathology',
--       'Radiology','Urology'
--     )
--     OR SECONDARY_SPECIALTY IN (
--       'Child & Adolescent Psychiatry','Psychiatry','Adolescent Medicine','Developmental - Behavioral Pediatrics',
--       'Neonatal-Perinatal Medicine','Nutrition, Pediatric','Oncology, Pediatrics','Pediatric Cardiology',
--       'Pediatric Critical Care Medicine','Pediatric Dermatology','Pediatric Emergency Medicine',
--       'Pediatric Endocrinology','Pediatric Gastroenterology','Pediatric Hematology-Oncology',
--       'Pediatric Infectious Diseases','Pediatric Nephrology','Pediatric Ophthalmology and Strabismus Specialist',
--       'Pediatric Orthopaedic Surgery','Pediatric Otolaryngology','Pediatric Pulmonology','Pediatric Radiology',
--       'Pediatric Rehabilitation Medicine','Pediatric Rheumatology','Pediatric Surgery','Pediatrics',
--       'Clinical Biochemical Genetics','Clinical Genetics (M.D.)','Clinical Molecular Genetics',
--       'Ph.D. Medical Genetics','Neurodevelopmental Disabilities','Neurology',
--       'Neurology with Special Qualifications in Child Neurology','Neuroradiology'
--     )
--   );


-- CREATE OR REPLACE TEMPORARY VIEW all_patient_claims AS
-- SELECT *
-- FROM (
--   SELECT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
--   FROM all_dx_claims
--   WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--   UNION

--   SELECT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
--   FROM all_tx_claims
--   WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
-- )
-- WHERE npi IN (SELECT DISTINCT npi FROM cohort_3_learnings);


-- CREATE OR REPLACE TEMPORARY VIEW primary_hcp AS
-- WITH hcp_metrics AS (
--     SELECT
--         a.PATIENT_ID,
--         a.NPI,

--         CASE
--             WHEN p.primary_specialty LIKE '%Genetic%'
--               OR p.secondary_specialty LIKE '%Genetic%'
--                 THEN 'Geneticist'
--             WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
--               OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
--               OR p.primary_specialty LIKE '%Neurological Surgery%'
--                 THEN 'Psychiatry & Neurology'
--             WHEN p.primary_specialty LIKE '%Pediatrics%'
--                 THEN 'Pediatrician'
--             WHEN p.primary_specialty LIKE '%Internal Medicine%'
--               OR p.secondary_specialty LIKE '%Internal Medicine%'
--               OR p.primary_specialty LIKE '%Family Medicine%'
--               OR p.secondary_specialty LIKE '%Family Medicine%'
--                 THEN 'PCP'
--             WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
--               OR p.primary_specialty LIKE '%Physician Assistant%'
--                 THEN 'NPPA'
--             WHEN a.NPI IS NULL
--                 THEN 'NA'
--             ELSE 'Others'
--         END AS SPECIALTY,

--         CASE
--             WHEN p.primary_specialty LIKE '%Genetic%'
--               OR p.secondary_specialty LIKE '%Genetic%'
--                 THEN 1
--             WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
--               OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
--               OR p.primary_specialty LIKE '%Neurological Surgery%'
--                 THEN 2
--             WHEN p.primary_specialty LIKE '%Pediatrics%'
--                 THEN 3
--             WHEN p.primary_specialty LIKE '%Internal Medicine%'
--               OR p.secondary_specialty LIKE '%Internal Medicine%'
--               OR p.primary_specialty LIKE '%Family Medicine%'
--               OR p.secondary_specialty LIKE '%Family Medicine%'
--                 THEN 4
--             WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
--               OR p.primary_specialty LIKE '%Physician Assistant%'
--                 THEN 5
--             WHEN a.NPI IS NULL
--                 THEN 7
--             ELSE 6
--         END AS SPECIALTY_PRIORITY,

--         COUNT(DISTINCT a.FILL_DATE) AS NO_OF_VISITS,

--         COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'DX' THEN a.FILL_DATE END) AS DX_VISITS,
--         COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'TX' THEN a.FILL_DATE END) AS TX_VISITS,

--         MAX(a.FILL_DATE) AS MOST_RECENT_VISIT

--     FROM all_patient_claims a
--     LEFT JOIN com_edp_prd.com_raw.kom_providers p
--         ON a.NPI = p.NPI
--     GROUP BY
--         a.PATIENT_ID,
--         a.NPI,
--         p.primary_specialty,
--         p.secondary_specialty
-- ),
-- ranked_hcps AS (
--     SELECT
--         *,
--         RANK() OVER (
--             PARTITION BY PATIENT_ID
--             ORDER BY
--                 SPECIALTY_PRIORITY ASC,
--                 NO_OF_VISITS DESC,
--                 MOST_RECENT_VISIT DESC,
--                 NPI ASC
--         ) AS HCP_RANK
--     FROM hcp_metrics
-- )
-- SELECT
--     PATIENT_ID,
--     NPI AS PRIMARY_HCP_NPI,
--     SPECIALTY AS PRIMARY_HCP_SPECIALTY,
--     SPECIALTY_PRIORITY,
--     NO_OF_VISITS,
--     DX_VISITS,
--     TX_VISITS,
--     MOST_RECENT_VISIT,
--     HCP_RANK
-- FROM ranked_hcps
-- WHERE HCP_RANK = 1;


-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.primary_hcp AS
-- SELECT
--     ph.*,
--     COALESCE(p.FIRST_NAME, '') || ' ' || COALESCE(p.LAST_NAME, '') AS primary_hcp_name_2yr,

--     ref.HCO_NAME AS primary_hcp_hco_name_2yr,
--     ref.HCO_CITY AS primary_hcp_hco_city_2yr,
--     ref.HCO_STATE AS primary_hcp_hco_state_2yr,

--     ref.mapped_territory_id AS primary_hcp_territory_id_2yr,
--     ref.TERRITORY           AS primary_hcp_territory_2yr,
--     ref.mapped_region_id    AS primary_hcp_region_id_2yr,
--     ref.region              AS primary_hcp_region_2yr

-- FROM primary_hcp ph

-- LEFT JOIN com_edp_prd.com_raw.kom_providers p
--     ON ph.PRIMARY_HCP_NPI = p.NPI


-- LEFT JOIN (
--   SELECT
--     a.hcp_npi,
--     a.hco_name,
--     a.hco_city,
--     a.hco_state,
--     a.territory,
--     a.region,
--     b.territory_id AS mapped_territory_id,
--     c.region_id AS mapped_region_id,
--     a.hcp_primary_specialty AS hcp_specialty
--   FROM cmpa_insights_internal_schema.reference_file a
--   LEFT JOIN (
--       SELECT DISTINCT try_cast(territory_id AS BIGINT) AS territory_id, territory_name
--       FROM cmpa_insights_internal_schema.zip_to_territory_mapping
--   ) b
--     ON try_cast(a.territory_id AS BIGINT) = b.territory_id
--   LEFT JOIN (
--       SELECT DISTINCT try_cast(region_id AS BIGINT) AS region_id, region_name
--       FROM cmpa_insights_internal_schema.zip_to_territory_mapping
--   ) c
--     ON try_cast(a.region_id AS BIGINT) = c.region_id
-- ) ref
-- ON ph.PRIMARY_HCP_NPI = ref.hcp_npi;


-- -- =============================================================================
-- -- CELL 5: patient360_master initial join (UNCHANGED)
-- -- =============================================================================
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master AS
-- SELECT DISTINCT
--     a.PATIENT_ID,
--     a.PATIENT_YOB,
--     a.PATIENT_AGE,
--     a.PATIENT_GENDER,
--     a.patient_state,

--     a.incidence_date,
--     a.first_incidence_treatment_date,
--     a.latest_claim_date,

--     a.latest_claim_hcp_npi,
--     a.latest_claim_hcp_name,
--     a.latest_claim_hcp_specialty,
--     a.latest_claim_hcp_visit_count,
--     a.latest_claim_hcp_hco_name,

--     a.latest_treatment_date,
--     a.latest_mpsii_tx_type,
--     a.latest_treatment_type,
--     a.first_tx_after_diagnosis,
--     a.time_dx_to_first_tx_in_months,
--     a.treatment_period_months,
--     a.elaprase_fills,
--     a.avlayah_status,
--     a.avlayah_switch_date,
--     a.latest_treatment_hcp_npi,
--     a.latest_treatment_hcp_name,
--     a.latest_treatment_hcp_specialty,
--     a.latest_treatment_hcp_visit_count,
--     a.latest_treatment_hcp_hco_name,

--     a.first_dx_hcp_5yr,
--     a.first_dx_all_visit_count_5yr,
--     a.first_dx_last_visit_5yr,
--     a.first_tx_hcp_5yr,
--     a.first_tx_all_visit_count_5yr,
--     a.first_tx_treatment_visit_count_5yr,
--     a.first_tx_last_visit_5yr,

--     a.most_seen_hcp1_3yr_ranked,
--     a.most_seen_hcp1_visit_count_5yr,
--     a.most_seen_hcp1_last_visit_5yr,
--     a.most_seen_hcp2_3yr_ranked,
--     a.most_seen_hcp2_visit_count_5yr,
--     a.most_seen_hcp2_last_visit_5yr,
--     a.most_seen_hcp3_3yr_ranked,
--     a.most_seen_hcp3_visit_count_5yr,
--     a.most_seen_hcp3_last_visit_5yr,
--     a.most_seen_hcp4_3yr_ranked,
--     a.most_seen_hcp4_visit_count_5yr,
--     a.most_seen_hcp4_last_visit_5yr,
--     a.most_seen_hcp5_3yr_ranked,
--     a.most_seen_hcp5_visit_count_5yr,
--     a.most_seen_hcp5_last_visit_5yr,

--     b.* EXCEPT (patient_id),

--     c.* EXCEPT (patient_id)

-- FROM com_edp_prd.cmpa_insights_internal_schema.patient360_base AS a

-- LEFT JOIN most_recently_treated_hcp AS b
--     ON a.PATIENT_ID = b.patient_id

-- LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.primary_hcp AS c
--     ON a.PATIENT_ID = c.patient_id;


-- -- =============================================================================
-- -- CELL 6: severity flagging (UNCHANGED)
-- -- =============================================================================
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master AS
-- WITh  base AS (
--   SELECT DISTINCT
--     a.PATIENT_ID,
--     b.SERVICE_DATE,
--     b.DIAGNOSIS_CODES
--   FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master a
--   LEFT JOIN com_edp_prd.com_raw.kom_medical_events b
--     ON a.PATIENT_ID = b.PATIENT_ID
--    AND b.SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
-- ),

-- flagged AS (
--   SELECT
--     PATIENT_ID,
--     SERVICE_DATE,
--     CASE
--       WHEN DIAGNOSIS_CODES IS NOT NULL AND (
--            DIAGNOSIS_CODES ILIKE '%|G910|%' OR DIAGNOSIS_CODES ILIKE '%|G911|%' OR DIAGNOSIS_CODES ILIKE '%|G912|%'
--         OR DIAGNOSIS_CODES ILIKE '%|G913|%' OR DIAGNOSIS_CODES ILIKE '%|G914|%' OR DIAGNOSIS_CODES ILIKE '%|G918|%'
--         OR DIAGNOSIS_CODES ILIKE '%|G919|%' OR DIAGNOSIS_CODES ILIKE '%|Q038|%' OR DIAGNOSIS_CODES ILIKE '%|Q039|%'
--         OR DIAGNOSIS_CODES ILIKE '%|Q050|%' OR DIAGNOSIS_CODES ILIKE '%|Q051|%' OR DIAGNOSIS_CODES ILIKE '%|Q052|%'
--         OR DIAGNOSIS_CODES ILIKE '%|Q053|%' OR DIAGNOSIS_CODES ILIKE '%|Q054|%' OR DIAGNOSIS_CODES ILIKE '%|Q055|%'
--         OR DIAGNOSIS_CODES ILIKE '%|Q056|%' OR DIAGNOSIS_CODES ILIKE '%|Q057|%' OR DIAGNOSIS_CODES ILIKE '%|Q058|%'
--         OR DIAGNOSIS_CODES ILIKE '%|Q0700|%' OR DIAGNOSIS_CODES ILIKE '%|Q0702|%' OR DIAGNOSIS_CODES ILIKE '%|Q0703|%'
--         OR DIAGNOSIS_CODES ILIKE '%|F445|%' OR DIAGNOSIS_CODES ILIKE '%|F639|%' OR DIAGNOSIS_CODES ILIKE '%|F70|%'
--         OR DIAGNOSIS_CODES ILIKE '%|F71|%' OR DIAGNOSIS_CODES ILIKE '%|F72|%' OR DIAGNOSIS_CODES ILIKE '%|F73|%'
--         OR DIAGNOSIS_CODES ILIKE '%|F78|%' OR DIAGNOSIS_CODES ILIKE '%|F78A1|%' OR DIAGNOSIS_CODES ILIKE '%|F78A9|%'
--         OR DIAGNOSIS_CODES ILIKE '%|F79|%' OR DIAGNOSIS_CODES ILIKE '%|F800|%' OR DIAGNOSIS_CODES ILIKE '%|F801|%'
--         OR DIAGNOSIS_CODES ILIKE '%|F802|%' OR DIAGNOSIS_CODES ILIKE '%|F804|%' OR DIAGNOSIS_CODES ILIKE '%|F8081|%'
--         OR DIAGNOSIS_CODES ILIKE '%|F8082|%' OR DIAGNOSIS_CODES ILIKE '%|F8089|%' OR DIAGNOSIS_CODES ILIKE '%|F809|%'
--         OR DIAGNOSIS_CODES ILIKE '%|F810|%' OR DIAGNOSIS_CODES ILIKE '%|F812|%' OR DIAGNOSIS_CODES ILIKE '%|F8181|%'
--         OR DIAGNOSIS_CODES ILIKE '%|F8189|%' OR DIAGNOSIS_CODES ILIKE '%|F819|%' OR DIAGNOSIS_CODES ILIKE '%|F82|%'
--         OR DIAGNOSIS_CODES ILIKE '%|F840|%' OR DIAGNOSIS_CODES ILIKE '%|F843|%' OR DIAGNOSIS_CODES ILIKE '%|F845|%'
--         OR DIAGNOSIS_CODES ILIKE '%|F848|%' OR DIAGNOSIS_CODES ILIKE '%|F849|%' OR DIAGNOSIS_CODES ILIKE '%|F88|%'
--         OR DIAGNOSIS_CODES ILIKE '%|F89|%' OR DIAGNOSIS_CODES ILIKE '%|R6250|%' OR DIAGNOSIS_CODES ILIKE '%|R620|%'
--         OR DIAGNOSIS_CODES ILIKE '%|R6251|%' OR DIAGNOSIS_CODES ILIKE '%|R6259|%' OR DIAGNOSIS_CODES ILIKE '%|R62|%'
--       )
--       THEN 1 ELSE 0
--     END AS has_severity_code
--   FROM base
-- ),

-- patient_with_severity_outcome AS (
--   SELECT
--     PATIENT_ID,
--     COUNT(DISTINCT SERVICE_DATE) AS count_fill_date,
--     COUNT(DISTINCT CASE
--                      WHEN has_severity_code = 1
--                      THEN SERVICE_DATE
--                    END) AS severity_dx_distinct_dates,
--     CASE
--       WHEN COUNT(DISTINCT CASE
--                             WHEN has_severity_code = 1
--                             THEN SERVICE_DATE
--                           END) >= 2
--       THEN 'Severe'
--       ELSE 'Attenuated'
--     END AS severity
--   FROM flagged
--   GROUP BY PATIENT_ID
--   ORDER BY severity_dx_distinct_dates DESC, PATIENT_ID
-- )

-- SELECT
--   a.*,
--   b.severity
-- FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master AS a
-- LEFT JOIN patient_with_severity_outcome AS b
--   ON a.PATIENT_ID = b.patient_id;


-- -- =============================================================================
-- -- CELL 7: Comorbidity flagging (UNCHANGED)
-- -- =============================================================================
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master AS
-- WITH base_patients AS (
--     SELECT DISTINCT patient_id
--     FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
-- ),

-- exploded_codes AS (
--     SELECT
--         m.patient_id,
--         code
--     FROM com_edp_prd.com_raw.kom_medical_events m
--     INNER JOIN base_patients bp
--         ON m.patient_id = bp.patient_id
--     LATERAL VIEW explode(split(m.DIAGNOSIS_CODES, '\\|')) s AS code
--     WHERE m.SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--     AND m.DIAGNOSIS_CODES IS NOT NULL
-- ),

-- comorbidity_raw AS (
--     SELECT
--         patient_id,
--         CASE
--             WHEN code IN (
--                 'J45909','J4520','J4530','J4540','J4541','J4531','J45901','J45998',
--                 'J4521','J4542','J45990','J4550','J45902','J4551','J4532','J45991'
--             ) THEN 'Asthma'

--             WHEN code IN ('G4733','G4730','G479','G4731','G4739')
--                 THEN 'Sleep Apnea'

--             WHEN code IN (
--                 'J988','J300','J301','J302','J305','J3081','J3089','J309','J310',
--                 'J311','J312','J320','J321','J322','J323','J324','J328','J329',
--                 'J330','J331','J338','J339','J340','J341','J342','J343','J3481',
--                 'J348200','J348201','J348202','J348210','J348211','J348212',
--                 'J34829','J3489','J349','J3501','J3502','J3503','J351','J352',
--                 'J353','J358','J359','J36','J370','J371','J3800','J3801','J3802',
--                 'J381','J382','J383','J384','J385','J386','J387','J390','J391',
--                 'J392','J393','J398','J399'
--             ) THEN 'Other diseases of upper respiratory tract'

--             WHEN code IN (
--                 'G4700','G4734','G4761','G478','G4710','G4701','G4736','G4720',
--                 'G4709','G4719','G4721','G4729','G4723','G47'
--             ) THEN 'Sleep related disorders'

--             WHEN code IN (
--                 'J00','J0100','J0101','J0110','J0111','J0120','J0121','J0130',
--                 'J0131','J0140','J0141','J0180','J0181','J0190','J0191','J020',
--                 'J028','J029','J0300','J0301','J0380','J0381','J0390','J0391',
--                 'J040','J0410','J0411','J042','J0430','J0431','J050','J0510',
--                 'J0511','J060','J069'
--             ) THEN 'Acute upper respiratory infections'

--             WHEN code IN (
--                 'H6000','H6001','H6002','H6003','H6010','H6011','H6012','H6013',
--                 'H6020','H6021','H6022','H60311','H60312','H60319','H60321',
--                 'H60322','H60329','H60391','H60392','H60399','H6040','H6041',
--                 'H6042','H6043','H60501','H60502','H60509','H60511','H60512',
--                 'H60519','H60551','H60552','H60559','H60591','H60592','H60599',
--                 'H6060','H6061','H6062','H608X1','H608X2','H608X9','H6090',
--                 'H6091','H6092','H61001','H61002','H61003','H61009','H61011',
--                 'H61012','H61013','H61019','H61021','H61022','H61023','H61029',
--                 'H61031','H61032','H61033','H61039','H61321','H61322','H61323',
--                 'H61329','H6240','H6241','H6242','H6500','H6501','H6502','H6504',
--                 'H6505','H6507','H65111','H65112','H65114','H65115','H65117',
--                 'H65119','H65191','H65192','H65194','H65195','H65197','H65199',
--                 'H6520','H6521','H6522','H6530','H6531','H6532','H65411',
--                 'H65412','H65419','H65491','H65492','H65499','H6590','H6591',
--                 'H6592','H66001','H66002','H66003','H66004','H66005','H66006',
--                 'H66007','H66009','H66011','H66012','H66013','H66014','H66015',
--                 'H66016','H66017','H66019','H6611','H6612','H6620','H6621',
--                 'H6622','H663X1','H663X2','H663X9','H6640','H6641','H6642',
--                 'H6690','H6691','H6692','H671','H672','H679','H68001','H68002',
--                 'H68009','H68011','H68012','H68019','H68021','H68022','H68029',
--                 'H70001','H70002','H70009','H70011','H70012','H70019','H70091',
--                 'H70092','H70099','H7010','H7011','H7012','H70201','H70202',
--                 'H70209','H70211','H70212','H70219','H70221','H70222','H70229',
--                 'H70811','H70812','H70819','H70891','H70892','H70899','H7090',
--                 'H7091','H7092','H7100','H7101','H7102','H7110','H7111','H7112',
--                 'H7120','H7121','H7122','H7130','H7131','H7132','H7190','H7191',
--                 'H7192','H73001','H73002','H73009','H73011','H73012','H73019',
--                 'H73091','H73092','H73099','H7310','H7311','H7312','H7320',
--                 'H7321','H7322','H7411','H7412','H7413','H7419','H7440','H7441',
--                 'H7442','H7443','H748X1','H748X2','H748X3','H748X9','H7490',
--                 'H7491','H7492','H7493','H8120','H8121','H8122','H8301','H8302',
--                 'H8309','H9210','H9211','H9212','H9500','H9501','H9502','H9503'
--             ) THEN 'Ear Infections'

--             WHEN code IN (
--                 'H900','H9011','H9012','H902','H903','H9041','H9042','H905',
--                 'H906','H9071','H9072','H908','H90A11','H90A12','H90A21',
--                 'H90A22','H90A31','H90A32','H9101','H9102','H9103','H9109',
--                 'H9120','H9121','H9122','H9123','H918X1','H918X2','H918X3',
--                 'H918X9','H9190','H9191','H9192','H9193','P096'
--             ) THEN 'Hearing loss'

--             WHEN code IN (
--                 'K5900','K5909','K5901','K5904','K5903','K5902','K590','K5939'
--             ) THEN 'Constipation'

--             WHEN code IN ('R197','K591','K580','K529')
--                 THEN 'Diarrhea'

--             WHEN code IN (
--                 'K4000','K4001','K4010','K4011','K4020','K4021','K4030','K4031',
--                 'K4040','K4041','K4090','K4091','K450','K451','K458','K460',
--                 'K461','K469'
--             ) THEN 'Abdominal/inguinal hernia'

--             WHEN code IN ('Q751','Q754','Q755')
--                 THEN 'Dysostosis Complex'

--             WHEN code IN (
--                 'M2560','M25611','M25612','M25619','M25621','M25622','M25629',
--                 'M25631','M25632','M25639','M25641','M25642','M25649','M25651',
--                 'M25652','M25659','M25661','M25662','M25669','M25671','M25672',
--                 'M25673','M25674','M25675','M25676','M2569'
--             ) THEN 'Joint Stiffness'

--             WHEN code IN ('G5600','G5601','G5602','G5603')
--                 THEN 'Carpal tunnel syndrome'

--             WHEN code IN (
--                 'F05','F060','F061','F062','F0630','F0631','F0632','F0633',
--                 'F0634','F064','F0670','F0671','F068','F070','F0781','F0789',
--                 'F079','F09','F22','F23','F24','F28','F29','F3010','F3011',
--                 'F3012','F3013','F302','F303','F304','F308','F309','F320',
--                 'F321','F322','F323','F324','F325','F328','F3289','F329','F32A',
--                 'F330','F331','F332','F333','F3340','F3341','F3342','F338',
--                 'F339','F340','F348','F3481','F3489','F349','F39','F410','F411',
--                 'F413','F418','F419','F430','F4310','F4311','F4312','F4320',
--                 'F4321','F4322','F4323','F4324','F4325','F4329','F438','F4389',
--                 'F439','F441','F442','F450','F451','F4522','F4541','F4542',
--                 'F54','F59','F600','F602','F603','F604','F605','F606','F6089',
--                 'F609','F6381','F6389','F639','F70','F71','F72','F73','F78',
--                 'F78A1','F78A9','F79','F800','F801','F802','F804','F8081',
--                 'F8082','F8089','F809','F810','F812','F8181','F8189','F819',
--                 'F82','F840','F843','F845','F848','F849','F88','F89','F900',
--                 'F901','F902','F908','F909','F910','F911','F912','F913','F918',
--                 'F919','F930','F938','F939','F940','F941','F942','F948','F949',
--                 'F950','F951','F9821','F9829','F983','F984','F985','F988',
--                 'F989','F99'
--             ) THEN 'Behavioral Issues'

--             WHEN code IN (
--                 'G910','G911','G912','G913','G914','G918','G919','Q038','Q039',
--                 'Q050','Q051','Q052','Q053','Q054','Q055','Q056','Q057','Q058',
--                 'Q0700','Q0702','Q0703','F445','G40001','G40009','G40011',
--                 'G40019','G40101','G40109','G40111','G40119','G40201','G40209',
--                 'G40211','G40219','G40501','G40509','G4089','R561'
--             ) THEN 'CNS Issues'

--             WHEN code IN ('R6250','R620','R6252','R6251','R6259','R627','R62')
--                 THEN 'Lack of Physiological Development'

--             WHEN code IN (
--                 'I10','I110','I129','I130','I119','I159','I160','I158','I120',
--                 'I161','I1310','I150'
--             ) THEN 'Hypertension'

--             WHEN code IN (
--                 'I050','I051','I052','I058','I059','I060','I061','I062','I068',
--                 'I069','I070','I071','I072','I078','I079','I080','I081','I082',
--                 'I083','I088','I089','I340','I341','I342','I348','I3481',
--                 'I3489','I349','I350','I351','I352','I358','I359','I360',
--                 'I361','I362','I368','I369','I370','I371','I372','I378','I379'
--             ) THEN 'Valvular Heart Disease'

--         END AS comorbidity_name
--     FROM exploded_codes
--     WHERE code IS NOT NULL AND code <> ''
-- ),

-- comorbidity_with_category AS (
--     SELECT
--         patient_id,
--         comorbidity_name,
--         CASE
--             WHEN comorbidity_name IN (
--                 'Asthma','Sleep Apnea','Other diseases of upper respiratory tract',
--                 'Sleep related disorders','Acute upper respiratory infections'
--             ) THEN 'Respiratory issues'

--             WHEN comorbidity_name IN ('Ear Infections','Hearing loss')
--                 THEN 'Ear-related disorders'

--             WHEN comorbidity_name IN ('Constipation','Diarrhea','Abdominal/inguinal hernia')
--                 THEN 'Gastrointestinal disorders'

--             WHEN comorbidity_name IN ('Dysostosis Complex','Joint Stiffness','Carpal tunnel syndrome')
--                 THEN 'Mobility issues'

--             WHEN comorbidity_name IN ('Behavioral Issues','CNS Issues','Lack of Physiological Development')
--                 THEN 'Neurological disorders'

--             WHEN comorbidity_name IN ('Hypertension','Valvular Heart Disease')
--                 THEN 'Other chronic conditions'
--         END AS comorbidity_category
--     FROM comorbidity_raw
--     WHERE comorbidity_name IS NOT NULL
-- ),

-- patient_with_comorbidity_outcome as (
--     SELECT
--     patient_id,

--     array_join(
--         array_sort(collect_set(comorbidity_category)),
--         ', '
--     ) AS comorbidity_categories,

--     size(collect_set(comorbidity_category)) AS count_of_comorbidity_categories,

--     array_join(
--         array_sort(collect_set(comorbidity_name)),
--         ', '
--     ) AS distinct_comorbidities,

--     size(collect_set(comorbidity_name)) AS count_of_distinct_comorbidities

-- FROM comorbidity_with_category
-- GROUP BY patient_id
-- ORDER BY patient_id
-- )
-- select
--     a.*,
--     b.comorbidity_categories,
--     b.count_of_comorbidity_categories,
--     b.distinct_comorbidities,
--     b.count_of_distinct_comorbidities
-- from com_edp_prd.cmpa_insights_internal_schema.patient360_master as a
-- left join patient_with_comorbidity_outcome as b
--     on a.patient_id=b.patient_id;


-- -- =============================================================================
-- -- CELL 8: primary_hcp secondary specialty enrichment (UNCHANGED)
-- -- =============================================================================
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master as
--   select distinct a.*, b.SECONDARY_SPECIALTY as primary_hcp_secondary_specialty
-- from com_edp_prd.cmpa_insights_internal_schema.patient360_master as a
-- left join com_edp_prd.com_raw.kom_providers as b on a.PRIMARY_HCP_NPI = b.npi and b.provider_type = 'INDIVIDUAL';


-- -- =============================================================================
-- -- CELL 9: age_bucket / payer / insurance enrichment (UNCHANGED)
-- -- =============================================================================
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master AS

-- WITH base_table AS (
--   SELECT DISTINCT *
--   FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
-- ),
-- age_buckets AS (
--   SELECT DISTINCT
--     patient_id,
--     CASE
--       WHEN patient_age < 5 THEN '<5 years'
--       WHEN patient_age BETWEEN 5 AND 10 THEN '5 - 10 years'
--       WHEN patient_age BETWEEN 11 AND 16 THEN '11 - 16 years'
--       ELSE '>17 years'
--     END AS age_bucket
--   FROM base_table
-- ),

-- patient_geography AS (
--   SELECT DISTINCT *
--   FROM (
--     SELECT *,
--       ROW_NUMBER() OVER (
--         PARTITION BY patient_id
--         ORDER BY
--           CASE WHEN valid_to_date > CURRENT_DATE() THEN 1 ELSE 2 END,
--           valid_to_date DESC
--       ) AS rn
--     FROM com_edp_prd.com_raw.kom_patient_geography
--   )
--   WHERE rn = 1
-- ),
-- patient_zip AS (
--   SELECT DISTINCT
--     a.patient_id,
--     b.patient_zip AS zip3
--   FROM base_table a
--   LEFT JOIN patient_geography b
--     ON a.patient_id = b.patient_id
-- ),

-- tx_claims_payer_analysis AS (
--   SELECT DISTINCT
--     a.patient_id,
--     a.fill_date,
--     a.claim_id,
--     a.kh_plan_id,
--     b.payer_name,
--     b.insurance_group
--   FROM (
--     SELECT
--       patient_id,
--       service_date AS fill_date,
--       kh_plan_id,
--       medical_event_id AS claim_id
--     FROM com_edp_prd.com_raw.kom_medical_events

--     UNION ALL

--     SELECT
--       patient_id,
--       fill_date,
--       COALESCE(primary_kh_plan_id, secondary_kh_plan_id) AS kh_plan_id,
--       pharmacy_event_id AS claim_id
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE transaction_result = 'PAID'
--   ) a
--   LEFT JOIN com_edp_prd.com_raw.kom_plans b
--     ON a.kh_plan_id = b.kh_plan_id
-- ),

-- payer_claims_count AS (
--   SELECT *,
--     ROW_NUMBER() OVER (
--       PARTITION BY patient_id
--       ORDER BY num_claims DESC
--     ) AS rn
--   FROM (
--     SELECT
--       patient_id,
--       payer_name,
--       COUNT(DISTINCT claim_id) AS num_claims
--     FROM tx_claims_payer_analysis
--     WHERE payer_name IS NOT NULL
--       AND fill_date BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--     GROUP BY patient_id, payer_name
--   )
-- ),

-- patient_primary_secondary_payer AS (
--   SELECT
--     patient_id,
--     MAX(CASE WHEN rn = 1 THEN payer_name END) AS primary_payer,
--     MAX(CASE WHEN rn = 2 THEN payer_name END) AS secondary_payer
--   FROM payer_claims_count
--   GROUP BY patient_id
-- ),

-- patient_latest_insurance_type AS (
--   SELECT
--     patient_id,
--     insurance_group AS latest_insurance_type
--   FROM (
--     SELECT
--       patient_id,
--       insurance_group,
--       fill_date,
--       ROW_NUMBER() OVER (
--         PARTITION BY patient_id
--         ORDER BY fill_date DESC
--       ) AS rn
--     FROM tx_claims_payer_analysis
--     WHERE insurance_group IS NOT NULL
--   )
--   WHERE rn = 1
-- )

-- SELECT
--   a.*,
--   b.age_bucket,
--   c.zip3,
--   f.primary_payer,
--   f.secondary_payer,
--   g.latest_insurance_type

-- FROM base_table a
-- LEFT JOIN age_buckets b
--   ON a.patient_id = b.patient_id
-- LEFT JOIN patient_zip c
--   ON a.patient_id = c.patient_id
-- LEFT JOIN patient_primary_secondary_payer f
--   ON a.patient_id = f.patient_id
-- LEFT JOIN patient_latest_insurance_type g
--   ON a.patient_id = g.patient_id;


-- -- =============================================================================
-- -- CELL 10: specialty re-enrichment for most_recently_treated and primary HCPs (UNCHANGED)
-- -- =============================================================================
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master as
-- select a.*except(a.most_recently_treated_hcp_specialty_2yr, a.PRIMARY_HCP_SPECIALTY, a.primary_hcp_secondary_specialty),
-- b.PRIMARY_SPECIALTY as most_recently_treated_hcp_specialty_2yr, b.SECONDARY_SPECIALTY as most_recently_treated_hcp_secondary_specialty_2yr, c.PRIMARY_SPECIALTY as primary_hcp_specialty, c.SECONDARY_SPECIALTY as primary_hcp_secondary_specialty
-- from cmpa_insights_internal_schema.patient360_master as a
-- left join com_raw.kom_providers as b on a.most_recently_treated_hcp_2yr = b.npi and b.PROVIDER_TYPE = 'INDIVIDUAL'
-- left join com_raw.kom_providers as c on a.PRIMARY_HCP_NPI = c.npi and c.PROVIDER_TYPE = 'INDIVIDUAL';


-- -- =============================================================================
-- -- CELL 11: newborn screening flag (UNCHANGED)
-- -- =============================================================================
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master as
-- with base_table as (
--   select distinct * from com_edp_prd.cmpa_insights_internal_schema.patient360_master
-- ),
-- newborn_screening_flag as (
--   select *,
--   case when patient_state in ('IL', 'MO', 'WV', 'PA', 'KY', 'MD', 'CA', 'DE', 'FL', 'KS', 'AZ', 'MA', 'RI', 'AR', 'IA', 'NC', 'TX', 'CT') then 1 else 0 end as newborn_screening_flag
--   from base_table
-- )
-- select * from newborn_screening_flag;


-- -- =============================================================================
-- -- CELL 12: most recent infusion location (UNCHANGED)
-- -- =============================================================================
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master as
-- WITH base_table as (
--   select distinct *
--   from com_edp_prd.cmpa_insights_internal_schema.patient360_master
-- ),

-- tx_patients as (
--   select distinct
--       patient_id,
--       coalesce(rendering_npi, referring_npi) as npi,
--       service_date as fill_date,
--       place_of_service
--   from com_edp_prd.com_raw.kom_medical_events
--   where ndc11 in ('54092070001','540920700')
--     and service_date between '2023-08-01' and (SELECT end_date FROM runtime_parameters)

--   union

--   select distinct
--       patient_id,
--       prescriber_npi as npi,
--       fill_date,
--       '' as place_of_service
--   from com_edp_prd.com_raw.kom_pharmacy_events
--   where ndc11 in ('54092070001','540920700')
--     and transaction_result = 'PAID'
--     and fill_date between '2023-08-01' and (SELECT end_date FROM runtime_parameters)

--   union

--   select distinct
--       patient_id,
--       rendering_npi as npi,
--       service_date as fill_date,
--       place_of_service
--   from com_edp_prd.com_raw.kom_medical_events
--   where procedure_code in ('99601','99602','96365','96366','J1743','S9357','S9379',
--                            '38206','38230','38232','38240','38241','38242','38243','38250')
--     and service_date between '2023-08-01' and (SELECT end_date FROM runtime_parameters)
-- ),

-- latest_pos_patients as (
--   select
--       patient_id,
--       nullif(trim(place_of_service), '') as most_recent_infusion_location
--   from (
--     select
--         patient_id,
--         place_of_service,
--         fill_date,
--         row_number() over (
--           partition by patient_id
--           order by fill_date desc
--         ) as rn
--     from tx_patients
--   ) t
--   where rn = 1
-- )

-- select
--     a.*,
--     b.most_recent_infusion_location,
--     c.description as most_recent_infusion_description
-- from base_table a
-- left join latest_pos_patients b
--   on a.patient_id = b.patient_id
-- left join com_edp_prd.cmpa_insights_internal_schema.pos_description c
--   on try_cast(nullif(trim(b.most_recent_infusion_location), '') as int) = c.code;


-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master AS
-- SELECT *
-- FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
-- WHERE NOT (
--     UPPER(PATIENT_GENDER) = 'F'
--     AND UPPER(latest_mpsii_tx_type) = 'OTHER_TX'
-- );

In [0]:
-- -- =============================================================================
-- -- CELL 1: patient_hcp_visit_summary (patient360_base)
-- -- MODIFIED: Patient cohort definition replaced with new Elaprase-specific logic
-- -- =============================================================================
-- CREATE OR REPLACE TEMP VIEW patient_hcp_visit_summary AS
-- WITH MPSII_Diagnoses_Specified AS (

--     SELECT DISTINCT
--         PATIENT_ID,
--         SERVICE_DATE AS DX_DATE
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE DIAGNOSIS_CODES LIKE '%E761%'

--     UNION

--     SELECT DISTINCT
--         PATIENT_ID,
--         FILL_DATE AS DX_DATE
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE DIAGNOSIS_CODE = 'E761'
--       AND TRANSACTION_STATUS = 'PAID'
-- ),

-- Patients_2Dx_Specified AS (

--     SELECT PATIENT_ID
--     FROM MPSII_Diagnoses_Specified
--     GROUP BY PATIENT_ID
--     HAVING COUNT(DISTINCT DX_DATE) >= 2
-- ),

-- /* ============================================================================
--    MPSII Treatment - Elaprase Ever (NEW COHORT DEFINITION)
--    ========================================================================== */
-- MPSII_Treatment_Elaprase_Ever AS (

--     SELECT DISTINCT PATIENT_ID
--     FROM (

--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE NDC11 IN ('54092070001','540920700')

--         UNION ALL

--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_pharmacy_events
--         WHERE NDC11 IN ('54092070001','540920700')
--           AND TRANSACTION_RESULT = 'PAID'

--         UNION ALL

--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE PROCEDURE_CODE = 'J1743'

--     ) t
-- ),

-- /* ============================================================================
--    Base MPS II Universe (NEW COHORT DEFINITION)
--    ========================================================================== */
-- Base_MPSII_Universe AS (

--     SELECT PATIENT_ID
--     FROM Patients_2Dx_Specified

--     UNION

--     SELECT PATIENT_ID
--     FROM MPSII_Treatment_Elaprase_Ever
-- ),

-- /* ============================================================================
--    Recent Treatment Activity (NEW COHORT DEFINITION - using global_runtime_dates)
--    ========================================================================== */
-- Recent_Treatment_2Y AS (

--     SELECT DISTINCT PATIENT_ID
--     FROM (

--         -- Elaprase NDC (Medical)
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE NDC11 IN ('54092070001','540920700')
--           AND SERVICE_DATE >= (SELECT elaprase_tx_start_date FROM global_runtime_dates)

--         UNION ALL

--         -- Elaprase NDC (Pharmacy)
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_pharmacy_events
--         WHERE NDC11 IN ('54092070001','540920700')
--           AND TRANSACTION_RESULT = 'PAID'
--           AND FILL_DATE >= (SELECT elaprase_tx_start_date FROM global_runtime_dates)

--         UNION ALL

--         -- Elaprase J-code
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE PROCEDURE_CODE = 'J1743'
--           AND SERVICE_DATE >= (SELECT elaprase_tx_start_date FROM global_runtime_dates)

--         UNION ALL

--         -- Other ERT / infusion procedure codes
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
--           AND SERVICE_DATE >= (SELECT elaprase_tx_start_date FROM global_runtime_dates)

--     ) t
-- ),

-- /* ============================================================================
--    FINAL ELIGIBLE PATIENTS (NEW COHORT DEFINITION)
--    ========================================================================== */
-- eligible_patients AS (

--     SELECT DISTINCT b.PATIENT_ID
--     FROM Base_MPSII_Universe b
--     INNER JOIN Recent_Treatment_2Y r
--         ON b.PATIENT_ID = r.PATIENT_ID
-- ),

-- /* ============================================================================
--    2) PROVIDER FILTER ("COHORT 3 LEARNINGS")
--    ========================================================================== */
-- cohort_3_learnings AS (
--   SELECT DISTINCT npi
--   FROM com_raw.kom_providers
--   WHERE provider_type = 'INDIVIDUAL'
--     AND (
--       PRIMARY_SPECIALTY NOT IN (
--         'Anesthesiologist Assistant','Anesthesiology','Dentist','Dietitian, Registered',
--         'Emergency Medical Technician, Basic','Emergency Medicine','General Acute Care Hospital',
--         'Nurse Anesthetist, Certified Registered','Obstetrics & Gynecology','Pathology',
--         'Radiology','Urology'
--       )
--       OR SECONDARY_SPECIALTY IN (
--         'Child & Adolescent Psychiatry','Psychiatry','Adolescent Medicine','Developmental - Behavioral Pediatrics',
--         'Neonatal-Perinatal Medicine','Nutrition, Pediatric','Oncology, Pediatrics','Pediatric Cardiology',
--         'Pediatric Critical Care Medicine','Pediatric Dermatology','Pediatric Emergency Medicine',
--         'Pediatric Endocrinology','Pediatric Gastroenterology','Pediatric Hematology-Oncology',
--         'Pediatric Infectious Diseases','Pediatric Nephrology','Pediatric Ophthalmology and Strabismus Specialist',
--         'Pediatric Orthopaedic Surgery','Pediatric Otolaryngology','Pediatric Pulmonology','Pediatric Radiology',
--         'Pediatric Rehabilitation Medicine','Pediatric Rheumatology','Pediatric Surgery','Pediatrics',
--         'Clinical Biochemical Genetics','Clinical Genetics (M.D.)','Clinical Molecular Genetics',
--         'Ph.D. Medical Genetics','Neurodevelopmental Disabilities','Neurology',
--         'Neurology with Special Qualifications in Child Neurology','Neuroradiology'
--       )
--     )
-- ),

-- /* ============================================================================
--    3) CLAIMS UNIVERSES (DX + TX) WITH NPI ATTRIBUTION
--    ========================================================================== */

-- all_dx_claims_5yr AS (
--     SELECT DISTINCT *
--     FROM (
--       SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
--         AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--       UNION
--       SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
--       FROM com_edp_prd.com_raw.kom_pharmacy_events
--       WHERE DIAGNOSIS_CODE IN ('E761','E763')
--         AND TRANSACTION_STATUS = 'PAID'
--         AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     )
--     WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
-- ),

-- all_tx_claims_5yr AS (
--     SELECT DISTINCT *
--     FROM (
--       SELECT DISTINCT
--           PATIENT_ID,
--           COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
--           SERVICE_DATE AS FILL_DATE,
--           NDC11 AS TX_CODE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE NDC11 IN ('54092070001','540920700')
--         AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--       UNION

--       SELECT DISTINCT
--           PATIENT_ID,
--           PRESCRIBER_NPI AS NPI,
--           FILL_DATE,
--           NDC11 AS TX_CODE
--       FROM com_edp_prd.com_raw.kom_pharmacy_events
--       WHERE NDC11 IN ('54092070001','540920700')
--         AND TRANSACTION_RESULT = 'PAID'
--         AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--       UNION

--       SELECT DISTINCT
--           PATIENT_ID,
--           RENDERING_NPI AS NPI,
--           SERVICE_DATE AS FILL_DATE,
--           PROCEDURE_CODE AS TX_CODE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
--                                '38206','38230','38232','38240','38241','38242','38243','38250')
--         AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     )
--     WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
-- ),

-- all_tx_claims_alltime AS (
--     SELECT DISTINCT *
--     FROM (
--       SELECT DISTINCT
--           PATIENT_ID,
--           COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
--           SERVICE_DATE AS FILL_DATE,
--           NDC11 AS TX_CODE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE NDC11 IN ('54092070001','540920700')
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--       UNION

--       SELECT DISTINCT
--           PATIENT_ID,
--           PRESCRIBER_NPI AS NPI,
--           FILL_DATE,
--           NDC11 AS TX_CODE
--       FROM com_edp_prd.com_raw.kom_pharmacy_events
--       WHERE NDC11 IN ('54092070001','540920700')
--         AND TRANSACTION_RESULT = 'PAID'
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--       UNION

--       SELECT DISTINCT
--           PATIENT_ID,
--           RENDERING_NPI AS NPI,
--           SERVICE_DATE AS FILL_DATE,
--           PROCEDURE_CODE AS TX_CODE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
--                                '38206','38230','38232','38240','38241','38242','38243','38250')
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     )
--     WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
-- ),

-- all_claims_5yr AS (
--     SELECT DISTINCT
--         PATIENT_ID,
--         NPI,
--         FILL_DATE,
--         CAST(NULL AS STRING) AS TX_CODE
--     FROM all_dx_claims_5yr
--     UNION
--     SELECT DISTINCT
--         PATIENT_ID,
--         NPI,
--         FILL_DATE,
--         TX_CODE
--     FROM all_tx_claims_5yr
-- ),

-- all_claims_3yr AS (
--     SELECT DISTINCT *
--     FROM (
--       SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
--         AND SERVICE_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--       UNION
--       SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
--       FROM com_edp_prd.com_raw.kom_pharmacy_events
--       WHERE DIAGNOSIS_CODE IN ('E761','E763')
--         AND TRANSACTION_STATUS = 'PAID'
--         AND FILL_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--       UNION
--       SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE NDC11 IN ('54092070001','540920700')
--         AND SERVICE_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--       UNION
--       SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
--       FROM com_edp_prd.com_raw.kom_pharmacy_events
--       WHERE NDC11 IN ('54092070001','540920700')
--         AND TRANSACTION_RESULT = 'PAID'
--         AND FILL_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--       UNION
--       SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI, SERVICE_DATE AS FILL_DATE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
--                                '38206','38230','38232','38240','38241','38242','38243','38250')
--         AND SERVICE_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     )
--     WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
-- ),

-- /* ============================================================================
--    4) FIRST DX / FIRST TX HCP ATTRIBUTION (5Y WINDOW)
--    ========================================================================== */

-- first_dx_hcp_ranked AS (
--     SELECT PATIENT_ID, NPI, MIN(FILL_DATE) AS first_dx_date,
--            ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY MIN(FILL_DATE) ASC, NPI) AS rn
--     FROM all_dx_claims_5yr
--     WHERE NPI IS NOT NULL
--     GROUP BY PATIENT_ID, NPI
-- ),
-- first_dx_hcp AS (
--     SELECT PATIENT_ID, NPI AS first_dx_hcp, first_dx_date
--     FROM first_dx_hcp_ranked
--     WHERE rn = 1
-- ),

-- provider_dim AS (
--     SELECT
--         npi,
--         CONCAT(FIRST_NAME, ' ', LAST_NAME) AS provider_name,
--         primary_specialty
--     FROM com_raw.kom_providers
--     WHERE provider_type = 'INDIVIDUAL'
-- ),

-- first_dx_hcp_5yr_stats AS (
--     SELECT fdh.PATIENT_ID, fdh.first_dx_hcp,
--            COUNT(DISTINCT ac.FILL_DATE) AS first_dx_all_visit_count_5yr,
--            MAX(ac.FILL_DATE) AS first_dx_last_visit_5yr
--     FROM first_dx_hcp fdh
--     LEFT JOIN all_claims_5yr ac
--       ON fdh.PATIENT_ID = ac.PATIENT_ID AND fdh.first_dx_hcp = ac.NPI
--     GROUP BY fdh.PATIENT_ID, fdh.first_dx_hcp
-- ),

-- first_tx_hcp_ranked AS (
--     SELECT PATIENT_ID, NPI, MIN(FILL_DATE) AS first_tx_date,
--            ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY MIN(FILL_DATE) ASC, NPI) AS rn
--     FROM all_tx_claims_5yr
--     WHERE NPI IS NOT NULL
--     GROUP BY PATIENT_ID, NPI
-- ),
-- first_tx_hcp AS (
--     SELECT PATIENT_ID, NPI AS first_tx_hcp, first_tx_date
--     FROM first_tx_hcp_ranked
--     WHERE rn = 1
-- ),

-- first_tx_hcp_5yr_stats AS (
--     SELECT fth.PATIENT_ID, fth.first_tx_hcp,
--            COUNT(DISTINCT ac.FILL_DATE) AS first_tx_all_visit_count_5yr,
--            MAX(ac.FILL_DATE) AS first_tx_last_visit_5yr
--     FROM first_tx_hcp fth
--     LEFT JOIN all_claims_5yr ac
--       ON fth.PATIENT_ID = ac.PATIENT_ID AND fth.first_tx_hcp = ac.NPI
--     GROUP BY fth.PATIENT_ID, fth.first_tx_hcp
-- ),

-- first_tx_hcp_5yr_tx_only AS (
--     SELECT fth.PATIENT_ID, fth.first_tx_hcp,
--            COUNT(DISTINCT tx.FILL_DATE) AS first_tx_treatment_visit_count_5yr
--     FROM first_tx_hcp fth
--     LEFT JOIN all_tx_claims_5yr tx
--       ON fth.PATIENT_ID = tx.PATIENT_ID AND fth.first_tx_hcp = tx.NPI
--     GROUP BY fth.PATIENT_ID, fth.first_tx_hcp
-- ),

-- /* ============================================================================
--    5) MOST-SEEN HCP RANKING (TOP 5) USING 3Y ACTIVITY
--    ========================================================================== */

-- most_seen_3yr_ranking AS (
--     SELECT PATIENT_ID,
--            NPI,
--            COUNT(DISTINCT FILL_DATE) AS visit_count_3yr,
--            MAX(FILL_DATE) AS last_visit_3yr,
--            ROW_NUMBER() OVER (
--              PARTITION BY PATIENT_ID
--              ORDER BY COUNT(DISTINCT FILL_DATE) DESC,
--                       MAX(FILL_DATE) DESC,
--                       NPI ASC
--            ) AS rank
--     FROM all_claims_3yr
--     WHERE NPI IS NOT NULL
--     GROUP BY PATIENT_ID, NPI
-- ),

-- most_seen_combined_stats AS (
--     SELECT
--         ms3.PATIENT_ID,
--         ms3.NPI,
--         ms3.rank,
--         ms3.visit_count_3yr,
--         ms3.last_visit_3yr,
--         COUNT(DISTINCT ac5.FILL_DATE) AS visit_count_5yr,
--         MAX(ac5.FILL_DATE)          AS last_visit_5yr
--     FROM most_seen_3yr_ranking ms3
--     LEFT JOIN all_claims_5yr ac5
--       ON ms3.PATIENT_ID = ac5.PATIENT_ID AND ms3.NPI = ac5.NPI
--     WHERE ms3.rank <= 5
--     GROUP BY ms3.PATIENT_ID, ms3.NPI, ms3.rank, ms3.visit_count_3yr, ms3.last_visit_3yr
-- ),

-- /* ============================================================================
--    6) HISTORICAL (ALL-TIME) FIRST DX / FIRST TX DATES (PATIENT LEVEL)
--    ========================================================================== */

-- historical_first_dx AS (
--     SELECT PATIENT_ID, MIN(FILL_DATE) AS incidence_date
--     FROM (
--         SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE DIAGNOSIS_CODES LIKE '%E761%'
--           AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID, FILL_DATE
--         FROM com_edp_prd.com_raw.kom_pharmacy_events
--         WHERE DIAGNOSIS_CODE = 'E761'
--           AND TRANSACTION_STATUS = 'PAID'
--           AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE DIAGNOSIS_CODES LIKE '%E763%'
--           AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID, FILL_DATE
--         FROM com_edp_prd.com_raw.kom_pharmacy_events
--         WHERE DIAGNOSIS_CODE = 'E763'
--           AND TRANSACTION_STATUS = 'PAID'
--           AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     ) all_dx
--     GROUP BY PATIENT_ID
-- ),

-- historical_first_tx AS (
--     SELECT PATIENT_ID, MIN(FILL_DATE) AS first_incidence_treatment_date
--     FROM (
--         SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE NDC11 IN ('54092070001','540920700')
--           AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID, FILL_DATE
--         FROM com_edp_prd.com_raw.kom_pharmacy_events
--         WHERE NDC11 IN ('54092070001','540920700')
--           AND TRANSACTION_RESULT = 'PAID'
--           AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
--                                  '38206','38230','38232','38240','38241','38242','38243','38250')
--           AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     ) all_tx
--     GROUP BY PATIENT_ID
-- ),

-- /* ============================================================================
--    7) LATEST HCP ATTRIBUTION (5Y WINDOW)
--    ========================================================================== */

-- all_claims_5yr_specialty_removed AS (
--     SELECT DISTINCT *
--     FROM (
--       SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
--         AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--       UNION
--       SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
--       FROM com_edp_prd.com_raw.kom_pharmacy_events
--       WHERE DIAGNOSIS_CODE IN ('E761','E763')
--         AND TRANSACTION_STATUS = 'PAID'
--         AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--       UNION
--       SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE NDC11 IN ('54092070001','540920700')
--         AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--       UNION
--       SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
--       FROM com_edp_prd.com_raw.kom_pharmacy_events
--       WHERE NDC11 IN ('54092070001','540920700')
--         AND TRANSACTION_RESULT = 'PAID'
--         AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--       UNION
--       SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI, SERVICE_DATE AS FILL_DATE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
--                                '38206','38230','38232','38240','38241','38242','38243','38250')
--         AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     )
-- ),

-- latest_claim_hcp_ranked as (
--   select patient_id, npi, fill_date, row_number() over(partition by patient_id order by fill_date desc, npi asc) as rn
--   from all_claims_5yr
-- ),

-- latest_claim_hcp as (
--   select patient_id, npi as latest_claim_hcp_npi, fill_date as latest_claim_date from latest_claim_hcp_ranked where rn = 1
-- ),

-- latest_claim_hcp_visit_count_5yr as (
--   select a.patient_id, a.latest_claim_hcp_npi, count(distinct fill_date) as latest_claim_hcp_visit_count_5yr
--   from latest_claim_hcp as a
--   left join all_claims_5yr as b on a.patient_id = b.patient_id and a.latest_claim_hcp_npi = b.npi
--   group by 1,2
-- ),

-- latest_claim_hcp_final as (
--   select a.patient_id, a.latest_claim_hcp_npi, a.latest_claim_date, b.latest_claim_hcp_visit_count_5yr
--   from latest_claim_hcp as a
--   left join latest_claim_hcp_visit_count_5yr as b on a.patient_id = b.patient_id
-- ),

-- all_tx_claims_5yr_specialty_removed AS (
--     SELECT DISTINCT *
--     FROM (
--       SELECT DISTINCT
--           PATIENT_ID,
--           COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
--           SERVICE_DATE AS FILL_DATE,
--           NDC11 AS TX_CODE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE NDC11 IN ('54092070001','540920700')
--         AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--       UNION

--       SELECT DISTINCT
--           PATIENT_ID,
--           PRESCRIBER_NPI AS NPI,
--           FILL_DATE,
--           NDC11 AS TX_CODE
--       FROM com_edp_prd.com_raw.kom_pharmacy_events
--       WHERE NDC11 IN ('54092070001','540920700')
--         AND TRANSACTION_RESULT = 'PAID'
--         AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--       UNION

--       SELECT DISTINCT
--           PATIENT_ID,
--           RENDERING_NPI AS NPI,
--           SERVICE_DATE AS FILL_DATE,
--           PROCEDURE_CODE AS TX_CODE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
--                                '38206','38230','38232','38240','38241','38242','38243','38250')
--         AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     )
-- ),

-- most_recent_tx_hcp_ranked as (
--   select patient_id, npi, fill_date, row_number() over(partition by patient_id order by fill_date desc, npi asc) as rn
--   from all_tx_claims_5yr
-- ),

-- most_recent_tx_hcp as (
--   select patient_id, npi as latest_treatment_hcp_npi, fill_date as latest_treatment_date
--   from most_recent_tx_hcp_ranked where rn = 1
-- ),

-- latest_treatment_hcp_visit_count as (
--   select a.patient_id, a.latest_treatment_hcp_npi, count(distinct fill_date) as latest_treatment_hcp_visit_count_5yr
--   from most_recent_tx_hcp as a
--   left join all_tx_claims_5yr as b on a.patient_id = b.patient_id and a.latest_treatment_hcp_npi = b.npi
--   group by 1, 2
-- ),

-- most_recent_tx_hcp_final as (
--   select a.patient_id, a.latest_treatment_hcp_npi, a.latest_treatment_date, b.latest_treatment_hcp_visit_count_5yr
--   from most_recent_tx_hcp as a
--   left join latest_treatment_hcp_visit_count as b on a.patient_id = b.patient_id
-- ),

-- /* ============================================================================
--    9) LATEST TX TYPE
--    Derived directly from recent 2Y treatment claims:
--      - Elaprase NDC, pharmacy NDC, or J1743 -> 'ELAPRASE'
--      - Other infusion/transplant procedure codes -> 'OTHER_TX'
--    Self-contained so every eligible patient gets a non-null label.
--    ========================================================================== */

-- recent_2yr_treatment_claims AS (

--     -- Elaprase Medical NDC
--     SELECT
--         PATIENT_ID,
--         SERVICE_DATE AS TX_DATE,
--         'ELAPRASE' AS TX_TYPE
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE NDC11 IN ('54092070001','540920700')
--       AND SERVICE_DATE >= (SELECT elaprase_tx_start_date FROM global_runtime_dates)
--       AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--     UNION ALL

--     -- Elaprase Pharmacy NDC
--     SELECT
--         PATIENT_ID,
--         FILL_DATE AS TX_DATE,
--         'ELAPRASE' AS TX_TYPE
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE NDC11 IN ('54092070001','540920700')
--       AND TRANSACTION_RESULT = 'PAID'
--       AND FILL_DATE >= (SELECT elaprase_tx_start_date FROM global_runtime_dates)
--       AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--     UNION ALL

--     -- Elaprase J-code
--     SELECT
--         PATIENT_ID,
--         SERVICE_DATE AS TX_DATE,
--         'ELAPRASE' AS TX_TYPE
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE PROCEDURE_CODE = 'J1743'
--       AND SERVICE_DATE >= (SELECT elaprase_tx_start_date FROM global_runtime_dates)
--       AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--     UNION ALL

--     -- Other infusion / transplant procedure codes
--     SELECT
--         PATIENT_ID,
--         SERVICE_DATE AS TX_DATE,
--         'OTHER_TX' AS TX_TYPE
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','S9357','S9379',
--                              '38206','38230','38232','38240','38241','38242','38243','38250')
--       AND SERVICE_DATE >= (SELECT elaprase_tx_start_date FROM global_runtime_dates)
--       AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
-- ),

-- latest_mpsii_treatment_type AS (
--     SELECT PATIENT_ID, latest_mpsii_tx_type
--     FROM (
--         SELECT
--             PATIENT_ID,
--             TX_TYPE AS latest_mpsii_tx_type,
--             TX_DATE,
--             ROW_NUMBER() OVER (
--                 PARTITION BY PATIENT_ID
--                 ORDER BY TX_DATE DESC
--             ) AS rn
--         FROM recent_2yr_treatment_claims
--     ) t
--     WHERE rn = 1
-- ),

-- /* ============================================================================
--    10) FIRST TX AFTER DIAGNOSIS (ALL-TIME TX, BUT MUST BE AFTER DX)
--    ========================================================================== */

-- first_tx_after_diagnosis AS (
--   SELECT
--     tx.patient_id,
--     MIN(tx.fill_date) AS first_tx_after_diagnosis
--   FROM all_tx_claims_alltime tx
--   INNER JOIN historical_first_dx dx
--     ON tx.patient_id = dx.patient_id
--   WHERE tx.fill_date >= dx.incidence_date
--   GROUP BY tx.patient_id
-- ),

-- /* ============================================================================
--    11) ELAPRASE FILL COUNTS (WINDOWED)
--    ========================================================================== */

-- elaprase_fills AS (
--   SELECT
--     patient_id,
--     COUNT(DISTINCT fill_date) AS elaprase_fills
--   FROM all_tx_claims_5yr
--   WHERE fill_date BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--     AND tx_code IN ('54092070001','540920700','J1743')
--   GROUP BY patient_id
-- ),

-- /* ============================================================================
--    12) AVLAYAH STATUS LOGIC
--    ========================================================================== */

-- avlayah_status_base AS (
--   SELECT
--     patient_id,

--     MIN(
--       CASE
--         WHEN tx_code IN ('8497600101')
--           OR (tx_code IN ('J3490','J3590','J9999')
--               AND fill_date >= '2026-03-01')
--         THEN fill_date
--       END
--     ) AS first_avlayah_date,

--     MAX(
--       CASE
--         WHEN tx_code IN ('8497600101')
--           OR (tx_code IN ('J3490','J3590','J9999')
--               AND fill_date >= '2026-03-01')
--         THEN 1 ELSE 0
--       END
--     ) AS has_avlayah,

--     MAX(
--       CASE
--         WHEN tx_code IN (
--           '99601','99602','96365','96366','J1743','S9357','S9379',
--           '38206','38230','38232','38240','38241','38242','38243','38250',
--           '54092070001','540920700'
--         )
--         THEN 1 ELSE 0
--       END
--     ) AS has_elaprase

--   FROM all_tx_claims_5yr
--   GROUP BY patient_id
-- ),

-- avlayah_status AS (
--   SELECT
--     patient_id,
--     first_avlayah_date,
--     has_avlayah,
--     has_elaprase,
--     CASE
--       WHEN has_avlayah = 1 AND has_elaprase = 0 THEN 'New'
--       WHEN has_avlayah = 1 AND has_elaprase = 1 THEN 'Switched'
--       WHEN has_avlayah = 0 AND has_elaprase = 1 THEN 'Non-Avlayah'
--       ELSE NULL
--     END AS avlayah_status
--   FROM avlayah_status_base
-- ),

-- /* ============================================================================
--    13) PATIENT DIMENSIONS
--    ========================================================================== */

-- patient_demographics AS (
--     SELECT *
--     FROM (
--         SELECT DISTINCT PATIENT_ID, PATIENT_YOB, PATIENT_GENDER,
--                ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY PATIENT_YOB ASC) AS rn
--         FROM com_edp_prd.com_raw.kom_patient_demographics
--     )
--     WHERE rn = 1
-- ),
-- patient_geography AS (
--     SELECT patient_id, patient_state
--     FROM (
--         SELECT
--             PATIENT_ID,
--             patient_state,
--             ROW_NUMBER() OVER (
--                 PARTITION BY PATIENT_ID
--                 ORDER BY
--                     CASE WHEN VALID_TO_DATE > CURRENT_DATE() THEN 1 ELSE 2 END,
--                     VALID_TO_DATE DESC
--             ) AS rn
--         FROM COM_EDP_PRD.COM_RAW.KOM_PATIENT_GEOGRAPHY
--     )
--     WHERE rn = 1
-- ),

-- /* ============================================================================
--    14) PIVOT TOP-5 MOST-SEEN HCPs INTO WIDE FORMAT
--    ========================================================================== */
-- most_seen_pivot AS (
--     SELECT
--         PATIENT_ID,

--         MAX(CASE WHEN rank = 1 THEN NPI END)              AS most_seen_hcp1_3yr_ranked,
--         MAX(CASE WHEN rank = 1 THEN visit_count_5yr END)  AS most_seen_hcp1_visit_count_5yr,
--         MAX(CASE WHEN rank = 1 THEN last_visit_5yr END)   AS most_seen_hcp1_last_visit_5yr,

--         MAX(CASE WHEN rank = 2 THEN NPI END)              AS most_seen_hcp2_3yr_ranked,
--         MAX(CASE WHEN rank = 2 THEN visit_count_5yr END)  AS most_seen_hcp2_visit_count_5yr,
--         MAX(CASE WHEN rank = 2 THEN last_visit_5yr END)   AS most_seen_hcp2_last_visit_5yr,

--         MAX(CASE WHEN rank = 3 THEN NPI END)              AS most_seen_hcp3_3yr_ranked,
--         MAX(CASE WHEN rank = 3 THEN visit_count_5yr END)  AS most_seen_hcp3_visit_count_5yr,
--         MAX(CASE WHEN rank = 3 THEN last_visit_5yr END)   AS most_seen_hcp3_last_visit_5yr,

--         MAX(CASE WHEN rank = 4 THEN NPI END)              AS most_seen_hcp4_3yr_ranked,
--         MAX(CASE WHEN rank = 4 THEN visit_count_5yr END)  AS most_seen_hcp4_visit_count_5yr,
--         MAX(CASE WHEN rank = 4 THEN last_visit_5yr END)   AS most_seen_hcp4_last_visit_5yr,

--         MAX(CASE WHEN rank = 5 THEN NPI END)              AS most_seen_hcp5_3yr_ranked,
--         MAX(CASE WHEN rank = 5 THEN visit_count_5yr END)  AS most_seen_hcp5_visit_count_5yr,
--         MAX(CASE WHEN rank = 5 THEN last_visit_5yr END)   AS most_seen_hcp5_last_visit_5yr

--     FROM most_seen_combined_stats
--     GROUP BY PATIENT_ID
-- )

-- /* ============================================================================
--    FINAL SELECT
--    ========================================================================== */
-- SELECT
--     ep.PATIENT_ID,
--     pd.PATIENT_YOB,
--     YEAR(CURRENT_DATE) - YEAR(pd.PATIENT_YOB) AS PATIENT_AGE,
--     pd.PATIENT_GENDER,
--     pg.patient_state,

--     hfdx.incidence_date,
--     hftx.first_incidence_treatment_date,

--     lch.latest_claim_date AS latest_claim_date,

--     lch.latest_claim_hcp_npi,
--     pdlch.provider_name     AS latest_claim_hcp_name,
--     pdlch.primary_specialty AS latest_claim_hcp_specialty,
--     COALESCE(lch.latest_claim_hcp_visit_count_5yr, 0) AS latest_claim_hcp_visit_count,

--     ref1.hco_name AS latest_claim_hcp_hco_name,

--     mrt.latest_treatment_date AS latest_treatment_date,

--     lmt.latest_mpsii_tx_type,

--     fta.first_tx_after_diagnosis,

--     ROUND(MONTHS_BETWEEN(fta.first_tx_after_diagnosis, hfdx.incidence_date), 0)
--       AS time_dx_to_first_tx_in_months,

--     ROUND(MONTHS_BETWEEN(mrt.latest_treatment_date, fta.first_tx_after_diagnosis), 0) AS treatment_period_months,

--     COALESCE(ef.elaprase_fills, 0) AS elaprase_fills,
--     avs.avlayah_status,

--     CASE
--       WHEN avs.avlayah_status = 'Switched'
--       THEN avs.first_avlayah_date
--     END AS avlayah_switch_date,
--     mrt.latest_treatment_hcp_npi,
--     pdtch.provider_name     AS latest_treatment_hcp_name,
--     pdtch.primary_specialty AS latest_treatment_hcp_specialty,
--     COALESCE(mrt.latest_treatment_hcp_visit_count_5yr, 0) AS latest_treatment_hcp_visit_count,

--     ref2.hco_name AS latest_treatment_hcp_hco_name,

--     fdh.first_dx_hcp AS first_dx_hcp_5yr,
--     COALESCE(fdhs.first_dx_all_visit_count_5yr, 0) AS first_dx_all_visit_count_5yr,
--     fdhs.first_dx_last_visit_5yr AS first_dx_last_visit_5yr,

--     fth.first_tx_hcp AS first_tx_hcp_5yr,
--     COALESCE(fths.first_tx_all_visit_count_5yr, 0) AS first_tx_all_visit_count_5yr,
--     COALESCE(fthtx.first_tx_treatment_visit_count_5yr, 0) AS first_tx_treatment_visit_count_5yr,
--     fths.first_tx_last_visit_5yr AS first_tx_last_visit_5yr,

--     msp.most_seen_hcp1_3yr_ranked,
--     COALESCE(msp.most_seen_hcp1_visit_count_5yr, 0) AS most_seen_hcp1_visit_count_5yr,
--     msp.most_seen_hcp1_last_visit_5yr,

--     msp.most_seen_hcp2_3yr_ranked,
--     COALESCE(msp.most_seen_hcp2_visit_count_5yr, 0) AS most_seen_hcp2_visit_count_5yr,
--     msp.most_seen_hcp2_last_visit_5yr,

--     msp.most_seen_hcp3_3yr_ranked,
--     COALESCE(msp.most_seen_hcp3_visit_count_5yr, 0) AS most_seen_hcp3_visit_count_5yr,
--     msp.most_seen_hcp3_last_visit_5yr,

--     msp.most_seen_hcp4_3yr_ranked,
--     COALESCE(msp.most_seen_hcp4_visit_count_5yr, 0) AS most_seen_hcp4_visit_count_5yr,
--     msp.most_seen_hcp4_last_visit_5yr,

--     msp.most_seen_hcp5_3yr_ranked,
--     COALESCE(msp.most_seen_hcp5_visit_count_5yr, 0) AS most_seen_hcp5_visit_count_5yr,
--     msp.most_seen_hcp5_last_visit_5yr

-- FROM eligible_patients ep
-- LEFT JOIN patient_demographics pd
--     ON ep.PATIENT_ID = pd.PATIENT_ID
-- LEFT JOIN patient_geography pg
--     ON ep.PATIENT_ID = pg.PATIENT_ID

-- LEFT JOIN historical_first_dx hfdx
--     ON ep.PATIENT_ID = hfdx.PATIENT_ID
-- LEFT JOIN historical_first_tx hftx
--     ON ep.PATIENT_ID = hftx.PATIENT_ID

-- LEFT JOIN first_tx_after_diagnosis fta
--     ON ep.PATIENT_ID = fta.PATIENT_ID

-- LEFT JOIN elaprase_fills ef
--     ON ep.PATIENT_ID = ef.PATIENT_ID

-- LEFT JOIN latest_mpsii_treatment_type lmt
--     ON ep.PATIENT_ID = lmt.PATIENT_ID

-- LEFT JOIN latest_claim_hcp_final lch
--     ON ep.PATIENT_ID = lch.PATIENT_ID
-- LEFT JOIN provider_dim pdlch
--     ON lch.latest_claim_hcp_npi = pdlch.npi
-- LEFT JOIN cmpa_insights_internal_schema.reference_file ref1
--     ON lch.latest_claim_hcp_npi = ref1.hcp_npi

-- LEFT JOIN most_recent_tx_hcp_final mrt
--     ON ep.PATIENT_ID = mrt.PATIENT_ID
-- LEFT JOIN provider_dim pdtch
--     ON mrt.latest_treatment_hcp_npi = pdtch.npi
-- LEFT JOIN cmpa_insights_internal_schema.reference_file ref2
--     ON mrt.latest_treatment_hcp_npi = ref2.hcp_npi

-- LEFT JOIN first_dx_hcp fdh
--     ON ep.PATIENT_ID = fdh.PATIENT_ID
-- LEFT JOIN first_dx_hcp_5yr_stats fdhs
--     ON ep.PATIENT_ID = fdhs.PATIENT_ID
-- LEFT JOIN first_tx_hcp fth
--     ON ep.PATIENT_ID = fth.PATIENT_ID
-- LEFT JOIN first_tx_hcp_5yr_stats fths
--     ON ep.PATIENT_ID = fths.PATIENT_ID
-- LEFT JOIN first_tx_hcp_5yr_tx_only fthtx
--     ON ep.PATIENT_ID = fthtx.PATIENT_ID

-- LEFT JOIN most_seen_pivot msp
--     ON ep.PATIENT_ID = msp.PATIENT_ID
-- LEFT JOIN avlayah_status avs
--   ON ep.PATIENT_ID = avs.patient_id
-- ORDER BY ep.PATIENT_ID;

-- -- Materialize the temp view into the persistent base table.
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_base AS
-- SELECT DISTINCT * FROM patient_hcp_visit_summary;


-- -- =============================================================================
-- -- CELL 2: mpsii_tx_claims (UNCHANGED)
-- -- =============================================================================
-- CREATE OR REPLACE TEMP VIEW mpsii_tx_claims AS
-- WITH MPSII_Diagnoses_Specified AS (
--     SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE DIAGNOSIS_CODES LIKE '%E761%'
--       AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--     UNION
--     SELECT DISTINCT PATIENT_ID, FILL_DATE
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE DIAGNOSIS_CODE = 'E761'
--       AND TRANSACTION_RESULT = 'PAID'
--       AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
-- ),

-- Patients_2Dx_Specified AS (
--     SELECT PATIENT_ID
--     FROM MPSII_Diagnoses_Specified
--     GROUP BY PATIENT_ID
--     HAVING COUNT(DISTINCT FILL_DATE) >= 2
-- ),

-- MPSII_Diagnoses_Unspecified AS (
--     SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE DIAGNOSIS_CODES LIKE '%E763%'
--       AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--     UNION
--     SELECT DISTINCT PATIENT_ID, FILL_DATE
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE DIAGNOSIS_CODE = 'E763'
--       AND TRANSACTION_RESULT = 'PAID'
--       AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
-- ),

-- Patients_2Dx_Unspecified AS (
--     SELECT PATIENT_ID
--     FROM MPSII_Diagnoses_Unspecified
--     GROUP BY PATIENT_ID
--     HAVING COUNT(DISTINCT FILL_DATE) >= 2
-- ),

-- MPSII_Treatment_All AS (
--     SELECT DISTINCT PATIENT_ID FROM (
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE NDC11 IN ('54092070001','540920700')
--           AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_pharmacy_events
--         WHERE NDC11 IN ('54092070001','540920700')
--           AND TRANSACTION_RESULT = 'PAID'
--           AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
--                                  '38206','38230','38232','38240','38241','38242','38243','38250')
--           AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--     ) t
-- ),

-- MPSII_Treatment_Elaprase_Only AS (
--     SELECT DISTINCT PATIENT_ID FROM (
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE NDC11 IN ('54092070001','540920700')
--           AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_pharmacy_events
--         WHERE NDC11 IN ('54092070001','540920700')
--           AND TRANSACTION_RESULT = 'PAID'
--           AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE PROCEDURE_CODE = 'J1743'
--           AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--     ) t
-- ),

-- Patients_2Dx_Specified_With_Treatment AS (
--     SELECT DISTINCT p.PATIENT_ID
--     FROM Patients_2Dx_Specified p
--     INNER JOIN MPSII_Treatment_All t USING (PATIENT_ID)
-- ),

-- Patients_Incremental_Unspecified AS (
--     SELECT DISTINCT p.PATIENT_ID
--     FROM Patients_2Dx_Unspecified p
--     INNER JOIN MPSII_Treatment_Elaprase_Only t USING (PATIENT_ID)
--     WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
-- ),

-- eligible_patients AS (
--     SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment
--     UNION
--     SELECT PATIENT_ID FROM Patients_Incremental_Unspecified
-- ),

-- cohort_3_learnings AS (
--   SELECT DISTINCT npi
--   FROM com_raw.kom_providers
--   WHERE provider_type = 'INDIVIDUAL'
--     AND (
--       PRIMARY_SPECIALTY NOT IN (
--         'Anesthesiologist Assistant',
--         'Anesthesiology',
--         'Dentist',
--         'Dietitian, Registered',
--         'Emergency Medical Technician, Basic',
--         'Emergency Medicine',
--         'General Acute Care Hospital',
--         'Nurse Anesthetist, Certified Registered',
--         'Obstetrics & Gynecology',
--         'Pathology',
--         'Radiology',
--         'Urology'
--       )
--       OR SECONDARY_SPECIALTY IN (
--         'Child & Adolescent Psychiatry',
--         'Psychiatry',
--         'Adolescent Medicine',
--         'Developmental - Behavioral Pediatrics',
--         'Neonatal-Perinatal Medicine',
--         'Nutrition, Pediatric',
--         'Oncology, Pediatrics',
--         'Pediatric Cardiology',
--         'Pediatric Critical Care Medicine',
--         'Pediatric Dermatology',
--         'Pediatric Emergency Medicine',
--         'Pediatric Endocrinology',
--         'Pediatric Gastroenterology',
--         'Pediatric Hematology-Oncology',
--         'Pediatric Infectious Diseases',
--         'Pediatric Nephrology',
--         'Pediatric Ophthalmology and Strabismus Specialist',
--         'Pediatric Orthopaedic Surgery',
--         'Pediatric Otolaryngology',
--         'Pediatric Pulmonology',
--         'Pediatric Radiology',
--         'Pediatric Rehabilitation Medicine',
--         'Pediatric Rheumatology',
--         'Pediatric Surgery',
--         'Pediatrics',
--         'Clinical Biochemical Genetics',
--         'Clinical Genetics (M.D.)',
--         'Clinical Molecular Genetics',
--         'Ph.D. Medical Genetics',
--         'Neurodevelopmental Disabilities',
--         'Neurology',
--         'Neurology with Special Qualifications in Child Neurology',
--         'Neuroradiology'
--       )
--     )
-- ),

-- all_tx_claims_2yr AS (
--     SELECT DISTINCT *
--     FROM (
--       SELECT DISTINCT
--         PATIENT_ID,
--         COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
--         SERVICE_DATE AS FILL_DATE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE NDC11 IN ('54092070001','540920700')
--         AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--       UNION

--       SELECT DISTINCT
--         PATIENT_ID,
--         PRESCRIBER_NPI AS NPI,
--         FILL_DATE
--       FROM com_edp_prd.com_raw.kom_pharmacy_events
--       WHERE NDC11 IN ('54092070001','540920700')
--         AND TRANSACTION_RESULT = 'PAID'
--         AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--       UNION

--       SELECT DISTINCT
--         PATIENT_ID,
--         RENDERING_NPI AS NPI,
--         SERVICE_DATE AS FILL_DATE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
--                                '38206','38230','38232','38240','38241','38242','38243','38250')
--         AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     )
--     WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
-- )

-- SELECT * FROM all_tx_claims_2yr;


-- -- =============================================================================
-- -- CELL 3: most_recently_treated_hcp (UNCHANGED)
-- -- =============================================================================
-- CREATE OR REPLACE TEMPORARY VIEW most_recently_treated_hcp AS
-- WITH tx_claims AS (
--     SELECT DISTINCT *
--     FROM mpsii_tx_claims
-- ),

-- MPSII_Diagnoses_Specified AS (
--     SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE DIAGNOSIS_CODES LIKE '%E761%'
--       AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--     UNION
--     SELECT DISTINCT PATIENT_ID, FILL_DATE
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE DIAGNOSIS_CODE = 'E761'
--       AND TRANSACTION_RESULT = 'PAID'
--       AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
-- ),

-- Patients_2Dx_Specified AS (
--     SELECT PATIENT_ID
--     FROM MPSII_Diagnoses_Specified
--     GROUP BY PATIENT_ID
--     HAVING COUNT(DISTINCT FILL_DATE) >= 2
-- ),

-- MPSII_Diagnoses_Unspecified AS (
--     SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE DIAGNOSIS_CODES LIKE '%E763%'
--       AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--     UNION
--     SELECT DISTINCT PATIENT_ID, FILL_DATE
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE DIAGNOSIS_CODE = 'E763'
--       AND TRANSACTION_RESULT = 'PAID'
--       AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
-- ),

-- Patients_2Dx_Unspecified AS (
--     SELECT PATIENT_ID
--     FROM MPSII_Diagnoses_Unspecified
--     GROUP BY PATIENT_ID
--     HAVING COUNT(DISTINCT FILL_DATE) >= 2
-- ),

-- MPSII_Treatment_All AS (
--     SELECT DISTINCT PATIENT_ID FROM (
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE NDC11 IN ('54092070001','540920700')
--           AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_pharmacy_events
--         WHERE NDC11 IN ('54092070001','540920700')
--           AND TRANSACTION_RESULT = 'PAID'
--           AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
--                                  '38206','38230','38232','38240','38241','38242','38243','38250')
--           AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--     ) t
-- ),

-- MPSII_Treatment_Elaprase_Only AS (
--     SELECT DISTINCT PATIENT_ID FROM (
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE NDC11 IN ('54092070001','540920700')
--           AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_pharmacy_events
--         WHERE NDC11 IN ('54092070001','540920700')
--           AND TRANSACTION_RESULT = 'PAID'
--           AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE PROCEDURE_CODE = 'J1743'
--           AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--     ) t
-- ),

-- Patients_2Dx_Specified_With_Treatment AS (
--     SELECT DISTINCT p.PATIENT_ID
--     FROM Patients_2Dx_Specified p
--     INNER JOIN MPSII_Treatment_All t USING (PATIENT_ID)
-- ),

-- Patients_Incremental_Unspecified AS (
--     SELECT DISTINCT p.PATIENT_ID
--     FROM Patients_2Dx_Unspecified p
--     INNER JOIN MPSII_Treatment_Elaprase_Only t USING (PATIENT_ID)
--     WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
-- ),

-- eligible_patients AS (
--     SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment
--     UNION
--     SELECT PATIENT_ID FROM Patients_Incremental_Unspecified
-- ),

-- cohort_3_learnings AS (
--   SELECT DISTINCT npi
--   FROM com_raw.kom_providers
--   WHERE provider_type = 'INDIVIDUAL'
--     AND (
--       PRIMARY_SPECIALTY NOT IN (
--         'Anesthesiologist Assistant','Anesthesiology','Dentist','Dietitian, Registered',
--         'Emergency Medical Technician, Basic','Emergency Medicine','General Acute Care Hospital',
--         'Nurse Anesthetist, Certified Registered','Obstetrics & Gynecology','Pathology',
--         'Radiology','Urology'
--       )
--       OR SECONDARY_SPECIALTY IN (
--         'Child & Adolescent Psychiatry','Psychiatry','Adolescent Medicine','Developmental - Behavioral Pediatrics',
--         'Neonatal-Perinatal Medicine','Nutrition, Pediatric','Oncology, Pediatrics','Pediatric Cardiology',
--         'Pediatric Critical Care Medicine','Pediatric Dermatology','Pediatric Emergency Medicine',
--         'Pediatric Endocrinology','Pediatric Gastroenterology','Pediatric Hematology-Oncology',
--         'Pediatric Infectious Diseases','Pediatric Nephrology','Pediatric Ophthalmology and Strabismus Specialist',
--         'Pediatric Orthopaedic Surgery','Pediatric Otolaryngology','Pediatric Pulmonology','Pediatric Radiology',
--         'Pediatric Rehabilitation Medicine','Pediatric Rheumatology','Pediatric Surgery','Pediatrics',
--         'Clinical Biochemical Genetics','Clinical Genetics (M.D.)','Clinical Molecular Genetics',
--         'Ph.D. Medical Genetics','Neurodevelopmental Disabilities','Neurology',
--         'Neurology with Special Qualifications in Child Neurology','Neuroradiology'
--       )
--     )
-- ),

-- all_dx_claims_5yr AS (
--     SELECT DISTINCT *
--     FROM (
--       SELECT DISTINCT
--         PATIENT_ID,
--         COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
--         SERVICE_DATE AS FILL_DATE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
--         AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--       UNION
--       SELECT DISTINCT
--         PATIENT_ID,
--         PRESCRIBER_NPI AS NPI,
--         FILL_DATE
--       FROM com_edp_prd.com_raw.kom_pharmacy_events
--       WHERE DIAGNOSIS_CODE IN ('E761','E763')
--         AND TRANSACTION_RESULT = 'PAID'
--         AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     )
--     WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
-- ),

-- all_tx_claims_5yr AS (
--     SELECT DISTINCT *
--     FROM (
--       SELECT DISTINCT
--         PATIENT_ID,
--         COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
--         SERVICE_DATE AS FILL_DATE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE NDC11 IN ('54092070001','540920700')
--         AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--       UNION
--       SELECT DISTINCT
--         PATIENT_ID,
--         PRESCRIBER_NPI AS NPI,
--         FILL_DATE
--       FROM com_edp_prd.com_raw.kom_pharmacy_events
--       WHERE NDC11 IN ('54092070001','540920700')
--         AND TRANSACTION_RESULT = 'PAID'
--         AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--       UNION
--       SELECT DISTINCT
--         PATIENT_ID,
--         RENDERING_NPI AS NPI,
--         SERVICE_DATE AS FILL_DATE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
--                                '38206','38230','38232','38240','38241','38242','38243','38250')
--         AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     )
--     WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
-- ),

-- all_claims AS (
--     SELECT * FROM all_dx_claims_5yr
--     UNION
--     SELECT * FROM all_tx_claims_5yr
-- ),

-- latest_treating_hcp AS (
--     SELECT patient_id, npi
--     FROM (
--         SELECT
--             patient_id,
--             npi,
--             fill_date,
--             ROW_NUMBER() OVER (
--                 PARTITION BY patient_id
--                 ORDER BY
--                     CASE WHEN npi IS NOT NULL THEN 1 ELSE 2 END,
--                     fill_date DESC,
--                     npi DESC
--             ) AS rn
--         FROM tx_claims
--     ) t
--     WHERE rn = 1
--       AND npi IS NOT NULL
-- ),

-- visit_counts AS (
--     SELECT
--         patient_id,
--         npi,
--         COUNT(DISTINCT fill_date) AS visit_counts
--     FROM all_claims
--     WHERE npi IS NOT NULL
--     GROUP BY patient_id, npi
-- ),

-- last_visit_date AS (
--     SELECT
--         lth.patient_id,
--         lth.npi,
--         MAX(ac.fill_date) AS last_visit_date
--     FROM latest_treating_hcp lth
--     LEFT JOIN all_claims ac
--       ON lth.patient_id = ac.patient_id
--      AND lth.npi = ac.npi
--     GROUP BY lth.patient_id, lth.npi
-- ),

-- latest_treating_hcp_with_visits AS (
--     SELECT
--         a.patient_id,
--         a.npi AS most_recently_treated_hcp,
--         b.visit_counts AS no_of_visits,
--         c.last_visit_date AS last_visit_date_5yr
--     FROM latest_treating_hcp AS a
--     LEFT JOIN visit_counts AS b
--         ON a.patient_id = b.patient_id
--        AND a.npi        = b.npi
--     LEFT JOIN last_visit_date AS c
--         ON a.patient_id = c.patient_id
--        AND a.npi        = c.npi
-- ),

-- hcp_with_other_info AS (
--     SELECT
--         a.*,
--         CONCAT(b.FIRST_NAME, ' ', b.LAST_NAME) AS hcp_name,
--         b.PRIMARY_SPECIALTY AS hcp_specialty,
--         c.hco_name,
--         c.territory_id,
--         c.territory,
--         c.region_id,
--         c.region
--     FROM latest_treating_hcp_with_visits AS a

--     LEFT JOIN com_edp_prd.com_raw.kom_providers AS b
--         ON a.most_recently_treated_hcp = b.NPI
--        AND b.PROVIDER_TYPE = 'INDIVIDUAL'

--     LEFT JOIN (
--   SELECT
--     a.hcp_npi,
--     a.hco_name,
--     a.territory,
--     a.region,
--     b.territory_id,
--     c.region_id,
--     a.hcp_primary_specialty AS hcp_specialty
--   FROM cmpa_insights_internal_schema.reference_file a
--   LEFT JOIN (
--       SELECT DISTINCT territory_id, territory_name
--       FROM cmpa_insights_internal_schema.zip_to_territory_mapping
--   ) b
--     ON a.territory = b.territory_name
--   LEFT JOIN (
--       SELECT DISTINCT region_id, region_name
--       FROM cmpa_insights_internal_schema.zip_to_territory_mapping
--   ) c
--     ON a.region = c.region_name
-- ) c
-- ON a.most_recently_treated_hcp = c.hcp_npi)

-- SELECT
--     patient_id,
--     most_recently_treated_hcp AS most_recently_treated_hcp_2yr,
--     hcp_name AS most_recently_treated_hcp_name_2yr,

--     no_of_visits AS most_recently_treated_hcp_2yr_no_of_visits_5yr,
--     last_visit_date_5yr AS most_recent_tx_hcp_2yr_last_visit_5yr,

--     hcp_specialty AS most_recently_treated_hcp_specialty_2yr,
--     hco_name AS most_recently_treated_hcp_hco_name,
--     territory_id AS most_recently_treated_hcp_territory_id_2yr,
--     territory AS most_recently_treated_hcp_territory_2yr,
--     region_id AS most_recently_treated_hcp_region_id_2yr,
--     region AS most_recently_treated_hcp_region_2yr
--     FROM hcp_with_other_info;


-- -- =============================================================================
-- -- CELL 4: Build all_dx_claims, all_tx_claims views (UNCHANGED)
-- -- =============================================================================
-- CREATE OR REPLACE TEMPORARY VIEW all_dx_claims AS

-- SELECT DISTINCT
--     PATIENT_ID,
--     COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
--     SERVICE_DATE AS FILL_DATE,
--     'DX' AS CLAIM_TYPE
-- FROM com_edp_prd.com_raw.kom_medical_events
-- WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
--   AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)

-- UNION

-- SELECT DISTINCT
--     PATIENT_ID,
--     PRESCRIBER_NPI AS NPI,
--     FILL_DATE,
--     'DX' AS CLAIM_TYPE
-- FROM com_edp_prd.com_raw.kom_pharmacy_events
-- WHERE DIAGNOSIS_CODE IN ('E761', 'E763')
--   AND TRANSACTION_RESULT = 'PAID'
--   AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters);


-- CREATE OR REPLACE TEMPORARY VIEW all_tx_claims AS

-- SELECT DISTINCT
--     PATIENT_ID,
--     COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
--     SERVICE_DATE AS FILL_DATE,
--     'TX' AS CLAIM_TYPE,
--     NDC11 AS CODE
-- FROM com_edp_prd.com_raw.kom_medical_events
-- WHERE NDC11 IN ('54092070001', '540920700')
--   AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)

-- UNION

-- SELECT DISTINCT
--     PATIENT_ID,
--     RENDERING_NPI AS NPI,
--     SERVICE_DATE AS FILL_DATE,
--     'TX' AS CLAIM_TYPE,
--     PROCEDURE_CODE AS CODE
-- FROM com_edp_prd.com_raw.kom_medical_events
-- WHERE PROCEDURE_CODE IN ('99601', '99602', '96365', '96366', 'J1743',
--                          'S9357', 'S9379', '38206', '38230', '38232',
--                          '38240', '38241', '38242', '38243', '38250')
--   AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)

-- UNION

-- SELECT DISTINCT
--     PATIENT_ID,
--     PRESCRIBER_NPI AS NPI,
--     FILL_DATE,
--     'TX' AS CLAIM_TYPE,
--     NDC11 AS CODE
-- FROM com_edp_prd.com_raw.kom_pharmacy_events
-- WHERE NDC11 IN ('54092070001', '540920700')
--   AND TRANSACTION_RESULT = 'PAID'
--   AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters);


-- CREATE OR REPLACE TEMPORARY VIEW tx_claims_2yr AS
-- SELECT DISTINCT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE, CODE
-- FROM all_tx_claims
-- WHERE FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters);


-- CREATE OR REPLACE TEMPORARY VIEW e761_patients_2dx AS
-- SELECT PATIENT_ID
-- FROM (
--     SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE DIAGNOSIS_CODES LIKE '%E761%'
--       AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--     UNION
--     SELECT DISTINCT PATIENT_ID, FILL_DATE
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE DIAGNOSIS_CODE = 'E761'
--       AND TRANSACTION_RESULT = 'PAID'
--       AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
-- )
-- GROUP BY PATIENT_ID
-- HAVING COUNT(DISTINCT FILL_DATE) >= 2;

-- CREATE OR REPLACE TEMPORARY VIEW specified_patients AS
-- SELECT DISTINCT e.PATIENT_ID
-- FROM e761_patients_2dx e
-- INNER JOIN tx_claims_2yr t ON e.PATIENT_ID = t.PATIENT_ID;

-- CREATE OR REPLACE TEMPORARY VIEW e763_patients_2dx AS
-- SELECT PATIENT_ID
-- FROM (
--     SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE DIAGNOSIS_CODES LIKE '%E763%'
--       AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--     UNION
--     SELECT DISTINCT PATIENT_ID, FILL_DATE
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE DIAGNOSIS_CODE = 'E763'
--       AND TRANSACTION_RESULT = 'PAID'
--       AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
-- )
-- GROUP BY PATIENT_ID
-- HAVING COUNT(DISTINCT FILL_DATE) >= 2;

-- CREATE OR REPLACE TEMPORARY VIEW elaprase_tx_2yr AS
-- SELECT DISTINCT PATIENT_ID
-- FROM tx_claims_2yr
-- WHERE CODE IN ('54092070001', '540920700', 'J1743');

-- CREATE OR REPLACE TEMPORARY VIEW incremental_patients AS
-- SELECT DISTINCT e.PATIENT_ID
-- FROM e763_patients_2dx e
-- INNER JOIN elaprase_tx_2yr t ON e.PATIENT_ID = t.PATIENT_ID
-- WHERE e.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM specified_patients);

-- CREATE OR REPLACE TEMPORARY VIEW eligible_patients AS
-- SELECT PATIENT_ID FROM specified_patients
-- UNION
-- SELECT PATIENT_ID FROM incremental_patients;


-- CREATE OR REPLACE TEMPORARY VIEW cohort_3_learnings AS
-- SELECT DISTINCT npi
-- FROM com_raw.kom_providers
-- WHERE provider_type = 'INDIVIDUAL'
--   AND (
--     PRIMARY_SPECIALTY NOT IN (
--       'Anesthesiologist Assistant','Anesthesiology','Dentist','Dietitian, Registered',
--       'Emergency Medical Technician, Basic','Emergency Medicine','General Acute Care Hospital',
--       'Nurse Anesthetist, Certified Registered','Obstetrics & Gynecology','Pathology',
--       'Radiology','Urology'
--     )
--     OR SECONDARY_SPECIALTY IN (
--       'Child & Adolescent Psychiatry','Psychiatry','Adolescent Medicine','Developmental - Behavioral Pediatrics',
--       'Neonatal-Perinatal Medicine','Nutrition, Pediatric','Oncology, Pediatrics','Pediatric Cardiology',
--       'Pediatric Critical Care Medicine','Pediatric Dermatology','Pediatric Emergency Medicine',
--       'Pediatric Endocrinology','Pediatric Gastroenterology','Pediatric Hematology-Oncology',
--       'Pediatric Infectious Diseases','Pediatric Nephrology','Pediatric Ophthalmology and Strabismus Specialist',
--       'Pediatric Orthopaedic Surgery','Pediatric Otolaryngology','Pediatric Pulmonology','Pediatric Radiology',
--       'Pediatric Rehabilitation Medicine','Pediatric Rheumatology','Pediatric Surgery','Pediatrics',
--       'Clinical Biochemical Genetics','Clinical Genetics (M.D.)','Clinical Molecular Genetics',
--       'Ph.D. Medical Genetics','Neurodevelopmental Disabilities','Neurology',
--       'Neurology with Special Qualifications in Child Neurology','Neuroradiology'
--     )
--   );


-- CREATE OR REPLACE TEMPORARY VIEW all_patient_claims AS
-- SELECT *
-- FROM (
--   SELECT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
--   FROM all_dx_claims
--   WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--   UNION

--   SELECT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
--   FROM all_tx_claims
--   WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
-- )
-- WHERE npi IN (SELECT DISTINCT npi FROM cohort_3_learnings);


-- CREATE OR REPLACE TEMPORARY VIEW primary_hcp AS
-- WITH hcp_metrics AS (
--     SELECT
--         a.PATIENT_ID,
--         a.NPI,

--         CASE
--             WHEN p.primary_specialty LIKE '%Genetic%'
--               OR p.secondary_specialty LIKE '%Genetic%'
--                 THEN 'Geneticist'
--             WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
--               OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
--               OR p.primary_specialty LIKE '%Neurological Surgery%'
--                 THEN 'Psychiatry & Neurology'
--             WHEN p.primary_specialty LIKE '%Pediatrics%'
--                 THEN 'Pediatrician'
--             WHEN p.primary_specialty LIKE '%Internal Medicine%'
--               OR p.secondary_specialty LIKE '%Internal Medicine%'
--               OR p.primary_specialty LIKE '%Family Medicine%'
--               OR p.secondary_specialty LIKE '%Family Medicine%'
--                 THEN 'PCP'
--             WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
--               OR p.primary_specialty LIKE '%Physician Assistant%'
--                 THEN 'NPPA'
--             WHEN a.NPI IS NULL
--                 THEN 'NA'
--             ELSE 'Others'
--         END AS SPECIALTY,

--         CASE
--             WHEN p.primary_specialty LIKE '%Genetic%'
--               OR p.secondary_specialty LIKE '%Genetic%'
--                 THEN 1
--             WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
--               OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
--               OR p.primary_specialty LIKE '%Neurological Surgery%'
--                 THEN 2
--             WHEN p.primary_specialty LIKE '%Pediatrics%'
--                 THEN 3
--             WHEN p.primary_specialty LIKE '%Internal Medicine%'
--               OR p.secondary_specialty LIKE '%Internal Medicine%'
--               OR p.primary_specialty LIKE '%Family Medicine%'
--               OR p.secondary_specialty LIKE '%Family Medicine%'
--                 THEN 4
--             WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
--               OR p.primary_specialty LIKE '%Physician Assistant%'
--                 THEN 5
--             WHEN a.NPI IS NULL
--                 THEN 7
--             ELSE 6
--         END AS SPECIALTY_PRIORITY,

--         COUNT(DISTINCT a.FILL_DATE) AS NO_OF_VISITS,

--         COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'DX' THEN a.FILL_DATE END) AS DX_VISITS,
--         COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'TX' THEN a.FILL_DATE END) AS TX_VISITS,

--         MAX(a.FILL_DATE) AS MOST_RECENT_VISIT

--     FROM all_patient_claims a
--     LEFT JOIN com_edp_prd.com_raw.kom_providers p
--         ON a.NPI = p.NPI
--     GROUP BY
--         a.PATIENT_ID,
--         a.NPI,
--         p.primary_specialty,
--         p.secondary_specialty
-- ),
-- ranked_hcps AS (
--     SELECT
--         *,
--         RANK() OVER (
--             PARTITION BY PATIENT_ID
--             ORDER BY
--                 SPECIALTY_PRIORITY ASC,
--                 NO_OF_VISITS DESC,
--                 MOST_RECENT_VISIT DESC,
--                 NPI ASC
--         ) AS HCP_RANK
--     FROM hcp_metrics
-- )
-- SELECT
--     PATIENT_ID,
--     NPI AS PRIMARY_HCP_NPI,
--     SPECIALTY AS PRIMARY_HCP_SPECIALTY,
--     SPECIALTY_PRIORITY,
--     NO_OF_VISITS,
--     DX_VISITS,
--     TX_VISITS,
--     MOST_RECENT_VISIT,
--     HCP_RANK
-- FROM ranked_hcps
-- WHERE HCP_RANK = 1;


-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.primary_hcp AS
-- SELECT
--     ph.*,
--     COALESCE(p.FIRST_NAME, '') || ' ' || COALESCE(p.LAST_NAME, '') AS primary_hcp_name_2yr,

--     ref.HCO_NAME AS primary_hcp_hco_name_2yr,
--     ref.HCO_CITY AS primary_hcp_hco_city_2yr,
--     ref.HCO_STATE AS primary_hcp_hco_state_2yr,

--     ref.mapped_territory_id AS primary_hcp_territory_id_2yr,
--     ref.TERRITORY           AS primary_hcp_territory_2yr,
--     ref.mapped_region_id    AS primary_hcp_region_id_2yr,
--     ref.region              AS primary_hcp_region_2yr

-- FROM primary_hcp ph

-- LEFT JOIN com_edp_prd.com_raw.kom_providers p
--     ON ph.PRIMARY_HCP_NPI = p.NPI


-- LEFT JOIN (
--   SELECT
--     a.hcp_npi,
--     a.hco_name,
--     a.hco_city,
--     a.hco_state,
--     a.territory,
--     a.region,
--     b.territory_id AS mapped_territory_id,
--     c.region_id AS mapped_region_id,
--     a.hcp_primary_specialty AS hcp_specialty
--   FROM cmpa_insights_internal_schema.reference_file a
--   LEFT JOIN (
--       SELECT DISTINCT try_cast(territory_id AS BIGINT) AS territory_id, territory_name
--       FROM cmpa_insights_internal_schema.zip_to_territory_mapping
--   ) b
--     ON try_cast(a.territory_id AS BIGINT) = b.territory_id
--   LEFT JOIN (
--       SELECT DISTINCT try_cast(region_id AS BIGINT) AS region_id, region_name
--       FROM cmpa_insights_internal_schema.zip_to_territory_mapping
--   ) c
--     ON try_cast(a.region_id AS BIGINT) = c.region_id
-- ) ref
-- ON ph.PRIMARY_HCP_NPI = ref.hcp_npi;


-- -- =============================================================================
-- -- CELL 5: patient360_master initial join (UNCHANGED)
-- -- =============================================================================
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master AS
-- SELECT DISTINCT
--     a.PATIENT_ID,
--     a.PATIENT_YOB,
--     a.PATIENT_AGE,
--     a.PATIENT_GENDER,
--     a.patient_state,

--     a.incidence_date,
--     a.first_incidence_treatment_date,
--     a.latest_claim_date,

--     a.latest_claim_hcp_npi,
--     a.latest_claim_hcp_name,
--     a.latest_claim_hcp_specialty,
--     a.latest_claim_hcp_visit_count,
--     a.latest_claim_hcp_hco_name,

--     a.latest_treatment_date,
--     a.latest_mpsii_tx_type,
--     a.first_tx_after_diagnosis,
--     a.time_dx_to_first_tx_in_months,
--     a.treatment_period_months,
--     a.elaprase_fills,
--     a.avlayah_status,
--     a.avlayah_switch_date,
--     a.latest_treatment_hcp_npi,
--     a.latest_treatment_hcp_name,
--     a.latest_treatment_hcp_specialty,
--     a.latest_treatment_hcp_visit_count,
--     a.latest_treatment_hcp_hco_name,

--     a.first_dx_hcp_5yr,
--     a.first_dx_all_visit_count_5yr,
--     a.first_dx_last_visit_5yr,
--     a.first_tx_hcp_5yr,
--     a.first_tx_all_visit_count_5yr,
--     a.first_tx_treatment_visit_count_5yr,
--     a.first_tx_last_visit_5yr,

--     a.most_seen_hcp1_3yr_ranked,
--     a.most_seen_hcp1_visit_count_5yr,
--     a.most_seen_hcp1_last_visit_5yr,
--     a.most_seen_hcp2_3yr_ranked,
--     a.most_seen_hcp2_visit_count_5yr,
--     a.most_seen_hcp2_last_visit_5yr,
--     a.most_seen_hcp3_3yr_ranked,
--     a.most_seen_hcp3_visit_count_5yr,
--     a.most_seen_hcp3_last_visit_5yr,
--     a.most_seen_hcp4_3yr_ranked,
--     a.most_seen_hcp4_visit_count_5yr,
--     a.most_seen_hcp4_last_visit_5yr,
--     a.most_seen_hcp5_3yr_ranked,
--     a.most_seen_hcp5_visit_count_5yr,
--     a.most_seen_hcp5_last_visit_5yr,

--     b.* EXCEPT (patient_id),

--     c.* EXCEPT (patient_id)

-- FROM com_edp_prd.cmpa_insights_internal_schema.patient360_base AS a

-- LEFT JOIN most_recently_treated_hcp AS b
--     ON a.PATIENT_ID = b.patient_id

-- LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.primary_hcp AS c
--     ON a.PATIENT_ID = c.patient_id;


-- -- =============================================================================
-- -- CELL 6: severity flagging (UNCHANGED)
-- -- =============================================================================
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master AS
-- WITh  base AS (
--   SELECT DISTINCT
--     a.PATIENT_ID,
--     b.SERVICE_DATE,
--     b.DIAGNOSIS_CODES
--   FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master a
--   LEFT JOIN com_edp_prd.com_raw.kom_medical_events b
--     ON a.PATIENT_ID = b.PATIENT_ID
--    AND b.SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
-- ),

-- flagged AS (
--   SELECT
--     PATIENT_ID,
--     SERVICE_DATE,
--     CASE
--       WHEN DIAGNOSIS_CODES IS NOT NULL AND (
--            DIAGNOSIS_CODES ILIKE '%|G910|%' OR DIAGNOSIS_CODES ILIKE '%|G911|%' OR DIAGNOSIS_CODES ILIKE '%|G912|%'
--         OR DIAGNOSIS_CODES ILIKE '%|G913|%' OR DIAGNOSIS_CODES ILIKE '%|G914|%' OR DIAGNOSIS_CODES ILIKE '%|G918|%'
--         OR DIAGNOSIS_CODES ILIKE '%|G919|%' OR DIAGNOSIS_CODES ILIKE '%|Q038|%' OR DIAGNOSIS_CODES ILIKE '%|Q039|%'
--         OR DIAGNOSIS_CODES ILIKE '%|Q050|%' OR DIAGNOSIS_CODES ILIKE '%|Q051|%' OR DIAGNOSIS_CODES ILIKE '%|Q052|%'
--         OR DIAGNOSIS_CODES ILIKE '%|Q053|%' OR DIAGNOSIS_CODES ILIKE '%|Q054|%' OR DIAGNOSIS_CODES ILIKE '%|Q055|%'
--         OR DIAGNOSIS_CODES ILIKE '%|Q056|%' OR DIAGNOSIS_CODES ILIKE '%|Q057|%' OR DIAGNOSIS_CODES ILIKE '%|Q058|%'
--         OR DIAGNOSIS_CODES ILIKE '%|Q0700|%' OR DIAGNOSIS_CODES ILIKE '%|Q0702|%' OR DIAGNOSIS_CODES ILIKE '%|Q0703|%'
--         OR DIAGNOSIS_CODES ILIKE '%|F445|%' OR DIAGNOSIS_CODES ILIKE '%|F639|%' OR DIAGNOSIS_CODES ILIKE '%|F70|%'
--         OR DIAGNOSIS_CODES ILIKE '%|F71|%' OR DIAGNOSIS_CODES ILIKE '%|F72|%' OR DIAGNOSIS_CODES ILIKE '%|F73|%'
--         OR DIAGNOSIS_CODES ILIKE '%|F78|%' OR DIAGNOSIS_CODES ILIKE '%|F78A1|%' OR DIAGNOSIS_CODES ILIKE '%|F78A9|%'
--         OR DIAGNOSIS_CODES ILIKE '%|F79|%' OR DIAGNOSIS_CODES ILIKE '%|F800|%' OR DIAGNOSIS_CODES ILIKE '%|F801|%'
--         OR DIAGNOSIS_CODES ILIKE '%|F802|%' OR DIAGNOSIS_CODES ILIKE '%|F804|%' OR DIAGNOSIS_CODES ILIKE '%|F8081|%'
--         OR DIAGNOSIS_CODES ILIKE '%|F8082|%' OR DIAGNOSIS_CODES ILIKE '%|F8089|%' OR DIAGNOSIS_CODES ILIKE '%|F809|%'
--         OR DIAGNOSIS_CODES ILIKE '%|F810|%' OR DIAGNOSIS_CODES ILIKE '%|F812|%' OR DIAGNOSIS_CODES ILIKE '%|F8181|%'
--         OR DIAGNOSIS_CODES ILIKE '%|F8189|%' OR DIAGNOSIS_CODES ILIKE '%|F819|%' OR DIAGNOSIS_CODES ILIKE '%|F82|%'
--         OR DIAGNOSIS_CODES ILIKE '%|F840|%' OR DIAGNOSIS_CODES ILIKE '%|F843|%' OR DIAGNOSIS_CODES ILIKE '%|F845|%'
--         OR DIAGNOSIS_CODES ILIKE '%|F848|%' OR DIAGNOSIS_CODES ILIKE '%|F849|%' OR DIAGNOSIS_CODES ILIKE '%|F88|%'
--         OR DIAGNOSIS_CODES ILIKE '%|F89|%' OR DIAGNOSIS_CODES ILIKE '%|R6250|%' OR DIAGNOSIS_CODES ILIKE '%|R620|%'
--         OR DIAGNOSIS_CODES ILIKE '%|R6251|%' OR DIAGNOSIS_CODES ILIKE '%|R6259|%' OR DIAGNOSIS_CODES ILIKE '%|R62|%'
--       )
--       THEN 1 ELSE 0
--     END AS has_severity_code
--   FROM base
-- ),

-- patient_with_severity_outcome AS (
--   SELECT
--     PATIENT_ID,
--     COUNT(DISTINCT SERVICE_DATE) AS count_fill_date,
--     COUNT(DISTINCT CASE
--                      WHEN has_severity_code = 1
--                      THEN SERVICE_DATE
--                    END) AS severity_dx_distinct_dates,
--     CASE
--       WHEN COUNT(DISTINCT CASE
--                             WHEN has_severity_code = 1
--                             THEN SERVICE_DATE
--                           END) >= 2
--       THEN 'Severe'
--       ELSE 'Attenuated'
--     END AS severity
--   FROM flagged
--   GROUP BY PATIENT_ID
--   ORDER BY severity_dx_distinct_dates DESC, PATIENT_ID
-- )

-- SELECT
--   a.*,
--   b.severity
-- FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master AS a
-- LEFT JOIN patient_with_severity_outcome AS b
--   ON a.PATIENT_ID = b.patient_id;


-- -- =============================================================================
-- -- CELL 7: Comorbidity flagging (UNCHANGED)
-- -- =============================================================================
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master AS
-- WITH base_patients AS (
--     SELECT DISTINCT patient_id
--     FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
-- ),

-- exploded_codes AS (
--     SELECT
--         m.patient_id,
--         code
--     FROM com_edp_prd.com_raw.kom_medical_events m
--     INNER JOIN base_patients bp
--         ON m.patient_id = bp.patient_id
--     LATERAL VIEW explode(split(m.DIAGNOSIS_CODES, '\\|')) s AS code
--     WHERE m.SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--     AND m.DIAGNOSIS_CODES IS NOT NULL
-- ),

-- comorbidity_raw AS (
--     SELECT
--         patient_id,
--         CASE
--             WHEN code IN (
--                 'J45909','J4520','J4530','J4540','J4541','J4531','J45901','J45998',
--                 'J4521','J4542','J45990','J4550','J45902','J4551','J4532','J45991'
--             ) THEN 'Asthma'

--             WHEN code IN ('G4733','G4730','G479','G4731','G4739')
--                 THEN 'Sleep Apnea'

--             WHEN code IN (
--                 'J988','J300','J301','J302','J305','J3081','J3089','J309','J310',
--                 'J311','J312','J320','J321','J322','J323','J324','J328','J329',
--                 'J330','J331','J338','J339','J340','J341','J342','J343','J3481',
--                 'J348200','J348201','J348202','J348210','J348211','J348212',
--                 'J34829','J3489','J349','J3501','J3502','J3503','J351','J352',
--                 'J353','J358','J359','J36','J370','J371','J3800','J3801','J3802',
--                 'J381','J382','J383','J384','J385','J386','J387','J390','J391',
--                 'J392','J393','J398','J399'
--             ) THEN 'Other diseases of upper respiratory tract'

--             WHEN code IN (
--                 'G4700','G4734','G4761','G478','G4710','G4701','G4736','G4720',
--                 'G4709','G4719','G4721','G4729','G4723','G47'
--             ) THEN 'Sleep related disorders'

--             WHEN code IN (
--                 'J00','J0100','J0101','J0110','J0111','J0120','J0121','J0130',
--                 'J0131','J0140','J0141','J0180','J0181','J0190','J0191','J020',
--                 'J028','J029','J0300','J0301','J0380','J0381','J0390','J0391',
--                 'J040','J0410','J0411','J042','J0430','J0431','J050','J0510',
--                 'J0511','J060','J069'
--             ) THEN 'Acute upper respiratory infections'

--             WHEN code IN (
--                 'H6000','H6001','H6002','H6003','H6010','H6011','H6012','H6013',
--                 'H6020','H6021','H6022','H60311','H60312','H60319','H60321',
--                 'H60322','H60329','H60391','H60392','H60399','H6040','H6041',
--                 'H6042','H6043','H60501','H60502','H60509','H60511','H60512',
--                 'H60519','H60551','H60552','H60559','H60591','H60592','H60599',
--                 'H6060','H6061','H6062','H608X1','H608X2','H608X9','H6090',
--                 'H6091','H6092','H61001','H61002','H61003','H61009','H61011',
--                 'H61012','H61013','H61019','H61021','H61022','H61023','H61029',
--                 'H61031','H61032','H61033','H61039','H61321','H61322','H61323',
--                 'H61329','H6240','H6241','H6242','H6500','H6501','H6502','H6504',
--                 'H6505','H6507','H65111','H65112','H65114','H65115','H65117',
--                 'H65119','H65191','H65192','H65194','H65195','H65197','H65199',
--                 'H6520','H6521','H6522','H6530','H6531','H6532','H65411',
--                 'H65412','H65419','H65491','H65492','H65499','H6590','H6591',
--                 'H6592','H66001','H66002','H66003','H66004','H66005','H66006',
--                 'H66007','H66009','H66011','H66012','H66013','H66014','H66015',
--                 'H66016','H66017','H66019','H6611','H6612','H6620','H6621',
--                 'H6622','H663X1','H663X2','H663X9','H6640','H6641','H6642',
--                 'H6690','H6691','H6692','H671','H672','H679','H68001','H68002',
--                 'H68009','H68011','H68012','H68019','H68021','H68022','H68029',
--                 'H70001','H70002','H70009','H70011','H70012','H70019','H70091',
--                 'H70092','H70099','H7010','H7011','H7012','H70201','H70202',
--                 'H70209','H70211','H70212','H70219','H70221','H70222','H70229',
--                 'H70811','H70812','H70819','H70891','H70892','H70899','H7090',
--                 'H7091','H7092','H7100','H7101','H7102','H7110','H7111','H7112',
--                 'H7120','H7121','H7122','H7130','H7131','H7132','H7190','H7191',
--                 'H7192','H73001','H73002','H73009','H73011','H73012','H73019',
--                 'H73091','H73092','H73099','H7310','H7311','H7312','H7320',
--                 'H7321','H7322','H7411','H7412','H7413','H7419','H7440','H7441',
--                 'H7442','H7443','H748X1','H748X2','H748X3','H748X9','H7490',
--                 'H7491','H7492','H7493','H8120','H8121','H8122','H8301','H8302',
--                 'H8309','H9210','H9211','H9212','H9500','H9501','H9502','H9503'
--             ) THEN 'Ear Infections'

--             WHEN code IN (
--                 'H900','H9011','H9012','H902','H903','H9041','H9042','H905',
--                 'H906','H9071','H9072','H908','H90A11','H90A12','H90A21',
--                 'H90A22','H90A31','H90A32','H9101','H9102','H9103','H9109',
--                 'H9120','H9121','H9122','H9123','H918X1','H918X2','H918X3',
--                 'H918X9','H9190','H9191','H9192','H9193','P096'
--             ) THEN 'Hearing loss'

--             WHEN code IN (
--                 'K5900','K5909','K5901','K5904','K5903','K5902','K590','K5939'
--             ) THEN 'Constipation'

--             WHEN code IN ('R197','K591','K580','K529')
--                 THEN 'Diarrhea'

--             WHEN code IN (
--                 'K4000','K4001','K4010','K4011','K4020','K4021','K4030','K4031',
--                 'K4040','K4041','K4090','K4091','K450','K451','K458','K460',
--                 'K461','K469'
--             ) THEN 'Abdominal/inguinal hernia'

--             WHEN code IN ('Q751','Q754','Q755')
--                 THEN 'Dysostosis Complex'

--             WHEN code IN (
--                 'M2560','M25611','M25612','M25619','M25621','M25622','M25629',
--                 'M25631','M25632','M25639','M25641','M25642','M25649','M25651',
--                 'M25652','M25659','M25661','M25662','M25669','M25671','M25672',
--                 'M25673','M25674','M25675','M25676','M2569'
--             ) THEN 'Joint Stiffness'

--             WHEN code IN ('G5600','G5601','G5602','G5603')
--                 THEN 'Carpal tunnel syndrome'

--             WHEN code IN (
--                 'F05','F060','F061','F062','F0630','F0631','F0632','F0633',
--                 'F0634','F064','F0670','F0671','F068','F070','F0781','F0789',
--                 'F079','F09','F22','F23','F24','F28','F29','F3010','F3011',
--                 'F3012','F3013','F302','F303','F304','F308','F309','F320',
--                 'F321','F322','F323','F324','F325','F328','F3289','F329','F32A',
--                 'F330','F331','F332','F333','F3340','F3341','F3342','F338',
--                 'F339','F340','F348','F3481','F3489','F349','F39','F410','F411',
--                 'F413','F418','F419','F430','F4310','F4311','F4312','F4320',
--                 'F4321','F4322','F4323','F4324','F4325','F4329','F438','F4389',
--                 'F439','F441','F442','F450','F451','F4522','F4541','F4542',
--                 'F54','F59','F600','F602','F603','F604','F605','F606','F6089',
--                 'F609','F6381','F6389','F639','F70','F71','F72','F73','F78',
--                 'F78A1','F78A9','F79','F800','F801','F802','F804','F8081',
--                 'F8082','F8089','F809','F810','F812','F8181','F8189','F819',
--                 'F82','F840','F843','F845','F848','F849','F88','F89','F900',
--                 'F901','F902','F908','F909','F910','F911','F912','F913','F918',
--                 'F919','F930','F938','F939','F940','F941','F942','F948','F949',
--                 'F950','F951','F9821','F9829','F983','F984','F985','F988',
--                 'F989','F99'
--             ) THEN 'Behavioral Issues'

--             WHEN code IN (
--                 'G910','G911','G912','G913','G914','G918','G919','Q038','Q039',
--                 'Q050','Q051','Q052','Q053','Q054','Q055','Q056','Q057','Q058',
--                 'Q0700','Q0702','Q0703','F445','G40001','G40009','G40011',
--                 'G40019','G40101','G40109','G40111','G40119','G40201','G40209',
--                 'G40211','G40219','G40501','G40509','G4089','R561'
--             ) THEN 'CNS Issues'

--             WHEN code IN ('R6250','R620','R6252','R6251','R6259','R627','R62')
--                 THEN 'Lack of Physiological Development'

--             WHEN code IN (
--                 'I10','I110','I129','I130','I119','I159','I160','I158','I120',
--                 'I161','I1310','I150'
--             ) THEN 'Hypertension'

--             WHEN code IN (
--                 'I050','I051','I052','I058','I059','I060','I061','I062','I068',
--                 'I069','I070','I071','I072','I078','I079','I080','I081','I082',
--                 'I083','I088','I089','I340','I341','I342','I348','I3481',
--                 'I3489','I349','I350','I351','I352','I358','I359','I360',
--                 'I361','I362','I368','I369','I370','I371','I372','I378','I379'
--             ) THEN 'Valvular Heart Disease'

--         END AS comorbidity_name
--     FROM exploded_codes
--     WHERE code IS NOT NULL AND code <> ''
-- ),

-- comorbidity_with_category AS (
--     SELECT
--         patient_id,
--         comorbidity_name,
--         CASE
--             WHEN comorbidity_name IN (
--                 'Asthma','Sleep Apnea','Other diseases of upper respiratory tract',
--                 'Sleep related disorders','Acute upper respiratory infections'
--             ) THEN 'Respiratory issues'

--             WHEN comorbidity_name IN ('Ear Infections','Hearing loss')
--                 THEN 'Ear-related disorders'

--             WHEN comorbidity_name IN ('Constipation','Diarrhea','Abdominal/inguinal hernia')
--                 THEN 'Gastrointestinal disorders'

--             WHEN comorbidity_name IN ('Dysostosis Complex','Joint Stiffness','Carpal tunnel syndrome')
--                 THEN 'Mobility issues'

--             WHEN comorbidity_name IN ('Behavioral Issues','CNS Issues','Lack of Physiological Development')
--                 THEN 'Neurological disorders'

--             WHEN comorbidity_name IN ('Hypertension','Valvular Heart Disease')
--                 THEN 'Other chronic conditions'
--         END AS comorbidity_category
--     FROM comorbidity_raw
--     WHERE comorbidity_name IS NOT NULL
-- ),

-- patient_with_comorbidity_outcome as (
--     SELECT
--     patient_id,

--     array_join(
--         array_sort(collect_set(comorbidity_category)),
--         ', '
--     ) AS comorbidity_categories,

--     size(collect_set(comorbidity_category)) AS count_of_comorbidity_categories,

--     array_join(
--         array_sort(collect_set(comorbidity_name)),
--         ', '
--     ) AS distinct_comorbidities,

--     size(collect_set(comorbidity_name)) AS count_of_distinct_comorbidities

-- FROM comorbidity_with_category
-- GROUP BY patient_id
-- ORDER BY patient_id
-- )
-- select
--     a.*,
--     b.comorbidity_categories,
--     b.count_of_comorbidity_categories,
--     b.distinct_comorbidities,
--     b.count_of_distinct_comorbidities
-- from com_edp_prd.cmpa_insights_internal_schema.patient360_master as a
-- left join patient_with_comorbidity_outcome as b
--     on a.patient_id=b.patient_id;


-- -- =============================================================================
-- -- CELL 8: primary_hcp secondary specialty enrichment (UNCHANGED)
-- -- =============================================================================
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master as
--   select distinct a.*, b.SECONDARY_SPECIALTY as primary_hcp_secondary_specialty
-- from com_edp_prd.cmpa_insights_internal_schema.patient360_master as a
-- left join com_edp_prd.com_raw.kom_providers as b on a.PRIMARY_HCP_NPI = b.npi and b.provider_type = 'INDIVIDUAL';


-- -- =============================================================================
-- -- CELL 9: age_bucket / payer / insurance enrichment (UNCHANGED)
-- -- =============================================================================
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master AS

-- WITH base_table AS (
--   SELECT DISTINCT *
--   FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
-- ),
-- age_buckets AS (
--   SELECT DISTINCT
--     patient_id,
--     CASE
--       WHEN patient_age < 5 THEN '<5 years'
--       WHEN patient_age BETWEEN 5 AND 10 THEN '5 - 10 years'
--       WHEN patient_age BETWEEN 11 AND 16 THEN '11 - 16 years'
--       ELSE '>17 years'
--     END AS age_bucket
--   FROM base_table
-- ),

-- patient_geography AS (
--   SELECT DISTINCT *
--   FROM (
--     SELECT *,
--       ROW_NUMBER() OVER (
--         PARTITION BY patient_id
--         ORDER BY
--           CASE WHEN valid_to_date > CURRENT_DATE() THEN 1 ELSE 2 END,
--           valid_to_date DESC
--       ) AS rn
--     FROM com_edp_prd.com_raw.kom_patient_geography
--   )
--   WHERE rn = 1
-- ),
-- patient_zip AS (
--   SELECT DISTINCT
--     a.patient_id,
--     b.patient_zip AS zip3
--   FROM base_table a
--   LEFT JOIN patient_geography b
--     ON a.patient_id = b.patient_id
-- ),

-- tx_claims_payer_analysis AS (
--   SELECT DISTINCT
--     a.patient_id,
--     a.fill_date,
--     a.claim_id,
--     a.kh_plan_id,
--     b.payer_name,
--     b.insurance_group
--   FROM (
--     SELECT
--       patient_id,
--       service_date AS fill_date,
--       kh_plan_id,
--       medical_event_id AS claim_id
--     FROM com_edp_prd.com_raw.kom_medical_events

--     UNION ALL

--     SELECT
--       patient_id,
--       fill_date,
--       COALESCE(primary_kh_plan_id, secondary_kh_plan_id) AS kh_plan_id,
--       pharmacy_event_id AS claim_id
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE transaction_result = 'PAID'
--   ) a
--   LEFT JOIN com_edp_prd.com_raw.kom_plans b
--     ON a.kh_plan_id = b.kh_plan_id
-- ),

-- payer_claims_count AS (
--   SELECT *,
--     ROW_NUMBER() OVER (
--       PARTITION BY patient_id
--       ORDER BY num_claims DESC
--     ) AS rn
--   FROM (
--     SELECT
--       patient_id,
--       payer_name,
--       COUNT(DISTINCT claim_id) AS num_claims
--     FROM tx_claims_payer_analysis
--     WHERE payer_name IS NOT NULL
--       AND fill_date BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--     GROUP BY patient_id, payer_name
--   )
-- ),

-- patient_primary_secondary_payer AS (
--   SELECT
--     patient_id,
--     MAX(CASE WHEN rn = 1 THEN payer_name END) AS primary_payer,
--     MAX(CASE WHEN rn = 2 THEN payer_name END) AS secondary_payer
--   FROM payer_claims_count
--   GROUP BY patient_id
-- ),

-- patient_latest_insurance_type AS (
--   SELECT
--     patient_id,
--     insurance_group AS latest_insurance_type
--   FROM (
--     SELECT
--       patient_id,
--       insurance_group,
--       fill_date,
--       ROW_NUMBER() OVER (
--         PARTITION BY patient_id
--         ORDER BY fill_date DESC
--       ) AS rn
--     FROM tx_claims_payer_analysis
--     WHERE insurance_group IS NOT NULL
--   )
--   WHERE rn = 1
-- )

-- SELECT
--   a.*,
--   b.age_bucket,
--   c.zip3,
--   f.primary_payer,
--   f.secondary_payer,
--   g.latest_insurance_type

-- FROM base_table a
-- LEFT JOIN age_buckets b
--   ON a.patient_id = b.patient_id
-- LEFT JOIN patient_zip c
--   ON a.patient_id = c.patient_id
-- LEFT JOIN patient_primary_secondary_payer f
--   ON a.patient_id = f.patient_id
-- LEFT JOIN patient_latest_insurance_type g
--   ON a.patient_id = g.patient_id;


-- -- =============================================================================
-- -- CELL 10: specialty re-enrichment for most_recently_treated and primary HCPs (UNCHANGED)
-- -- =============================================================================
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master as
-- select a.*except(a.most_recently_treated_hcp_specialty_2yr, a.PRIMARY_HCP_SPECIALTY, a.primary_hcp_secondary_specialty),
-- b.PRIMARY_SPECIALTY as most_recently_treated_hcp_specialty_2yr, b.SECONDARY_SPECIALTY as most_recently_treated_hcp_secondary_specialty_2yr, c.PRIMARY_SPECIALTY as primary_hcp_specialty, c.SECONDARY_SPECIALTY as primary_hcp_secondary_specialty
-- from cmpa_insights_internal_schema.patient360_master as a
-- left join com_raw.kom_providers as b on a.most_recently_treated_hcp_2yr = b.npi and b.PROVIDER_TYPE = 'INDIVIDUAL'
-- left join com_raw.kom_providers as c on a.PRIMARY_HCP_NPI = c.npi and c.PROVIDER_TYPE = 'INDIVIDUAL';


-- -- =============================================================================
-- -- CELL 11: newborn screening flag (UNCHANGED)
-- -- =============================================================================
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master as
-- with base_table as (
--   select distinct * from com_edp_prd.cmpa_insights_internal_schema.patient360_master
-- ),
-- newborn_screening_flag as (
--   select *,
--   case when patient_state in ('IL', 'MO', 'WV', 'PA', 'KY', 'MD', 'CA', 'DE', 'FL', 'KS', 'AZ', 'MA', 'RI', 'AR', 'IA', 'NC', 'TX', 'CT') then 1 else 0 end as newborn_screening_flag
--   from base_table
-- )
-- select * from newborn_screening_flag;


-- -- =============================================================================
-- -- CELL 12: most recent infusion location (UNCHANGED)
-- -- =============================================================================
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master as
-- WITH base_table as (
--   select distinct *
--   from com_edp_prd.cmpa_insights_internal_schema.patient360_master
-- ),

-- tx_patients as (
--   select distinct
--       patient_id,
--       coalesce(rendering_npi, referring_npi) as npi,
--       service_date as fill_date,
--       place_of_service
--   from com_edp_prd.com_raw.kom_medical_events
--   where ndc11 in ('54092070001','540920700')
--     and service_date between '2023-08-01' and (SELECT end_date FROM runtime_parameters)

--   union

--   select distinct
--       patient_id,
--       prescriber_npi as npi,
--       fill_date,
--       '' as place_of_service
--   from com_edp_prd.com_raw.kom_pharmacy_events
--   where ndc11 in ('54092070001','540920700')
--     and transaction_result = 'PAID'
--     and fill_date between '2023-08-01' and (SELECT end_date FROM runtime_parameters)

--   union

--   select distinct
--       patient_id,
--       rendering_npi as npi,
--       service_date as fill_date,
--       place_of_service
--   from com_edp_prd.com_raw.kom_medical_events
--   where procedure_code in ('99601','99602','96365','96366','J1743','S9357','S9379',
--                            '38206','38230','38232','38240','38241','38242','38243','38250')
--     and service_date between '2023-08-01' and (SELECT end_date FROM runtime_parameters)
-- ),

-- latest_pos_patients as (
--   select
--       patient_id,
--       nullif(trim(place_of_service), '') as most_recent_infusion_location
--   from (
--     select
--         patient_id,
--         place_of_service,
--         fill_date,
--         row_number() over (
--           partition by patient_id
--           order by fill_date desc
--         ) as rn
--     from tx_patients
--   ) t
--   where rn = 1
-- )

-- select
--     a.*,
--     b.most_recent_infusion_location,
--     c.description as most_recent_infusion_description
-- from base_table a
-- left join latest_pos_patients b
--   on a.patient_id = b.patient_id
-- left join com_edp_prd.cmpa_insights_internal_schema.pos_description c
--   on try_cast(nullif(trim(b.most_recent_infusion_location), '') as int) = c.code;


-- -- =============================================================================
-- -- CELL 13: NEW — Remove female patients whose latest MPSII tx type is "OTHER_TX"
-- -- latest_mpsii_tx_type is now self-contained: it is either 'ELAPRASE' or 'OTHER_TX'
-- -- (NULL only if a patient somehow has no recent tx claim, which shouldn't happen
-- --  given eligibility requires recent treatment).
-- -- =============================================================================
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master AS
-- SELECT *
-- FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
-- WHERE NOT (
--     UPPER(PATIENT_GENDER) = 'F'
--     AND UPPER(latest_mpsii_tx_type) = 'OTHER_TX'
-- );


-- -- =============================================================================
-- -- =============================================================================
-- -- TIVI / AVLAYAH ADD-ON CELLS
-- -- These cells build a parallel Tivi cohort using NDC '8497600101' and
-- -- procedure codes 'J3490','J3590','J9999' (gated to >= '2026-03-01'), produce a
-- -- tivi_patient360_master view, then LEFT JOIN it onto the existing
-- -- patient360_master to add tivi_* prefixed columns.
-- -- =============================================================================
-- -- =============================================================================


-- -- =============================================================================
-- -- CELL 14 (TIVI): patient_hcp_visit_summary (Tivi base) + patient360_base view
-- -- =============================================================================
-- CREATE OR REPLACE TEMP VIEW patient_hcp_visit_summary AS
-- WITH MPSII_Diagnoses_Specified AS (
--     SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE DIAGNOSIS_CODES LIKE '%E761%'
--       AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--     UNION
--     SELECT DISTINCT PATIENT_ID, FILL_DATE
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE DIAGNOSIS_CODE = 'E761'
--       AND TRANSACTION_RESULT = 'PAID'
--       AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
-- ),

-- Patients_2Dx_Specified AS (
--     SELECT PATIENT_ID
--     FROM MPSII_Diagnoses_Specified
--     GROUP BY PATIENT_ID
--     HAVING COUNT(DISTINCT FILL_DATE) >= 2
-- ),

-- MPSII_Diagnoses_Unspecified AS (
--     SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE DIAGNOSIS_CODES LIKE '%E763%'
--       AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--     UNION
--     SELECT DISTINCT PATIENT_ID, FILL_DATE
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE DIAGNOSIS_CODE = 'E763'
--       AND TRANSACTION_RESULT = 'PAID'
--       AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
-- ),

-- Patients_2Dx_Unspecified AS (
--     SELECT PATIENT_ID
--     FROM MPSII_Diagnoses_Unspecified
--     GROUP BY PATIENT_ID
--     HAVING COUNT(DISTINCT FILL_DATE) >= 2
-- ),

-- MPSII_Treatment_All AS (
--     SELECT DISTINCT PATIENT_ID FROM (
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE NDC11 IN ('8497600101')
--           AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_pharmacy_events
--         WHERE NDC11 IN ('8497600101')
--           AND TRANSACTION_RESULT = 'PAID'
--           AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE PROCEDURE_CODE IN ('J3490','J3590','J9999')
--           AND SERVICE_DATE >= '2026-03-01'
--     ) t
-- ),

-- MPSII_Treatment_Tivi_Only AS (
--     SELECT DISTINCT PATIENT_ID FROM (
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE NDC11 IN ('8497600101')
--           AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_pharmacy_events
--         WHERE NDC11 IN ('8497600101')
--           AND TRANSACTION_RESULT = 'PAID'
--           AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE PROCEDURE_CODE IN ('J3490','J3590','J9999')
--           AND SERVICE_DATE >= '2026-03-01'
--     ) t
-- ),

-- Patients_2Dx_Specified_With_Treatment AS (
--     SELECT DISTINCT p.PATIENT_ID
--     FROM Patients_2Dx_Specified p
--     INNER JOIN MPSII_Treatment_All t USING (PATIENT_ID)
-- ),

-- Patients_Incremental_Unspecified AS (
--     SELECT DISTINCT p.PATIENT_ID
--     FROM Patients_2Dx_Unspecified p
--     INNER JOIN MPSII_Treatment_Tivi_Only t USING (PATIENT_ID)
--     WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
-- ),

-- eligible_patients AS (
--     SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment
--     UNION
--     SELECT PATIENT_ID FROM Patients_Incremental_Unspecified
-- ),

-- cohort_3_learnings AS (
--   SELECT DISTINCT npi
--   FROM com_raw.kom_providers
--   WHERE provider_type = 'INDIVIDUAL'
--     AND (
--       PRIMARY_SPECIALTY NOT IN (
--         'Anesthesiologist Assistant','Anesthesiology','Dentist','Dietitian, Registered',
--         'Emergency Medical Technician, Basic','Emergency Medicine','General Acute Care Hospital',
--         'Nurse Anesthetist, Certified Registered','Obstetrics & Gynecology','Pathology',
--         'Radiology','Urology'
--       )
--       OR SECONDARY_SPECIALTY IN (
--         'Child & Adolescent Psychiatry','Psychiatry','Adolescent Medicine','Developmental - Behavioral Pediatrics',
--         'Neonatal-Perinatal Medicine','Nutrition, Pediatric','Oncology, Pediatrics','Pediatric Cardiology',
--         'Pediatric Critical Care Medicine','Pediatric Dermatology','Pediatric Emergency Medicine',
--         'Pediatric Endocrinology','Pediatric Gastroenterology','Pediatric Hematology-Oncology',
--         'Pediatric Infectious Diseases','Pediatric Nephrology','Pediatric Ophthalmology and Strabismus Specialist',
--         'Pediatric Orthopaedic Surgery','Pediatric Otolaryngology','Pediatric Pulmonology','Pediatric Radiology',
--         'Pediatric Rehabilitation Medicine','Pediatric Rheumatology','Pediatric Surgery','Pediatrics',
--         'Clinical Biochemical Genetics','Clinical Genetics (M.D.)','Clinical Molecular Genetics',
--         'Ph.D. Medical Genetics','Neurodevelopmental Disabilities','Neurology',
--         'Neurology with Special Qualifications in Child Neurology','Neuroradiology'
--       )
--     )
-- ),

-- all_dx_claims_5yr AS (
--     SELECT DISTINCT *
--     FROM (
--       SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
--         AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--       UNION
--       SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
--       FROM com_edp_prd.com_raw.kom_pharmacy_events
--       WHERE DIAGNOSIS_CODE IN ('E761','E763')
--         AND TRANSACTION_RESULT = 'PAID'
--         AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     )
--     WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
-- ),
-- all_tx_claims_5yr AS (
--     SELECT DISTINCT *
--     FROM (
--       SELECT DISTINCT
--           PATIENT_ID,
--           COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
--           SERVICE_DATE AS FILL_DATE,
--           NDC11 AS TX_CODE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE NDC11 IN ('8497600101')
--         AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--       UNION

--       SELECT DISTINCT
--           PATIENT_ID,
--           PRESCRIBER_NPI AS NPI,
--           FILL_DATE,
--           NDC11 AS TX_CODE
--       FROM com_edp_prd.com_raw.kom_pharmacy_events
--       WHERE NDC11 IN ('8497600101')
--         AND TRANSACTION_RESULT = 'PAID'
--         AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--       UNION

--       SELECT DISTINCT
--           PATIENT_ID,
--           RENDERING_NPI AS NPI,
--           SERVICE_DATE AS FILL_DATE,
--           PROCEDURE_CODE AS TX_CODE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE PROCEDURE_CODE IN ('J3490','J3590','J9999')
--         AND  SERVICE_DATE >= '2026-03-01'
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     )
--     WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
-- ),

-- all_tx_claims_alltime AS (
--     SELECT DISTINCT *
--     FROM (
--       SELECT DISTINCT
--           PATIENT_ID,
--           COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
--           SERVICE_DATE AS FILL_DATE,
--           NDC11 AS TX_CODE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE NDC11 IN ('8497600101')
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--       UNION

--       SELECT DISTINCT
--           PATIENT_ID,
--           PRESCRIBER_NPI AS NPI,
--           FILL_DATE,
--           NDC11 AS TX_CODE
--       FROM com_edp_prd.com_raw.kom_pharmacy_events
--       WHERE NDC11 IN ('8497600101')
--         AND TRANSACTION_RESULT = 'PAID'
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--       UNION

--       SELECT DISTINCT
--           PATIENT_ID,
--           RENDERING_NPI AS NPI,
--           SERVICE_DATE AS FILL_DATE,
--           PROCEDURE_CODE AS TX_CODE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE PROCEDURE_CODE IN ('J3490','J3590','J9999')
--         AND  SERVICE_DATE >= '2026-03-01'
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     )
--     WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
-- ),

-- all_claims_5yr AS (
--     SELECT DISTINCT
--         PATIENT_ID,
--         NPI,
--         FILL_DATE,
--         CAST(NULL AS STRING) AS TX_CODE
--     FROM all_dx_claims_5yr
--     UNION
--     SELECT DISTINCT
--         PATIENT_ID,
--         NPI,
--         FILL_DATE,
--         TX_CODE
--     FROM all_tx_claims_5yr
-- ),

-- all_claims_3yr AS (
--     SELECT DISTINCT *
--     FROM (
--       SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
--         AND SERVICE_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--       UNION
--       SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
--       FROM com_edp_prd.com_raw.kom_pharmacy_events
--       WHERE DIAGNOSIS_CODE IN ('E761','E763')
--         AND TRANSACTION_RESULT = 'PAID'
--         AND FILL_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--       UNION
--       SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE NDC11 IN ('8497600101')
--         AND SERVICE_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--       UNION
--       SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
--       FROM com_edp_prd.com_raw.kom_pharmacy_events
--       WHERE NDC11 IN ('8497600101')
--         AND TRANSACTION_RESULT = 'PAID'
--         AND FILL_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--       UNION
--       SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI, SERVICE_DATE AS FILL_DATE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE PROCEDURE_CODE IN ('J3490','J3590','J9999')
--         AND SERVICE_DATE >= '2026-03-01'
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     )
--     WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
-- ),

-- first_dx_hcp_ranked AS (
--     SELECT PATIENT_ID, NPI, MIN(FILL_DATE) AS first_dx_date,
--            ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY MIN(FILL_DATE) ASC, NPI) AS rn
--     FROM all_dx_claims_5yr
--     WHERE NPI IS NOT NULL
--     GROUP BY PATIENT_ID, NPI
-- ),
-- first_dx_hcp AS (
--     SELECT PATIENT_ID, NPI AS first_dx_hcp, first_dx_date
--     FROM first_dx_hcp_ranked
--     WHERE rn = 1
-- ),

-- provider_dim AS (
--     SELECT
--         npi,
--         CONCAT(FIRST_NAME, ' ', LAST_NAME) AS provider_name,
--         primary_specialty
--     FROM com_raw.kom_providers
--     WHERE provider_type = 'INDIVIDUAL'
-- ),

-- first_dx_hcp_5yr_stats AS (
--     SELECT fdh.PATIENT_ID, fdh.first_dx_hcp,
--            COUNT(DISTINCT ac.FILL_DATE) AS first_dx_all_visit_count_5yr,
--            MAX(ac.FILL_DATE) AS first_dx_last_visit_5yr
--     FROM first_dx_hcp fdh
--     LEFT JOIN all_claims_5yr ac
--       ON fdh.PATIENT_ID = ac.PATIENT_ID AND fdh.first_dx_hcp = ac.NPI
--     GROUP BY fdh.PATIENT_ID, fdh.first_dx_hcp
-- ),

-- first_tx_hcp_ranked AS (
--     SELECT PATIENT_ID, NPI, MIN(FILL_DATE) AS first_tx_date,
--            ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY MIN(FILL_DATE) ASC, NPI) AS rn
--     FROM all_tx_claims_5yr
--     WHERE NPI IS NOT NULL
--     GROUP BY PATIENT_ID, NPI
-- ),
-- first_tx_hcp AS (
--     SELECT PATIENT_ID, NPI AS first_tx_hcp, first_tx_date
--     FROM first_tx_hcp_ranked
--     WHERE rn = 1
-- ),

-- first_tx_hcp_5yr_stats AS (
--     SELECT fth.PATIENT_ID, fth.first_tx_hcp,
--            COUNT(DISTINCT ac.FILL_DATE) AS first_tx_all_visit_count_5yr,
--            MAX(ac.FILL_DATE) AS first_tx_last_visit_5yr
--     FROM first_tx_hcp fth
--     LEFT JOIN all_claims_5yr ac
--       ON fth.PATIENT_ID = ac.PATIENT_ID AND fth.first_tx_hcp = ac.NPI
--     GROUP BY fth.PATIENT_ID, fth.first_tx_hcp
-- ),

-- first_tx_hcp_5yr_tx_only AS (
--     SELECT fth.PATIENT_ID, fth.first_tx_hcp,
--            COUNT(DISTINCT tx.FILL_DATE) AS first_tx_treatment_visit_count_5yr
--     FROM first_tx_hcp fth
--     LEFT JOIN all_tx_claims_5yr tx
--       ON fth.PATIENT_ID = tx.PATIENT_ID AND fth.first_tx_hcp = tx.NPI
--     GROUP BY fth.PATIENT_ID, fth.first_tx_hcp
-- ),

-- most_seen_3yr_ranking AS (
--     SELECT PATIENT_ID,
--            NPI,
--            COUNT(DISTINCT FILL_DATE) AS visit_count_3yr,
--            MAX(FILL_DATE) AS last_visit_3yr,
--            ROW_NUMBER() OVER (
--              PARTITION BY PATIENT_ID
--              ORDER BY COUNT(DISTINCT FILL_DATE) DESC,
--                       MAX(FILL_DATE) DESC,
--                       NPI ASC
--            ) AS rank
--     FROM all_claims_3yr
--     WHERE NPI IS NOT NULL
--     GROUP BY PATIENT_ID, NPI
-- ),

-- most_seen_combined_stats AS (
--     SELECT
--         ms3.PATIENT_ID,
--         ms3.NPI,
--         ms3.rank,
--         ms3.visit_count_3yr,
--         ms3.last_visit_3yr,
--         COUNT(DISTINCT ac5.FILL_DATE) AS visit_count_5yr,
--         MAX(ac5.FILL_DATE)          AS last_visit_5yr
--     FROM most_seen_3yr_ranking ms3
--     LEFT JOIN all_claims_5yr ac5
--       ON ms3.PATIENT_ID = ac5.PATIENT_ID AND ms3.NPI = ac5.NPI
--     WHERE ms3.rank <= 5
--     GROUP BY ms3.PATIENT_ID, ms3.NPI, ms3.rank, ms3.visit_count_3yr, ms3.last_visit_3yr
-- ),

-- historical_first_dx AS (
--     SELECT PATIENT_ID, MIN(FILL_DATE) AS incidence_date
--     FROM (
--         SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE DIAGNOSIS_CODES LIKE '%E761%'
--           AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID, FILL_DATE
--         FROM com_edp_prd.com_raw.kom_pharmacy_events
--         WHERE DIAGNOSIS_CODE = 'E761'
--           AND TRANSACTION_RESULT = 'PAID'
--           AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE DIAGNOSIS_CODES LIKE '%E763%'
--           AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID, FILL_DATE
--         FROM com_edp_prd.com_raw.kom_pharmacy_events
--         WHERE DIAGNOSIS_CODE = 'E763'
--           AND TRANSACTION_RESULT = 'PAID'
--           AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     ) all_dx
--     GROUP BY PATIENT_ID
-- ),

-- historical_first_tx AS (
--     SELECT PATIENT_ID, MIN(FILL_DATE) AS first_incidence_treatment_date
--     FROM (
--         SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE NDC11 IN ('8497600101')
--           AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID, FILL_DATE
--         FROM com_edp_prd.com_raw.kom_pharmacy_events
--         WHERE NDC11 IN ('8497600101')
--           AND TRANSACTION_RESULT = 'PAID'
--           AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE PROCEDURE_CODE IN ('J3490','J3590','J9999')
--           and SERVICE_DATE >= '2026-03-01'
--           AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     ) all_tx
--     GROUP BY PATIENT_ID
-- ),

-- all_claims_5yr_specialty_removed AS (
--     SELECT DISTINCT *
--     FROM (
--       SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
--         AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--       UNION
--       SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
--       FROM com_edp_prd.com_raw.kom_pharmacy_events
--       WHERE DIAGNOSIS_CODE IN ('E761','E763')
--         AND TRANSACTION_RESULT = 'PAID'
--         AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--       UNION
--       SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE NDC11 IN ('8497600101')
--         AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--       UNION
--       SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
--       FROM com_edp_prd.com_raw.kom_pharmacy_events
--       WHERE NDC11 IN ('8497600101')
--         AND TRANSACTION_RESULT = 'PAID'
--         AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--       UNION
--       SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI, SERVICE_DATE AS FILL_DATE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE PROCEDURE_CODE IN ('J3490','J3590','J9999')
--         AND SERVICE_DATE >= '2026-03-01'
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     )
-- ),

-- latest_claim_hcp_ranked as (
--   select patient_id, npi, fill_date, row_number() over(partition by patient_id order by fill_date desc, npi asc) as rn
--   from all_claims_5yr
-- ),

-- latest_claim_hcp as (
--   select patient_id, npi as latest_claim_hcp_npi, fill_date as latest_claim_date from latest_claim_hcp_ranked where rn = 1
-- ),

-- latest_claim_hcp_visit_count_5yr as (
--   select a.patient_id, a.latest_claim_hcp_npi, count(distinct fill_date) as latest_claim_hcp_visit_count_5yr
--   from latest_claim_hcp as a
--   left join all_claims_5yr as b on a.patient_id = b.patient_id and a.latest_claim_hcp_npi = b.npi
--   group by 1,2
-- ),

-- latest_claim_hcp_final as (
--   select a.patient_id, a.latest_claim_hcp_npi, a.latest_claim_date, b.latest_claim_hcp_visit_count_5yr
--   from latest_claim_hcp as a
--   left join latest_claim_hcp_visit_count_5yr as b on a.patient_id = b.patient_id
-- ),

-- all_tx_claims_5yr_specialty_removed AS (
--     SELECT DISTINCT *
--     FROM (
--       SELECT DISTINCT
--           PATIENT_ID,
--           COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
--           SERVICE_DATE AS FILL_DATE,
--           NDC11 AS TX_CODE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE NDC11 IN ('8497600101')
--         AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--       UNION

--       SELECT DISTINCT
--           PATIENT_ID,
--           PRESCRIBER_NPI AS NPI,
--           FILL_DATE,
--           NDC11 AS TX_CODE
--       FROM com_edp_prd.com_raw.kom_pharmacy_events
--       WHERE NDC11 IN ('8497600101')
--         AND TRANSACTION_RESULT = 'PAID'
--         AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--       UNION

--       SELECT DISTINCT
--           PATIENT_ID,
--           RENDERING_NPI AS NPI,
--           SERVICE_DATE AS FILL_DATE,
--           PROCEDURE_CODE AS TX_CODE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE PROCEDURE_CODE IN ('J3490','J3590','J9999')
--         AND SERVICE_DATE >= '2026-03-01'
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     )
-- ),

-- most_recent_tx_hcp_ranked as (
--   select patient_id, npi, fill_date, row_number() over(partition by patient_id order by fill_date desc, npi asc) as rn
--   from all_tx_claims_5yr
-- ),

-- most_recent_tx_hcp as (
--   select patient_id, npi as latest_treatment_hcp_npi, fill_date as latest_treatment_date
--   from most_recent_tx_hcp_ranked where rn = 1
-- ),

-- latest_treatment_hcp_visit_count as (
--   select a.patient_id, a.latest_treatment_hcp_npi, count(distinct fill_date) as latest_treatment_hcp_visit_count_5yr
--   from most_recent_tx_hcp as a
--   left join all_tx_claims_5yr as b on a.patient_id = b.patient_id and a.latest_treatment_hcp_npi = b.npi
--   group by 1, 2
-- ),

-- most_recent_tx_hcp_final as (
--   select a.patient_id, a.latest_treatment_hcp_npi, a.latest_treatment_date, b.latest_treatment_hcp_visit_count_5yr
--   from most_recent_tx_hcp as a
--   left join latest_treatment_hcp_visit_count as b on a.patient_id = b.patient_id
-- ),

-- latest_mpsii_treatment_type AS (
--   SELECT
--     patient_id,
--     CASE
--       WHEN tx_code IN ('8497600101') OR (tx_code IN ('J3490','J3590','J9999') AND fill_date >= '2026-03-01')
--         THEN 'AVLAYAH'
--     END AS latest_mpsii_tx_type
--   FROM (
--     SELECT
--       patient_id,
--       fill_date,
--       tx_code,
--       ROW_NUMBER() OVER (
--         PARTITION BY patient_id
--         ORDER BY fill_date DESC, tx_code ASC
--       ) AS rn
--     FROM all_tx_claims_5yr
--     WHERE tx_code IS NOT NULL
--   )
--   WHERE rn = 1
-- ),

-- first_tx_after_diagnosis AS (
--   SELECT
--     tx.patient_id,
--     MIN(tx.fill_date) AS first_tx_after_diagnosis
--   FROM all_tx_claims_alltime tx
--   INNER JOIN historical_first_dx dx
--     ON tx.patient_id = dx.patient_id
--   WHERE tx.fill_date >= dx.incidence_date
--   GROUP BY tx.patient_id
-- ),

-- Tivi_fills AS (
--   SELECT
--     patient_id,
--     COUNT(DISTINCT fill_date) AS Tivi_fills
--   FROM all_tx_claims_5yr
--   WHERE fill_date BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--     AND tx_code IN ('8497600101')
--   GROUP BY patient_id
-- ),

-- patient_demographics AS (
--     SELECT *
--     FROM (
--         SELECT DISTINCT PATIENT_ID, PATIENT_YOB, PATIENT_GENDER,
--                ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY PATIENT_YOB ASC) AS rn
--         FROM com_edp_prd.com_raw.kom_patient_demographics
--     )
--     WHERE rn = 1
-- ),
-- patient_geography AS (
--     SELECT patient_id, patient_state
--     FROM (
--         SELECT
--             PATIENT_ID,
--             patient_state,
--             ROW_NUMBER() OVER (
--                 PARTITION BY PATIENT_ID
--                 ORDER BY
--                     CASE WHEN VALID_TO_DATE > CURRENT_DATE() THEN 1 ELSE 2 END,
--                     VALID_TO_DATE DESC
--             ) AS rn
--         FROM COM_EDP_PRD.COM_RAW.KOM_PATIENT_GEOGRAPHY
--     )
--     WHERE rn = 1
-- ),

-- most_seen_pivot AS (
--     SELECT
--         PATIENT_ID,

--         MAX(CASE WHEN rank = 1 THEN NPI END)              AS most_seen_hcp1_3yr_ranked,
--         MAX(CASE WHEN rank = 1 THEN visit_count_5yr END)  AS most_seen_hcp1_visit_count_5yr,
--         MAX(CASE WHEN rank = 1 THEN last_visit_5yr END)   AS most_seen_hcp1_last_visit_5yr,

--         MAX(CASE WHEN rank = 2 THEN NPI END)              AS most_seen_hcp2_3yr_ranked,
--         MAX(CASE WHEN rank = 2 THEN visit_count_5yr END)  AS most_seen_hcp2_visit_count_5yr,
--         MAX(CASE WHEN rank = 2 THEN last_visit_5yr END)   AS most_seen_hcp2_last_visit_5yr,

--         MAX(CASE WHEN rank = 3 THEN NPI END)              AS most_seen_hcp3_3yr_ranked,
--         MAX(CASE WHEN rank = 3 THEN visit_count_5yr END)  AS most_seen_hcp3_visit_count_5yr,
--         MAX(CASE WHEN rank = 3 THEN last_visit_5yr END)   AS most_seen_hcp3_last_visit_5yr,

--         MAX(CASE WHEN rank = 4 THEN NPI END)              AS most_seen_hcp4_3yr_ranked,
--         MAX(CASE WHEN rank = 4 THEN visit_count_5yr END)  AS most_seen_hcp4_visit_count_5yr,
--         MAX(CASE WHEN rank = 4 THEN last_visit_5yr END)   AS most_seen_hcp4_last_visit_5yr,

--         MAX(CASE WHEN rank = 5 THEN NPI END)              AS most_seen_hcp5_3yr_ranked,
--         MAX(CASE WHEN rank = 5 THEN visit_count_5yr END)  AS most_seen_hcp5_visit_count_5yr,
--         MAX(CASE WHEN rank = 5 THEN last_visit_5yr END)   AS most_seen_hcp5_last_visit_5yr

--     FROM most_seen_combined_stats
--     GROUP BY PATIENT_ID
-- )
-- SELECT
--     ep.PATIENT_ID,
--     pd.PATIENT_YOB,
--     YEAR(CURRENT_DATE) - YEAR(pd.PATIENT_YOB) AS PATIENT_AGE,
--     pd.PATIENT_GENDER,
--     pg.patient_state,

--     hfdx.incidence_date,
--     hftx.first_incidence_treatment_date,

--     lch.latest_claim_date AS latest_claim_date,

--     lch.latest_claim_hcp_npi,
--     pdlch.provider_name     AS latest_claim_hcp_name,
--     pdlch.primary_specialty AS latest_claim_hcp_specialty,
--     COALESCE(lch.latest_claim_hcp_visit_count_5yr, 0) AS latest_claim_hcp_visit_count,

--     ref1.hco_name AS latest_claim_hcp_hco_name,

--     mrt.latest_treatment_date AS latest_treatment_date,

--     lmt.latest_mpsii_tx_type,

--     fta.first_tx_after_diagnosis,

--     ROUND(MONTHS_BETWEEN(fta.first_tx_after_diagnosis, hfdx.incidence_date), 0)
--       AS time_dx_to_first_tx_in_months,

--     ROUND(MONTHS_BETWEEN(mrt.latest_treatment_date, fta.first_tx_after_diagnosis), 0) AS treatment_period_months,

--     COALESCE(ef.Tivi_fills, 0) AS Tivi_fills,

--     mrt.latest_treatment_hcp_npi,
--     pdtch.provider_name     AS latest_treatment_hcp_name,
--     pdtch.primary_specialty AS latest_treatment_hcp_specialty,
--     COALESCE(mrt.latest_treatment_hcp_visit_count_5yr, 0) AS latest_treatment_hcp_visit_count,

--     ref2.hco_name AS latest_treatment_hcp_hco_name,

--     fdh.first_dx_hcp AS first_dx_hcp_5yr,
--     COALESCE(fdhs.first_dx_all_visit_count_5yr, 0) AS first_dx_all_visit_count_5yr,
--     fdhs.first_dx_last_visit_5yr AS first_dx_last_visit_5yr,

--     fth.first_tx_hcp AS first_tx_hcp_5yr,
--     COALESCE(fths.first_tx_all_visit_count_5yr, 0) AS first_tx_all_visit_count_5yr,
--     COALESCE(fthtx.first_tx_treatment_visit_count_5yr, 0) AS first_tx_treatment_visit_count_5yr,
--     fths.first_tx_last_visit_5yr AS first_tx_last_visit_5yr,

--     msp.most_seen_hcp1_3yr_ranked,
--     COALESCE(msp.most_seen_hcp1_visit_count_5yr, 0) AS most_seen_hcp1_visit_count_5yr,
--     msp.most_seen_hcp1_last_visit_5yr,

--     msp.most_seen_hcp2_3yr_ranked,
--     COALESCE(msp.most_seen_hcp2_visit_count_5yr, 0) AS most_seen_hcp2_visit_count_5yr,
--     msp.most_seen_hcp2_last_visit_5yr,

--     msp.most_seen_hcp3_3yr_ranked,
--     COALESCE(msp.most_seen_hcp3_visit_count_5yr, 0) AS most_seen_hcp3_visit_count_5yr,
--     msp.most_seen_hcp3_last_visit_5yr,

--     msp.most_seen_hcp4_3yr_ranked,
--     COALESCE(msp.most_seen_hcp4_visit_count_5yr, 0) AS most_seen_hcp4_visit_count_5yr,
--     msp.most_seen_hcp4_last_visit_5yr,

--     msp.most_seen_hcp5_3yr_ranked,
--     COALESCE(msp.most_seen_hcp5_visit_count_5yr, 0) AS most_seen_hcp5_visit_count_5yr,
--     msp.most_seen_hcp5_last_visit_5yr

-- FROM eligible_patients ep
-- LEFT JOIN patient_demographics pd
--     ON ep.PATIENT_ID = pd.PATIENT_ID
-- LEFT JOIN patient_geography pg
--     ON ep.PATIENT_ID = pg.PATIENT_ID

-- LEFT JOIN historical_first_dx hfdx
--     ON ep.PATIENT_ID = hfdx.PATIENT_ID
-- LEFT JOIN historical_first_tx hftx
--     ON ep.PATIENT_ID = hftx.PATIENT_ID

-- LEFT JOIN first_tx_after_diagnosis fta
--     ON ep.PATIENT_ID = fta.PATIENT_ID

-- LEFT JOIN Tivi_fills ef
--     ON ep.PATIENT_ID = ef.PATIENT_ID

-- LEFT JOIN latest_mpsii_treatment_type lmt
--     ON ep.PATIENT_ID = lmt.PATIENT_ID

-- LEFT JOIN latest_claim_hcp_final lch
--     ON ep.PATIENT_ID = lch.PATIENT_ID
-- LEFT JOIN provider_dim pdlch
--     ON lch.latest_claim_hcp_npi = pdlch.npi
-- LEFT JOIN cmpa_insights_internal_schema.reference_file ref1
--     ON lch.latest_claim_hcp_npi = ref1.hcp_npi

-- LEFT JOIN most_recent_tx_hcp_final mrt
--     ON ep.PATIENT_ID = mrt.PATIENT_ID
-- LEFT JOIN provider_dim pdtch
--     ON mrt.latest_treatment_hcp_npi = pdtch.npi
-- LEFT JOIN cmpa_insights_internal_schema.reference_file ref2
--     ON mrt.latest_treatment_hcp_npi = ref2.hcp_npi

-- LEFT JOIN first_dx_hcp fdh
--     ON ep.PATIENT_ID = fdh.PATIENT_ID
-- LEFT JOIN first_dx_hcp_5yr_stats fdhs
--     ON ep.PATIENT_ID = fdhs.PATIENT_ID
-- LEFT JOIN first_tx_hcp fth
--     ON ep.PATIENT_ID = fth.PATIENT_ID
-- LEFT JOIN first_tx_hcp_5yr_stats fths
--     ON ep.PATIENT_ID = fths.PATIENT_ID
-- LEFT JOIN first_tx_hcp_5yr_tx_only fthtx
--     ON ep.PATIENT_ID = fthtx.PATIENT_ID

-- LEFT JOIN most_seen_pivot msp
--     ON ep.PATIENT_ID = msp.PATIENT_ID

-- ORDER BY ep.PATIENT_ID;

-- CREATE OR REPLACE TEMPORARY VIEW patient360_base AS
-- SELECT DISTINCT * FROM patient_hcp_visit_summary;


-- -- =============================================================================
-- -- CELL 15 (TIVI): mpsii_tx_claims (Tivi)
-- -- =============================================================================
-- CREATE OR REPLACE TEMP VIEW mpsii_tx_claims AS
-- WITH MPSII_Diagnoses_Specified AS (
--     SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE DIAGNOSIS_CODES LIKE '%E761%'
--       AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--     UNION
--     SELECT DISTINCT PATIENT_ID, FILL_DATE
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE DIAGNOSIS_CODE = 'E761'
--       AND TRANSACTION_RESULT = 'PAID'
--       AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
-- ),

-- Patients_2Dx_Specified AS (
--     SELECT PATIENT_ID
--     FROM MPSII_Diagnoses_Specified
--     GROUP BY PATIENT_ID
--     HAVING COUNT(DISTINCT FILL_DATE) >= 2
-- ),

-- MPSII_Diagnoses_Unspecified AS (
--     SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE DIAGNOSIS_CODES LIKE '%E763%'
--       AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--     UNION
--     SELECT DISTINCT PATIENT_ID, FILL_DATE
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE DIAGNOSIS_CODE = 'E763'
--       AND TRANSACTION_RESULT = 'PAID'
--       AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
-- ),

-- Patients_2Dx_Unspecified AS (
--     SELECT PATIENT_ID
--     FROM MPSII_Diagnoses_Unspecified
--     GROUP BY PATIENT_ID
--     HAVING COUNT(DISTINCT FILL_DATE) >= 2
-- ),

-- MPSII_Treatment_All AS (
--     SELECT DISTINCT PATIENT_ID FROM (
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE NDC11 IN ('8497600101')
--           AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_pharmacy_events
--         WHERE NDC11 IN ('8497600101')
--           AND TRANSACTION_RESULT = 'PAID'
--           AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE PROCEDURE_CODE IN ('J3490','J3590','J9999')
--           AND SERVICE_DATE >= '2026-03-01'
--     ) t
-- ),

-- MPSII_Treatment_Tivi_Only AS (
--     SELECT DISTINCT PATIENT_ID FROM (
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE NDC11 IN ('8497600101')
--           AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_pharmacy_events
--         WHERE NDC11 IN ('8497600101')
--           AND TRANSACTION_RESULT = 'PAID'
--           AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE PROCEDURE_CODE IN ('J3490','J3590','J9999')
--           AND SERVICE_DATE >= '2026-03-01'
--     ) t
-- ),

-- Patients_2Dx_Specified_With_Treatment AS (
--     SELECT DISTINCT p.PATIENT_ID
--     FROM Patients_2Dx_Specified p
--     INNER JOIN MPSII_Treatment_All t USING (PATIENT_ID)
-- ),

-- Patients_Incremental_Unspecified AS (
--     SELECT DISTINCT p.PATIENT_ID
--     FROM Patients_2Dx_Unspecified p
--     INNER JOIN MPSII_Treatment_Tivi_Only t USING (PATIENT_ID)
--     WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
-- ),

-- eligible_patients AS (
--     SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment
--     UNION
--     SELECT PATIENT_ID FROM Patients_Incremental_Unspecified
-- ),

-- cohort_3_learnings AS (
--   SELECT DISTINCT npi
--   FROM com_raw.kom_providers
--   WHERE provider_type = 'INDIVIDUAL'
--     AND (
--       PRIMARY_SPECIALTY NOT IN (
--         'Anesthesiologist Assistant','Anesthesiology','Dentist','Dietitian, Registered',
--         'Emergency Medical Technician, Basic','Emergency Medicine','General Acute Care Hospital',
--         'Nurse Anesthetist, Certified Registered','Obstetrics & Gynecology','Pathology',
--         'Radiology','Urology'
--       )
--       OR SECONDARY_SPECIALTY IN (
--         'Child & Adolescent Psychiatry','Psychiatry','Adolescent Medicine','Developmental - Behavioral Pediatrics',
--         'Neonatal-Perinatal Medicine','Nutrition, Pediatric','Oncology, Pediatrics','Pediatric Cardiology',
--         'Pediatric Critical Care Medicine','Pediatric Dermatology','Pediatric Emergency Medicine',
--         'Pediatric Endocrinology','Pediatric Gastroenterology','Pediatric Hematology-Oncology',
--         'Pediatric Infectious Diseases','Pediatric Nephrology','Pediatric Ophthalmology and Strabismus Specialist',
--         'Pediatric Orthopaedic Surgery','Pediatric Otolaryngology','Pediatric Pulmonology','Pediatric Radiology',
--         'Pediatric Rehabilitation Medicine','Pediatric Rheumatology','Pediatric Surgery','Pediatrics',
--         'Clinical Biochemical Genetics','Clinical Genetics (M.D.)','Clinical Molecular Genetics',
--         'Ph.D. Medical Genetics','Neurodevelopmental Disabilities','Neurology',
--         'Neurology with Special Qualifications in Child Neurology','Neuroradiology'
--       )
--     )
-- ),

-- all_tx_claims_2yr AS (
--     SELECT DISTINCT *
--     FROM (
--       SELECT DISTINCT
--         PATIENT_ID,
--         COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
--         SERVICE_DATE AS FILL_DATE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE NDC11 IN ('8497600101')
--         AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--       UNION

--       SELECT DISTINCT
--         PATIENT_ID,
--         PRESCRIBER_NPI AS NPI,
--         FILL_DATE
--       FROM com_edp_prd.com_raw.kom_pharmacy_events
--       WHERE NDC11 IN ('8497600101')
--         AND TRANSACTION_RESULT = 'PAID'
--         AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--       UNION

--       SELECT DISTINCT
--         PATIENT_ID,
--         RENDERING_NPI AS NPI,
--         SERVICE_DATE AS FILL_DATE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE PROCEDURE_CODE IN ('J3490','J3590','J9999')
--         AND SERVICE_DATE >= '2026-03-01'
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     )
--     WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
-- )

-- SELECT * FROM all_tx_claims_2yr;


-- -- =============================================================================
-- -- CELL 16 (TIVI): most_recently_treated_hcp (Tivi)
-- -- =============================================================================
-- CREATE OR REPLACE TEMPORARY VIEW most_recently_treated_hcp AS
-- WITH tx_claims AS (
--     SELECT DISTINCT *
--     FROM mpsii_tx_claims
-- ),

-- MPSII_Diagnoses_Specified AS (
--     SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE DIAGNOSIS_CODES LIKE '%E761%'
--       AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--     UNION
--     SELECT DISTINCT PATIENT_ID, FILL_DATE
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE DIAGNOSIS_CODE = 'E761'
--       AND TRANSACTION_RESULT = 'PAID'
--       AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
-- ),

-- Patients_2Dx_Specified AS (
--     SELECT PATIENT_ID
--     FROM MPSII_Diagnoses_Specified
--     GROUP BY PATIENT_ID
--     HAVING COUNT(DISTINCT FILL_DATE) >= 2
-- ),

-- MPSII_Diagnoses_Unspecified AS (
--     SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE DIAGNOSIS_CODES LIKE '%E763%'
--       AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--     UNION
--     SELECT DISTINCT PATIENT_ID, FILL_DATE
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE DIAGNOSIS_CODE = 'E763'
--       AND TRANSACTION_RESULT = 'PAID'
--       AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
-- ),

-- Patients_2Dx_Unspecified AS (
--     SELECT PATIENT_ID
--     FROM MPSII_Diagnoses_Unspecified
--     GROUP BY PATIENT_ID
--     HAVING COUNT(DISTINCT FILL_DATE) >= 2
-- ),

-- MPSII_Treatment_All AS (
--     SELECT DISTINCT PATIENT_ID FROM (
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE NDC11 IN ('8497600101')
--           AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_pharmacy_events
--         WHERE NDC11 IN ('8497600101')
--           AND TRANSACTION_RESULT = 'PAID'
--           AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE PROCEDURE_CODE IN ('J3490','J3590','J9999')
--           AND SERVICE_DATE >= '2026-03-01'
--     ) t
-- ),

-- MPSII_Treatment_Tivi_Only AS (
--     SELECT DISTINCT PATIENT_ID FROM (
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE NDC11 IN ('8497600101')
--           AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_pharmacy_events
--         WHERE NDC11 IN ('8497600101')
--           AND TRANSACTION_RESULT = 'PAID'
--           AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
--         UNION ALL
--         SELECT DISTINCT PATIENT_ID
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE PROCEDURE_CODE  IN ('J3490','J3590','J9999')
--           AND SERVICE_DATE >= '2026-03-01'
--     ) t
-- ),

-- Patients_2Dx_Specified_With_Treatment AS (
--     SELECT DISTINCT p.PATIENT_ID
--     FROM Patients_2Dx_Specified p
--     INNER JOIN MPSII_Treatment_All t USING (PATIENT_ID)
-- ),

-- Patients_Incremental_Unspecified AS (
--     SELECT DISTINCT p.PATIENT_ID
--     FROM Patients_2Dx_Unspecified p
--     INNER JOIN MPSII_Treatment_Tivi_Only t USING (PATIENT_ID)
--     WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
-- ),

-- eligible_patients AS (
--     SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment
--     UNION
--     SELECT PATIENT_ID FROM Patients_Incremental_Unspecified
-- ),

-- cohort_3_learnings AS (
--   SELECT DISTINCT npi
--   FROM com_raw.kom_providers
--   WHERE provider_type = 'INDIVIDUAL'
--     AND (
--       PRIMARY_SPECIALTY NOT IN (
--         'Anesthesiologist Assistant','Anesthesiology','Dentist','Dietitian, Registered',
--         'Emergency Medical Technician, Basic','Emergency Medicine','General Acute Care Hospital',
--         'Nurse Anesthetist, Certified Registered','Obstetrics & Gynecology','Pathology',
--         'Radiology','Urology'
--       )
--       OR SECONDARY_SPECIALTY IN (
--         'Child & Adolescent Psychiatry','Psychiatry','Adolescent Medicine','Developmental - Behavioral Pediatrics',
--         'Neonatal-Perinatal Medicine','Nutrition, Pediatric','Oncology, Pediatrics','Pediatric Cardiology',
--         'Pediatric Critical Care Medicine','Pediatric Dermatology','Pediatric Emergency Medicine',
--         'Pediatric Endocrinology','Pediatric Gastroenterology','Pediatric Hematology-Oncology',
--         'Pediatric Infectious Diseases','Pediatric Nephrology','Pediatric Ophthalmology and Strabismus Specialist',
--         'Pediatric Orthopaedic Surgery','Pediatric Otolaryngology','Pediatric Pulmonology','Pediatric Radiology',
--         'Pediatric Rehabilitation Medicine','Pediatric Rheumatology','Pediatric Surgery','Pediatrics',
--         'Clinical Biochemical Genetics','Clinical Genetics (M.D.)','Clinical Molecular Genetics',
--         'Ph.D. Medical Genetics','Neurodevelopmental Disabilities','Neurology',
--         'Neurology with Special Qualifications in Child Neurology','Neuroradiology'
--       )
--     )
-- ),
-- all_dx_claims_5yr AS (
--     SELECT DISTINCT *
--     FROM (
--       SELECT DISTINCT
--         PATIENT_ID,
--         COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
--         SERVICE_DATE AS FILL_DATE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
--         AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--       UNION
--       SELECT DISTINCT
--         PATIENT_ID,
--         PRESCRIBER_NPI AS NPI,
--         FILL_DATE
--       FROM com_edp_prd.com_raw.kom_pharmacy_events
--       WHERE DIAGNOSIS_CODE IN ('E761','E763')
--         AND TRANSACTION_RESULT = 'PAID'
--         AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     )
--     WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
-- ),

-- all_tx_claims_5yr AS (
--     SELECT DISTINCT *
--     FROM (
--       SELECT DISTINCT
--         PATIENT_ID,
--         COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
--         SERVICE_DATE AS FILL_DATE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE NDC11 IN ('8497600101')
--         AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--       UNION
--       SELECT DISTINCT
--         PATIENT_ID,
--         PRESCRIBER_NPI AS NPI,
--         FILL_DATE
--       FROM com_edp_prd.com_raw.kom_pharmacy_events
--       WHERE NDC11 IN ('8497600101')
--         AND TRANSACTION_RESULT = 'PAID'
--         AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--       UNION
--       SELECT DISTINCT
--         PATIENT_ID,
--         RENDERING_NPI AS NPI,
--         SERVICE_DATE AS FILL_DATE
--       FROM com_edp_prd.com_raw.kom_medical_events
--       WHERE PROCEDURE_CODE IN ('J3490','J3590','J9999')
--         AND SERVICE_DATE >= '2026-03-01'
--         AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
--     )
--     WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
-- ),

-- all_claims AS (
--     SELECT * FROM all_dx_claims_5yr
--     UNION
--     SELECT * FROM all_tx_claims_5yr
-- ),

-- latest_treating_hcp AS (
--     SELECT patient_id, npi
--     FROM (
--         SELECT
--             patient_id,
--             npi,
--             fill_date,
--             ROW_NUMBER() OVER (
--                 PARTITION BY patient_id
--                 ORDER BY
--                     CASE WHEN npi IS NOT NULL THEN 1 ELSE 2 END,
--                     fill_date DESC,
--                     npi DESC
--             ) AS rn
--         FROM tx_claims
--     ) t
--     WHERE rn = 1
--       AND npi IS NOT NULL
-- ),

-- visit_counts AS (
--     SELECT
--         patient_id,
--         npi,
--         COUNT(DISTINCT fill_date) AS visit_counts
--     FROM all_claims
--     WHERE npi IS NOT NULL
--     GROUP BY patient_id, npi
-- ),

-- last_visit_date AS (
--     SELECT
--         lth.patient_id,
--         lth.npi,
--         MAX(ac.fill_date) AS last_visit_date
--     FROM latest_treating_hcp lth
--     LEFT JOIN all_claims ac
--       ON lth.patient_id = ac.patient_id
--      AND lth.npi = ac.npi
--     GROUP BY lth.patient_id, lth.npi
-- ),

-- latest_treating_hcp_with_visits AS (
--     SELECT
--         a.patient_id,
--         a.npi AS most_recently_treated_hcp,
--         b.visit_counts AS no_of_visits,
--         c.last_visit_date AS last_visit_date_5yr
--     FROM latest_treating_hcp AS a
--     LEFT JOIN visit_counts AS b
--         ON a.patient_id = b.patient_id
--        AND a.npi        = b.npi
--     LEFT JOIN last_visit_date AS c
--         ON a.patient_id = c.patient_id
--        AND a.npi        = c.npi
-- ),

-- hcp_with_other_info AS (
--     SELECT
--         a.*,
--         CONCAT(b.FIRST_NAME, ' ', b.LAST_NAME) AS hcp_name,
--         b.PRIMARY_SPECIALTY AS hcp_specialty,
--         c.hco_name,
--         c.mapped_territory_id AS territory_id,
--         c.territory,
--         c.mapped_region_id AS region_id,
--         c.region
--     FROM latest_treating_hcp_with_visits AS a

--     LEFT JOIN com_edp_prd.com_raw.kom_providers AS b
--         ON a.most_recently_treated_hcp = b.NPI
--        AND b.PROVIDER_TYPE = 'INDIVIDUAL'

--     LEFT JOIN (
--   SELECT
--     * EXCEPT (hcp_primary_specialty),
--     hcp_primary_specialty AS hcp_specialty
--   FROM (
--     SELECT
--       a.* EXCEPT (territory_id, region_id),
--       b.territory_id AS mapped_territory_id,
--       c.region_id AS mapped_region_id
--     FROM cmpa_insights_internal_schema.reference_file AS a
--     LEFT JOIN (
--       SELECT DISTINCT territory_id, territory_name
--       FROM cmpa_insights_internal_schema.zip_to_territory_mapping
--     ) AS b
--       ON a.territory = b.territory_name
--     LEFT JOIN (
--       SELECT DISTINCT region_id, region_name
--       FROM cmpa_insights_internal_schema.zip_to_territory_mapping
--     ) AS c
--       ON a.region = c.region_name
--   )
-- ) AS c
-- ON a.most_recently_treated_hcp = c.hcp_npi
-- )

-- SELECT
--     patient_id,
--     most_recently_treated_hcp AS most_recently_treated_hcp_2yr,
--     hcp_name AS most_recently_treated_hcp_name_2yr,

--     no_of_visits AS most_recently_treated_hcp_2yr_no_of_visits_5yr,
--     last_visit_date_5yr AS most_recent_tx_hcp_2yr_last_visit_5yr,

--     hcp_specialty AS most_recently_treated_hcp_specialty_2yr,
--     hco_name AS most_recently_treated_hcp_hco_name,
--     territory_id AS most_recently_treated_hcp_territory_id_2yr,
--     territory AS most_recently_treated_hcp_territory_2yr,
--     region_id AS most_recently_treated_hcp_region_id_2yr,
--     region AS most_recently_treated_hcp_region_2yr
-- FROM hcp_with_other_info;


-- -- =============================================================================
-- -- CELL 17 (TIVI): all_dx_claims / all_tx_claims / primary_hcp build (Tivi)
-- -- =============================================================================
-- CREATE OR REPLACE TEMPORARY VIEW all_dx_claims AS

-- SELECT DISTINCT
--     PATIENT_ID,
--     COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
--     SERVICE_DATE AS FILL_DATE,
--     'DX' AS CLAIM_TYPE
-- FROM com_edp_prd.com_raw.kom_medical_events
-- WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
--   AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)

-- UNION

-- SELECT DISTINCT
--     PATIENT_ID,
--     PRESCRIBER_NPI AS NPI,
--     FILL_DATE,
--     'DX' AS CLAIM_TYPE
-- FROM com_edp_prd.com_raw.kom_pharmacy_events
-- WHERE DIAGNOSIS_CODE IN ('E761', 'E763')
--   AND TRANSACTION_RESULT = 'PAID'
--   AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters);

-- CREATE OR REPLACE TEMPORARY VIEW all_tx_claims AS

-- SELECT DISTINCT
--     PATIENT_ID,
--     COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
--     SERVICE_DATE AS FILL_DATE,
--     'TX' AS CLAIM_TYPE,
--     NDC11 AS CODE
-- FROM com_edp_prd.com_raw.kom_medical_events
-- WHERE NDC11 IN ('8497600101')
--   AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)

-- UNION

-- SELECT DISTINCT
--     PATIENT_ID,
--     RENDERING_NPI AS NPI,
--     SERVICE_DATE AS FILL_DATE,
--     'TX' AS CLAIM_TYPE,
--     PROCEDURE_CODE AS CODE
-- FROM com_edp_prd.com_raw.kom_medical_events
-- WHERE PROCEDURE_CODE IN ('J3490','J3590','J9999')
--   AND SERVICE_DATE >= '2026-03-01'

-- UNION

-- SELECT DISTINCT
--     PATIENT_ID,
--     PRESCRIBER_NPI AS NPI,
--     FILL_DATE,
--     'TX' AS CLAIM_TYPE,
--     NDC11 AS CODE
-- FROM com_edp_prd.com_raw.kom_pharmacy_events
-- WHERE NDC11 IN ('8497600101')
--   AND TRANSACTION_RESULT = 'PAID'
--   AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters);

-- CREATE OR REPLACE TEMPORARY VIEW tx_claims_2yr AS
-- SELECT DISTINCT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE, CODE
-- FROM all_tx_claims
-- WHERE FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters);


-- CREATE OR REPLACE TEMPORARY VIEW e761_patients_2dx AS
-- SELECT PATIENT_ID
-- FROM (
--     SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE DIAGNOSIS_CODES LIKE '%E761%'
--       AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--     UNION
--     SELECT DISTINCT PATIENT_ID, FILL_DATE
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE DIAGNOSIS_CODE = 'E761'
--       AND TRANSACTION_RESULT = 'PAID'
--       AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
-- )
-- GROUP BY PATIENT_ID
-- HAVING COUNT(DISTINCT FILL_DATE) >= 2;

-- CREATE OR REPLACE TEMPORARY VIEW specified_patients AS
-- SELECT DISTINCT e.PATIENT_ID
-- FROM e761_patients_2dx e
-- INNER JOIN tx_claims_2yr t ON e.PATIENT_ID = t.PATIENT_ID;

-- CREATE OR REPLACE TEMPORARY VIEW e763_patients_2dx AS
-- SELECT PATIENT_ID
-- FROM (
--     SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE DIAGNOSIS_CODES LIKE '%E763%'
--       AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--     UNION
--     SELECT DISTINCT PATIENT_ID, FILL_DATE
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE DIAGNOSIS_CODE = 'E763'
--       AND TRANSACTION_RESULT = 'PAID'
--       AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
-- )
-- GROUP BY PATIENT_ID
-- HAVING COUNT(DISTINCT FILL_DATE) >= 2;

-- CREATE OR REPLACE TEMPORARY VIEW Tivi_tx_2yr AS
-- SELECT DISTINCT PATIENT_ID
-- FROM tx_claims_2yr
-- WHERE CODE IN ('8497600101');

-- CREATE OR REPLACE TEMPORARY VIEW incremental_patients AS
-- SELECT DISTINCT e.PATIENT_ID
-- FROM e763_patients_2dx e
-- INNER JOIN Tivi_tx_2yr t ON e.PATIENT_ID = t.PATIENT_ID
-- WHERE e.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM specified_patients);

-- CREATE OR REPLACE TEMPORARY VIEW eligible_patients AS
-- SELECT PATIENT_ID FROM specified_patients
-- UNION
-- SELECT PATIENT_ID FROM incremental_patients;


-- CREATE OR REPLACE TEMPORARY VIEW cohort_3_learnings AS
-- SELECT DISTINCT npi
-- FROM com_raw.kom_providers
-- WHERE provider_type = 'INDIVIDUAL'
--   AND (
--     PRIMARY_SPECIALTY NOT IN (
--       'Anesthesiologist Assistant','Anesthesiology','Dentist','Dietitian, Registered',
--       'Emergency Medical Technician, Basic','Emergency Medicine','General Acute Care Hospital',
--       'Nurse Anesthetist, Certified Registered','Obstetrics & Gynecology','Pathology',
--       'Radiology','Urology'
--     )
--     OR SECONDARY_SPECIALTY IN (
--       'Child & Adolescent Psychiatry','Psychiatry','Adolescent Medicine','Developmental - Behavioral Pediatrics',
--       'Neonatal-Perinatal Medicine','Nutrition, Pediatric','Oncology, Pediatrics','Pediatric Cardiology',
--       'Pediatric Critical Care Medicine','Pediatric Dermatology','Pediatric Emergency Medicine',
--       'Pediatric Endocrinology','Pediatric Gastroenterology','Pediatric Hematology-Oncology',
--       'Pediatric Infectious Diseases','Pediatric Nephrology','Pediatric Ophthalmology and Strabismus Specialist',
--       'Pediatric Orthopaedic Surgery','Pediatric Otolaryngology','Pediatric Pulmonology','Pediatric Radiology',
--       'Pediatric Rehabilitation Medicine','Pediatric Rheumatology','Pediatric Surgery','Pediatrics',
--       'Clinical Biochemical Genetics','Clinical Genetics (M.D.)','Clinical Molecular Genetics',
--       'Ph.D. Medical Genetics','Neurodevelopmental Disabilities','Neurology',
--       'Neurology with Special Qualifications in Child Neurology','Neuroradiology'
--     )
--   );


-- CREATE OR REPLACE TEMPORARY VIEW all_patient_claims AS
-- SELECT *
-- FROM (
--   SELECT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
--   FROM all_dx_claims
--   WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--   UNION

--   SELECT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
--   FROM all_tx_claims
--   WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
-- )
-- WHERE npi IN (SELECT DISTINCT npi FROM cohort_3_learnings);


-- CREATE OR REPLACE TEMPORARY VIEW primary_hcp AS
-- WITH hcp_metrics AS (
--     SELECT
--         a.PATIENT_ID,
--         a.NPI,

--         CASE
--             WHEN p.primary_specialty LIKE '%Genetic%'
--               OR p.secondary_specialty LIKE '%Genetic%'
--                 THEN 'Geneticist'
--             WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
--               OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
--               OR p.primary_specialty LIKE '%Neurological Surgery%'
--                 THEN 'Psychiatry & Neurology'
--             WHEN p.primary_specialty LIKE '%Pediatrics%'
--                 THEN 'Pediatrician'
--             WHEN p.primary_specialty LIKE '%Internal Medicine%'
--               OR p.secondary_specialty LIKE '%Internal Medicine%'
--               OR p.primary_specialty LIKE '%Family Medicine%'
--               OR p.secondary_specialty LIKE '%Family Medicine%'
--                 THEN 'PCP'
--             WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
--               OR p.primary_specialty LIKE '%Physician Assistant%'
--                 THEN 'NPPA'
--             WHEN a.NPI IS NULL
--                 THEN 'NA'
--             ELSE 'Others'
--         END AS SPECIALTY,

--         CASE
--             WHEN p.primary_specialty LIKE '%Genetic%'
--               OR p.secondary_specialty LIKE '%Genetic%'
--                 THEN 1
--             WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
--               OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
--               OR p.primary_specialty LIKE '%Neurological Surgery%'
--                 THEN 2
--             WHEN p.primary_specialty LIKE '%Pediatrics%'
--                 THEN 3
--             WHEN p.primary_specialty LIKE '%Internal Medicine%'
--               OR p.secondary_specialty LIKE '%Internal Medicine%'
--               OR p.primary_specialty LIKE '%Family Medicine%'
--               OR p.secondary_specialty LIKE '%Family Medicine%'
--                 THEN 4
--             WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
--               OR p.primary_specialty LIKE '%Physician Assistant%'
--                 THEN 5
--             WHEN a.NPI IS NULL
--                 THEN 7
--             ELSE 6
--         END AS SPECIALTY_PRIORITY,

--         COUNT(DISTINCT a.FILL_DATE) AS NO_OF_VISITS,

--         COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'DX' THEN a.FILL_DATE END) AS DX_VISITS,
--         COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'TX' THEN a.FILL_DATE END) AS TX_VISITS,

--         MAX(a.FILL_DATE) AS MOST_RECENT_VISIT

--     FROM all_patient_claims a
--     LEFT JOIN com_edp_prd.com_raw.kom_providers p
--         ON a.NPI = p.NPI
--     GROUP BY
--         a.PATIENT_ID,
--         a.NPI,
--         p.primary_specialty,
--         p.secondary_specialty
-- ),
-- ranked_hcps AS (
--     SELECT
--         *,
--         RANK() OVER (
--             PARTITION BY PATIENT_ID
--             ORDER BY
--                 SPECIALTY_PRIORITY ASC,
--                 NO_OF_VISITS DESC,
--                 MOST_RECENT_VISIT DESC,
--                 NPI ASC
--         ) AS HCP_RANK
--     FROM hcp_metrics
-- )
-- SELECT
--     PATIENT_ID,
--     NPI AS PRIMARY_HCP_NPI,
--     SPECIALTY AS PRIMARY_HCP_SPECIALTY,
--     SPECIALTY_PRIORITY,
--     NO_OF_VISITS,
--     DX_VISITS,
--     TX_VISITS,
--     MOST_RECENT_VISIT,
--     HCP_RANK
-- FROM ranked_hcps
-- WHERE HCP_RANK = 1;


-- -- =============================================================================
-- -- CELL 18 (TIVI): tivi_patient360_master view (all columns prefixed tivi_)
-- -- =============================================================================
-- CREATE OR REPLACE TEMPORARY VIEW tivi_patient360_master AS
-- select PATIENT_ID as tivi_patient_id, PATIENT_YOB as tivi_patient_yob, PATIENT_AGE as tivi_patient_age, PATIENT_GENDER as tivi_patient_gender, patient_state as tivi_patient_state, incidence_date as tivi_incidence_date, first_incidence_treatment_date as tivi_first_incidence_treatment_date, latest_claim_date as tivi_latest_claim_date, latest_claim_hcp_npi as tivi_latest_claim_hcp_npi, latest_claim_hcp_name as tivi_latest_claim_hcp_name, latest_claim_hcp_specialty as tivi_latest_claim_hcp_specialty, latest_claim_hcp_visit_count as tivi_latest_claim_hcp_visit_count, latest_claim_hcp_hco_name as tivi_latest_claim_hcp_hco_name, latest_treatment_date as tivi_latest_treatment_date, latest_mpsii_tx_type as tivi_latest_mpsii_tx_type, first_tx_after_diagnosis as tivi_first_tx_after_diagnosis, time_dx_to_first_tx_in_months as tivi_time_dx_to_first_tx_in_months, treatment_period_months as tivi_treatment_period_months, Tivi_fills as tivi_fills, latest_treatment_hcp_npi as tivi_latest_treatment_hcp_npi, latest_treatment_hcp_name as tivi_latest_treatment_hcp_name, latest_treatment_hcp_specialty as tivi_latest_treatment_hcp_specialty, latest_treatment_hcp_visit_count as tivi_latest_treatment_hcp_visit_count, latest_treatment_hcp_hco_name as tivi_latest_treatment_hcp_hco_name, first_dx_hcp_5yr as tivi_first_dx_hcp_5yr, first_dx_all_visit_count_5yr as tivi_first_dx_all_visit_count_5yr, first_dx_last_visit_5yr as tivi_first_dx_last_visit_5yr, first_tx_hcp_5yr as tivi_first_tx_hcp_5yr, first_tx_all_visit_count_5yr as tivi_first_tx_all_visit_count_5yr, first_tx_treatment_visit_count_5yr as tivi_first_tx_treatment_visit_count_5yr, first_tx_last_visit_5yr as tivi_first_tx_last_visit_5yr, most_seen_hcp1_3yr_ranked as tivi_most_seen_hcp1_3yr_ranked, most_seen_hcp1_visit_count_5yr as tivi_most_seen_hcp1_visit_count_5yr, most_seen_hcp1_last_visit_5yr as tivi_most_seen_hcp1_last_visit_5yr, most_seen_hcp2_3yr_ranked as tivi_most_seen_hcp2_3yr_ranked, most_seen_hcp2_visit_count_5yr as tivi_most_seen_hcp2_visit_count_5yr, most_seen_hcp2_last_visit_5yr as tivi_most_seen_hcp2_last_visit_5yr, most_seen_hcp3_3yr_ranked as tivi_most_seen_hcp3_3yr_ranked, most_seen_hcp3_visit_count_5yr as tivi_most_seen_hcp3_visit_count_5yr, most_seen_hcp3_last_visit_5yr as tivi_most_seen_hcp3_last_visit_5yr, most_seen_hcp4_3yr_ranked as tivi_most_seen_hcp4_3yr_ranked, most_seen_hcp4_visit_count_5yr as tivi_most_seen_hcp4_visit_count_5yr, most_seen_hcp4_last_visit_5yr as tivi_most_seen_hcp4_last_visit_5yr, most_seen_hcp5_3yr_ranked as tivi_most_seen_hcp5_3yr_ranked, most_seen_hcp5_visit_count_5yr as tivi_most_seen_hcp5_visit_count_5yr, most_seen_hcp5_last_visit_5yr as tivi_most_seen_hcp5_last_visit_5yr, most_recently_treated_hcp_2yr as tivi_most_recently_treated_hcp_2yr, most_recently_treated_hcp_name_2yr as tivi_most_recently_treated_hcp_name_2yr, most_recently_treated_hcp_2yr_no_of_visits_5yr as tivi_most_recently_treated_hcp_2yr_no_of_visits_5yr, most_recent_tx_hcp_2yr_last_visit_5yr as tivi_most_recent_tx_hcp_2yr_last_visit_5yr, most_recently_treated_hcp_specialty_2yr as tivi_most_recently_treated_hcp_specialty_2yr, most_recently_treated_hcp_hco_name as tivi_most_recently_treated_hcp_hco_name, most_recently_treated_hcp_territory_id_2yr as tivi_most_recently_treated_hcp_territory_id_2yr, most_recently_treated_hcp_territory_2yr as tivi_most_recently_treated_hcp_territory_2yr, most_recently_treated_hcp_region_id_2yr as tivi_most_recently_treated_hcp_region_id_2yr, most_recently_treated_hcp_region_2yr as tivi_most_recently_treated_hcp_region_2yr, PRIMARY_HCP_NPI as tivi_primary_hcp_npi, PRIMARY_HCP_SPECIALTY as tivi_primary_hcp_specialty, SPECIALTY_PRIORITY as tivi_specialty_priority, NO_OF_VISITS as tivi_no_of_visits, DX_VISITS as tivi_dx_visits, TX_VISITS as tivi_tx_visits, MOST_RECENT_VISIT as tivi_most_recent_visit, HCP_RANK as tivi_hcp_rank
-- from (SELECT DISTINCT
--     a.PATIENT_ID,
--     a.PATIENT_YOB,
--     a.PATIENT_AGE,
--     a.PATIENT_GENDER,
--     a.patient_state,

--     a.incidence_date,
--     a.first_incidence_treatment_date,

--     a.latest_claim_date,

--     a.latest_claim_hcp_npi,
--     a.latest_claim_hcp_name,
--     a.latest_claim_hcp_specialty,
--     a.latest_claim_hcp_visit_count,

--     a.latest_claim_hcp_hco_name,

--     a.latest_treatment_date,
--     a.latest_mpsii_tx_type,

--     a.first_tx_after_diagnosis,
--     a.time_dx_to_first_tx_in_months,
--     a.treatment_period_months,

--     a.Tivi_fills,

--     a.latest_treatment_hcp_npi,
--     a.latest_treatment_hcp_name,
--     a.latest_treatment_hcp_specialty,
--     a.latest_treatment_hcp_visit_count,

--     a.latest_treatment_hcp_hco_name,

--     a.first_dx_hcp_5yr,
--     a.first_dx_all_visit_count_5yr,
--     a.first_dx_last_visit_5yr,
--     a.first_tx_hcp_5yr,
--     a.first_tx_all_visit_count_5yr,
--     a.first_tx_treatment_visit_count_5yr,
--     a.first_tx_last_visit_5yr,

--     a.most_seen_hcp1_3yr_ranked,
--     a.most_seen_hcp1_visit_count_5yr,
--     a.most_seen_hcp1_last_visit_5yr,
--     a.most_seen_hcp2_3yr_ranked,
--     a.most_seen_hcp2_visit_count_5yr,
--     a.most_seen_hcp2_last_visit_5yr,
--     a.most_seen_hcp3_3yr_ranked,
--     a.most_seen_hcp3_visit_count_5yr,
--     a.most_seen_hcp3_last_visit_5yr,
--     a.most_seen_hcp4_3yr_ranked,
--     a.most_seen_hcp4_visit_count_5yr,
--     a.most_seen_hcp4_last_visit_5yr,
--     a.most_seen_hcp5_3yr_ranked,
--     a.most_seen_hcp5_visit_count_5yr,
--     a.most_seen_hcp5_last_visit_5yr,

--     b.* EXCEPT (patient_id),

--     c.* EXCEPT (patient_id)

-- FROM patient360_base AS a

-- LEFT JOIN most_recently_treated_hcp AS b
--     ON a.PATIENT_ID = b.patient_id

-- LEFT JOIN primary_hcp AS c
--     ON a.PATIENT_ID = c.patient_id);


-- -- =============================================================================
-- -- CELL 19 (TIVI): Merge tivi_* columns into the existing patient360_master
-- -- =============================================================================
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master AS
-- WITH base AS (
--   SELECT * FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
-- ),
-- t2 AS (
--   SELECT a.*, b.* EXCEPT (b.tivi_patient_id, b.tivi_patient_yob, b.tivi_patient_age, b.tivi_patient_gender, b.tivi_patient_state)
--   FROM base AS a
--   LEFT JOIN tivi_patient360_master AS b
--     ON a.patient_id = b.tivi_patient_id
-- )
-- SELECT DISTINCT * FROM t2;


In [0]:
CREATE OR REPLACE TEMP VIEW patient_hcp_visit_summary AS
WITH MPSII_Diagnoses_Specified AS (

    SELECT DISTINCT
        PATIENT_ID,
        SERVICE_DATE AS DX_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'

    UNION

    SELECT DISTINCT
        PATIENT_ID,
        FILL_DATE AS DX_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
),

Patients_2Dx_Specified AS (

    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Specified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT DX_DATE) >= 2
),

/* ============================================================================
   MPSII Treatment - Elaprase Ever (NEW COHORT DEFINITION)
   ========================================================================== */
MPSII_Treatment_Elaprase_Ever AS (

    SELECT DISTINCT PATIENT_ID
    FROM (

        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')

        UNION ALL

        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'

        UNION ALL

        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE = 'J1743'

    ) t
),

/* ============================================================================
   Base MPS II Universe (NEW COHORT DEFINITION)
   ========================================================================== */
Base_MPSII_Universe AS (

    SELECT PATIENT_ID
    FROM Patients_2Dx_Specified

    UNION

    SELECT PATIENT_ID
    FROM MPSII_Treatment_Elaprase_Ever
),

/* ============================================================================
   Recent Treatment Activity (NEW COHORT DEFINITION - using global_runtime_dates)
   ========================================================================== */
Recent_Treatment_2Y AS (

    SELECT DISTINCT PATIENT_ID
    FROM (

        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE >= (SELECT elaprase_tx_start_date FROM global_runtime_dates)

        UNION ALL

        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE >= (SELECT elaprase_tx_start_date FROM global_runtime_dates)

        UNION ALL

        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE = 'J1743'
          AND SERVICE_DATE >= (SELECT elaprase_tx_start_date FROM global_runtime_dates)

        UNION ALL

        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
          AND SERVICE_DATE >= (SELECT elaprase_tx_start_date FROM global_runtime_dates)

    ) t
),

/* ============================================================================
   FINAL ELIGIBLE PATIENTS (NEW COHORT DEFINITION)
   ========================================================================== */
eligible_patients AS (

    SELECT DISTINCT b.PATIENT_ID
    FROM Base_MPSII_Universe b
    INNER JOIN Recent_Treatment_2Y r
        ON b.PATIENT_ID = r.PATIENT_ID
),

/* ============================================================================
   2) PROVIDER FILTER ("COHORT 3 LEARNINGS")
   ========================================================================== */
cohort_3_learnings AS (
  SELECT DISTINCT npi
  FROM com_raw.kom_providers
  WHERE provider_type = 'INDIVIDUAL'
    AND (
      PRIMARY_SPECIALTY NOT IN (
        'Anesthesiologist Assistant','Anesthesiology','Dentist','Dietitian, Registered',
        'Emergency Medical Technician, Basic','Emergency Medicine','General Acute Care Hospital',
        'Nurse Anesthetist, Certified Registered','Obstetrics & Gynecology','Pathology',
        'Radiology','Urology'
      )
      OR SECONDARY_SPECIALTY IN (
        'Child & Adolescent Psychiatry','Psychiatry','Adolescent Medicine','Developmental - Behavioral Pediatrics',
        'Neonatal-Perinatal Medicine','Nutrition, Pediatric','Oncology, Pediatrics','Pediatric Cardiology',
        'Pediatric Critical Care Medicine','Pediatric Dermatology','Pediatric Emergency Medicine',
        'Pediatric Endocrinology','Pediatric Gastroenterology','Pediatric Hematology-Oncology',
        'Pediatric Infectious Diseases','Pediatric Nephrology','Pediatric Ophthalmology and Strabismus Specialist',
        'Pediatric Orthopaedic Surgery','Pediatric Otolaryngology','Pediatric Pulmonology','Pediatric Radiology',
        'Pediatric Rehabilitation Medicine','Pediatric Rheumatology','Pediatric Surgery','Pediatrics',
        'Clinical Biochemical Genetics','Clinical Genetics (M.D.)','Clinical Molecular Genetics',
        'Ph.D. Medical Genetics','Neurodevelopmental Disabilities','Neurology',
        'Neurology with Special Qualifications in Child Neurology','Neuroradiology'
      )
    )
),

/* ============================================================================
   3) CLAIMS UNIVERSES (DX + TX) WITH NPI ATTRIBUTION
   ========================================================================== */

all_dx_claims_5yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE DIAGNOSIS_CODE IN ('E761','E763')
        AND TRANSACTION_STATUS = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

all_tx_claims_5yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT
          PATIENT_ID,
          COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
          SERVICE_DATE AS FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          PRESCRIBER_NPI AS NPI,
          FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          RENDERING_NPI AS NPI,
          SERVICE_DATE AS FILL_DATE,
          PROCEDURE_CODE AS TX_CODE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                               '38206','38230','38232','38240','38241','38242','38243','38250')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

all_tx_claims_alltime AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT
          PATIENT_ID,
          COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
          SERVICE_DATE AS FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          PRESCRIBER_NPI AS NPI,
          FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND TRANSACTION_RESULT = 'PAID'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          RENDERING_NPI AS NPI,
          SERVICE_DATE AS FILL_DATE,
          PROCEDURE_CODE AS TX_CODE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                               '38206','38230','38232','38240','38241','38242','38243','38250')
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

all_claims_5yr AS (
    SELECT DISTINCT
        PATIENT_ID,
        NPI,
        FILL_DATE,
        CAST(NULL AS STRING) AS TX_CODE
    FROM all_dx_claims_5yr
    UNION
    SELECT DISTINCT
        PATIENT_ID,
        NPI,
        FILL_DATE,
        TX_CODE
    FROM all_tx_claims_5yr
),

all_claims_3yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
        AND SERVICE_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE DIAGNOSIS_CODE IN ('E761','E763')
        AND TRANSACTION_STATUS = 'PAID'
        AND FILL_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND SERVICE_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                               '38206','38230','38232','38240','38241','38242','38243','38250')
        AND SERVICE_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

/* ============================================================================
   3b) AVLAYAH NDC EVENTS (PULLED DIRECTLY FROM SOURCE TABLES)
   The original avlayah_status_base read from all_tx_claims_5yr -- but that
   view excludes Avlayah codes, so has_avlayah was always 0. This CTE pulls
   confirmed Avlayah NDC events (8497600101) from medical and pharmacy events
   directly. Used by both avlayah_status_base and latest_treatment_type.
   Per business rule: NDC ONLY -- no proxy J-codes.
   ========================================================================== */
avlayah_ndc_events AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE NDC11 = '8497600101'
      OR PROCEDURE_CODE in ('J3490','J3590','J9999')
      AND SERVICE_DATE >= '2026-03-01'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

    UNION

    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 = '8497600101'
      AND TRANSACTION_RESULT = 'PAID'
      AND FILL_DATE >= '2026-03-01'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
),

/* ============================================================================
   4) FIRST DX / FIRST TX HCP ATTRIBUTION (5Y WINDOW)
   ========================================================================== */

first_dx_hcp_ranked AS (
    SELECT PATIENT_ID, NPI, MIN(FILL_DATE) AS first_dx_date,
           ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY MIN(FILL_DATE) ASC, NPI) AS rn
    FROM all_dx_claims_5yr
    WHERE NPI IS NOT NULL
    GROUP BY PATIENT_ID, NPI
),
first_dx_hcp AS (
    SELECT PATIENT_ID, NPI AS first_dx_hcp, first_dx_date
    FROM first_dx_hcp_ranked
    WHERE rn = 1
),

provider_dim AS (
    SELECT
        npi,
        CONCAT(FIRST_NAME, ' ', LAST_NAME) AS provider_name,
        primary_specialty
    FROM com_raw.kom_providers
    WHERE provider_type = 'INDIVIDUAL'
),

first_dx_hcp_5yr_stats AS (
    SELECT fdh.PATIENT_ID, fdh.first_dx_hcp,
           COUNT(DISTINCT ac.FILL_DATE) AS first_dx_all_visit_count_5yr,
           MAX(ac.FILL_DATE) AS first_dx_last_visit_5yr
    FROM first_dx_hcp fdh
    LEFT JOIN all_claims_5yr ac
      ON fdh.PATIENT_ID = ac.PATIENT_ID AND fdh.first_dx_hcp = ac.NPI
    GROUP BY fdh.PATIENT_ID, fdh.first_dx_hcp
),

first_tx_hcp_ranked AS (
    SELECT PATIENT_ID, NPI, MIN(FILL_DATE) AS first_tx_date,
           ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY MIN(FILL_DATE) ASC, NPI) AS rn
    FROM all_tx_claims_5yr
    WHERE NPI IS NOT NULL
    GROUP BY PATIENT_ID, NPI
),
first_tx_hcp AS (
    SELECT PATIENT_ID, NPI AS first_tx_hcp, first_tx_date
    FROM first_tx_hcp_ranked
    WHERE rn = 1
),

first_tx_hcp_5yr_stats AS (
    SELECT fth.PATIENT_ID, fth.first_tx_hcp,
           COUNT(DISTINCT ac.FILL_DATE) AS first_tx_all_visit_count_5yr,
           MAX(ac.FILL_DATE) AS first_tx_last_visit_5yr
    FROM first_tx_hcp fth
    LEFT JOIN all_claims_5yr ac
      ON fth.PATIENT_ID = ac.PATIENT_ID AND fth.first_tx_hcp = ac.NPI
    GROUP BY fth.PATIENT_ID, fth.first_tx_hcp
),

first_tx_hcp_5yr_tx_only AS (
    SELECT fth.PATIENT_ID, fth.first_tx_hcp,
           COUNT(DISTINCT tx.FILL_DATE) AS first_tx_treatment_visit_count_5yr
    FROM first_tx_hcp fth
    LEFT JOIN all_tx_claims_5yr tx
      ON fth.PATIENT_ID = tx.PATIENT_ID AND fth.first_tx_hcp = tx.NPI
    GROUP BY fth.PATIENT_ID, fth.first_tx_hcp
),

/* ============================================================================
   5) MOST-SEEN HCP RANKING (TOP 5) USING 3Y ACTIVITY
   ========================================================================== */

most_seen_3yr_ranking AS (
    SELECT PATIENT_ID,
           NPI,
           COUNT(DISTINCT FILL_DATE) AS visit_count_3yr,
           MAX(FILL_DATE) AS last_visit_3yr,
           ROW_NUMBER() OVER (
             PARTITION BY PATIENT_ID
             ORDER BY COUNT(DISTINCT FILL_DATE) DESC,
                      MAX(FILL_DATE) DESC,
                      NPI ASC
           ) AS rank
    FROM all_claims_3yr
    WHERE NPI IS NOT NULL
    GROUP BY PATIENT_ID, NPI
),

most_seen_combined_stats AS (
    SELECT
        ms3.PATIENT_ID,
        ms3.NPI,
        ms3.rank,
        ms3.visit_count_3yr,
        ms3.last_visit_3yr,
        COUNT(DISTINCT ac5.FILL_DATE) AS visit_count_5yr,
        MAX(ac5.FILL_DATE)          AS last_visit_5yr
    FROM most_seen_3yr_ranking ms3
    LEFT JOIN all_claims_5yr ac5
      ON ms3.PATIENT_ID = ac5.PATIENT_ID AND ms3.NPI = ac5.NPI
    WHERE ms3.rank <= 5
    GROUP BY ms3.PATIENT_ID, ms3.NPI, ms3.rank, ms3.visit_count_3yr, ms3.last_visit_3yr
),

/* ============================================================================
   6) HISTORICAL (ALL-TIME) FIRST DX / FIRST TX DATES (PATIENT LEVEL)
   ========================================================================== */

historical_first_dx AS (
    SELECT PATIENT_ID, MIN(FILL_DATE) AS incidence_date
    FROM (
        SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE DIAGNOSIS_CODES LIKE '%E761%'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, FILL_DATE
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE DIAGNOSIS_CODE = 'E761'
          AND TRANSACTION_STATUS = 'PAID'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE DIAGNOSIS_CODES LIKE '%E763%'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, FILL_DATE
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE DIAGNOSIS_CODE = 'E763'
          AND TRANSACTION_STATUS = 'PAID'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    ) all_dx
    GROUP BY PATIENT_ID
),

historical_first_tx AS (
    SELECT PATIENT_ID, MIN(FILL_DATE) AS first_incidence_treatment_date
    FROM (
        SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, FILL_DATE
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                                 '38206','38230','38232','38240','38241','38242','38243','38250')
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    ) all_tx
    GROUP BY PATIENT_ID
),

/* ============================================================================
   7) LATEST HCP ATTRIBUTION (5Y WINDOW)
   ========================================================================== */

all_claims_5yr_specialty_removed AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE DIAGNOSIS_CODE IN ('E761','E763')
        AND TRANSACTION_STATUS = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                               '38206','38230','38232','38240','38241','38242','38243','38250')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
),

latest_claim_hcp_ranked as (
  select patient_id, npi, fill_date, row_number() over(partition by patient_id order by fill_date desc, npi asc) as rn
  from all_claims_5yr
),

latest_claim_hcp as (
  select patient_id, npi as latest_claim_hcp_npi, fill_date as latest_claim_date from latest_claim_hcp_ranked where rn = 1
),

latest_claim_hcp_visit_count_5yr as (
  select a.patient_id, a.latest_claim_hcp_npi, count(distinct fill_date) as latest_claim_hcp_visit_count_5yr
  from latest_claim_hcp as a
  left join all_claims_5yr as b on a.patient_id = b.patient_id and a.latest_claim_hcp_npi = b.npi
  group by 1,2
),

latest_claim_hcp_final as (
  select a.patient_id, a.latest_claim_hcp_npi, a.latest_claim_date, b.latest_claim_hcp_visit_count_5yr
  from latest_claim_hcp as a
  left join latest_claim_hcp_visit_count_5yr as b on a.patient_id = b.patient_id
),

all_tx_claims_5yr_specialty_removed AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT
          PATIENT_ID,
          COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
          SERVICE_DATE AS FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          PRESCRIBER_NPI AS NPI,
          FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          RENDERING_NPI AS NPI,
          SERVICE_DATE AS FILL_DATE,
          PROCEDURE_CODE AS TX_CODE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                               '38206','38230','38232','38240','38241','38242','38243','38250')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
),

most_recent_tx_hcp_ranked as (
  select patient_id, npi, fill_date, row_number() over(partition by patient_id order by fill_date desc, npi asc) as rn
  from all_tx_claims_5yr
),

most_recent_tx_hcp as (
  select patient_id, npi as latest_treatment_hcp_npi, fill_date as latest_treatment_date
  from most_recent_tx_hcp_ranked where rn = 1
),

latest_treatment_hcp_visit_count as (
  select a.patient_id, a.latest_treatment_hcp_npi, count(distinct b.fill_date) as latest_treatment_hcp_visit_count_5yr
  from most_recent_tx_hcp as a
  left join all_claims_5yr as b on a.patient_id = b.patient_id and a.latest_treatment_hcp_npi = b.npi
  group by 1, 2
),

last_infusion_hcp_last_visit_cte as (
  select
      a.patient_id,
      a.latest_treatment_hcp_npi,
      max(b.fill_date) as last_infusion_hcp_last_visit
  from most_recent_tx_hcp as a
  left join all_claims_5yr as b
      on a.patient_id = b.patient_id
     and a.latest_treatment_hcp_npi = b.npi
  group by
      a.patient_id,
      a.latest_treatment_hcp_npi
),

most_recent_tx_hcp_final as (
  select a.patient_id, a.latest_treatment_hcp_npi, a.latest_treatment_date, b.latest_treatment_hcp_visit_count_5yr, c.last_infusion_hcp_last_visit
  from most_recent_tx_hcp as a
  left join latest_treatment_hcp_visit_count as b on a.patient_id = b.patient_id
  left join last_infusion_hcp_last_visit_cte as c
      on a.patient_id = c.patient_id
),

recent_2yr_treatment_claims AS (

    SELECT
        PATIENT_ID,
        SERVICE_DATE AS TX_DATE,
        'ELAPRASE' AS TX_TYPE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND SERVICE_DATE >= (SELECT elaprase_tx_start_date FROM global_runtime_dates)
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

    UNION ALL

    SELECT
        PATIENT_ID,
        FILL_DATE AS TX_DATE,
        'ELAPRASE' AS TX_TYPE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND TRANSACTION_RESULT = 'PAID'
      AND FILL_DATE >= (SELECT elaprase_tx_start_date FROM global_runtime_dates)
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

    UNION ALL

    SELECT
        PATIENT_ID,
        SERVICE_DATE AS TX_DATE,
        'ELAPRASE' AS TX_TYPE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PROCEDURE_CODE = 'J1743'
      AND SERVICE_DATE >= (SELECT elaprase_tx_start_date FROM global_runtime_dates)
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

    UNION ALL

    SELECT
        PATIENT_ID,
        SERVICE_DATE AS TX_DATE,
        'OTHER ERT' AS TX_TYPE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','S9357','S9379',
                             '38206','38230','38232','38240','38241','38242','38243','38250')
      AND SERVICE_DATE >= (SELECT elaprase_tx_start_date FROM global_runtime_dates)
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    UNION ALL

    SELECT
        PATIENT_ID,
        SERVICE_DATE AS TX_DATE,
        'AVLAYAH' AS TX_TYPE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE NDC11 IN ('8497600101')
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

    UNION ALL

    SELECT
        PATIENT_ID,
        FILL_DATE AS TX_DATE,
        'AVLAYAH' AS TX_TYPE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('8497600101')
      AND TRANSACTION_RESULT = 'PAID'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

    UNION ALL

    SELECT
        PATIENT_ID,
        SERVICE_DATE AS TX_DATE,
        'AVLAYAH' AS TX_TYPE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PROCEDURE_CODE IN ('J3490','J3590','J9999')
      AND SERVICE_DATE >= '2026-03-01'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
),

latest_mpsii_treatment_type AS (
    SELECT PATIENT_ID, latest_mpsii_tx_type
    FROM (
        SELECT
            PATIENT_ID,
            TX_TYPE AS latest_mpsii_tx_type,
            TX_DATE,
            ROW_NUMBER() OVER (
                PARTITION BY PATIENT_ID
                ORDER BY TX_DATE DESC
            ) AS rn
        FROM recent_2yr_treatment_claims
    ) t
    WHERE rn = 1
),
elaprase_latest_event AS (
    SELECT
        PATIENT_ID,
        MAX(FILL_DATE) AS latest_elaprase_date
    FROM all_tx_claims_5yr
    WHERE TX_CODE IN ('54092070001','540920700','J1743')
    GROUP BY PATIENT_ID
),
other_ert_latest_event AS (
    SELECT
        PATIENT_ID,
        MAX(FILL_DATE) AS latest_other_ert_date
    FROM all_tx_claims_5yr
    WHERE TX_CODE IN ('99601','99602','96365','96366','S9357','S9379',
                      '38206','38230','38232','38240','38241','38242','38243','38250')
    GROUP BY PATIENT_ID
),
avlayah_latest_event AS (
    SELECT
        PATIENT_ID,
        MAX(FILL_DATE) AS latest_avlayah_date
    FROM avlayah_ndc_events
    GROUP BY PATIENT_ID
),
latest_treatment_type AS (
    SELECT
        ep.PATIENT_ID,
        CASE
            -- Has Avlayah, AND has Elaprase strictly AFTER the latest Avlayah event -> ELAPRASE
            WHEN av.latest_avlayah_date IS NOT NULL
             AND el.latest_elaprase_date IS NOT NULL
             AND el.latest_elaprase_date > av.latest_avlayah_date
                THEN 'ELAPRASE'

            -- Has Avlayah (and no later Elaprase) -> AVLAYAH
            WHEN av.latest_avlayah_date IS NOT NULL
                THEN 'AVLAYAH'

            -- No Avlayah: latest by date between Elaprase and Other ERT
            WHEN el.latest_elaprase_date IS NOT NULL
             AND (
                    oe.latest_other_ert_date IS NULL
                    OR el.latest_elaprase_date >= oe.latest_other_ert_date
                 )
                THEN 'ELAPRASE'

            WHEN oe.latest_other_ert_date IS NOT NULL
                THEN 'OTHER ERT'

            ELSE lm.latest_mpsii_tx_type
        END AS latest_treatment_type
    FROM eligible_patients ep
    LEFT JOIN avlayah_latest_event av
        ON ep.PATIENT_ID = av.PATIENT_ID
    LEFT JOIN elaprase_latest_event el
        ON ep.PATIENT_ID = el.PATIENT_ID
    LEFT JOIN other_ert_latest_event oe
        ON ep.PATIENT_ID = oe.PATIENT_ID
    LEFT JOIN latest_mpsii_treatment_type lm
        ON ep.PATIENT_ID = lm.PATIENT_ID
),

/* ============================================================================
   10) FIRST TX AFTER DIAGNOSIS (ALL-TIME TX, BUT MUST BE AFTER DX)
   ========================================================================== */

first_tx_after_diagnosis AS (
  SELECT
    tx.patient_id,
    MIN(tx.fill_date) AS first_tx_after_diagnosis
  FROM all_tx_claims_alltime tx
  INNER JOIN historical_first_dx dx
    ON tx.patient_id = dx.patient_id
  WHERE tx.fill_date >= dx.incidence_date
  GROUP BY tx.patient_id
),

/* ============================================================================
   11) ELAPRASE FILL COUNTS (WINDOWED)
   ========================================================================== */

elaprase_fills AS (
  SELECT
    patient_id,
    COUNT(DISTINCT fill_date) AS elaprase_fills
  FROM all_tx_claims_5yr
  WHERE fill_date BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
    AND tx_code IN ('54092070001','540920700','J1743')
  GROUP BY patient_id
),

/* ============================================================================
   12) AVLAYAH STATUS LOGIC
   FIXED: avlayah_status_base now reads from avlayah_ndc_events (NDC-only
   confirmed Avlayah events from raw tables) for has_avlayah and
   first_avlayah_date. has_elaprase keeps original definition (any prior
   Elaprase/J1743/Other ERT proc) so that Switched fires on ANY conversion
   to Avlayah NDC, per business rule.
   ========================================================================== */

avlayah_status_base AS (
  SELECT
    ep.patient_id,
    av.first_avlayah_date,
    CASE WHEN av.first_avlayah_date IS NOT NULL THEN 1 ELSE 0 END AS has_avlayah,
    MAX(
      CASE
        WHEN tx.tx_code IN (
          '99601','99602','96365','96366','J1743','S9357','S9379',
          '38206','38230','38232','38240','38241','38242','38243','38250',
          '54092070001','540920700'
        )
        THEN 1 ELSE 0
      END
    ) AS has_elaprase
  FROM eligible_patients ep
  LEFT JOIN (
    SELECT patient_id, MIN(fill_date) AS first_avlayah_date
    FROM avlayah_ndc_events
    GROUP BY patient_id
  ) av ON ep.patient_id = av.patient_id
  LEFT JOIN all_tx_claims_5yr tx ON ep.patient_id = tx.patient_id
  GROUP BY ep.patient_id, av.first_avlayah_date
),

avlayah_status AS (
  SELECT
    patient_id,
    first_avlayah_date,
    has_avlayah,
    has_elaprase,
    CASE
      WHEN has_avlayah = 1 AND has_elaprase = 0 THEN 'New'
      WHEN has_avlayah = 1 AND has_elaprase = 1 THEN 'Switched'
      WHEN has_avlayah = 0 AND has_elaprase = 1 THEN 'Non-Avlayah'
      ELSE NULL
    END AS avlayah_status
  FROM avlayah_status_base
),

/* ============================================================================
   13) PATIENT DIMENSIONS
   ========================================================================== */

patient_demographics AS (
    SELECT *
    FROM (
        SELECT DISTINCT PATIENT_ID, PATIENT_YOB, PATIENT_GENDER,
               ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY PATIENT_YOB ASC) AS rn
        FROM com_edp_prd.com_raw.kom_patient_demographics
    )
    WHERE rn = 1
),
patient_geography AS (
    SELECT patient_id, patient_state
    FROM (
        SELECT
            PATIENT_ID,
            patient_state,
            ROW_NUMBER() OVER (
                PARTITION BY PATIENT_ID
                ORDER BY
                    CASE WHEN VALID_TO_DATE > CURRENT_DATE() THEN 1 ELSE 2 END,
                    VALID_TO_DATE DESC
            ) AS rn
        FROM COM_EDP_PRD.COM_RAW.KOM_PATIENT_GEOGRAPHY
    )
    WHERE rn = 1
),

/* ============================================================================
   14) PIVOT TOP-5 MOST-SEEN HCPs INTO WIDE FORMAT
   ========================================================================== */
most_seen_pivot AS (
    SELECT
        PATIENT_ID,

        MAX(CASE WHEN rank = 1 THEN NPI END)              AS most_seen_hcp1_3yr_ranked,
        MAX(CASE WHEN rank = 1 THEN visit_count_5yr END)  AS most_seen_hcp1_visit_count_5yr,
        MAX(CASE WHEN rank = 1 THEN last_visit_5yr END)   AS most_seen_hcp1_last_visit_5yr,

        MAX(CASE WHEN rank = 2 THEN NPI END)              AS most_seen_hcp2_3yr_ranked,
        MAX(CASE WHEN rank = 2 THEN visit_count_5yr END)  AS most_seen_hcp2_visit_count_5yr,
        MAX(CASE WHEN rank = 2 THEN last_visit_5yr END)   AS most_seen_hcp2_last_visit_5yr,

        MAX(CASE WHEN rank = 3 THEN NPI END)              AS most_seen_hcp3_3yr_ranked,
        MAX(CASE WHEN rank = 3 THEN visit_count_5yr END)  AS most_seen_hcp3_visit_count_5yr,
        MAX(CASE WHEN rank = 3 THEN last_visit_5yr END)   AS most_seen_hcp3_last_visit_5yr,

        MAX(CASE WHEN rank = 4 THEN NPI END)              AS most_seen_hcp4_3yr_ranked,
        MAX(CASE WHEN rank = 4 THEN visit_count_5yr END)  AS most_seen_hcp4_visit_count_5yr,
        MAX(CASE WHEN rank = 4 THEN last_visit_5yr END)   AS most_seen_hcp4_last_visit_5yr,

        MAX(CASE WHEN rank = 5 THEN NPI END)              AS most_seen_hcp5_3yr_ranked,
        MAX(CASE WHEN rank = 5 THEN visit_count_5yr END)  AS most_seen_hcp5_visit_count_5yr,
        MAX(CASE WHEN rank = 5 THEN last_visit_5yr END)   AS most_seen_hcp5_last_visit_5yr

    FROM most_seen_combined_stats
    GROUP BY PATIENT_ID
)

/* ============================================================================
   FINAL SELECT
   ========================================================================== */
SELECT
    ep.PATIENT_ID,
    pd.PATIENT_YOB,
    YEAR(CURRENT_DATE) - YEAR(pd.PATIENT_YOB) AS PATIENT_AGE,
    pd.PATIENT_GENDER,
    pg.patient_state,

    hfdx.incidence_date,
    hftx.first_incidence_treatment_date,

    lch.latest_claim_date AS latest_claim_date,

    lch.latest_claim_hcp_npi,
    pdlch.provider_name     AS latest_claim_hcp_name,
    pdlch.primary_specialty AS latest_claim_hcp_specialty,
    COALESCE(lch.latest_claim_hcp_visit_count_5yr, 0) AS latest_claim_hcp_visit_count,

    ref1.hco_name AS latest_claim_hcp_hco_name,

    mrt.latest_treatment_date AS latest_treatment_date,

    lmt.latest_mpsii_tx_type,
    ltt.latest_treatment_type,

    fta.first_tx_after_diagnosis,

    ROUND(MONTHS_BETWEEN(fta.first_tx_after_diagnosis, hfdx.incidence_date), 0)
      AS time_dx_to_first_tx_in_months,

    ROUND(MONTHS_BETWEEN(mrt.latest_treatment_date, fta.first_tx_after_diagnosis), 0) AS treatment_period_months,

    COALESCE(ef.elaprase_fills, 0) AS elaprase_fills,
    avs.avlayah_status,

    CASE
      WHEN avs.avlayah_status = 'Switched'
      THEN avs.first_avlayah_date
    END AS avlayah_switch_date,
    mrt.latest_treatment_hcp_npi,
    pdtch.provider_name     AS latest_treatment_hcp_name,
    pdtch.primary_specialty AS latest_treatment_hcp_specialty,
    COALESCE(mrt.latest_treatment_hcp_visit_count_5yr, 0) AS latest_treatment_hcp_visit_count,
    mrt.last_infusion_hcp_last_visit,
    ref2.hco_name AS latest_treatment_hcp_hco_name,

    fdh.first_dx_hcp AS first_dx_hcp_5yr,
    COALESCE(fdhs.first_dx_all_visit_count_5yr, 0) AS first_dx_all_visit_count_5yr,
    fdhs.first_dx_last_visit_5yr AS first_dx_last_visit_5yr,

    fth.first_tx_hcp AS first_tx_hcp_5yr,
    COALESCE(fths.first_tx_all_visit_count_5yr, 0) AS first_tx_all_visit_count_5yr,
    COALESCE(fthtx.first_tx_treatment_visit_count_5yr, 0) AS first_tx_treatment_visit_count_5yr,
    fths.first_tx_last_visit_5yr AS first_tx_last_visit_5yr,

    msp.most_seen_hcp1_3yr_ranked,
    COALESCE(msp.most_seen_hcp1_visit_count_5yr, 0) AS most_seen_hcp1_visit_count_5yr,
    msp.most_seen_hcp1_last_visit_5yr,

    msp.most_seen_hcp2_3yr_ranked,
    COALESCE(msp.most_seen_hcp2_visit_count_5yr, 0) AS most_seen_hcp2_visit_count_5yr,
    msp.most_seen_hcp2_last_visit_5yr,

    msp.most_seen_hcp3_3yr_ranked,
    COALESCE(msp.most_seen_hcp3_visit_count_5yr, 0) AS most_seen_hcp3_visit_count_5yr,
    msp.most_seen_hcp3_last_visit_5yr,

    msp.most_seen_hcp4_3yr_ranked,
    COALESCE(msp.most_seen_hcp4_visit_count_5yr, 0) AS most_seen_hcp4_visit_count_5yr,
    msp.most_seen_hcp4_last_visit_5yr,

    msp.most_seen_hcp5_3yr_ranked,
    COALESCE(msp.most_seen_hcp5_visit_count_5yr, 0) AS most_seen_hcp5_visit_count_5yr,
    msp.most_seen_hcp5_last_visit_5yr

FROM eligible_patients ep
LEFT JOIN patient_demographics pd
    ON ep.PATIENT_ID = pd.PATIENT_ID
LEFT JOIN patient_geography pg
    ON ep.PATIENT_ID = pg.PATIENT_ID

LEFT JOIN historical_first_dx hfdx
    ON ep.PATIENT_ID = hfdx.PATIENT_ID
LEFT JOIN historical_first_tx hftx
    ON ep.PATIENT_ID = hftx.PATIENT_ID

LEFT JOIN first_tx_after_diagnosis fta
    ON ep.PATIENT_ID = fta.PATIENT_ID

LEFT JOIN elaprase_fills ef
    ON ep.PATIENT_ID = ef.PATIENT_ID

LEFT JOIN latest_mpsii_treatment_type lmt
    ON ep.PATIENT_ID = lmt.PATIENT_ID

LEFT JOIN latest_treatment_type ltt
    ON ep.PATIENT_ID = ltt.PATIENT_ID

LEFT JOIN latest_claim_hcp_final lch
    ON ep.PATIENT_ID = lch.PATIENT_ID
LEFT JOIN provider_dim pdlch
    ON lch.latest_claim_hcp_npi = pdlch.npi
LEFT JOIN cmpa_insights_internal_schema.reference_file ref1
    ON lch.latest_claim_hcp_npi = ref1.hcp_npi

LEFT JOIN most_recent_tx_hcp_final mrt
    ON ep.PATIENT_ID = mrt.PATIENT_ID
LEFT JOIN provider_dim pdtch
    ON mrt.latest_treatment_hcp_npi = pdtch.npi
LEFT JOIN cmpa_insights_internal_schema.reference_file ref2
    ON mrt.latest_treatment_hcp_npi = ref2.hcp_npi

LEFT JOIN first_dx_hcp fdh
    ON ep.PATIENT_ID = fdh.PATIENT_ID
LEFT JOIN first_dx_hcp_5yr_stats fdhs
    ON ep.PATIENT_ID = fdhs.PATIENT_ID
LEFT JOIN first_tx_hcp fth
    ON ep.PATIENT_ID = fth.PATIENT_ID
LEFT JOIN first_tx_hcp_5yr_stats fths
    ON ep.PATIENT_ID = fths.PATIENT_ID
LEFT JOIN first_tx_hcp_5yr_tx_only fthtx
    ON ep.PATIENT_ID = fthtx.PATIENT_ID

LEFT JOIN most_seen_pivot msp
    ON ep.PATIENT_ID = msp.PATIENT_ID
LEFT JOIN avlayah_status avs
  ON ep.PATIENT_ID = avs.patient_id
ORDER BY ep.PATIENT_ID;

-- Materialize the temp view into the persistent base table.
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_base AS
SELECT DISTINCT * FROM patient_hcp_visit_summary;


-- =============================================================================
-- CELL 2: mpsii_tx_claims (UNCHANGED)
-- =============================================================================
CREATE OR REPLACE TEMP VIEW mpsii_tx_claims AS
WITH MPSII_Diagnoses_Specified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_RESULT = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
),

Patients_2Dx_Specified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Specified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

MPSII_Diagnoses_Unspecified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_RESULT = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
),

Patients_2Dx_Unspecified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Unspecified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

MPSII_Treatment_All AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                                 '38206','38230','38232','38240','38241','38242','38243','38250')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
    ) t
),

MPSII_Treatment_Elaprase_Only AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE = 'J1743'
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
    ) t
),

Patients_2Dx_Specified_With_Treatment AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Specified p
    INNER JOIN MPSII_Treatment_All t USING (PATIENT_ID)
),

Patients_Incremental_Unspecified AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Unspecified p
    INNER JOIN MPSII_Treatment_Elaprase_Only t USING (PATIENT_ID)
    WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
),

eligible_patients AS (
    SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment
    UNION
    SELECT PATIENT_ID FROM Patients_Incremental_Unspecified
),

cohort_3_learnings AS (
  SELECT DISTINCT npi
  FROM com_raw.kom_providers
  WHERE provider_type = 'INDIVIDUAL'
    AND (
      PRIMARY_SPECIALTY NOT IN (
        'Anesthesiologist Assistant','Anesthesiology','Dentist','Dietitian, Registered',
        'Emergency Medical Technician, Basic','Emergency Medicine','General Acute Care Hospital',
        'Nurse Anesthetist, Certified Registered','Obstetrics & Gynecology','Pathology',
        'Radiology','Urology'
      )
      OR SECONDARY_SPECIALTY IN (
        'Child & Adolescent Psychiatry','Psychiatry','Adolescent Medicine','Developmental - Behavioral Pediatrics',
        'Neonatal-Perinatal Medicine','Nutrition, Pediatric','Oncology, Pediatrics','Pediatric Cardiology',
        'Pediatric Critical Care Medicine','Pediatric Dermatology','Pediatric Emergency Medicine',
        'Pediatric Endocrinology','Pediatric Gastroenterology','Pediatric Hematology-Oncology',
        'Pediatric Infectious Diseases','Pediatric Nephrology','Pediatric Ophthalmology and Strabismus Specialist',
        'Pediatric Orthopaedic Surgery','Pediatric Otolaryngology','Pediatric Pulmonology','Pediatric Radiology',
        'Pediatric Rehabilitation Medicine','Pediatric Rheumatology','Pediatric Surgery','Pediatrics',
        'Clinical Biochemical Genetics','Clinical Genetics (M.D.)','Clinical Molecular Genetics',
        'Ph.D. Medical Genetics','Neurodevelopmental Disabilities','Neurology',
        'Neurology with Special Qualifications in Child Neurology','Neuroradiology'
      )
    )
),

all_tx_claims_2yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
        PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
        PATIENT_ID,
        RENDERING_NPI AS NPI,
        SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                               '38206','38230','38232','38240','38241','38242','38243','38250')
        AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
)

SELECT * FROM all_tx_claims_2yr;


-- =============================================================================
-- CELL 3: most_recently_treated_hcp (UNCHANGED)
-- =============================================================================
CREATE OR REPLACE TEMPORARY VIEW most_recently_treated_hcp AS
WITH tx_claims AS (
    SELECT DISTINCT *
    FROM mpsii_tx_claims
),

MPSII_Diagnoses_Specified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_RESULT = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
),

Patients_2Dx_Specified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Specified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

MPSII_Diagnoses_Unspecified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_RESULT = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
),

Patients_2Dx_Unspecified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Unspecified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

MPSII_Treatment_All AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                                 '38206','38230','38232','38240','38241','38242','38243','38250')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
    ) t
),

MPSII_Treatment_Elaprase_Only AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE = 'J1743'
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
    ) t
),

Patients_2Dx_Specified_With_Treatment AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Specified p
    INNER JOIN MPSII_Treatment_All t USING (PATIENT_ID)
),

Patients_Incremental_Unspecified AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Unspecified p
    INNER JOIN MPSII_Treatment_Elaprase_Only t USING (PATIENT_ID)
    WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
),

eligible_patients AS (
    SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment
    UNION
    SELECT PATIENT_ID FROM Patients_Incremental_Unspecified
),

cohort_3_learnings AS (
  SELECT DISTINCT npi
  FROM com_raw.kom_providers
  WHERE provider_type = 'INDIVIDUAL'
    AND (
      PRIMARY_SPECIALTY NOT IN (
        'Anesthesiologist Assistant','Anesthesiology','Dentist','Dietitian, Registered',
        'Emergency Medical Technician, Basic','Emergency Medicine','General Acute Care Hospital',
        'Nurse Anesthetist, Certified Registered','Obstetrics & Gynecology','Pathology',
        'Radiology','Urology'
      )
      OR SECONDARY_SPECIALTY IN (
        'Child & Adolescent Psychiatry','Psychiatry','Adolescent Medicine','Developmental - Behavioral Pediatrics',
        'Neonatal-Perinatal Medicine','Nutrition, Pediatric','Oncology, Pediatrics','Pediatric Cardiology',
        'Pediatric Critical Care Medicine','Pediatric Dermatology','Pediatric Emergency Medicine',
        'Pediatric Endocrinology','Pediatric Gastroenterology','Pediatric Hematology-Oncology',
        'Pediatric Infectious Diseases','Pediatric Nephrology','Pediatric Ophthalmology and Strabismus Specialist',
        'Pediatric Orthopaedic Surgery','Pediatric Otolaryngology','Pediatric Pulmonology','Pediatric Radiology',
        'Pediatric Rehabilitation Medicine','Pediatric Rheumatology','Pediatric Surgery','Pediatrics',
        'Clinical Biochemical Genetics','Clinical Genetics (M.D.)','Clinical Molecular Genetics',
        'Ph.D. Medical Genetics','Neurodevelopmental Disabilities','Neurology',
        'Neurology with Special Qualifications in Child Neurology','Neuroradiology'
      )
    )
),

all_dx_claims_5yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT
        PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE DIAGNOSIS_CODE IN ('E761','E763')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

all_tx_claims_5yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT
        PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT
        PATIENT_ID,
        RENDERING_NPI AS NPI,
        SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                               '38206','38230','38232','38240','38241','38242','38243','38250')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

all_claims AS (
    SELECT * FROM all_dx_claims_5yr
    UNION
    SELECT * FROM all_tx_claims_5yr
),

latest_treating_hcp AS (
    SELECT patient_id, npi
    FROM (
        SELECT
            patient_id,
            npi,
            fill_date,
            ROW_NUMBER() OVER (
                PARTITION BY patient_id
                ORDER BY
                    CASE WHEN npi IS NOT NULL THEN 1 ELSE 2 END,
                    fill_date DESC,
                    npi DESC
            ) AS rn
        FROM tx_claims
    ) t
    WHERE rn = 1
      AND npi IS NOT NULL
),

visit_counts AS (
    SELECT
        patient_id,
        npi,
        COUNT(DISTINCT fill_date) AS visit_counts
    FROM all_claims
    WHERE npi IS NOT NULL
    GROUP BY patient_id, npi
),

last_visit_date AS (
    SELECT
        lth.patient_id,
        lth.npi,
        MAX(ac.fill_date) AS last_visit_date
    FROM latest_treating_hcp lth
    LEFT JOIN all_claims ac
      ON lth.patient_id = ac.patient_id
     AND lth.npi = ac.npi
    GROUP BY lth.patient_id, lth.npi
),

latest_treating_hcp_with_visits AS (
    SELECT
        a.patient_id,
        a.npi AS most_recently_treated_hcp,
        b.visit_counts AS no_of_visits,
        c.last_visit_date AS last_visit_date_5yr
    FROM latest_treating_hcp AS a
    LEFT JOIN visit_counts AS b
        ON a.patient_id = b.patient_id
       AND a.npi        = b.npi
    LEFT JOIN last_visit_date AS c
        ON a.patient_id = c.patient_id
       AND a.npi        = c.npi
),

hcp_with_other_info AS (
    SELECT
        a.*,
        CONCAT(b.FIRST_NAME, ' ', b.LAST_NAME) AS hcp_name,
        b.PRIMARY_SPECIALTY AS hcp_specialty,
        c.hco_name,
        c.territory_id,
        c.territory,
        c.region_id,
        c.region
    FROM latest_treating_hcp_with_visits AS a

    LEFT JOIN com_edp_prd.com_raw.kom_providers AS b
        ON a.most_recently_treated_hcp = b.NPI
       AND b.PROVIDER_TYPE = 'INDIVIDUAL'

    LEFT JOIN (
  SELECT
    a.hcp_npi,
    a.hco_name,
    a.territory,
    a.region,
    b.territory_id,
    c.region_id,
    a.hcp_primary_specialty AS hcp_specialty
  FROM cmpa_insights_internal_schema.reference_file a
  LEFT JOIN (
      SELECT DISTINCT territory_id, territory_name
      FROM cmpa_insights_internal_schema.zip_to_territory_mapping
  ) b
    ON a.territory = b.territory_name
  LEFT JOIN (
      SELECT DISTINCT region_id, region_name
      FROM cmpa_insights_internal_schema.zip_to_territory_mapping
  ) c
    ON a.region = c.region_name
) c
ON a.most_recently_treated_hcp = c.hcp_npi)

SELECT
    patient_id,
    most_recently_treated_hcp AS most_recently_treated_hcp_2yr,
    hcp_name AS most_recently_treated_hcp_name_2yr,

    no_of_visits AS most_recently_treated_hcp_2yr_no_of_visits_5yr,
    last_visit_date_5yr AS most_recent_tx_hcp_2yr_last_visit_5yr,

    hcp_specialty AS most_recently_treated_hcp_specialty_2yr,
    hco_name AS most_recently_treated_hcp_hco_name,
    territory_id AS most_recently_treated_hcp_territory_id_2yr,
    territory AS most_recently_treated_hcp_territory_2yr,
    region_id AS most_recently_treated_hcp_region_id_2yr,
    region AS most_recently_treated_hcp_region_2yr
    FROM hcp_with_other_info;


-- =============================================================================
-- CELL 4: Build all_dx_claims, all_tx_claims views (UNCHANGED)
-- =============================================================================
CREATE OR REPLACE TEMPORARY VIEW all_dx_claims AS

SELECT DISTINCT
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'DX' AS CLAIM_TYPE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)

UNION

SELECT DISTINCT
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    'DX' AS CLAIM_TYPE
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE DIAGNOSIS_CODE IN ('E761', 'E763')
  AND TRANSACTION_RESULT = 'PAID'
  AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters);


CREATE OR REPLACE TEMPORARY VIEW all_tx_claims AS

SELECT DISTINCT
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'TX' AS CLAIM_TYPE,
    NDC11 AS CODE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE NDC11 IN ('54092070001', '540920700')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)

UNION

SELECT DISTINCT
    PATIENT_ID,
    RENDERING_NPI AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'TX' AS CLAIM_TYPE,
    PROCEDURE_CODE AS CODE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE PROCEDURE_CODE IN ('99601', '99602', '96365', '96366', 'J1743',
                         'S9357', 'S9379', '38206', '38230', '38232',
                         '38240', '38241', '38242', '38243', '38250')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)

UNION

SELECT DISTINCT
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    'TX' AS CLAIM_TYPE,
    NDC11 AS CODE
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE NDC11 IN ('54092070001', '540920700')
  AND TRANSACTION_RESULT = 'PAID'
  AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters);


CREATE OR REPLACE TEMPORARY VIEW tx_claims_2yr AS
SELECT DISTINCT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE, CODE
FROM all_tx_claims
WHERE FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters);


CREATE OR REPLACE TEMPORARY VIEW e761_patients_2dx AS
SELECT PATIENT_ID
FROM (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_RESULT = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;

CREATE OR REPLACE TEMPORARY VIEW specified_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e761_patients_2dx e
INNER JOIN tx_claims_2yr t ON e.PATIENT_ID = t.PATIENT_ID;

CREATE OR REPLACE TEMPORARY VIEW e763_patients_2dx AS
SELECT PATIENT_ID
FROM (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_RESULT = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;

CREATE OR REPLACE TEMPORARY VIEW elaprase_tx_2yr AS
SELECT DISTINCT PATIENT_ID
FROM tx_claims_2yr
WHERE CODE IN ('54092070001', '540920700', 'J1743');

CREATE OR REPLACE TEMPORARY VIEW incremental_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e763_patients_2dx e
INNER JOIN elaprase_tx_2yr t ON e.PATIENT_ID = t.PATIENT_ID
WHERE e.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM specified_patients);

CREATE OR REPLACE TEMPORARY VIEW eligible_patients AS
SELECT PATIENT_ID FROM specified_patients
UNION
SELECT PATIENT_ID FROM incremental_patients;


CREATE OR REPLACE TEMPORARY VIEW cohort_3_learnings AS
SELECT DISTINCT npi
FROM com_raw.kom_providers
WHERE provider_type = 'INDIVIDUAL'
  AND (
    PRIMARY_SPECIALTY NOT IN (
      'Anesthesiologist Assistant','Anesthesiology','Dentist','Dietitian, Registered',
      'Emergency Medical Technician, Basic','Emergency Medicine','General Acute Care Hospital',
      'Nurse Anesthetist, Certified Registered','Obstetrics & Gynecology','Pathology',
      'Radiology','Urology'
    )
    OR SECONDARY_SPECIALTY IN (
      'Child & Adolescent Psychiatry','Psychiatry','Adolescent Medicine','Developmental - Behavioral Pediatrics',
      'Neonatal-Perinatal Medicine','Nutrition, Pediatric','Oncology, Pediatrics','Pediatric Cardiology',
      'Pediatric Critical Care Medicine','Pediatric Dermatology','Pediatric Emergency Medicine',
      'Pediatric Endocrinology','Pediatric Gastroenterology','Pediatric Hematology-Oncology',
      'Pediatric Infectious Diseases','Pediatric Nephrology','Pediatric Ophthalmology and Strabismus Specialist',
      'Pediatric Orthopaedic Surgery','Pediatric Otolaryngology','Pediatric Pulmonology','Pediatric Radiology',
      'Pediatric Rehabilitation Medicine','Pediatric Rheumatology','Pediatric Surgery','Pediatrics',
      'Clinical Biochemical Genetics','Clinical Genetics (M.D.)','Clinical Molecular Genetics',
      'Ph.D. Medical Genetics','Neurodevelopmental Disabilities','Neurology',
      'Neurology with Special Qualifications in Child Neurology','Neuroradiology'
    )
  );


CREATE OR REPLACE TEMPORARY VIEW all_patient_claims AS
SELECT *
FROM (
  SELECT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
  FROM all_dx_claims
  WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

  UNION

  SELECT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
  FROM all_tx_claims
  WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
)
WHERE npi IN (SELECT DISTINCT npi FROM cohort_3_learnings);


CREATE OR REPLACE TEMPORARY VIEW primary_hcp AS
WITH hcp_metrics AS (
    SELECT
        a.PATIENT_ID,
        a.NPI,

        CASE
            WHEN p.primary_specialty LIKE '%Genetic%'
              OR p.secondary_specialty LIKE '%Genetic%'
                THEN 'Geneticist'
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
              OR p.primary_specialty LIKE '%Neurological Surgery%'
                THEN 'Psychiatry & Neurology'
            WHEN p.primary_specialty LIKE '%Pediatrics%'
                THEN 'Pediatrician'
            WHEN p.primary_specialty LIKE '%Internal Medicine%'
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%'
              OR p.secondary_specialty LIKE '%Family Medicine%'
                THEN 'PCP'
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
              OR p.primary_specialty LIKE '%Physician Assistant%'
                THEN 'NPPA'
            WHEN a.NPI IS NULL
                THEN 'NA'
            ELSE 'Others'
        END AS SPECIALTY,

        CASE
            WHEN p.primary_specialty LIKE '%Genetic%'
              OR p.secondary_specialty LIKE '%Genetic%'
                THEN 1
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
              OR p.primary_specialty LIKE '%Neurological Surgery%'
                THEN 2
            WHEN p.primary_specialty LIKE '%Pediatrics%'
                THEN 3
            WHEN p.primary_specialty LIKE '%Internal Medicine%'
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%'
              OR p.secondary_specialty LIKE '%Family Medicine%'
                THEN 4
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
              OR p.primary_specialty LIKE '%Physician Assistant%'
                THEN 5
            WHEN a.NPI IS NULL
                THEN 7
            ELSE 6
        END AS SPECIALTY_PRIORITY,

        COUNT(DISTINCT a.FILL_DATE) AS NO_OF_VISITS,

        COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'DX' THEN a.FILL_DATE END) AS DX_VISITS,
        COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'TX' THEN a.FILL_DATE END) AS TX_VISITS,

        MAX(a.FILL_DATE) AS MOST_RECENT_VISIT

    FROM all_patient_claims a
    LEFT JOIN com_edp_prd.com_raw.kom_providers p
        ON a.NPI = p.NPI
    GROUP BY
        a.PATIENT_ID,
        a.NPI,
        p.primary_specialty,
        p.secondary_specialty
),
ranked_hcps AS (
    SELECT
        *,
        RANK() OVER (
            PARTITION BY PATIENT_ID
            ORDER BY
                SPECIALTY_PRIORITY ASC,
                NO_OF_VISITS DESC,
                MOST_RECENT_VISIT DESC,
                NPI ASC
        ) AS HCP_RANK
    FROM hcp_metrics
)
SELECT
    PATIENT_ID,
    NPI AS PRIMARY_HCP_NPI,
    SPECIALTY AS PRIMARY_HCP_SPECIALTY,
    SPECIALTY_PRIORITY,
    NO_OF_VISITS,
    DX_VISITS,
    TX_VISITS,
    MOST_RECENT_VISIT,
    HCP_RANK
FROM ranked_hcps
WHERE HCP_RANK = 1;


CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.primary_hcp AS
SELECT
    ph.*,
    COALESCE(p.FIRST_NAME, '') || ' ' || COALESCE(p.LAST_NAME, '') AS primary_hcp_name_2yr,

    ref.HCO_NAME AS primary_hcp_hco_name_2yr,
    ref.HCO_CITY AS primary_hcp_hco_city_2yr,
    ref.HCO_STATE AS primary_hcp_hco_state_2yr,

    ref.mapped_territory_id AS primary_hcp_territory_id_2yr,
    ref.TERRITORY           AS primary_hcp_territory_2yr,
    ref.mapped_region_id    AS primary_hcp_region_id_2yr,
    ref.region              AS primary_hcp_region_2yr

FROM primary_hcp ph

LEFT JOIN com_edp_prd.com_raw.kom_providers p
    ON ph.PRIMARY_HCP_NPI = p.NPI


LEFT JOIN (
  SELECT
    a.hcp_npi,
    a.hco_name,
    a.hco_city,
    a.hco_state,
    a.territory,
    a.region,
    b.territory_id AS mapped_territory_id,
    c.region_id AS mapped_region_id,
    a.hcp_primary_specialty AS hcp_specialty
  FROM cmpa_insights_internal_schema.reference_file a
  LEFT JOIN (
      SELECT DISTINCT try_cast(territory_id AS BIGINT) AS territory_id, territory_name
      FROM cmpa_insights_internal_schema.zip_to_territory_mapping
  ) b
    ON try_cast(a.territory_id AS BIGINT) = b.territory_id
  LEFT JOIN (
      SELECT DISTINCT try_cast(region_id AS BIGINT) AS region_id, region_name
      FROM cmpa_insights_internal_schema.zip_to_territory_mapping
  ) c
    ON try_cast(a.region_id AS BIGINT) = c.region_id
) ref
ON ph.PRIMARY_HCP_NPI = ref.hcp_npi;


-- =============================================================================
-- CELL 5: patient360_master initial join (UNCHANGED)
-- =============================================================================
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master AS
SELECT DISTINCT
    a.PATIENT_ID,
    a.PATIENT_YOB,
    a.PATIENT_AGE,
    a.PATIENT_GENDER,
    a.patient_state,

    a.incidence_date,
    a.first_incidence_treatment_date,
    a.latest_claim_date,

    a.latest_claim_hcp_npi,
    a.latest_claim_hcp_name,
    a.latest_claim_hcp_specialty,
    a.latest_claim_hcp_visit_count,
    a.latest_claim_hcp_hco_name,

    a.latest_treatment_date,
    a.latest_mpsii_tx_type,
    a.latest_treatment_type,
    a.first_tx_after_diagnosis,
    a.time_dx_to_first_tx_in_months,
    a.treatment_period_months,
    a.elaprase_fills,
    a.avlayah_status,
    a.avlayah_switch_date,
    a.latest_treatment_hcp_npi,
    a.latest_treatment_hcp_name,
    a.latest_treatment_hcp_specialty,
    a.latest_treatment_hcp_visit_count,
    a.last_infusion_hcp_last_visit,
    a.latest_treatment_hcp_hco_name,

    a.first_dx_hcp_5yr,
    a.first_dx_all_visit_count_5yr,
    a.first_dx_last_visit_5yr,
    a.first_tx_hcp_5yr,
    a.first_tx_all_visit_count_5yr,
    a.first_tx_treatment_visit_count_5yr,
    a.first_tx_last_visit_5yr,

    a.most_seen_hcp1_3yr_ranked,
    a.most_seen_hcp1_visit_count_5yr,
    a.most_seen_hcp1_last_visit_5yr,
    a.most_seen_hcp2_3yr_ranked,
    a.most_seen_hcp2_visit_count_5yr,
    a.most_seen_hcp2_last_visit_5yr,
    a.most_seen_hcp3_3yr_ranked,
    a.most_seen_hcp3_visit_count_5yr,
    a.most_seen_hcp3_last_visit_5yr,
    a.most_seen_hcp4_3yr_ranked,
    a.most_seen_hcp4_visit_count_5yr,
    a.most_seen_hcp4_last_visit_5yr,
    a.most_seen_hcp5_3yr_ranked,
    a.most_seen_hcp5_visit_count_5yr,
    a.most_seen_hcp5_last_visit_5yr,

    b.* EXCEPT (patient_id),

    c.* EXCEPT (patient_id)

FROM com_edp_prd.cmpa_insights_internal_schema.patient360_base AS a

LEFT JOIN most_recently_treated_hcp AS b
    ON a.PATIENT_ID = b.patient_id

LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.primary_hcp AS c
    ON a.PATIENT_ID = c.patient_id;


-- =============================================================================
-- CELL 6: severity flagging (UNCHANGED)
-- =============================================================================
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master AS
WITh  base AS (
  SELECT DISTINCT
    a.PATIENT_ID,
    b.SERVICE_DATE,
    b.DIAGNOSIS_CODES
  FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master a
  LEFT JOIN com_edp_prd.com_raw.kom_medical_events b
    ON a.PATIENT_ID = b.PATIENT_ID
   AND b.SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
),

flagged AS (
  SELECT
    PATIENT_ID,
    SERVICE_DATE,
    CASE
      WHEN DIAGNOSIS_CODES IS NOT NULL AND (
           DIAGNOSIS_CODES ILIKE '%|G910|%' OR DIAGNOSIS_CODES ILIKE '%|G911|%' OR DIAGNOSIS_CODES ILIKE '%|G912|%'
        OR DIAGNOSIS_CODES ILIKE '%|G913|%' OR DIAGNOSIS_CODES ILIKE '%|G914|%' OR DIAGNOSIS_CODES ILIKE '%|G918|%'
        OR DIAGNOSIS_CODES ILIKE '%|G919|%' OR DIAGNOSIS_CODES ILIKE '%|Q038|%' OR DIAGNOSIS_CODES ILIKE '%|Q039|%'
        OR DIAGNOSIS_CODES ILIKE '%|Q050|%' OR DIAGNOSIS_CODES ILIKE '%|Q051|%' OR DIAGNOSIS_CODES ILIKE '%|Q052|%'
        OR DIAGNOSIS_CODES ILIKE '%|Q053|%' OR DIAGNOSIS_CODES ILIKE '%|Q054|%' OR DIAGNOSIS_CODES ILIKE '%|Q055|%'
        OR DIAGNOSIS_CODES ILIKE '%|Q056|%' OR DIAGNOSIS_CODES ILIKE '%|Q057|%' OR DIAGNOSIS_CODES ILIKE '%|Q058|%'
        OR DIAGNOSIS_CODES ILIKE '%|Q0700|%' OR DIAGNOSIS_CODES ILIKE '%|Q0702|%' OR DIAGNOSIS_CODES ILIKE '%|Q0703|%'
        OR DIAGNOSIS_CODES ILIKE '%|F445|%' OR DIAGNOSIS_CODES ILIKE '%|F639|%' OR DIAGNOSIS_CODES ILIKE '%|F70|%'
        OR DIAGNOSIS_CODES ILIKE '%|F71|%' OR DIAGNOSIS_CODES ILIKE '%|F72|%' OR DIAGNOSIS_CODES ILIKE '%|F73|%'
        OR DIAGNOSIS_CODES ILIKE '%|F78|%' OR DIAGNOSIS_CODES ILIKE '%|F78A1|%' OR DIAGNOSIS_CODES ILIKE '%|F78A9|%'
        OR DIAGNOSIS_CODES ILIKE '%|F79|%' OR DIAGNOSIS_CODES ILIKE '%|F800|%' OR DIAGNOSIS_CODES ILIKE '%|F801|%'
        OR DIAGNOSIS_CODES ILIKE '%|F802|%' OR DIAGNOSIS_CODES ILIKE '%|F804|%' OR DIAGNOSIS_CODES ILIKE '%|F8081|%'
        OR DIAGNOSIS_CODES ILIKE '%|F8082|%' OR DIAGNOSIS_CODES ILIKE '%|F8089|%' OR DIAGNOSIS_CODES ILIKE '%|F809|%'
        OR DIAGNOSIS_CODES ILIKE '%|F810|%' OR DIAGNOSIS_CODES ILIKE '%|F812|%' OR DIAGNOSIS_CODES ILIKE '%|F8181|%'
        OR DIAGNOSIS_CODES ILIKE '%|F8189|%' OR DIAGNOSIS_CODES ILIKE '%|F819|%' OR DIAGNOSIS_CODES ILIKE '%|F82|%'
        OR DIAGNOSIS_CODES ILIKE '%|F840|%' OR DIAGNOSIS_CODES ILIKE '%|F843|%' OR DIAGNOSIS_CODES ILIKE '%|F845|%'
        OR DIAGNOSIS_CODES ILIKE '%|F848|%' OR DIAGNOSIS_CODES ILIKE '%|F849|%' OR DIAGNOSIS_CODES ILIKE '%|F88|%'
        OR DIAGNOSIS_CODES ILIKE '%|F89|%' OR DIAGNOSIS_CODES ILIKE '%|R6250|%' OR DIAGNOSIS_CODES ILIKE '%|R620|%'
        OR DIAGNOSIS_CODES ILIKE '%|R6251|%' OR DIAGNOSIS_CODES ILIKE '%|R6259|%' OR DIAGNOSIS_CODES ILIKE '%|R62|%'
      )
      THEN 1 ELSE 0
    END AS has_severity_code
  FROM base
),

patient_with_severity_outcome AS (
  SELECT
    PATIENT_ID,
    COUNT(DISTINCT SERVICE_DATE) AS count_fill_date,
    COUNT(DISTINCT CASE
                     WHEN has_severity_code = 1
                     THEN SERVICE_DATE
                   END) AS severity_dx_distinct_dates,
    CASE
      WHEN COUNT(DISTINCT CASE
                            WHEN has_severity_code = 1
                            THEN SERVICE_DATE
                          END) >= 2
      THEN 'Severe'
      ELSE 'Attenuated'
    END AS severity
  FROM flagged
  GROUP BY PATIENT_ID
  ORDER BY severity_dx_distinct_dates DESC, PATIENT_ID
)

SELECT
  a.*,
  b.severity
FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master AS a
LEFT JOIN patient_with_severity_outcome AS b
  ON a.PATIENT_ID = b.patient_id;


-- =============================================================================
-- CELL 7: Comorbidity flagging (UNCHANGED)
-- =============================================================================
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master AS
WITH base_patients AS (
    SELECT DISTINCT patient_id
    FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
),

exploded_codes AS (
    SELECT
        m.patient_id,
        code
    FROM com_edp_prd.com_raw.kom_medical_events m
    INNER JOIN base_patients bp
        ON m.patient_id = bp.patient_id
    LATERAL VIEW explode(split(m.DIAGNOSIS_CODES, '\\|')) s AS code
    WHERE m.SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    AND m.DIAGNOSIS_CODES IS NOT NULL
),

comorbidity_raw AS (
    SELECT
        patient_id,
        CASE
            WHEN code IN (
                'J45909','J4520','J4530','J4540','J4541','J4531','J45901','J45998',
                'J4521','J4542','J45990','J4550','J45902','J4551','J4532','J45991'
            ) THEN 'Asthma'

            WHEN code IN ('G4733','G4730','G479','G4731','G4739')
                THEN 'Sleep Apnea'

            WHEN code IN (
                'J988','J300','J301','J302','J305','J3081','J3089','J309','J310',
                'J311','J312','J320','J321','J322','J323','J324','J328','J329',
                'J330','J331','J338','J339','J340','J341','J342','J343','J3481',
                'J348200','J348201','J348202','J348210','J348211','J348212',
                'J34829','J3489','J349','J3501','J3502','J3503','J351','J352',
                'J353','J358','J359','J36','J370','J371','J3800','J3801','J3802',
                'J381','J382','J383','J384','J385','J386','J387','J390','J391',
                'J392','J393','J398','J399'
            ) THEN 'Other diseases of upper respiratory tract'

            WHEN code IN (
                'G4700','G4734','G4761','G478','G4710','G4701','G4736','G4720',
                'G4709','G4719','G4721','G4729','G4723','G47'
            ) THEN 'Sleep related disorders'

            WHEN code IN (
                'J00','J0100','J0101','J0110','J0111','J0120','J0121','J0130',
                'J0131','J0140','J0141','J0180','J0181','J0190','J0191','J020',
                'J028','J029','J0300','J0301','J0380','J0381','J0390','J0391',
                'J040','J0410','J0411','J042','J0430','J0431','J050','J0510',
                'J0511','J060','J069'
            ) THEN 'Acute upper respiratory infections'

            WHEN code IN (
                'H6000','H6001','H6002','H6003','H6010','H6011','H6012','H6013',
                'H6020','H6021','H6022','H60311','H60312','H60319','H60321',
                'H60322','H60329','H60391','H60392','H60399','H6040','H6041',
                'H6042','H6043','H60501','H60502','H60509','H60511','H60512',
                'H60519','H60551','H60552','H60559','H60591','H60592','H60599',
                'H6060','H6061','H6062','H608X1','H608X2','H608X9','H6090',
                'H6091','H6092','H61001','H61002','H61003','H61009','H61011',
                'H61012','H61013','H61019','H61021','H61022','H61023','H61029',
                'H61031','H61032','H61033','H61039','H61321','H61322','H61323',
                'H61329','H6240','H6241','H6242','H6500','H6501','H6502','H6504',
                'H6505','H6507','H65111','H65112','H65114','H65115','H65117',
                'H65119','H65191','H65192','H65194','H65195','H65197','H65199',
                'H6520','H6521','H6522','H6530','H6531','H6532','H65411',
                'H65412','H65419','H65491','H65492','H65499','H6590','H6591',
                'H6592','H66001','H66002','H66003','H66004','H66005','H66006',
                'H66007','H66009','H66011','H66012','H66013','H66014','H66015',
                'H66016','H66017','H66019','H6611','H6612','H6620','H6621',
                'H6622','H663X1','H663X2','H663X9','H6640','H6641','H6642',
                'H6690','H6691','H6692','H671','H672','H679','H68001','H68002',
                'H68009','H68011','H68012','H68019','H68021','H68022','H68029',
                'H70001','H70002','H70009','H70011','H70012','H70019','H70091',
                'H70092','H70099','H7010','H7011','H7012','H70201','H70202',
                'H70209','H70211','H70212','H70219','H70221','H70222','H70229',
                'H70811','H70812','H70819','H70891','H70892','H70899','H7090',
                'H7091','H7092','H7100','H7101','H7102','H7110','H7111','H7112',
                'H7120','H7121','H7122','H7130','H7131','H7132','H7190','H7191',
                'H7192','H73001','H73002','H73009','H73011','H73012','H73019',
                'H73091','H73092','H73099','H7310','H7311','H7312','H7320',
                'H7321','H7322','H7411','H7412','H7413','H7419','H7440','H7441',
                'H7442','H7443','H748X1','H748X2','H748X3','H748X9','H7490',
                'H7491','H7492','H7493','H8120','H8121','H8122','H8301','H8302',
                'H8309','H9210','H9211','H9212','H9500','H9501','H9502','H9503'
            ) THEN 'Ear Infections'

            WHEN code IN (
                'H900','H9011','H9012','H902','H903','H9041','H9042','H905',
                'H906','H9071','H9072','H908','H90A11','H90A12','H90A21',
                'H90A22','H90A31','H90A32','H9101','H9102','H9103','H9109',
                'H9120','H9121','H9122','H9123','H918X1','H918X2','H918X3',
                'H918X9','H9190','H9191','H9192','H9193','P096'
            ) THEN 'Hearing loss'

            WHEN code IN (
                'K5900','K5909','K5901','K5904','K5903','K5902','K590','K5939'
            ) THEN 'Constipation'

            WHEN code IN ('R197','K591','K580','K529')
                THEN 'Diarrhea'

            WHEN code IN (
                'K4000','K4001','K4010','K4011','K4020','K4021','K4030','K4031',
                'K4040','K4041','K4090','K4091','K450','K451','K458','K460',
                'K461','K469'
            ) THEN 'Abdominal/inguinal hernia'

            WHEN code IN ('Q751','Q754','Q755')
                THEN 'Dysostosis Complex'

            WHEN code IN (
                'M2560','M25611','M25612','M25619','M25621','M25622','M25629',
                'M25631','M25632','M25639','M25641','M25642','M25649','M25651',
                'M25652','M25659','M25661','M25662','M25669','M25671','M25672',
                'M25673','M25674','M25675','M25676','M2569'
            ) THEN 'Joint Stiffness'

            WHEN code IN ('G5600','G5601','G5602','G5603')
                THEN 'Carpal tunnel syndrome'

            WHEN code IN (
                'F05','F060','F061','F062','F0630','F0631','F0632','F0633',
                'F0634','F064','F0670','F0671','F068','F070','F0781','F0789',
                'F079','F09','F22','F23','F24','F28','F29','F3010','F3011',
                'F3012','F3013','F302','F303','F304','F308','F309','F320',
                'F321','F322','F323','F324','F325','F328','F3289','F329','F32A',
                'F330','F331','F332','F333','F3340','F3341','F3342','F338',
                'F339','F340','F348','F3481','F3489','F349','F39','F410','F411',
                'F413','F418','F419','F430','F4310','F4311','F4312','F4320',
                'F4321','F4322','F4323','F4324','F4325','F4329','F438','F4389',
                'F439','F441','F442','F450','F451','F4522','F4541','F4542',
                'F54','F59','F600','F602','F603','F604','F605','F606','F6089',
                'F609','F6381','F6389','F639','F70','F71','F72','F73','F78',
                'F78A1','F78A9','F79','F800','F801','F802','F804','F8081',
                'F8082','F8089','F809','F810','F812','F8181','F8189','F819',
                'F82','F840','F843','F845','F848','F849','F88','F89','F900',
                'F901','F902','F908','F909','F910','F911','F912','F913','F918',
                'F919','F930','F938','F939','F940','F941','F942','F948','F949',
                'F950','F951','F9821','F9829','F983','F984','F985','F988',
                'F989','F99'
            ) THEN 'Behavioral Issues'

            WHEN code IN (
                'G910','G911','G912','G913','G914','G918','G919','Q038','Q039',
                'Q050','Q051','Q052','Q053','Q054','Q055','Q056','Q057','Q058',
                'Q0700','Q0702','Q0703','F445','G40001','G40009','G40011',
                'G40019','G40101','G40109','G40111','G40119','G40201','G40209',
                'G40211','G40219','G40501','G40509','G4089','R561'
            ) THEN 'CNS Issues'

            WHEN code IN ('R6250','R620','R6252','R6251','R6259','R627','R62')
                THEN 'Lack of Physiological Development'

            WHEN code IN (
                'I10','I110','I129','I130','I119','I159','I160','I158','I120',
                'I161','I1310','I150'
            ) THEN 'Hypertension'

            WHEN code IN (
                'I050','I051','I052','I058','I059','I060','I061','I062','I068',
                'I069','I070','I071','I072','I078','I079','I080','I081','I082',
                'I083','I088','I089','I340','I341','I342','I348','I3481',
                'I3489','I349','I350','I351','I352','I358','I359','I360',
                'I361','I362','I368','I369','I370','I371','I372','I378','I379'
            ) THEN 'Valvular Heart Disease'

        END AS comorbidity_name
    FROM exploded_codes
    WHERE code IS NOT NULL AND code <> ''
),

comorbidity_with_category AS (
    SELECT
        patient_id,
        comorbidity_name,
        CASE
            WHEN comorbidity_name IN (
                'Asthma','Sleep Apnea','Other diseases of upper respiratory tract',
                'Sleep related disorders','Acute upper respiratory infections'
            ) THEN 'Respiratory issues'

            WHEN comorbidity_name IN ('Ear Infections','Hearing loss')
                THEN 'Ear-related disorders'

            WHEN comorbidity_name IN ('Constipation','Diarrhea','Abdominal/inguinal hernia')
                THEN 'Gastrointestinal disorders'

            WHEN comorbidity_name IN ('Dysostosis Complex','Joint Stiffness','Carpal tunnel syndrome')
                THEN 'Mobility issues'

            WHEN comorbidity_name IN ('Behavioral Issues','CNS Issues','Lack of Physiological Development')
                THEN 'Neurological disorders'

            WHEN comorbidity_name IN ('Hypertension','Valvular Heart Disease')
                THEN 'Other chronic conditions'
        END AS comorbidity_category
    FROM comorbidity_raw
    WHERE comorbidity_name IS NOT NULL
),

patient_with_comorbidity_outcome as (
    SELECT
    patient_id,

    array_join(
        array_sort(collect_set(comorbidity_category)),
        ', '
    ) AS comorbidity_categories,

    size(collect_set(comorbidity_category)) AS count_of_comorbidity_categories,

    array_join(
        array_sort(collect_set(comorbidity_name)),
        ', '
    ) AS distinct_comorbidities,

    size(collect_set(comorbidity_name)) AS count_of_distinct_comorbidities

FROM comorbidity_with_category
GROUP BY patient_id
ORDER BY patient_id
)
select
    a.*,
    b.comorbidity_categories,
    b.count_of_comorbidity_categories,
    b.distinct_comorbidities,
    b.count_of_distinct_comorbidities
from com_edp_prd.cmpa_insights_internal_schema.patient360_master as a
left join patient_with_comorbidity_outcome as b
    on a.patient_id=b.patient_id;


-- =============================================================================
-- CELL 8: primary_hcp secondary specialty enrichment (UNCHANGED)
-- =============================================================================
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master as
  select distinct a.*, b.SECONDARY_SPECIALTY as primary_hcp_secondary_specialty
from com_edp_prd.cmpa_insights_internal_schema.patient360_master as a
left join com_edp_prd.com_raw.kom_providers as b on a.PRIMARY_HCP_NPI = b.npi and b.provider_type = 'INDIVIDUAL';


-- =============================================================================
-- CELL 9: age_bucket / payer / insurance enrichment (UNCHANGED)
-- =============================================================================
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master AS

WITH base_table AS (
  SELECT DISTINCT *
  FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
),
age_buckets AS (
  SELECT DISTINCT
    patient_id,
    CASE
      WHEN patient_age < 5 THEN '<5 years'
      WHEN patient_age BETWEEN 5 AND 10 THEN '5 - 10 years'
      WHEN patient_age BETWEEN 11 AND 16 THEN '11 - 16 years'
      ELSE '>17 years'
    END AS age_bucket
  FROM base_table
),

patient_geography AS (
  SELECT DISTINCT *
  FROM (
    SELECT *,
      ROW_NUMBER() OVER (
        PARTITION BY patient_id
        ORDER BY
          CASE WHEN valid_to_date > CURRENT_DATE() THEN 1 ELSE 2 END,
          valid_to_date DESC
      ) AS rn
    FROM com_edp_prd.com_raw.kom_patient_geography
  )
  WHERE rn = 1
),
patient_zip AS (
  SELECT DISTINCT
    a.patient_id,
    b.patient_zip AS zip3
  FROM base_table a
  LEFT JOIN patient_geography b
    ON a.patient_id = b.patient_id
),

tx_claims_payer_analysis AS (
  SELECT DISTINCT
    a.patient_id,
    a.fill_date,
    a.claim_id,
    a.kh_plan_id,
    b.payer_name,
    b.insurance_group
  FROM (
    SELECT
      patient_id,
      service_date AS fill_date,
      kh_plan_id,
      medical_event_id AS claim_id
    FROM com_edp_prd.com_raw.kom_medical_events

    UNION ALL

    SELECT
      patient_id,
      fill_date,
      COALESCE(primary_kh_plan_id, secondary_kh_plan_id) AS kh_plan_id,
      pharmacy_event_id AS claim_id
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE transaction_result = 'PAID'
  ) a
  LEFT JOIN com_edp_prd.com_raw.kom_plans b
    ON a.kh_plan_id = b.kh_plan_id
),

payer_claims_count AS (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY patient_id
      ORDER BY num_claims DESC
    ) AS rn
  FROM (
    SELECT
      patient_id,
      payer_name,
      COUNT(DISTINCT claim_id) AS num_claims
    FROM tx_claims_payer_analysis
    WHERE payer_name IS NOT NULL
      AND fill_date BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
    GROUP BY patient_id, payer_name
  )
),

patient_primary_secondary_payer AS (
  SELECT
    patient_id,
    MAX(CASE WHEN rn = 1 THEN payer_name END) AS primary_payer,
    MAX(CASE WHEN rn = 2 THEN payer_name END) AS secondary_payer
  FROM payer_claims_count
  GROUP BY patient_id
),

/* Only for latest_insurance_type.
   Does not affect primary_payer or secondary_payer. */
mpsii_insurance_claims AS (

  /* MPS II medical diagnosis claims */
  SELECT DISTINCT
    m.patient_id,
    m.service_date AS fill_date,
    m.medical_event_id AS claim_id,
    m.kh_plan_id AS plan_id,
    COALESCE(m.rendering_npi, m.referring_npi) AS npi
  FROM com_edp_prd.com_raw.kom_medical_events m
  INNER JOIN base_table b
    ON m.patient_id = b.patient_id
  WHERE (
          m.diagnosis_codes LIKE '%E761%'
       OR m.diagnosis_codes LIKE '%E763%'
        )
    AND m.service_date BETWEEN DATE '2020-08-01'
                           AND (SELECT end_date FROM runtime_parameters)

  UNION

  /* MPS II pharmacy diagnosis claims */
  SELECT DISTINCT
    p.patient_id,
    p.fill_date,
    p.pharmacy_event_id AS claim_id,
    COALESCE(
      p.primary_kh_plan_id,
      p.secondary_kh_plan_id
    ) AS plan_id,
    p.prescriber_npi AS npi
  FROM com_edp_prd.com_raw.kom_pharmacy_events p
  INNER JOIN base_table b
    ON p.patient_id = b.patient_id
  WHERE p.diagnosis_code IN ('E761', 'E763')
    AND p.transaction_status = 'PAID'
    AND p.fill_date BETWEEN DATE '2020-08-01'
                        AND (SELECT end_date FROM runtime_parameters)

  UNION

  /* Medical Elaprase NDC claims */
  SELECT DISTINCT
    m.patient_id,
    m.service_date AS fill_date,
    m.medical_event_id AS claim_id,
    m.kh_plan_id AS plan_id,
    COALESCE(m.rendering_npi, m.referring_npi) AS npi
  FROM com_edp_prd.com_raw.kom_medical_events m
  INNER JOIN base_table b
    ON m.patient_id = b.patient_id
  WHERE m.ndc11 IN ('54092070001', '540920700')
    AND m.service_date BETWEEN DATE '2023-08-01'
                           AND (SELECT end_date FROM runtime_parameters)

  UNION

  /* Pharmacy Elaprase NDC claims */
  SELECT DISTINCT
    p.patient_id,
    p.fill_date,
    p.pharmacy_event_id AS claim_id,
    COALESCE(
      p.primary_kh_plan_id,
      p.secondary_kh_plan_id
    ) AS plan_id,
    p.prescriber_npi AS npi
  FROM com_edp_prd.com_raw.kom_pharmacy_events p
  INNER JOIN base_table b
    ON p.patient_id = b.patient_id
  WHERE p.ndc11 IN ('54092070001', '540920700')
    AND p.transaction_result = 'PAID'
    AND p.fill_date BETWEEN DATE '2023-08-01'
                        AND (SELECT end_date FROM runtime_parameters)

  UNION

  /* Medical treatment procedure claims */
  SELECT DISTINCT
    m.patient_id,
    m.service_date AS fill_date,
    m.medical_event_id AS claim_id,
    m.kh_plan_id AS plan_id,
    m.rendering_npi AS npi
  FROM com_edp_prd.com_raw.kom_medical_events m
  INNER JOIN base_table b
    ON m.patient_id = b.patient_id
  WHERE m.procedure_code IN (
          '99601', '99602', '96365', '96366', 'J1743',
          'S9357', 'S9379', '38206', '38230', '38232',
          '38240', '38241', '38242', '38243', '38250'
        )
    AND m.service_date BETWEEN DATE '2023-08-01'
                           AND (SELECT end_date FROM runtime_parameters)
),

latest_mpsii_plan AS (
  SELECT
    patient_id,
    fill_date,
    claim_id,
    plan_id,
    npi
  FROM (
    SELECT
      patient_id,
      fill_date,
      claim_id,
      plan_id,
      npi,
      ROW_NUMBER() OVER (
        PARTITION BY patient_id
        ORDER BY
          fill_date DESC,
          npi ASC,
          claim_id DESC
      ) AS rn
    FROM mpsii_insurance_claims
    WHERE plan_id IS NOT NULL
  )
  WHERE rn = 1
),

patient_latest_insurance_type AS (
  SELECT
    b.patient_id,

    COALESCE(
      kp.insurance_group,
      'Unknown'
    ) AS latest_insurance_type

  FROM base_table b

  LEFT JOIN latest_mpsii_plan lmp
    ON b.patient_id = lmp.patient_id

  LEFT JOIN com_edp_prd.com_raw.kom_plans kp
    ON lmp.plan_id = kp.kh_plan_id
)

SELECT
  a.*,
  b.age_bucket,
  c.zip3,
  f.primary_payer,
  f.secondary_payer,
  g.latest_insurance_type

FROM base_table a
LEFT JOIN age_buckets b
  ON a.patient_id = b.patient_id
LEFT JOIN patient_zip c
  ON a.patient_id = c.patient_id
LEFT JOIN patient_primary_secondary_payer f
  ON a.patient_id = f.patient_id
LEFT JOIN patient_latest_insurance_type g
  ON a.patient_id = g.patient_id;


-- =============================================================================
-- CELL 10: specialty re-enrichment for most_recently_treated and primary HCPs (UNCHANGED)
-- =============================================================================
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master as
select a.*except(a.most_recently_treated_hcp_specialty_2yr, a.PRIMARY_HCP_SPECIALTY, a.primary_hcp_secondary_specialty),
b.PRIMARY_SPECIALTY as most_recently_treated_hcp_specialty_2yr, b.SECONDARY_SPECIALTY as most_recently_treated_hcp_secondary_specialty_2yr, c.PRIMARY_SPECIALTY as primary_hcp_specialty, c.SECONDARY_SPECIALTY as primary_hcp_secondary_specialty
from cmpa_insights_internal_schema.patient360_master as a
left join com_raw.kom_providers as b on a.most_recently_treated_hcp_2yr = b.npi and b.PROVIDER_TYPE = 'INDIVIDUAL'
left join com_raw.kom_providers as c on a.PRIMARY_HCP_NPI = c.npi and c.PROVIDER_TYPE = 'INDIVIDUAL';


-- =============================================================================
-- CELL 11: newborn screening flag (UNCHANGED)
-- =============================================================================
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master as
with base_table as (
  select distinct * from com_edp_prd.cmpa_insights_internal_schema.patient360_master
),
newborn_screening_flag as (
  select *,
  case when patient_state in ('IL', 'MO', 'WV', 'PA', 'KY', 'MD', 'CA', 'DE', 'FL', 'KS', 'AZ', 'MA', 'RI', 'AR', 'IA', 'NC', 'TX', 'CT') then 1 else 0 end as newborn_screening_flag
  from base_table
)
select * from newborn_screening_flag;


-- =============================================================================
-- CELL 12: most recent infusion location (UNCHANGED)
-- =============================================================================
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master as
WITH base_table as (
  select distinct *
  from com_edp_prd.cmpa_insights_internal_schema.patient360_master
),

tx_patients as (
  select distinct
      patient_id,
      coalesce(rendering_npi, referring_npi) as npi,
      service_date as fill_date,
      place_of_service
  from com_edp_prd.com_raw.kom_medical_events
  where ndc11 in ('54092070001','540920700')
    and service_date between '2023-08-01' and (SELECT end_date FROM runtime_parameters)

  union

  select distinct
      patient_id,
      prescriber_npi as npi,
      fill_date,
      '' as place_of_service
  from com_edp_prd.com_raw.kom_pharmacy_events
  where ndc11 in ('54092070001','540920700')
    and transaction_result = 'PAID'
    and fill_date between '2023-08-01' and (SELECT end_date FROM runtime_parameters)

  union

  select distinct
      patient_id,
      rendering_npi as npi,
      service_date as fill_date,
      place_of_service
  from com_edp_prd.com_raw.kom_medical_events
  where procedure_code in ('99601','99602','96365','96366','J1743','S9357','S9379',
                           '38206','38230','38232','38240','38241','38242','38243','38250')
    and service_date between '2023-08-01' and (SELECT end_date FROM runtime_parameters)
),

latest_pos_patients as (
  select
      patient_id,
      nullif(trim(place_of_service), '') as most_recent_infusion_location
  from (
    select
        patient_id,
        place_of_service,
        fill_date,
        row_number() over (
          partition by patient_id
          order by fill_date desc
        ) as rn
    from tx_patients
  ) t
  where rn = 1
)

select
    a.*,
    b.most_recent_infusion_location,
    c.description as most_recent_infusion_description
from base_table a
left join latest_pos_patients b
  on a.patient_id = b.patient_id
left join com_edp_prd.cmpa_insights_internal_schema.pos_description c
  on try_cast(nullif(trim(b.most_recent_infusion_location), '') as int) = c.code;


CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master AS
SELECT *
FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
WHERE NOT (
    UPPER(PATIENT_GENDER) = 'F'
    AND UPPER(latest_mpsii_tx_type) = 'OTHER ERT'
);


-- =============================================================================
-- =============================================================================
-- TIVI / AVLAYAH ADD-ON CELLS (14-19)
-- These cells build a parallel Tivi cohort using NDC '8497600101' and
-- procedure codes 'J3490','J3590','J9999' (gated to >= '2026-03-01'), produce a
-- tivi_patient360_master view, then LEFT JOIN it onto the existing
-- patient360_master to add tivi_* prefixed columns.
-- =============================================================================
-- =============================================================================


-- =============================================================================
-- CELL 14 (TIVI): patient_hcp_visit_summary (Tivi base) + patient360_base view
-- =============================================================================
CREATE OR REPLACE TEMP VIEW patient_hcp_visit_summary AS
WITH MPSII_Diagnoses_Specified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_RESULT = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
),

Patients_2Dx_Specified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Specified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

MPSII_Diagnoses_Unspecified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_RESULT = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
),

Patients_2Dx_Unspecified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Unspecified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

MPSII_Treatment_All AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('8497600101')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('8497600101')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE IN ('J3490','J3590','J9999')
          AND SERVICE_DATE >= '2026-03-01'
    ) t
),

MPSII_Treatment_Tivi_Only AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('8497600101')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('8497600101')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE IN ('J3490','J3590','J9999')
          AND SERVICE_DATE >= '2026-03-01'
    ) t
),

Patients_2Dx_Specified_With_Treatment AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Specified p
    INNER JOIN MPSII_Treatment_All t USING (PATIENT_ID)
),

Patients_Incremental_Unspecified AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Unspecified p
    INNER JOIN MPSII_Treatment_Tivi_Only t USING (PATIENT_ID)
    WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
),

eligible_patients AS (
    SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment
    UNION
    SELECT PATIENT_ID FROM Patients_Incremental_Unspecified
),

cohort_3_learnings AS (
  SELECT DISTINCT npi
  FROM com_raw.kom_providers
  WHERE provider_type = 'INDIVIDUAL'
    AND (
      PRIMARY_SPECIALTY NOT IN (
        'Anesthesiologist Assistant','Anesthesiology','Dentist','Dietitian, Registered',
        'Emergency Medical Technician, Basic','Emergency Medicine','General Acute Care Hospital',
        'Nurse Anesthetist, Certified Registered','Obstetrics & Gynecology','Pathology',
        'Radiology','Urology'
      )
      OR SECONDARY_SPECIALTY IN (
        'Child & Adolescent Psychiatry','Psychiatry','Adolescent Medicine','Developmental - Behavioral Pediatrics',
        'Neonatal-Perinatal Medicine','Nutrition, Pediatric','Oncology, Pediatrics','Pediatric Cardiology',
        'Pediatric Critical Care Medicine','Pediatric Dermatology','Pediatric Emergency Medicine',
        'Pediatric Endocrinology','Pediatric Gastroenterology','Pediatric Hematology-Oncology',
        'Pediatric Infectious Diseases','Pediatric Nephrology','Pediatric Ophthalmology and Strabismus Specialist',
        'Pediatric Orthopaedic Surgery','Pediatric Otolaryngology','Pediatric Pulmonology','Pediatric Radiology',
        'Pediatric Rehabilitation Medicine','Pediatric Rheumatology','Pediatric Surgery','Pediatrics',
        'Clinical Biochemical Genetics','Clinical Genetics (M.D.)','Clinical Molecular Genetics',
        'Ph.D. Medical Genetics','Neurodevelopmental Disabilities','Neurology',
        'Neurology with Special Qualifications in Child Neurology','Neuroradiology'
      )
    )
),

all_dx_claims_5yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE DIAGNOSIS_CODE IN ('E761','E763')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),
all_tx_claims_5yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT
          PATIENT_ID,
          COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
          SERVICE_DATE AS FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('8497600101')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          PRESCRIBER_NPI AS NPI,
          FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('8497600101')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          RENDERING_NPI AS NPI,
          SERVICE_DATE AS FILL_DATE,
          PROCEDURE_CODE AS TX_CODE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE PROCEDURE_CODE IN ('J3490','J3590','J9999')
        AND  SERVICE_DATE >= '2026-03-01'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

all_tx_claims_alltime AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT
          PATIENT_ID,
          COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
          SERVICE_DATE AS FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('8497600101')
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          PRESCRIBER_NPI AS NPI,
          FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('8497600101')
        AND TRANSACTION_RESULT = 'PAID'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          RENDERING_NPI AS NPI,
          SERVICE_DATE AS FILL_DATE,
          PROCEDURE_CODE AS TX_CODE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE PROCEDURE_CODE IN ('J3490','J3590','J9999')
        AND  SERVICE_DATE >= '2026-03-01'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

all_claims_5yr AS (
    SELECT DISTINCT
        PATIENT_ID,
        NPI,
        FILL_DATE,
        CAST(NULL AS STRING) AS TX_CODE
    FROM all_dx_claims_5yr
    UNION
    SELECT DISTINCT
        PATIENT_ID,
        NPI,
        FILL_DATE,
        TX_CODE
    FROM all_tx_claims_5yr
),

all_claims_3yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
        AND SERVICE_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE DIAGNOSIS_CODE IN ('E761','E763')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('8497600101')
        AND SERVICE_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('8497600101')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE PROCEDURE_CODE IN ('J3490','J3590','J9999')
        AND SERVICE_DATE >= '2026-03-01'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

first_dx_hcp_ranked AS (
    SELECT PATIENT_ID, NPI, MIN(FILL_DATE) AS first_dx_date,
           ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY MIN(FILL_DATE) ASC, NPI) AS rn
    FROM all_dx_claims_5yr
    WHERE NPI IS NOT NULL
    GROUP BY PATIENT_ID, NPI
),
first_dx_hcp AS (
    SELECT PATIENT_ID, NPI AS first_dx_hcp, first_dx_date
    FROM first_dx_hcp_ranked
    WHERE rn = 1
),

provider_dim AS (
    SELECT
        npi,
        CONCAT(FIRST_NAME, ' ', LAST_NAME) AS provider_name,
        primary_specialty
    FROM com_raw.kom_providers
    WHERE provider_type = 'INDIVIDUAL'
),

first_dx_hcp_5yr_stats AS (
    SELECT fdh.PATIENT_ID, fdh.first_dx_hcp,
           COUNT(DISTINCT ac.FILL_DATE) AS first_dx_all_visit_count_5yr,
           MAX(ac.FILL_DATE) AS first_dx_last_visit_5yr
    FROM first_dx_hcp fdh
    LEFT JOIN all_claims_5yr ac
      ON fdh.PATIENT_ID = ac.PATIENT_ID AND fdh.first_dx_hcp = ac.NPI
    GROUP BY fdh.PATIENT_ID, fdh.first_dx_hcp
),

first_tx_hcp_ranked AS (
    SELECT PATIENT_ID, NPI, MIN(FILL_DATE) AS first_tx_date,
           ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY MIN(FILL_DATE) ASC, NPI) AS rn
    FROM all_tx_claims_5yr
    WHERE NPI IS NOT NULL
    GROUP BY PATIENT_ID, NPI
),
first_tx_hcp AS (
    SELECT PATIENT_ID, NPI AS first_tx_hcp, first_tx_date
    FROM first_tx_hcp_ranked
    WHERE rn = 1
),

first_tx_hcp_5yr_stats AS (
    SELECT fth.PATIENT_ID, fth.first_tx_hcp,
           COUNT(DISTINCT ac.FILL_DATE) AS first_tx_all_visit_count_5yr,
           MAX(ac.FILL_DATE) AS first_tx_last_visit_5yr
    FROM first_tx_hcp fth
    LEFT JOIN all_claims_5yr ac
      ON fth.PATIENT_ID = ac.PATIENT_ID AND fth.first_tx_hcp = ac.NPI
    GROUP BY fth.PATIENT_ID, fth.first_tx_hcp
),

first_tx_hcp_5yr_tx_only AS (
    SELECT fth.PATIENT_ID, fth.first_tx_hcp,
           COUNT(DISTINCT tx.FILL_DATE) AS first_tx_treatment_visit_count_5yr
    FROM first_tx_hcp fth
    LEFT JOIN all_tx_claims_5yr tx
      ON fth.PATIENT_ID = tx.PATIENT_ID AND fth.first_tx_hcp = tx.NPI
    GROUP BY fth.PATIENT_ID, fth.first_tx_hcp
),

most_seen_3yr_ranking AS (
    SELECT PATIENT_ID,
           NPI,
           COUNT(DISTINCT FILL_DATE) AS visit_count_3yr,
           MAX(FILL_DATE) AS last_visit_3yr,
           ROW_NUMBER() OVER (
             PARTITION BY PATIENT_ID
             ORDER BY COUNT(DISTINCT FILL_DATE) DESC,
                      MAX(FILL_DATE) DESC,
                      NPI ASC
           ) AS rank
    FROM all_claims_3yr
    WHERE NPI IS NOT NULL
    GROUP BY PATIENT_ID, NPI
),

most_seen_combined_stats AS (
    SELECT
        ms3.PATIENT_ID,
        ms3.NPI,
        ms3.rank,
        ms3.visit_count_3yr,
        ms3.last_visit_3yr,
        COUNT(DISTINCT ac5.FILL_DATE) AS visit_count_5yr,
        MAX(ac5.FILL_DATE)          AS last_visit_5yr
    FROM most_seen_3yr_ranking ms3
    LEFT JOIN all_claims_5yr ac5
      ON ms3.PATIENT_ID = ac5.PATIENT_ID AND ms3.NPI = ac5.NPI
    WHERE ms3.rank <= 5
    GROUP BY ms3.PATIENT_ID, ms3.NPI, ms3.rank, ms3.visit_count_3yr, ms3.last_visit_3yr
),

historical_first_dx AS (
    SELECT PATIENT_ID, MIN(FILL_DATE) AS incidence_date
    FROM (
        SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE DIAGNOSIS_CODES LIKE '%E761%'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, FILL_DATE
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE DIAGNOSIS_CODE = 'E761'
          AND TRANSACTION_RESULT = 'PAID'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE DIAGNOSIS_CODES LIKE '%E763%'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, FILL_DATE
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE DIAGNOSIS_CODE = 'E763'
          AND TRANSACTION_RESULT = 'PAID'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    ) all_dx
    GROUP BY PATIENT_ID
),

historical_first_tx AS (
    SELECT PATIENT_ID, MIN(FILL_DATE) AS first_incidence_treatment_date
    FROM (
        SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('8497600101')
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, FILL_DATE
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('8497600101')
          AND TRANSACTION_RESULT = 'PAID'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE IN ('J3490','J3590','J9999')
          and SERVICE_DATE >= '2026-03-01'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    ) all_tx
    GROUP BY PATIENT_ID
),

all_claims_5yr_specialty_removed AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE DIAGNOSIS_CODE IN ('E761','E763')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('8497600101')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('8497600101')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE PROCEDURE_CODE IN ('J3490','J3590','J9999')
        AND SERVICE_DATE >= '2026-03-01'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
),

latest_claim_hcp_ranked as (
  select patient_id, npi, fill_date, row_number() over(partition by patient_id order by fill_date desc, npi asc) as rn
  from all_claims_5yr
),

latest_claim_hcp as (
  select patient_id, npi as latest_claim_hcp_npi, fill_date as latest_claim_date from latest_claim_hcp_ranked where rn = 1
),

latest_claim_hcp_visit_count_5yr as (
  select a.patient_id, a.latest_claim_hcp_npi, count(distinct fill_date) as latest_claim_hcp_visit_count_5yr
  from latest_claim_hcp as a
  left join all_claims_5yr as b on a.patient_id = b.patient_id and a.latest_claim_hcp_npi = b.npi
  group by 1,2
),

latest_claim_hcp_final as (
  select a.patient_id, a.latest_claim_hcp_npi, a.latest_claim_date, b.latest_claim_hcp_visit_count_5yr
  from latest_claim_hcp as a
  left join latest_claim_hcp_visit_count_5yr as b on a.patient_id = b.patient_id
),

all_tx_claims_5yr_specialty_removed AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT
          PATIENT_ID,
          COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
          SERVICE_DATE AS FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('8497600101')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          PRESCRIBER_NPI AS NPI,
          FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('8497600101')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          RENDERING_NPI AS NPI,
          SERVICE_DATE AS FILL_DATE,
          PROCEDURE_CODE AS TX_CODE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE PROCEDURE_CODE IN ('J3490','J3590','J9999')
        AND SERVICE_DATE >= '2026-03-01'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
),

most_recent_tx_hcp_ranked as (
  select patient_id, npi, fill_date, row_number() over(partition by patient_id order by fill_date desc, npi asc) as rn
  from all_tx_claims_5yr
),

most_recent_tx_hcp as (
  select patient_id, npi as latest_treatment_hcp_npi, fill_date as latest_treatment_date
  from most_recent_tx_hcp_ranked where rn = 1
),

latest_treatment_hcp_visit_count as (
  select a.patient_id, a.latest_treatment_hcp_npi, count(distinct b.fill_date) as latest_treatment_hcp_visit_count_5yr
  from most_recent_tx_hcp as a
  left join all_claims_5yr as b on a.patient_id = b.patient_id and a.latest_treatment_hcp_npi = b.npi
  group by 1, 2
),

last_infusion_hcp_last_visit_cte as (
  select
      a.patient_id,
      a.latest_treatment_hcp_npi,
      max(b.fill_date) as last_infusion_hcp_last_visit
  from most_recent_tx_hcp as a
  left join all_claims_5yr as b
      on a.patient_id = b.patient_id
     and a.latest_treatment_hcp_npi = b.npi
  group by
      a.patient_id,
      a.latest_treatment_hcp_npi
),

most_recent_tx_hcp_final as (
  select a.patient_id, a.latest_treatment_hcp_npi, a.latest_treatment_date, b.latest_treatment_hcp_visit_count_5yr, c.last_infusion_hcp_last_visit
  from most_recent_tx_hcp as a
  left join latest_treatment_hcp_visit_count as b on a.patient_id = b.patient_id
  left join last_infusion_hcp_last_visit_cte as c
      on a.patient_id = c.patient_id
),

latest_mpsii_treatment_type AS (
  SELECT
    patient_id,
    CASE
      WHEN tx_code IN ('8497600101') OR (tx_code IN ('J3490','J3590','J9999') AND fill_date >= '2026-03-01')
        THEN 'AVLAYAH'
    END AS latest_mpsii_tx_type
  FROM (
    SELECT
      patient_id,
      fill_date,
      tx_code,
      ROW_NUMBER() OVER (
        PARTITION BY patient_id
        ORDER BY fill_date DESC, tx_code ASC
      ) AS rn
    FROM all_tx_claims_5yr
    WHERE tx_code IS NOT NULL
  )
  WHERE rn = 1
),

first_tx_after_diagnosis AS (
  SELECT
    tx.patient_id,
    MIN(tx.fill_date) AS first_tx_after_diagnosis
  FROM all_tx_claims_alltime tx
  INNER JOIN historical_first_dx dx
    ON tx.patient_id = dx.patient_id
  WHERE tx.fill_date >= dx.incidence_date
  GROUP BY tx.patient_id
),

Tivi_fills AS (
  SELECT
    patient_id,
    COUNT(DISTINCT fill_date) AS Tivi_fills
  FROM all_tx_claims_5yr
  WHERE fill_date BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
    AND tx_code IN ('8497600101')
  GROUP BY patient_id
),

patient_demographics AS (
    SELECT *
    FROM (
        SELECT DISTINCT PATIENT_ID, PATIENT_YOB, PATIENT_GENDER,
               ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY PATIENT_YOB ASC) AS rn
        FROM com_edp_prd.com_raw.kom_patient_demographics
    )
    WHERE rn = 1
),
patient_geography AS (
    SELECT patient_id, patient_state
    FROM (
        SELECT
            PATIENT_ID,
            patient_state,
            ROW_NUMBER() OVER (
                PARTITION BY PATIENT_ID
                ORDER BY
                    CASE WHEN VALID_TO_DATE > CURRENT_DATE() THEN 1 ELSE 2 END,
                    VALID_TO_DATE DESC
            ) AS rn
        FROM COM_EDP_PRD.COM_RAW.KOM_PATIENT_GEOGRAPHY
    )
    WHERE rn = 1
),

most_seen_pivot AS (
    SELECT
        PATIENT_ID,

        MAX(CASE WHEN rank = 1 THEN NPI END)              AS most_seen_hcp1_3yr_ranked,
        MAX(CASE WHEN rank = 1 THEN visit_count_5yr END)  AS most_seen_hcp1_visit_count_5yr,
        MAX(CASE WHEN rank = 1 THEN last_visit_5yr END)   AS most_seen_hcp1_last_visit_5yr,

        MAX(CASE WHEN rank = 2 THEN NPI END)              AS most_seen_hcp2_3yr_ranked,
        MAX(CASE WHEN rank = 2 THEN visit_count_5yr END)  AS most_seen_hcp2_visit_count_5yr,
        MAX(CASE WHEN rank = 2 THEN last_visit_5yr END)   AS most_seen_hcp2_last_visit_5yr,

        MAX(CASE WHEN rank = 3 THEN NPI END)              AS most_seen_hcp3_3yr_ranked,
        MAX(CASE WHEN rank = 3 THEN visit_count_5yr END)  AS most_seen_hcp3_visit_count_5yr,
        MAX(CASE WHEN rank = 3 THEN last_visit_5yr END)   AS most_seen_hcp3_last_visit_5yr,

        MAX(CASE WHEN rank = 4 THEN NPI END)              AS most_seen_hcp4_3yr_ranked,
        MAX(CASE WHEN rank = 4 THEN visit_count_5yr END)  AS most_seen_hcp4_visit_count_5yr,
        MAX(CASE WHEN rank = 4 THEN last_visit_5yr END)   AS most_seen_hcp4_last_visit_5yr,

        MAX(CASE WHEN rank = 5 THEN NPI END)              AS most_seen_hcp5_3yr_ranked,
        MAX(CASE WHEN rank = 5 THEN visit_count_5yr END)  AS most_seen_hcp5_visit_count_5yr,
        MAX(CASE WHEN rank = 5 THEN last_visit_5yr END)   AS most_seen_hcp5_last_visit_5yr

    FROM most_seen_combined_stats
    GROUP BY PATIENT_ID
)
SELECT
    ep.PATIENT_ID,
    pd.PATIENT_YOB,
    YEAR(CURRENT_DATE) - YEAR(pd.PATIENT_YOB) AS PATIENT_AGE,
    pd.PATIENT_GENDER,
    pg.patient_state,

    hfdx.incidence_date,
    hftx.first_incidence_treatment_date,

    lch.latest_claim_date AS latest_claim_date,

    lch.latest_claim_hcp_npi,
    pdlch.provider_name     AS latest_claim_hcp_name,
    pdlch.primary_specialty AS latest_claim_hcp_specialty,
    COALESCE(lch.latest_claim_hcp_visit_count_5yr, 0) AS latest_claim_hcp_visit_count,

    ref1.hco_name AS latest_claim_hcp_hco_name,

    mrt.latest_treatment_date AS latest_treatment_date,

    lmt.latest_mpsii_tx_type,

    fta.first_tx_after_diagnosis,

    ROUND(MONTHS_BETWEEN(fta.first_tx_after_diagnosis, hfdx.incidence_date), 0)
      AS time_dx_to_first_tx_in_months,

    ROUND(MONTHS_BETWEEN(mrt.latest_treatment_date, fta.first_tx_after_diagnosis), 0) AS treatment_period_months,

    COALESCE(ef.Tivi_fills, 0) AS Tivi_fills,

    mrt.latest_treatment_hcp_npi,
    pdtch.provider_name     AS latest_treatment_hcp_name,
    pdtch.primary_specialty AS latest_treatment_hcp_specialty,
    COALESCE(mrt.latest_treatment_hcp_visit_count_5yr, 0) AS latest_treatment_hcp_visit_count,
    mrt.last_infusion_hcp_last_visit,
    ref2.hco_name AS latest_treatment_hcp_hco_name,

    fdh.first_dx_hcp AS first_dx_hcp_5yr,
    COALESCE(fdhs.first_dx_all_visit_count_5yr, 0) AS first_dx_all_visit_count_5yr,
    fdhs.first_dx_last_visit_5yr AS first_dx_last_visit_5yr,

    fth.first_tx_hcp AS first_tx_hcp_5yr,
    COALESCE(fths.first_tx_all_visit_count_5yr, 0) AS first_tx_all_visit_count_5yr,
    COALESCE(fthtx.first_tx_treatment_visit_count_5yr, 0) AS first_tx_treatment_visit_count_5yr,
    fths.first_tx_last_visit_5yr AS first_tx_last_visit_5yr,

    msp.most_seen_hcp1_3yr_ranked,
    COALESCE(msp.most_seen_hcp1_visit_count_5yr, 0) AS most_seen_hcp1_visit_count_5yr,
    msp.most_seen_hcp1_last_visit_5yr,

    msp.most_seen_hcp2_3yr_ranked,
    COALESCE(msp.most_seen_hcp2_visit_count_5yr, 0) AS most_seen_hcp2_visit_count_5yr,
    msp.most_seen_hcp2_last_visit_5yr,

    msp.most_seen_hcp3_3yr_ranked,
    COALESCE(msp.most_seen_hcp3_visit_count_5yr, 0) AS most_seen_hcp3_visit_count_5yr,
    msp.most_seen_hcp3_last_visit_5yr,

    msp.most_seen_hcp4_3yr_ranked,
    COALESCE(msp.most_seen_hcp4_visit_count_5yr, 0) AS most_seen_hcp4_visit_count_5yr,
    msp.most_seen_hcp4_last_visit_5yr,

    msp.most_seen_hcp5_3yr_ranked,
    COALESCE(msp.most_seen_hcp5_visit_count_5yr, 0) AS most_seen_hcp5_visit_count_5yr,
    msp.most_seen_hcp5_last_visit_5yr

FROM eligible_patients ep
LEFT JOIN patient_demographics pd
    ON ep.PATIENT_ID = pd.PATIENT_ID
LEFT JOIN patient_geography pg
    ON ep.PATIENT_ID = pg.PATIENT_ID

LEFT JOIN historical_first_dx hfdx
    ON ep.PATIENT_ID = hfdx.PATIENT_ID
LEFT JOIN historical_first_tx hftx
    ON ep.PATIENT_ID = hftx.PATIENT_ID

LEFT JOIN first_tx_after_diagnosis fta
    ON ep.PATIENT_ID = fta.PATIENT_ID

LEFT JOIN Tivi_fills ef
    ON ep.PATIENT_ID = ef.PATIENT_ID

LEFT JOIN latest_mpsii_treatment_type lmt
    ON ep.PATIENT_ID = lmt.PATIENT_ID

LEFT JOIN latest_claim_hcp_final lch
    ON ep.PATIENT_ID = lch.PATIENT_ID
LEFT JOIN provider_dim pdlch
    ON lch.latest_claim_hcp_npi = pdlch.npi
LEFT JOIN cmpa_insights_internal_schema.reference_file ref1
    ON lch.latest_claim_hcp_npi = ref1.hcp_npi

LEFT JOIN most_recent_tx_hcp_final mrt
    ON ep.PATIENT_ID = mrt.PATIENT_ID
LEFT JOIN provider_dim pdtch
    ON mrt.latest_treatment_hcp_npi = pdtch.npi
LEFT JOIN cmpa_insights_internal_schema.reference_file ref2
    ON mrt.latest_treatment_hcp_npi = ref2.hcp_npi

LEFT JOIN first_dx_hcp fdh
    ON ep.PATIENT_ID = fdh.PATIENT_ID
LEFT JOIN first_dx_hcp_5yr_stats fdhs
    ON ep.PATIENT_ID = fdhs.PATIENT_ID
LEFT JOIN first_tx_hcp fth
    ON ep.PATIENT_ID = fth.PATIENT_ID
LEFT JOIN first_tx_hcp_5yr_stats fths
    ON ep.PATIENT_ID = fths.PATIENT_ID
LEFT JOIN first_tx_hcp_5yr_tx_only fthtx
    ON ep.PATIENT_ID = fthtx.PATIENT_ID

LEFT JOIN most_seen_pivot msp
    ON ep.PATIENT_ID = msp.PATIENT_ID

ORDER BY ep.PATIENT_ID;

CREATE OR REPLACE TEMPORARY VIEW patient360_base AS
SELECT DISTINCT * FROM patient_hcp_visit_summary;


-- =============================================================================
-- CELL 15 (TIVI): mpsii_tx_claims (Tivi)
-- =============================================================================
CREATE OR REPLACE TEMP VIEW mpsii_tx_claims AS
WITH MPSII_Diagnoses_Specified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_RESULT = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
),

Patients_2Dx_Specified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Specified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

MPSII_Diagnoses_Unspecified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_RESULT = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
),

Patients_2Dx_Unspecified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Unspecified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

MPSII_Treatment_All AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('8497600101')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('8497600101')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE IN ('J3490','J3590','J9999')
          AND SERVICE_DATE >= '2026-03-01'
    ) t
),

MPSII_Treatment_Tivi_Only AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('8497600101')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('8497600101')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE IN ('J3490','J3590','J9999')
          AND SERVICE_DATE >= '2026-03-01'
    ) t
),

Patients_2Dx_Specified_With_Treatment AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Specified p
    INNER JOIN MPSII_Treatment_All t USING (PATIENT_ID)
),

Patients_Incremental_Unspecified AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Unspecified p
    INNER JOIN MPSII_Treatment_Tivi_Only t USING (PATIENT_ID)
    WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
),

eligible_patients AS (
    SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment
    UNION
    SELECT PATIENT_ID FROM Patients_Incremental_Unspecified
),

cohort_3_learnings AS (
  SELECT DISTINCT npi
  FROM com_raw.kom_providers
  WHERE provider_type = 'INDIVIDUAL'
    AND (
      PRIMARY_SPECIALTY NOT IN (
        'Anesthesiologist Assistant','Anesthesiology','Dentist','Dietitian, Registered',
        'Emergency Medical Technician, Basic','Emergency Medicine','General Acute Care Hospital',
        'Nurse Anesthetist, Certified Registered','Obstetrics & Gynecology','Pathology',
        'Radiology','Urology'
      )
      OR SECONDARY_SPECIALTY IN (
        'Child & Adolescent Psychiatry','Psychiatry','Adolescent Medicine','Developmental - Behavioral Pediatrics',
        'Neonatal-Perinatal Medicine','Nutrition, Pediatric','Oncology, Pediatrics','Pediatric Cardiology',
        'Pediatric Critical Care Medicine','Pediatric Dermatology','Pediatric Emergency Medicine',
        'Pediatric Endocrinology','Pediatric Gastroenterology','Pediatric Hematology-Oncology',
        'Pediatric Infectious Diseases','Pediatric Nephrology','Pediatric Ophthalmology and Strabismus Specialist',
        'Pediatric Orthopaedic Surgery','Pediatric Otolaryngology','Pediatric Pulmonology','Pediatric Radiology',
        'Pediatric Rehabilitation Medicine','Pediatric Rheumatology','Pediatric Surgery','Pediatrics',
        'Clinical Biochemical Genetics','Clinical Genetics (M.D.)','Clinical Molecular Genetics',
        'Ph.D. Medical Genetics','Neurodevelopmental Disabilities','Neurology',
        'Neurology with Special Qualifications in Child Neurology','Neuroradiology'
      )
    )
),

all_tx_claims_2yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('8497600101')
        AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
        PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('8497600101')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
        PATIENT_ID,
        RENDERING_NPI AS NPI,
        SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE PROCEDURE_CODE IN ('J3490','J3590','J9999')
        AND SERVICE_DATE >= '2026-03-01'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
)

SELECT * FROM all_tx_claims_2yr;


-- =============================================================================
-- CELL 16 (TIVI): most_recently_treated_hcp (Tivi)
-- =============================================================================
CREATE OR REPLACE TEMPORARY VIEW most_recently_treated_hcp AS
WITH tx_claims AS (
    SELECT DISTINCT *
    FROM mpsii_tx_claims
),

MPSII_Diagnoses_Specified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_RESULT = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
),

Patients_2Dx_Specified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Specified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

MPSII_Diagnoses_Unspecified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_RESULT = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
),

Patients_2Dx_Unspecified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Unspecified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

MPSII_Treatment_All AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('8497600101')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('8497600101')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE IN ('J3490','J3590','J9999')
          AND SERVICE_DATE >= '2026-03-01'
    ) t
),

MPSII_Treatment_Tivi_Only AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('8497600101')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('8497600101')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE  IN ('J3490','J3590','J9999')
          AND SERVICE_DATE >= '2026-03-01'
    ) t
),

Patients_2Dx_Specified_With_Treatment AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Specified p
    INNER JOIN MPSII_Treatment_All t USING (PATIENT_ID)
),

Patients_Incremental_Unspecified AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Unspecified p
    INNER JOIN MPSII_Treatment_Tivi_Only t USING (PATIENT_ID)
    WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
),

eligible_patients AS (
    SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment
    UNION
    SELECT PATIENT_ID FROM Patients_Incremental_Unspecified
),

cohort_3_learnings AS (
  SELECT DISTINCT npi
  FROM com_raw.kom_providers
  WHERE provider_type = 'INDIVIDUAL'
    AND (
      PRIMARY_SPECIALTY NOT IN (
        'Anesthesiologist Assistant','Anesthesiology','Dentist','Dietitian, Registered',
        'Emergency Medical Technician, Basic','Emergency Medicine','General Acute Care Hospital',
        'Nurse Anesthetist, Certified Registered','Obstetrics & Gynecology','Pathology',
        'Radiology','Urology'
      )
      OR SECONDARY_SPECIALTY IN (
        'Child & Adolescent Psychiatry','Psychiatry','Adolescent Medicine','Developmental - Behavioral Pediatrics',
        'Neonatal-Perinatal Medicine','Nutrition, Pediatric','Oncology, Pediatrics','Pediatric Cardiology',
        'Pediatric Critical Care Medicine','Pediatric Dermatology','Pediatric Emergency Medicine',
        'Pediatric Endocrinology','Pediatric Gastroenterology','Pediatric Hematology-Oncology',
        'Pediatric Infectious Diseases','Pediatric Nephrology','Pediatric Ophthalmology and Strabismus Specialist',
        'Pediatric Orthopaedic Surgery','Pediatric Otolaryngology','Pediatric Pulmonology','Pediatric Radiology',
        'Pediatric Rehabilitation Medicine','Pediatric Rheumatology','Pediatric Surgery','Pediatrics',
        'Clinical Biochemical Genetics','Clinical Genetics (M.D.)','Clinical Molecular Genetics',
        'Ph.D. Medical Genetics','Neurodevelopmental Disabilities','Neurology',
        'Neurology with Special Qualifications in Child Neurology','Neuroradiology'
      )
    )
),
all_dx_claims_5yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT
        PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE DIAGNOSIS_CODE IN ('E761','E763')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

all_tx_claims_5yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('8497600101')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT
        PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('8497600101')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT
        PATIENT_ID,
        RENDERING_NPI AS NPI,
        SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE PROCEDURE_CODE IN ('J3490','J3590','J9999')
        AND SERVICE_DATE >= '2026-03-01'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

all_claims AS (
    SELECT * FROM all_dx_claims_5yr
    UNION
    SELECT * FROM all_tx_claims_5yr
),

latest_treating_hcp AS (
    SELECT patient_id, npi
    FROM (
        SELECT
            patient_id,
            npi,
            fill_date,
            ROW_NUMBER() OVER (
                PARTITION BY patient_id
                ORDER BY
                    CASE WHEN npi IS NOT NULL THEN 1 ELSE 2 END,
                    fill_date DESC,
                    npi DESC
            ) AS rn
        FROM tx_claims
    ) t
    WHERE rn = 1
      AND npi IS NOT NULL
),

visit_counts AS (
    SELECT
        patient_id,
        npi,
        COUNT(DISTINCT fill_date) AS visit_counts
    FROM all_claims
    WHERE npi IS NOT NULL
    GROUP BY patient_id, npi
),

last_visit_date AS (
    SELECT
        lth.patient_id,
        lth.npi,
        MAX(ac.fill_date) AS last_visit_date
    FROM latest_treating_hcp lth
    LEFT JOIN all_claims ac
      ON lth.patient_id = ac.patient_id
     AND lth.npi = ac.npi
    GROUP BY lth.patient_id, lth.npi
),

latest_treating_hcp_with_visits AS (
    SELECT
        a.patient_id,
        a.npi AS most_recently_treated_hcp,
        b.visit_counts AS no_of_visits,
        c.last_visit_date AS last_visit_date_5yr
    FROM latest_treating_hcp AS a
    LEFT JOIN visit_counts AS b
        ON a.patient_id = b.patient_id
       AND a.npi        = b.npi
    LEFT JOIN last_visit_date AS c
        ON a.patient_id = c.patient_id
       AND a.npi        = c.npi
),

hcp_with_other_info AS (
    SELECT
        a.*,
        CONCAT(b.FIRST_NAME, ' ', b.LAST_NAME) AS hcp_name,
        b.PRIMARY_SPECIALTY AS hcp_specialty,
        c.hco_name,
        c.mapped_territory_id AS territory_id,
        c.territory,
        c.mapped_region_id AS region_id,
        c.region
    FROM latest_treating_hcp_with_visits AS a

    LEFT JOIN com_edp_prd.com_raw.kom_providers AS b
        ON a.most_recently_treated_hcp = b.NPI
       AND b.PROVIDER_TYPE = 'INDIVIDUAL'

    LEFT JOIN (
  SELECT
    * EXCEPT (hcp_primary_specialty),
    hcp_primary_specialty AS hcp_specialty
  FROM (
    SELECT
      a.* EXCEPT (territory_id, region_id),
      b.territory_id AS mapped_territory_id,
      c.region_id AS mapped_region_id
    FROM cmpa_insights_internal_schema.reference_file AS a
    LEFT JOIN (
      SELECT DISTINCT territory_id, territory_name
      FROM cmpa_insights_internal_schema.zip_to_territory_mapping
    ) AS b
      ON a.territory = b.territory_name
    LEFT JOIN (
      SELECT DISTINCT region_id, region_name
      FROM cmpa_insights_internal_schema.zip_to_territory_mapping
    ) AS c
      ON a.region = c.region_name
  )
) AS c
ON a.most_recently_treated_hcp = c.hcp_npi
)

SELECT
    patient_id,
    most_recently_treated_hcp AS most_recently_treated_hcp_2yr,
    hcp_name AS most_recently_treated_hcp_name_2yr,

    no_of_visits AS most_recently_treated_hcp_2yr_no_of_visits_5yr,
    last_visit_date_5yr AS most_recent_tx_hcp_2yr_last_visit_5yr,

    hcp_specialty AS most_recently_treated_hcp_specialty_2yr,
    hco_name AS most_recently_treated_hcp_hco_name,
    territory_id AS most_recently_treated_hcp_territory_id_2yr,
    territory AS most_recently_treated_hcp_territory_2yr,
    region_id AS most_recently_treated_hcp_region_id_2yr,
    region AS most_recently_treated_hcp_region_2yr
FROM hcp_with_other_info;


-- =============================================================================
-- CELL 17 (TIVI): all_dx_claims / all_tx_claims / primary_hcp build (Tivi)
-- =============================================================================
CREATE OR REPLACE TEMPORARY VIEW all_dx_claims AS

SELECT DISTINCT
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'DX' AS CLAIM_TYPE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)

UNION

SELECT DISTINCT
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    'DX' AS CLAIM_TYPE
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE DIAGNOSIS_CODE IN ('E761', 'E763')
  AND TRANSACTION_RESULT = 'PAID'
  AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters);

CREATE OR REPLACE TEMPORARY VIEW all_tx_claims AS

SELECT DISTINCT
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'TX' AS CLAIM_TYPE,
    NDC11 AS CODE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE NDC11 IN ('8497600101')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)

UNION

SELECT DISTINCT
    PATIENT_ID,
    RENDERING_NPI AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'TX' AS CLAIM_TYPE,
    PROCEDURE_CODE AS CODE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE PROCEDURE_CODE IN ('J3490','J3590','J9999')
  AND SERVICE_DATE >= '2026-03-01'

UNION

SELECT DISTINCT
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    'TX' AS CLAIM_TYPE,
    NDC11 AS CODE
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE NDC11 IN ('8497600101')
  AND TRANSACTION_RESULT = 'PAID'
  AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters);

CREATE OR REPLACE TEMPORARY VIEW tx_claims_2yr AS
SELECT DISTINCT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE, CODE
FROM all_tx_claims
WHERE FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters);


CREATE OR REPLACE TEMPORARY VIEW e761_patients_2dx AS
SELECT PATIENT_ID
FROM (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_RESULT = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;

CREATE OR REPLACE TEMPORARY VIEW specified_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e761_patients_2dx e
INNER JOIN tx_claims_2yr t ON e.PATIENT_ID = t.PATIENT_ID;

CREATE OR REPLACE TEMPORARY VIEW e763_patients_2dx AS
SELECT PATIENT_ID
FROM (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_RESULT = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;

CREATE OR REPLACE TEMPORARY VIEW Tivi_tx_2yr AS
SELECT DISTINCT PATIENT_ID
FROM tx_claims_2yr
WHERE CODE IN ('8497600101');

CREATE OR REPLACE TEMPORARY VIEW incremental_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e763_patients_2dx e
INNER JOIN Tivi_tx_2yr t ON e.PATIENT_ID = t.PATIENT_ID
WHERE e.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM specified_patients);

CREATE OR REPLACE TEMPORARY VIEW eligible_patients AS
SELECT PATIENT_ID FROM specified_patients
UNION
SELECT PATIENT_ID FROM incremental_patients;


CREATE OR REPLACE TEMPORARY VIEW cohort_3_learnings AS
SELECT DISTINCT npi
FROM com_raw.kom_providers
WHERE provider_type = 'INDIVIDUAL'
  AND (
    PRIMARY_SPECIALTY NOT IN (
      'Anesthesiologist Assistant','Anesthesiology','Dentist','Dietitian, Registered',
      'Emergency Medical Technician, Basic','Emergency Medicine','General Acute Care Hospital',
      'Nurse Anesthetist, Certified Registered','Obstetrics & Gynecology','Pathology',
      'Radiology','Urology'
    )
    OR SECONDARY_SPECIALTY IN (
      'Child & Adolescent Psychiatry','Psychiatry','Adolescent Medicine','Developmental - Behavioral Pediatrics',
      'Neonatal-Perinatal Medicine','Nutrition, Pediatric','Oncology, Pediatrics','Pediatric Cardiology',
      'Pediatric Critical Care Medicine','Pediatric Dermatology','Pediatric Emergency Medicine',
      'Pediatric Endocrinology','Pediatric Gastroenterology','Pediatric Hematology-Oncology',
      'Pediatric Infectious Diseases','Pediatric Nephrology','Pediatric Ophthalmology and Strabismus Specialist',
      'Pediatric Orthopaedic Surgery','Pediatric Otolaryngology','Pediatric Pulmonology','Pediatric Radiology',
      'Pediatric Rehabilitation Medicine','Pediatric Rheumatology','Pediatric Surgery','Pediatrics',
      'Clinical Biochemical Genetics','Clinical Genetics (M.D.)','Clinical Molecular Genetics',
      'Ph.D. Medical Genetics','Neurodevelopmental Disabilities','Neurology',
      'Neurology with Special Qualifications in Child Neurology','Neuroradiology'
    )
  );


CREATE OR REPLACE TEMPORARY VIEW all_patient_claims AS
SELECT *
FROM (
  SELECT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
  FROM all_dx_claims
  WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

  UNION

  SELECT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
  FROM all_tx_claims
  WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
)
WHERE npi IN (SELECT DISTINCT npi FROM cohort_3_learnings);


CREATE OR REPLACE TEMPORARY VIEW primary_hcp AS
WITH hcp_metrics AS (
    SELECT
        a.PATIENT_ID,
        a.NPI,

        CASE
            WHEN p.primary_specialty LIKE '%Genetic%'
              OR p.secondary_specialty LIKE '%Genetic%'
                THEN 'Geneticist'
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
              OR p.primary_specialty LIKE '%Neurological Surgery%'
                THEN 'Psychiatry & Neurology'
            WHEN p.primary_specialty LIKE '%Pediatrics%'
                THEN 'Pediatrician'
            WHEN p.primary_specialty LIKE '%Internal Medicine%'
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%'
              OR p.secondary_specialty LIKE '%Family Medicine%'
                THEN 'PCP'
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
              OR p.primary_specialty LIKE '%Physician Assistant%'
                THEN 'NPPA'
            WHEN a.NPI IS NULL
                THEN 'NA'
            ELSE 'Others'
        END AS SPECIALTY,

        CASE
            WHEN p.primary_specialty LIKE '%Genetic%'
              OR p.secondary_specialty LIKE '%Genetic%'
                THEN 1
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
              OR p.primary_specialty LIKE '%Neurological Surgery%'
                THEN 2
            WHEN p.primary_specialty LIKE '%Pediatrics%'
                THEN 3
            WHEN p.primary_specialty LIKE '%Internal Medicine%'
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%'
              OR p.secondary_specialty LIKE '%Family Medicine%'
                THEN 4
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
              OR p.primary_specialty LIKE '%Physician Assistant%'
                THEN 5
            WHEN a.NPI IS NULL
                THEN 7
            ELSE 6
        END AS SPECIALTY_PRIORITY,

        COUNT(DISTINCT a.FILL_DATE) AS NO_OF_VISITS,

        COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'DX' THEN a.FILL_DATE END) AS DX_VISITS,
        COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'TX' THEN a.FILL_DATE END) AS TX_VISITS,

        MAX(a.FILL_DATE) AS MOST_RECENT_VISIT

    FROM all_patient_claims a
    LEFT JOIN com_edp_prd.com_raw.kom_providers p
        ON a.NPI = p.NPI
    GROUP BY
        a.PATIENT_ID,
        a.NPI,
        p.primary_specialty,
        p.secondary_specialty
),
ranked_hcps AS (
    SELECT
        *,
        RANK() OVER (
            PARTITION BY PATIENT_ID
            ORDER BY
                SPECIALTY_PRIORITY ASC,
                NO_OF_VISITS DESC,
                MOST_RECENT_VISIT DESC,
                NPI ASC
        ) AS HCP_RANK
    FROM hcp_metrics
)
SELECT
    PATIENT_ID,
    NPI AS PRIMARY_HCP_NPI,
    SPECIALTY AS PRIMARY_HCP_SPECIALTY,
    SPECIALTY_PRIORITY,
    NO_OF_VISITS,
    DX_VISITS,
    TX_VISITS,
    MOST_RECENT_VISIT,
    HCP_RANK
FROM ranked_hcps
WHERE HCP_RANK = 1;


-- =============================================================================
-- CELL 18 (TIVI): tivi_patient360_master view (all columns prefixed tivi_)
-- =============================================================================
CREATE OR REPLACE TEMPORARY VIEW tivi_patient360_master AS
select PATIENT_ID as tivi_patient_id, PATIENT_YOB as tivi_patient_yob, PATIENT_AGE as tivi_patient_age, PATIENT_GENDER as tivi_patient_gender, patient_state as tivi_patient_state, incidence_date as tivi_incidence_date, first_incidence_treatment_date as tivi_first_incidence_treatment_date, latest_claim_date as tivi_latest_claim_date, latest_claim_hcp_npi as tivi_latest_claim_hcp_npi, latest_claim_hcp_name as tivi_latest_claim_hcp_name, latest_claim_hcp_specialty as tivi_latest_claim_hcp_specialty, latest_claim_hcp_visit_count as tivi_latest_claim_hcp_visit_count, latest_claim_hcp_hco_name as tivi_latest_claim_hcp_hco_name, latest_treatment_date as tivi_latest_treatment_date, latest_mpsii_tx_type as tivi_latest_mpsii_tx_type, first_tx_after_diagnosis as tivi_first_tx_after_diagnosis, time_dx_to_first_tx_in_months as tivi_time_dx_to_first_tx_in_months, treatment_period_months as tivi_treatment_period_months, Tivi_fills as tivi_fills, latest_treatment_hcp_npi as tivi_latest_treatment_hcp_npi, latest_treatment_hcp_name as tivi_latest_treatment_hcp_name, latest_treatment_hcp_specialty as tivi_latest_treatment_hcp_specialty, latest_treatment_hcp_visit_count as tivi_latest_treatment_hcp_visit_count, last_infusion_hcp_last_visit as tivi_last_infusion_hcp_last_visit,latest_treatment_hcp_hco_name as tivi_latest_treatment_hcp_hco_name, first_dx_hcp_5yr as tivi_first_dx_hcp_5yr, first_dx_all_visit_count_5yr as tivi_first_dx_all_visit_count_5yr, first_dx_last_visit_5yr as tivi_first_dx_last_visit_5yr, first_tx_hcp_5yr as tivi_first_tx_hcp_5yr, first_tx_all_visit_count_5yr as tivi_first_tx_all_visit_count_5yr, first_tx_treatment_visit_count_5yr as tivi_first_tx_treatment_visit_count_5yr, first_tx_last_visit_5yr as tivi_first_tx_last_visit_5yr, most_seen_hcp1_3yr_ranked as tivi_most_seen_hcp1_3yr_ranked, most_seen_hcp1_visit_count_5yr as tivi_most_seen_hcp1_visit_count_5yr, most_seen_hcp1_last_visit_5yr as tivi_most_seen_hcp1_last_visit_5yr, most_seen_hcp2_3yr_ranked as tivi_most_seen_hcp2_3yr_ranked, most_seen_hcp2_visit_count_5yr as tivi_most_seen_hcp2_visit_count_5yr, most_seen_hcp2_last_visit_5yr as tivi_most_seen_hcp2_last_visit_5yr, most_seen_hcp3_3yr_ranked as tivi_most_seen_hcp3_3yr_ranked, most_seen_hcp3_visit_count_5yr as tivi_most_seen_hcp3_visit_count_5yr, most_seen_hcp3_last_visit_5yr as tivi_most_seen_hcp3_last_visit_5yr, most_seen_hcp4_3yr_ranked as tivi_most_seen_hcp4_3yr_ranked, most_seen_hcp4_visit_count_5yr as tivi_most_seen_hcp4_visit_count_5yr, most_seen_hcp4_last_visit_5yr as tivi_most_seen_hcp4_last_visit_5yr, most_seen_hcp5_3yr_ranked as tivi_most_seen_hcp5_3yr_ranked, most_seen_hcp5_visit_count_5yr as tivi_most_seen_hcp5_visit_count_5yr, most_seen_hcp5_last_visit_5yr as tivi_most_seen_hcp5_last_visit_5yr, most_recently_treated_hcp_2yr as tivi_most_recently_treated_hcp_2yr, most_recently_treated_hcp_name_2yr as tivi_most_recently_treated_hcp_name_2yr, most_recently_treated_hcp_2yr_no_of_visits_5yr as tivi_most_recently_treated_hcp_2yr_no_of_visits_5yr, most_recent_tx_hcp_2yr_last_visit_5yr as tivi_most_recent_tx_hcp_2yr_last_visit_5yr, most_recently_treated_hcp_specialty_2yr as tivi_most_recently_treated_hcp_specialty_2yr, most_recently_treated_hcp_hco_name as tivi_most_recently_treated_hcp_hco_name, most_recently_treated_hcp_territory_id_2yr as tivi_most_recently_treated_hcp_territory_id_2yr, most_recently_treated_hcp_territory_2yr as tivi_most_recently_treated_hcp_territory_2yr, most_recently_treated_hcp_region_id_2yr as tivi_most_recently_treated_hcp_region_id_2yr, most_recently_treated_hcp_region_2yr as tivi_most_recently_treated_hcp_region_2yr, PRIMARY_HCP_NPI as tivi_primary_hcp_npi, PRIMARY_HCP_SPECIALTY as tivi_primary_hcp_specialty, SPECIALTY_PRIORITY as tivi_specialty_priority, NO_OF_VISITS as tivi_no_of_visits, DX_VISITS as tivi_dx_visits, TX_VISITS as tivi_tx_visits, MOST_RECENT_VISIT as tivi_most_recent_visit, HCP_RANK as tivi_hcp_rank
from (SELECT DISTINCT
    a.PATIENT_ID,
    a.PATIENT_YOB,
    a.PATIENT_AGE,
    a.PATIENT_GENDER,
    a.patient_state,

    a.incidence_date,
    a.first_incidence_treatment_date,

    a.latest_claim_date,

    a.latest_claim_hcp_npi,
    a.latest_claim_hcp_name,
    a.latest_claim_hcp_specialty,
    a.latest_claim_hcp_visit_count,

    a.latest_claim_hcp_hco_name,

    a.latest_treatment_date,
    a.latest_mpsii_tx_type,

    a.first_tx_after_diagnosis,
    a.time_dx_to_first_tx_in_months,
    a.treatment_period_months,

    a.Tivi_fills,

    a.latest_treatment_hcp_npi,
    a.latest_treatment_hcp_name,
    a.latest_treatment_hcp_specialty,
    a.latest_treatment_hcp_visit_count,
    a.last_infusion_hcp_last_visit,
    a.latest_treatment_hcp_hco_name,

    a.first_dx_hcp_5yr,
    a.first_dx_all_visit_count_5yr,
    a.first_dx_last_visit_5yr,
    a.first_tx_hcp_5yr,
    a.first_tx_all_visit_count_5yr,
    a.first_tx_treatment_visit_count_5yr,
    a.first_tx_last_visit_5yr,

    a.most_seen_hcp1_3yr_ranked,
    a.most_seen_hcp1_visit_count_5yr,
    a.most_seen_hcp1_last_visit_5yr,
    a.most_seen_hcp2_3yr_ranked,
    a.most_seen_hcp2_visit_count_5yr,
    a.most_seen_hcp2_last_visit_5yr,
    a.most_seen_hcp3_3yr_ranked,
    a.most_seen_hcp3_visit_count_5yr,
    a.most_seen_hcp3_last_visit_5yr,
    a.most_seen_hcp4_3yr_ranked,
    a.most_seen_hcp4_visit_count_5yr,
    a.most_seen_hcp4_last_visit_5yr,
    a.most_seen_hcp5_3yr_ranked,
    a.most_seen_hcp5_visit_count_5yr,
    a.most_seen_hcp5_last_visit_5yr,

    b.* EXCEPT (patient_id),

    c.* EXCEPT (patient_id)

FROM patient360_base AS a

LEFT JOIN most_recently_treated_hcp AS b
    ON a.PATIENT_ID = b.patient_id

LEFT JOIN primary_hcp AS c
    ON a.PATIENT_ID = c.patient_id);


-- =============================================================================
-- CELL 19 (TIVI): Merge tivi_* columns into the existing patient360_master
-- Note: EXCEPT() in Spark SQL takes unqualified column names, not table.col.
-- =============================================================================
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master AS
WITH base AS (
  SELECT * FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
),
t2 AS (
  SELECT a.*, b.* EXCEPT (tivi_patient_id, tivi_patient_yob, tivi_patient_age, tivi_patient_gender, tivi_patient_state)
  FROM base AS a
  LEFT JOIN tivi_patient360_master AS b
    ON a.patient_id = b.tivi_patient_id
)
SELECT DISTINCT * FROM t2;

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.patient360_master

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.patient360_master
where PATIENT_ID = 'LZT622TW'

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.patient360_master
where primary_hcp_territory_id_2yr is null and latest_treatment_type in ('ELAPRASE', 'OTHER ERT')

In [0]:
SELECT 
    latest_insurance_type, 
    COUNT(DISTINCT PATIENT_ID) AS patient_count
FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
GROUP BY latest_insurance_type
ORDER BY patient_count DESC;

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient_360_hcp_table AS
WITH reference_dedup AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY hcp_npi
                   ORDER BY territory_id
               ) rn
        FROM cmpa_insights_internal_schema.reference_file
    )
    WHERE rn = 1
),
 
hcp_base AS (
    SELECT DISTINCT
        r.hcp_npi AS hcp_id,
        r.hcp_name,
        r.reporting_parent_tier,
        r.hco_veeva_crm_id AS hco_id,
        r.hco_name,
        r.hco_city,
        r.hco_state,
        NULLIF(r.territory_id, '-') AS territory_id,
        NULLIF(r.region_id, '-') AS region_id,
        r.territory AS territory_name,
        r.region AS region_name
    FROM reference_dedup r
),
 
patient_hcp AS (
    SELECT
        patient_id,
        Primary_HCP_NPI AS hcp_id,
        patient_age,
        latest_treatment_type
    FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
),
 
hcp_universe AS (
    SELECT DISTINCT hcp_id
    FROM patient_hcp
),
 
total_patients AS (
    SELECT
        hcp_id,
        COUNT(DISTINCT patient_id) AS total_mps_ii_patient
    FROM patient_hcp
    GROUP BY hcp_id
),
 
avlayah_claims_base AS (
    SELECT DISTINCT patient_id
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE (
            NDC11 IN ('8497600101')
            OR PROCEDURE_CODE IN ('J3490','J3590','J9999')
          )
      AND service_date >= '2026-03-01'
 
    UNION
 
    SELECT DISTINCT patient_id
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('8497600101')
      AND transaction_result = 'PAID'
      AND fill_date >= '2026-03-01'
),
 
avlayah_claims AS (
    SELECT
        p.hcp_id,
        COUNT(DISTINCT p.patient_id) AS avlayah_claims_pt_ct
    FROM patient_hcp p
    INNER JOIN avlayah_claims_base a
        ON p.patient_id = a.patient_id
    WHERE p.patient_age < 17
    GROUP BY p.hcp_id
),
 
elaprase_claims_base AS (
    SELECT DISTINCT patient_id
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE procedure_code IN ('J1743')
      AND service_date BETWEEN '2020-08-01'
      AND (SELECT MAX(end_date) FROM runtime_parameters)
 
    UNION
 
    SELECT DISTINCT patient_id
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND transaction_result = 'PAID'
      AND fill_date BETWEEN '2020-08-01'
      AND (SELECT MAX(end_date) FROM runtime_parameters)
),
 
elaprase_claims AS (
    SELECT
        p.hcp_id,
        COUNT(DISTINCT p.patient_id) AS elaprase_claims_pt_ct
    FROM patient_hcp p
    INNER JOIN elaprase_claims_base e
        ON p.patient_id = e.patient_id
    WHERE p.patient_age < 17
    GROUP BY p.hcp_id
),
 
avlayah_sp AS (
    SELECT
        b.hcp_id,
        COUNT(DISTINCT s.patient_id) AS avlayah_sp_pt_ct
    FROM hcp_base b
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.sp_patients_avlayah s
        ON b.territory_id = s.territory_id
    INNER JOIN com_edp_prd.cmpa_insights_internal_schema.patient360_master p
        ON s.patient_id = p.patient_id
    WHERE p.patient_age < 17
    GROUP BY b.hcp_id
),

avlayah_hub AS (
    SELECT
        b.hcp_id,
        COUNT(DISTINCT h.patient_id) AS avlayah_hub_pt_ct
    FROM hcp_base b
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.hub_patients_avlayah h
        ON b.territory_id = h.territory_id
    INNER JOIN com_edp_prd.cmpa_insights_internal_schema.patient360_master p
        ON h.patient_id = p.patient_id
    WHERE p.patient_age < 17
    GROUP BY b.hcp_id
),

avlayah_crm AS (
    SELECT
        b.hcp_id,
        SUM(c.crm_patient_count) AS avlayah_crm_pt_ct
    FROM hcp_base b
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.crm_patients_avlayah c
        ON b.territory_id = c.territory_id
    GROUP BY b.hcp_id
),
 
total_mpsii_avlayah_elaprase AS (
    SELECT
        hcp_id,
        COUNT(DISTINCT patient_id) AS total_mpsii_avlayah_elaprase
    FROM patient_hcp
    WHERE UPPER(latest_treatment_type) IN ('AVLAYAH','ELAPRASE')
    GROUP BY hcp_id
),
 
total_mpsii_avlayah AS (
    SELECT
        hcp_id,
        COUNT(DISTINCT patient_id) AS total_mpsii_avlayah
    FROM patient_hcp
    WHERE UPPER(latest_treatment_type) = 'AVLAYAH'
    GROUP BY hcp_id
),
 
total_mpsii_elaprase AS (
    SELECT
        hcp_id,
        COUNT(DISTINCT patient_id) AS total_mpsii_elaprase
    FROM patient_hcp
    WHERE UPPER(latest_treatment_type) = 'ELAPRASE'
    GROUP BY hcp_id
),
 
total_mpsii_avlayah_u17 AS (
    SELECT
        hcp_id,
        COUNT(DISTINCT patient_id) AS total_mpsii_avlayah_u17
    FROM patient_hcp
    WHERE UPPER(latest_treatment_type) = 'AVLAYAH'
      AND patient_age < 17
    GROUP BY hcp_id
),
 
total_mpsii_elaprase_u17 AS (
    SELECT
        hcp_id,
        COUNT(DISTINCT patient_id) AS total_mpsii_elaprase_u17
    FROM patient_hcp
    WHERE UPPER(latest_treatment_type) = 'ELAPRASE'
      AND patient_age < 17
    GROUP BY hcp_id
),
 
total_mpsii_avlayah_elaprase_u17 AS (
    SELECT
        hcp_id,
        COUNT(DISTINCT patient_id) AS total_mpsii_avlayah_elaprase_u17
    FROM patient_hcp
    WHERE UPPER(latest_treatment_type) IN ('AVLAYAH','ELAPRASE')
      AND patient_age < 17
    GROUP BY hcp_id
),
 
total_mpsii_u17 AS (
    SELECT
        hcp_id,
        COUNT(DISTINCT patient_id) AS total_mpsii_u17
    FROM patient_hcp
    WHERE patient_age < 17
      AND latest_treatment_type IS NOT NULL
    GROUP BY hcp_id
)
 
SELECT
    u.hcp_id,
    b.hco_id,
    b.hco_name,
    b.reporting_parent_tier,
    b.hcp_name,
    b.hco_city,
    b.hco_state,
    b.territory_id,
    b.territory_name,
    b.region_id,
    b.region_name,
    COALESCE(t.total_mps_ii_patient,0) AS total_mps_ii_patient,
    COALESCE(a.avlayah_claims_pt_ct,0) AS avlayah_claims_pt_ct,
    COALESCE(e.elaprase_claims_pt_ct,0) AS elaprase_claims_pt_ct,
    COALESCE(u17.total_mpsii_u17,0) AS total_mpsii_u17,
    COALESCE(sp.avlayah_sp_pt_ct,0) AS avlayah_sp_pt_ct,
    COALESCE(h.avlayah_hub_pt_ct,0) AS avlayah_hub_pt_ct,
    COALESCE(c.avlayah_crm_pt_ct,0) AS avlayah_crm_pt_ct,
    COALESCE(m.total_mpsii_avlayah_elaprase,0) AS total_mpsii_avlayah_elaprase,
    COALESCE(av.total_mpsii_avlayah,0) AS total_mpsii_avlayah,
    COALESCE(el.total_mpsii_elaprase,0) AS total_mpsii_elaprase,
    COALESCE(av17.total_mpsii_avlayah_u17,0) AS total_mpsii_avlayah_u17,
    COALESCE(el17.total_mpsii_elaprase_u17,0) AS total_mpsii_elaprase_u17,
    COALESCE(m17.total_mpsii_avlayah_elaprase_u17,0) AS total_mpsii_avlayah_elaprase_u17
FROM hcp_universe u
LEFT JOIN hcp_base b ON u.hcp_id = b.hcp_id
LEFT JOIN total_mpsii_u17 u17 ON u.hcp_id <=> u17.hcp_id
LEFT JOIN total_patients t ON u.hcp_id <=> t.hcp_id
LEFT JOIN avlayah_claims a ON u.hcp_id <=> a.hcp_id
LEFT JOIN elaprase_claims e ON u.hcp_id <=> e.hcp_id
LEFT JOIN avlayah_sp sp ON u.hcp_id <=> sp.hcp_id
LEFT JOIN avlayah_hub h ON u.hcp_id <=> h.hcp_id
LEFT JOIN avlayah_crm c ON u.hcp_id <=> c.hcp_id
LEFT JOIN total_mpsii_avlayah_elaprase m ON u.hcp_id <=> m.hcp_id
LEFT JOIN total_mpsii_avlayah av ON u.hcp_id <=> av.hcp_id
LEFT JOIN total_mpsii_elaprase el ON u.hcp_id <=> el.hcp_id
LEFT JOIN total_mpsii_avlayah_u17 av17 ON u.hcp_id <=> av17.hcp_id
LEFT JOIN total_mpsii_elaprase_u17 el17 ON u.hcp_id <=> el17.hcp_id
LEFT JOIN total_mpsii_avlayah_elaprase_u17 m17 ON u.hcp_id <=> m17.hcp_id;
 
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient_360_hco_table AS
 
WITH reference_dedup AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY hco_veeva_crm_id
                   ORDER BY territory_id
               ) rn
        FROM cmpa_insights_internal_schema.reference_file
    )
    WHERE rn = 1
),
 
hco_base AS (
    SELECT DISTINCT
        r.hco_veeva_crm_id,
        r.hco_name,
        r.reporting_parent_tier,
        r.hco_city,
        r.hco_state,
        NULLIF(r.territory_id, '-') AS territory_id,
        NULLIF(r.region_id, '-') AS region_id,
        r.territory AS territory_name,
        r.region AS region_name
    FROM reference_dedup r
),
 
patient_hco AS (
    SELECT
        patient_id,
        primary_hcp_hco_name_2yr AS hco_name,
        patient_age,
        latest_treatment_type
    FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
),
 
hco_universe AS (
    SELECT DISTINCT hco_name
    FROM patient_hco
),
 
total_patients AS (
    SELECT
        hco_name,
        COUNT(DISTINCT patient_id) AS total_mps_ii_patient
    FROM patient_hco
    GROUP BY hco_name
),
 
avlayah_claims_base AS (
    SELECT DISTINCT patient_id
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE (
            NDC11 IN ('8497600101')
            OR PROCEDURE_CODE IN ('J3490','J3590','J9999')
          )
      AND service_date >= '2026-03-01'
 
    UNION
 
    SELECT DISTINCT patient_id
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('8497600101')
      AND transaction_result = 'PAID'
      AND fill_date >= '2026-03-01'
),
 
avlayah_claims AS (
    SELECT
        p.hco_name,
        COUNT(DISTINCT p.patient_id) AS avlayah_claims_pt_ct
    FROM patient_hco p
    INNER JOIN avlayah_claims_base a
        ON p.patient_id = a.patient_id
    WHERE p.patient_age < 17
    GROUP BY p.hco_name
),
 
elaprase_claims_base AS (
    SELECT DISTINCT patient_id
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE procedure_code IN ('J1743')
      AND service_date BETWEEN '2020-08-01'
      AND (SELECT MAX(end_date) FROM runtime_parameters)
 
    UNION
 
    SELECT DISTINCT patient_id
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND transaction_result = 'PAID'
      AND fill_date BETWEEN '2020-08-01'
      AND (SELECT MAX(end_date) FROM runtime_parameters)
),
 
elaprase_claims AS (
    SELECT
        p.hco_name,
        COUNT(DISTINCT p.patient_id) AS elaprase_claims_pt_ct
    FROM patient_hco p
    INNER JOIN elaprase_claims_base e
        ON p.patient_id = e.patient_id
    WHERE p.patient_age < 17
    GROUP BY p.hco_name
),
 
total_mpsii_avlayah_elaprase AS (
    SELECT
        hco_name,
        COUNT(DISTINCT patient_id) AS total_mpsii_avlayah_elaprase
    FROM patient_hco
    WHERE UPPER(latest_treatment_type) IN ('AVLAYAH','ELAPRASE')
    GROUP BY hco_name
),
 
total_mpsii_avlayah AS (
    SELECT
        hco_name,
        COUNT(DISTINCT patient_id) AS total_mpsii_avlayah
    FROM patient_hco
    WHERE UPPER(latest_treatment_type) = 'AVLAYAH'
    GROUP BY hco_name
),
 
total_mpsii_elaprase AS (
    SELECT
        hco_name,
        COUNT(DISTINCT patient_id) AS total_mpsii_elaprase
    FROM patient_hco
    WHERE UPPER(latest_treatment_type) = 'ELAPRASE'
    GROUP BY hco_name
),
 
total_mpsii_avlayah_u17 AS (
    SELECT
        hco_name,
        COUNT(DISTINCT patient_id) AS total_mpsii_avlayah_u17
    FROM patient_hco
    WHERE UPPER(latest_treatment_type) = 'AVLAYAH'
      AND patient_age < 17
    GROUP BY hco_name
),
 
total_mpsii_elaprase_u17 AS (
    SELECT
        hco_name,
        COUNT(DISTINCT patient_id) AS total_mpsii_elaprase_u17
    FROM patient_hco
    WHERE UPPER(latest_treatment_type) = 'ELAPRASE'
      AND patient_age < 17
    GROUP BY hco_name
),
 
total_mpsii_avlayah_elaprase_u17 AS (
    SELECT
        hco_name,
        COUNT(DISTINCT patient_id) AS total_mpsii_avlayah_elaprase_u17
    FROM patient_hco
    WHERE UPPER(latest_treatment_type) IN ('AVLAYAH','ELAPRASE')
      AND patient_age < 17
    GROUP BY hco_name
),
 avlayah_sp AS (
    SELECT
        b.hco_name,
        COUNT(DISTINCT s.patient_id) AS avlayah_sp_pt_ct
    FROM hco_base b
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.sp_patients_avlayah s
        ON b.territory_id = s.territory_id
    INNER JOIN com_edp_prd.cmpa_insights_internal_schema.patient360_master p
        ON s.patient_id = p.patient_id
    WHERE p.patient_age < 17
    GROUP BY b.hco_name
),

avlayah_hub AS (
    SELECT
        b.hco_name,
        COUNT(DISTINCT h.patient_id) AS avlayah_hub_pt_ct
    FROM hco_base b
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.hub_patients_avlayah h
        ON b.territory_id = h.territory_id
    INNER JOIN com_edp_prd.cmpa_insights_internal_schema.patient360_master p
        ON h.patient_id = p.patient_id
    WHERE p.patient_age < 17
    GROUP BY b.hco_name
),

avlayah_crm AS (
    SELECT
        b.hco_name,
        SUM(c.crm_patient_count) AS avlayah_crm_pt_ct
    FROM hco_base b
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.crm_patients_avlayah c
        ON b.territory_id = c.territory_id
    GROUP BY b.hco_name
),
 
-- FIXED: added UPPER(TRIM()) in PARTITION BY to normalize name casing/spacing
hco_ordered AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY UPPER(TRIM(account_facility_name))
                   ORDER BY WTD DESC
               ) AS rn
        FROM com_edp_prd.cmpa_insights_internal_schema.HCO_Ordered_Account_Level
    )
    WHERE rn = 1
),
 
total_mpsii_u17 AS (
    SELECT
        hco_name,
        COUNT(DISTINCT patient_id) AS total_mpsii_u17
    FROM patient_hco
    WHERE patient_age < 17
      AND latest_treatment_type IS NOT NULL
    GROUP BY hco_name
),

hco_ordered_flag AS (
    SELECT DISTINCT
        UPPER(TRIM(account_facility_name)) AS hco_name_norm,
        'YES' AS hco_ordered_flag
    FROM com_edp_prd.cmpa_insights_internal_schema.HCO_Ordered_Account_Level
    WHERE account_type = 'HCO'
)
 
SELECT
    b.hco_veeva_crm_id,
    u.hco_name,
    b.reporting_parent_tier,
    b.hco_city,
    b.hco_state,
    b.territory_id,
    b.territory_name,
    b.region_id,
    b.region_name,
    COALESCE(t.total_mps_ii_patient,0) AS total_mps_ii_patient,
    COALESCE(a.avlayah_claims_pt_ct,0) AS avlayah_claims_pt_ct,
    COALESCE(e.elaprase_claims_pt_ct,0) AS elaprase_claims_pt_ct,
    COALESCE(u17.total_mpsii_u17,0) AS total_mpsii_u17,
    COALESCE(sp.avlayah_sp_pt_ct,0) AS avlayah_sp_pt_ct,
    COALESCE(h.avlayah_hub_pt_ct,0) AS avlayah_hub_pt_ct,
    COALESCE(c.avlayah_crm_pt_ct,0) AS avlayah_crm_pt_ct,
    COALESCE(m.total_mpsii_avlayah_elaprase,0) AS total_mpsii_avlayah_elaprase,
    COALESCE(av.total_mpsii_avlayah,0) AS total_mpsii_avlayah,
    COALESCE(el.total_mpsii_elaprase,0) AS total_mpsii_elaprase,
    COALESCE(av17.total_mpsii_avlayah_u17,0) AS total_mpsii_avlayah_u17,
    COALESCE(el17.total_mpsii_elaprase_u17,0) AS total_mpsii_elaprase_u17,
    COALESCE(m17.total_mpsii_avlayah_elaprase_u17,0) AS total_mpsii_avlayah_elaprase_u17,
    COALESCE(hof.hco_ordered_flag,'NO') AS hco_ordered_flag,
    ho.crx_account_id,
    ho.account_type AS ordered_account_type,
    ho.postal_code AS ordered_postal_code,
    ho.LTD AS ordered_ltd_vials,
    ho.MTD AS ordered_mtd_vials,
    ho.QTD AS ordered_qtd_vials,
    ho.YTD AS ordered_ytd_vials,
    ho.WTD_START_DATE,
    ho.WTD_END_DATE,
    ho.WTD AS ordered_wtd_vials
FROM hco_universe u
LEFT JOIN hco_base b
    ON u.hco_name = b.hco_name
LEFT JOIN total_patients t ON u.hco_name <=> t.hco_name
LEFT JOIN avlayah_claims a ON u.hco_name <=> a.hco_name
LEFT JOIN elaprase_claims e ON u.hco_name <=> e.hco_name
LEFT JOIN avlayah_sp sp ON u.hco_name <=> sp.hco_name
LEFT JOIN avlayah_hub h ON u.hco_name <=> h.hco_name
LEFT JOIN avlayah_crm c ON u.hco_name <=> c.hco_name
LEFT JOIN total_mpsii_u17 u17 ON u.hco_name <=> u17.hco_name
LEFT JOIN total_mpsii_avlayah_elaprase m ON u.hco_name <=> m.hco_name
LEFT JOIN total_mpsii_avlayah av ON u.hco_name <=> av.hco_name
LEFT JOIN total_mpsii_elaprase el ON u.hco_name <=> el.hco_name
LEFT JOIN total_mpsii_avlayah_u17 av17 ON u.hco_name <=> av17.hco_name
LEFT JOIN total_mpsii_elaprase_u17 el17 ON u.hco_name <=> el17.hco_name
LEFT JOIN total_mpsii_avlayah_elaprase_u17 m17 ON u.hco_name <=> m17.hco_name
LEFT JOIN hco_ordered_flag hof ON UPPER(TRIM(b.hco_name)) = hof.hco_name_norm
LEFT JOIN hco_ordered ho ON UPPER(TRIM(b.hco_name)) = UPPER(TRIM(ho.account_facility_name));

In [0]:
WITH patient_hcos AS (
    SELECT DISTINCT
        UPPER(TRIM(primary_hcp_hco_name_2yr)) AS hco_name
    FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
    WHERE primary_hcp_hco_name_2yr IS NOT NULL
),

ordered_hcos AS (
    SELECT DISTINCT
        UPPER(TRIM(account_facility_name)) AS hco_name
    FROM com_edp_prd.cmpa_insights_internal_schema.HCO_Ordered_Account_Level
    WHERE account_type = 'HCO'
)

SELECT
    p.hco_name,
    CASE
        WHEN o.hco_name IS NOT NULL THEN 'YES'
        ELSE 'NO'
    END AS hco_ordered_flag
FROM patient_hcos p
LEFT JOIN ordered_hcos o
    ON p.hco_name = o.hco_name
ORDER BY hco_ordered_flag DESC, p.hco_name;

In [0]:
WITH patient_hcos AS (
    SELECT DISTINCT
        UPPER(TRIM(primary_hcp_hco_name_2yr)) AS hco_name
    FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
    WHERE primary_hcp_hco_name_2yr IS NOT NULL
),

ordered_hcos AS (
    SELECT DISTINCT
        UPPER(TRIM(account_facility_name)) AS hco_name
    FROM com_edp_prd.cmpa_insights_internal_schema.HCO_Ordered_Account_Level
    WHERE account_type = 'HCO'
)

SELECT
    COUNT(DISTINCT p.hco_name) AS total_hcos,
    COUNT(DISTINCT o.hco_name) AS matched_hcos,
    COUNT(DISTINCT p.hco_name) - COUNT(DISTINCT o.hco_name) AS unmatched_hcos
FROM patient_hcos p
LEFT JOIN ordered_hcos o
    ON p.hco_name = o.hco_name;

In [0]:
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient_360_hcp_table AS
-- WITH reference_dedup AS (

--     SELECT *

--     FROM (

--         SELECT *,

--                ROW_NUMBER() OVER (

--                    PARTITION BY hcp_npi

--                    ORDER BY territory_id

--                ) rn

--         FROM cmpa_insights_internal_schema.reference_file

--     )

--     WHERE rn = 1

-- ),
 
-- hcp_base AS (

--     SELECT DISTINCT

--         r.hcp_npi AS hcp_id,

--         r.hcp_name,

--         r.reporting_parent_tier,

--         r.hco_veeva_crm_id AS hco_id,

--         r.hco_name,

--         r.hco_city,

--         r.hco_state,

--         r.territory_id,

--         r.territory AS territory_name,

--         r.region_id,

--         r.region AS region_name

--     FROM reference_dedup r

-- ),
 
-- patient_hcp AS (

--     SELECT

--         patient_id,

--         Primary_HCP_NPI AS hcp_id,

--         patient_age,

--         latest_mpsii_tx_type

--     FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master

-- ),
 
-- hcp_universe AS (

--     SELECT DISTINCT hcp_id

--     FROM patient_hcp

-- ),
 
-- total_patients AS (

--     SELECT

--         hcp_id,

--         COUNT(DISTINCT patient_id) AS total_mps_ii_patient

--     FROM patient_hcp

--     GROUP BY hcp_id

-- ),
 
-- avlayah_claims_base AS (

--     SELECT DISTINCT patient_id

--     FROM com_edp_prd.com_raw.kom_medical_events

--     WHERE (

--             NDC11 IN ('8497600101')

--             OR PROCEDURE_CODE IN ('J3490','J3590','J9999')

--           )

--       AND service_date >= '2026-03-01'
 
--     UNION
 
--     SELECT DISTINCT patient_id

--     FROM com_edp_prd.com_raw.kom_pharmacy_events

--     WHERE NDC11 IN ('8497600101')

--       AND transaction_result = 'PAID'

--       AND fill_date >= '2026-03-01'

-- ),
 
-- avlayah_claims AS (

--     SELECT

--         p.hcp_id,

--         COUNT(DISTINCT p.patient_id) AS avlayah_claims_pt_ct

--     FROM patient_hcp p

--     INNER JOIN avlayah_claims_base a

--         ON p.patient_id = a.patient_id

--     WHERE p.patient_age < 17

--     GROUP BY p.hcp_id

-- ),
 
-- elaprase_claims_base AS (

--     SELECT DISTINCT patient_id

--     FROM com_edp_prd.com_raw.kom_medical_events

--     WHERE procedure_code IN ('J1743')

--       AND service_date BETWEEN '2020-08-01'

--       AND (SELECT MAX(end_date) FROM runtime_parameters)
 
--     UNION
 
--     SELECT DISTINCT patient_id

--     FROM com_edp_prd.com_raw.kom_pharmacy_events

--     WHERE NDC11 IN ('54092070001','540920700')

--       AND transaction_result = 'PAID'

--       AND fill_date BETWEEN '2020-08-01'

--       AND (SELECT MAX(end_date) FROM runtime_parameters)

-- ),
 
-- elaprase_claims AS (

--     SELECT

--         p.hcp_id,

--         COUNT(DISTINCT p.patient_id) AS elaprase_claims_pt_ct

--     FROM patient_hcp p

--     INNER JOIN elaprase_claims_base e

--         ON p.patient_id = e.patient_id

--     WHERE p.patient_age < 17

--     GROUP BY p.hcp_id

-- ),
 
-- avlayah_sp AS (

--     SELECT hcp_id, 0 AS avlayah_sp_pt_ct

--     FROM hcp_universe

-- ),
 
-- avlayah_hub AS (

--     SELECT hcp_id, 0 AS avlayah_hub_pt_ct

--     FROM hcp_universe

-- ),
 
-- avlayah_crm AS (

--     SELECT hcp_id, 0 AS avlayah_crm_pt_ct

--     FROM hcp_universe

-- ),
 
-- total_mpsii_avlayah_elaprase AS (

--     SELECT

--         hcp_id,

--         COUNT(DISTINCT patient_id) AS total_mpsii_avlayah_elaprase

--     FROM patient_hcp

--     WHERE UPPER(latest_mpsii_tx_type) IN ('AVLAYAH','ELAPRASE')

--     GROUP BY hcp_id

-- ),
 
-- total_mpsii_avlayah AS (

--     SELECT

--         hcp_id,

--         COUNT(DISTINCT patient_id) AS total_mpsii_avlayah

--     FROM patient_hcp

--     WHERE UPPER(latest_mpsii_tx_type) = 'AVLAYAH'

--     GROUP BY hcp_id

-- ),
 
-- total_mpsii_elaprase AS (

--     SELECT

--         hcp_id,

--         COUNT(DISTINCT patient_id) AS total_mpsii_elaprase

--     FROM patient_hcp

--     WHERE UPPER(latest_mpsii_tx_type) = 'ELAPRASE'

--     GROUP BY hcp_id

-- ),
 
-- total_mpsii_avlayah_u17 AS (

--     SELECT

--         hcp_id,

--         COUNT(DISTINCT patient_id) AS total_mpsii_avlayah_u17

--     FROM patient_hcp

--     WHERE UPPER(latest_mpsii_tx_type) = 'AVLAYAH'

--       AND patient_age < 17

--     GROUP BY hcp_id

-- ),
 
-- total_mpsii_elaprase_u17 AS (

--     SELECT

--         hcp_id,

--         COUNT(DISTINCT patient_id) AS total_mpsii_elaprase_u17

--     FROM patient_hcp

--     WHERE UPPER(latest_mpsii_tx_type) = 'ELAPRASE'

--       AND patient_age < 17

--     GROUP BY hcp_id

-- ),
 
-- total_mpsii_avlayah_elaprase_u17 AS (

--     SELECT

--         hcp_id,

--         COUNT(DISTINCT patient_id) AS total_mpsii_avlayah_elaprase_u17

--     FROM patient_hcp

--     WHERE UPPER(latest_mpsii_tx_type) IN ('AVLAYAH','ELAPRASE')

--       AND patient_age < 17

--     GROUP BY hcp_id

-- ),
 
-- total_mpsii_u17 AS (

--     SELECT
--         hcp_id,

--         COUNT(DISTINCT patient_id) AS total_mpsii_u17

--     FROM patient_hcp

--     WHERE patient_age < 17
--       AND latest_mpsii_tx_type IS NOT NULL

--     GROUP BY hcp_id

-- )
 
-- SELECT

--     u.hcp_id,

--     b.hco_id,

--     b.hco_name,

--     b.reporting_parent_tier,

--     b.hcp_name,

--     b.hco_city,

--     b.hco_state,

--     b.territory_id,

--     b.territory_name,

--     b.region_id,

--     b.region_name,

--     COALESCE(t.total_mps_ii_patient,0) AS total_mps_ii_patient,

--     COALESCE(u17.total_mpsii_u17,0) AS total_mpsii_u17,

--     COALESCE(a.avlayah_claims_pt_ct,0) AS avlayah_claims_pt_ct,

--     COALESCE(e.elaprase_claims_pt_ct,0) AS elaprase_claims_pt_ct,

--     COALESCE(sp.avlayah_sp_pt_ct,0) AS avlayah_sp_pt_ct,

--     COALESCE(h.avlayah_hub_pt_ct,0) AS avlayah_hub_pt_ct,

--     COALESCE(c.avlayah_crm_pt_ct,0) AS avlayah_crm_pt_ct,

--     COALESCE(m.total_mpsii_avlayah_elaprase,0) AS total_mpsii_avlayah_elaprase,

--     COALESCE(av.total_mpsii_avlayah,0) AS total_mpsii_avlayah,

--     COALESCE(el.total_mpsii_elaprase,0) AS total_mpsii_elaprase,

--     COALESCE(av17.total_mpsii_avlayah_u17,0) AS total_mpsii_avlayah_u17,

--     COALESCE(el17.total_mpsii_elaprase_u17,0) AS total_mpsii_elaprase_u17,

--     COALESCE(m17.total_mpsii_avlayah_elaprase_u17,0) AS total_mpsii_avlayah_elaprase_u17

-- FROM hcp_universe u

-- LEFT JOIN hcp_base b ON u.hcp_id = b.hcp_id

-- LEFT JOIN total_patients t ON u.hcp_id <=> t.hcp_id

-- LEFT JOIN total_mpsii_u17 u17 ON u.hcp_id <=> u17.hcp_id

-- LEFT JOIN avlayah_claims a ON u.hcp_id <=> a.hcp_id

-- LEFT JOIN elaprase_claims e ON u.hcp_id <=> e.hcp_id

-- LEFT JOIN avlayah_sp sp ON u.hcp_id <=> sp.hcp_id

-- LEFT JOIN avlayah_hub h ON u.hcp_id <=> h.hcp_id

-- LEFT JOIN avlayah_crm c ON u.hcp_id <=> c.hcp_id

-- LEFT JOIN total_mpsii_avlayah_elaprase m ON u.hcp_id <=> m.hcp_id

-- LEFT JOIN total_mpsii_avlayah av ON u.hcp_id <=> av.hcp_id

-- LEFT JOIN total_mpsii_elaprase el ON u.hcp_id <=> el.hcp_id

-- LEFT JOIN total_mpsii_avlayah_u17 av17 ON u.hcp_id <=> av17.hcp_id

-- LEFT JOIN total_mpsii_elaprase_u17 el17 ON u.hcp_id <=> el17.hcp_id

-- LEFT JOIN total_mpsii_avlayah_elaprase_u17 m17 ON u.hcp_id <=> m17.hcp_id;
 



 
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient_360_hco_table AS
 
-- WITH reference_dedup AS (

--     SELECT *

--     FROM (

--         SELECT *,

--                ROW_NUMBER() OVER (

--                    PARTITION BY hco_veeva_crm_id

--                    ORDER BY territory_id

--                ) rn

--         FROM cmpa_insights_internal_schema.reference_file

--     )

--     WHERE rn = 1

-- ),
 
-- hco_base AS (

--     SELECT DISTINCT

--         r.hco_veeva_crm_id,

--         r.hco_name,

--         r.reporting_parent_tier,

--         r.hco_city,

--         r.hco_state,

--         r.territory_id,

--         r.territory AS territory_name,

--         r.region_id,

--         r.region AS region_name

--     FROM reference_dedup r

-- ),
 
-- patient_hco AS (

--     SELECT

--         patient_id,

--         primary_hcp_hco_name_2yr AS hco_name,

--         patient_age,

--         latest_mpsii_tx_type

--     FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master

-- ),
 
-- hco_universe AS (

--     SELECT DISTINCT hco_name

--     FROM patient_hco

-- ),
 
-- total_patients AS (

--     SELECT

--         hco_name,

--         COUNT(DISTINCT patient_id) AS total_mps_ii_patient

--     FROM patient_hco

--     GROUP BY hco_name

-- ),
 
-- avlayah_claims_base AS (

--     SELECT DISTINCT patient_id

--     FROM com_edp_prd.com_raw.kom_medical_events

--     WHERE (

--             NDC11 IN ('8497600101')

--             OR PROCEDURE_CODE IN ('J3490','J3590','J9999')

--           )

--       AND service_date >= '2026-03-01'
 
--     UNION
 
--     SELECT DISTINCT patient_id

--     FROM com_edp_prd.com_raw.kom_pharmacy_events

--     WHERE NDC11 IN ('8497600101')

--       AND transaction_result = 'PAID'

--       AND fill_date >= '2026-03-01'

-- ),
 
-- avlayah_claims AS (

--     SELECT

--         p.hco_name,

--         COUNT(DISTINCT p.patient_id) AS avlayah_claims_pt_ct

--     FROM patient_hco p

--     INNER JOIN avlayah_claims_base a

--         ON p.patient_id = a.patient_id

--     WHERE p.patient_age < 17

--     GROUP BY p.hco_name

-- ),
 
-- elaprase_claims_base AS (

--     SELECT DISTINCT patient_id

--     FROM com_edp_prd.com_raw.kom_medical_events

--     WHERE procedure_code IN ('J1743')

--       AND service_date BETWEEN '2020-08-01'

--       AND (SELECT MAX(end_date) FROM runtime_parameters)
 
--     UNION
 
--     SELECT DISTINCT patient_id

--     FROM com_edp_prd.com_raw.kom_pharmacy_events

--     WHERE NDC11 IN ('54092070001','540920700')

--       AND transaction_result = 'PAID'

--       AND fill_date BETWEEN '2020-08-01'

--       AND (SELECT MAX(end_date) FROM runtime_parameters)

-- ),
 
-- elaprase_claims AS (

--     SELECT

--         p.hco_name,

--         COUNT(DISTINCT p.patient_id) AS elaprase_claims_pt_ct

--     FROM patient_hco p

--     INNER JOIN elaprase_claims_base e

--         ON p.patient_id = e.patient_id

--     WHERE p.patient_age < 17

--     GROUP BY p.hco_name

-- ),
 
-- total_mpsii_avlayah_elaprase AS (

--     SELECT

--         hco_name,

--         COUNT(DISTINCT patient_id) AS total_mpsii_avlayah_elaprase

--     FROM patient_hco

--     WHERE UPPER(latest_mpsii_tx_type) IN ('AVLAYAH','ELAPRASE')

--     GROUP BY hco_name

-- ),
 
-- total_mpsii_avlayah AS (

--     SELECT

--         hco_name,

--         COUNT(DISTINCT patient_id) AS total_mpsii_avlayah

--     FROM patient_hco

--     WHERE UPPER(latest_mpsii_tx_type) = 'AVLAYAH'

--     GROUP BY hco_name

-- ),
 
-- total_mpsii_elaprase AS (

--     SELECT

--         hco_name,

--         COUNT(DISTINCT patient_id) AS total_mpsii_elaprase

--     FROM patient_hco

--     WHERE UPPER(latest_mpsii_tx_type) = 'ELAPRASE'

--     GROUP BY hco_name

-- ),
 
-- total_mpsii_avlayah_u17 AS (

--     SELECT

--         hco_name,

--         COUNT(DISTINCT patient_id) AS total_mpsii_avlayah_u17

--     FROM patient_hco

--     WHERE UPPER(latest_mpsii_tx_type) = 'AVLAYAH'

--       AND patient_age < 17

--     GROUP BY hco_name

-- ),
 
-- total_mpsii_elaprase_u17 AS (

--     SELECT

--         hco_name,

--         COUNT(DISTINCT patient_id) AS total_mpsii_elaprase_u17

--     FROM patient_hco

--     WHERE UPPER(latest_mpsii_tx_type) = 'ELAPRASE'

--       AND patient_age < 17

--     GROUP BY hco_name

-- ),
 
-- total_mpsii_avlayah_elaprase_u17 AS (

--     SELECT

--         hco_name,

--         COUNT(DISTINCT patient_id) AS total_mpsii_avlayah_elaprase_u17

--     FROM patient_hco

--     WHERE UPPER(latest_mpsii_tx_type) IN ('AVLAYAH','ELAPRASE')

--       AND patient_age < 17

--     GROUP BY hco_name

-- ),
 
-- avlayah_sp AS (

--     SELECT hco_name, 0 AS avlayah_sp_pt_ct

--     FROM hco_universe

-- ),
 
-- avlayah_hub AS (

--     SELECT hco_name, 0 AS avlayah_hub_pt_ct

--     FROM hco_universe

-- ),
 
-- avlayah_crm AS (

--     SELECT hco_name, 0 AS avlayah_crm_pt_ct

--     FROM hco_universe

-- ),
 
-- hco_ordered AS (

--     SELECT *

--     FROM (

--         SELECT *,

--                ROW_NUMBER() OVER (

--                    PARTITION BY account_facility_name

--                    ORDER BY WTD DESC

--                ) AS rn

--         FROM com_edp_prd.cmpa_insights_internal_schema.HCO_Ordered_Account_Level

--     )

--     WHERE rn = 1

-- ),
 
-- total_mpsii_u17 AS (

--     SELECT
--         hco_name,

--         COUNT(DISTINCT patient_id) AS total_mpsii_u17

--     FROM patient_hco

--     WHERE patient_age < 17
--       AND latest_mpsii_tx_type IS NOT NULL

--     GROUP BY hco_name

-- )
 
-- SELECT

--     b.hco_veeva_crm_id,

--     u.hco_name,

--     b.reporting_parent_tier,

--     b.hco_city,

--     b.hco_state,

--     b.territory_id,

--     b.territory_name,

--     b.region_id,

--     b.region_name,

--     COALESCE(t.total_mps_ii_patient,0) AS total_mps_ii_patient,

--     COALESCE(u17.total_mpsii_u17,0) AS total_mpsii_u17,

--     COALESCE(a.avlayah_claims_pt_ct,0) AS avlayah_claims_pt_ct,

--     COALESCE(e.elaprase_claims_pt_ct,0) AS elaprase_claims_pt_ct,

--     COALESCE(sp.avlayah_sp_pt_ct,0) AS avlayah_sp_pt_ct,

--     COALESCE(h.avlayah_hub_pt_ct,0) AS avlayah_hub_pt_ct,

--     COALESCE(c.avlayah_crm_pt_ct,0) AS avlayah_crm_pt_ct,

--     COALESCE(m.total_mpsii_avlayah_elaprase,0) AS total_mpsii_avlayah_elaprase,

--     COALESCE(av.total_mpsii_avlayah,0) AS total_mpsii_avlayah,

--     COALESCE(el.total_mpsii_elaprase,0) AS total_mpsii_elaprase,

--     COALESCE(av17.total_mpsii_avlayah_u17,0) AS total_mpsii_avlayah_u17,

--     COALESCE(el17.total_mpsii_elaprase_u17,0) AS total_mpsii_elaprase_u17,

--     COALESCE(m17.total_mpsii_avlayah_elaprase_u17,0) AS total_mpsii_avlayah_elaprase_u17,

--     ho.crx_account_id,

--     ho.account_type AS ordered_account_type,

--     ho.postal_code AS ordered_postal_code,

--     ho.LTD AS ordered_ltd_vials,

--     ho.MTD AS ordered_mtd_vials,

--     ho.QTD AS ordered_qtd_vials,

--     ho.YTD AS ordered_ytd_vials,

--     ho.WTD_START_DATE,

--     ho.WTD_END_DATE,

--     ho.WTD AS ordered_wtd_vials

-- FROM hco_universe u

-- LEFT JOIN hco_base b

--     ON u.hco_name = b.hco_name

-- LEFT JOIN total_patients t ON u.hco_name <=> t.hco_name

-- LEFT JOIN total_mpsii_u17 u17 ON u.hco_name <=> u17.hco_name

-- LEFT JOIN avlayah_claims a ON u.hco_name <=> a.hco_name

-- LEFT JOIN elaprase_claims e ON u.hco_name <=> e.hco_name

-- LEFT JOIN avlayah_sp sp ON u.hco_name <=> sp.hco_name

-- LEFT JOIN avlayah_hub h ON u.hco_name <=> h.hco_name

-- LEFT JOIN avlayah_crm c ON u.hco_name <=> c.hco_name

-- LEFT JOIN total_mpsii_avlayah_elaprase m ON u.hco_name <=> m.hco_name

-- LEFT JOIN total_mpsii_avlayah av ON u.hco_name <=> av.hco_name

-- LEFT JOIN total_mpsii_elaprase el ON u.hco_name <=> el.hco_name

-- LEFT JOIN total_mpsii_avlayah_u17 av17 ON u.hco_name <=> av17.hco_name

-- LEFT JOIN total_mpsii_elaprase_u17 el17 ON u.hco_name <=> el17.hco_name

-- LEFT JOIN total_mpsii_avlayah_elaprase_u17 m17 ON u.hco_name <=> m17.hco_name

-- LEFT JOIN hco_ordered ho

--     ON b.hco_name = ho.account_facility_name;
--  CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient_360_hcp_table AS

-- WITH reference_dedup AS (
--     SELECT *
--     FROM (
--         SELECT *,
--                ROW_NUMBER() OVER (
--                    PARTITION BY hcp_npi
--                    ORDER BY territory_id
--                ) rn
--         FROM cmpa_insights_internal_schema.reference_file
--     )
--     WHERE rn = 1
-- ),

-- hcp_base AS (
--     SELECT DISTINCT
--         r.hcp_npi AS hcp_id,
--         r.hcp_name,
--         r.reporting_parent_tier,
--         r.hco_veeva_crm_id AS hco_id,
--         r.hco_name,
--         r.hco_city,
--         r.hco_state,
--         r.territory_id,
--         r.territory AS territory_name,
--         r.region_id,
--         r.region AS region_name
--     FROM reference_dedup r
-- ),

-- patient_hcp AS (
--     SELECT
--         patient_id,
--         Primary_HCP_NPI AS hcp_id,
--         patient_age,
--         latest_mpsii_tx_type
--     FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
-- ),

-- hcp_universe AS (
--     SELECT DISTINCT hcp_id
--     FROM patient_hcp
-- ),

-- total_patients AS (
--     SELECT
--         hcp_id,
--         COUNT(DISTINCT patient_id) AS total_mps_ii_patient
--     FROM patient_hcp
--     GROUP BY hcp_id
-- ),

-- avlayah_claims_base AS (
--     SELECT DISTINCT patient_id
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE (
--             NDC11 IN ('8497600101')
--             OR PROCEDURE_CODE IN ('J3490','J3590','J9999')
--           )
--       AND service_date >= '2026-03-01'

--     UNION

--     SELECT DISTINCT patient_id
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE NDC11 IN ('8497600101')
--       AND transaction_result = 'PAID'
--       AND fill_date >= '2026-03-01'
-- ),

-- avlayah_claims AS (
--     SELECT
--         p.hcp_id,
--         COUNT(DISTINCT p.patient_id) AS avlayah_claims_pt_ct
--     FROM patient_hcp p
--     INNER JOIN avlayah_claims_base a
--         ON p.patient_id = a.patient_id
--     WHERE p.patient_age < 17
--     GROUP BY p.hcp_id
-- ),

-- elaprase_claims_base AS (
--     SELECT DISTINCT patient_id
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE procedure_code IN ('J1743')
--       AND service_date BETWEEN '2020-08-01'
--       AND (SELECT MAX(end_date) FROM runtime_parameters)

--     UNION

--     SELECT DISTINCT patient_id
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE NDC11 IN ('54092070001','540920700')
--       AND transaction_result = 'PAID'
--       AND fill_date BETWEEN '2020-08-01'
--       AND (SELECT MAX(end_date) FROM runtime_parameters)
-- ),

-- elaprase_claims AS (
--     SELECT
--         p.hcp_id,
--         COUNT(DISTINCT p.patient_id) AS elaprase_claims_pt_ct
--     FROM patient_hcp p
--     INNER JOIN elaprase_claims_base e
--         ON p.patient_id = e.patient_id
--     WHERE p.patient_age < 17
--     GROUP BY p.hcp_id
-- ),

-- avlayah_sp AS (
--     SELECT hcp_id, 0 AS avlayah_sp_pt_ct
--     FROM hcp_universe
-- ),

-- avlayah_hub AS (
--     SELECT hcp_id, 0 AS avlayah_hub_pt_ct
--     FROM hcp_universe
-- ),

-- avlayah_crm AS (
--     SELECT hcp_id, 0 AS avlayah_crm_pt_ct
--     FROM hcp_universe
-- ),

-- total_mpsii_avlayah_elaprase AS (
--     SELECT
--         hcp_id,
--         COUNT(DISTINCT patient_id) AS total_mpsii_avlayah_elaprase
--     FROM patient_hcp
--     WHERE UPPER(latest_mpsii_tx_type) IN ('AVLAYAH','ELAPRASE')
--     GROUP BY hcp_id
-- ),

-- total_mpsii_avlayah AS (
--     SELECT
--         hcp_id,
--         COUNT(DISTINCT patient_id) AS total_mpsii_avlayah
--     FROM patient_hcp
--     WHERE UPPER(latest_mpsii_tx_type) = 'AVLAYAH'
--     GROUP BY hcp_id
-- ),

-- total_mpsii_elaprase AS (
--     SELECT
--         hcp_id,
--         COUNT(DISTINCT patient_id) AS total_mpsii_elaprase
--     FROM patient_hcp
--     WHERE UPPER(latest_mpsii_tx_type) = 'ELAPRASE'
--     GROUP BY hcp_id
-- ),

-- total_mpsii_avlayah_u17 AS (
--     SELECT
--         hcp_id,
--         COUNT(DISTINCT patient_id) AS total_mpsii_avlayah_u17
--     FROM patient_hcp
--     WHERE UPPER(latest_mpsii_tx_type) = 'AVLAYAH'
--       AND patient_age < 17
--     GROUP BY hcp_id
-- ),

-- total_mpsii_elaprase_u17 AS (
--     SELECT
--         hcp_id,
--         COUNT(DISTINCT patient_id) AS total_mpsii_elaprase_u17
--     FROM patient_hcp
--     WHERE UPPER(latest_mpsii_tx_type) = 'ELAPRASE'
--       AND patient_age < 17
--     GROUP BY hcp_id
-- ),

-- total_mpsii_avlayah_elaprase_u17 AS (
--     SELECT
--         hcp_id,
--         COUNT(DISTINCT patient_id) AS total_mpsii_avlayah_elaprase_u17
--     FROM patient_hcp
--     WHERE UPPER(latest_mpsii_tx_type) IN ('AVLAYAH','ELAPRASE')
--       AND patient_age < 17
--     GROUP BY hcp_id
-- ),

-- total_mpsii_u17 AS (

--     SELECT
--         hcp_id,
--         COUNT(DISTINCT patient_id) AS total_mpsii_u17
--     FROM patient_hcp
--     WHERE patient_age < 17
--       AND latest_mpsii_tx_type IS NOT NULL
--     GROUP BY hcp_id

-- )

-- SELECT
--     u.hcp_id,
--     b.hco_id,
--     b.hco_name,
--     b.reporting_parent_tier,
--     b.hcp_name,
--     b.hco_city,
--     b.hco_state,
--     b.territory_id,
--     b.territory_name,
--     b.region_id,
--     b.region_name,
--     COALESCE(t.total_mps_ii_patient,0) AS total_mps_ii_patient,
--     COALESCE(a.avlayah_claims_pt_ct,0) AS avlayah_claims_pt_ct,
--     COALESCE(e.elaprase_claims_pt_ct,0) AS elaprase_claims_pt_ct,
--     COALESCE(u17.total_mpsii_u17,0) AS total_mpsii_u17,
--     COALESCE(sp.avlayah_sp_pt_ct,0) AS avlayah_sp_pt_ct,
--     COALESCE(h.avlayah_hub_pt_ct,0) AS avlayah_hub_pt_ct,
--     COALESCE(c.avlayah_crm_pt_ct,0) AS avlayah_crm_pt_ct,
--     COALESCE(m.total_mpsii_avlayah_elaprase,0) AS total_mpsii_avlayah_elaprase,
--     COALESCE(av.total_mpsii_avlayah,0) AS total_mpsii_avlayah,
--     COALESCE(el.total_mpsii_elaprase,0) AS total_mpsii_elaprase,
--     COALESCE(av17.total_mpsii_avlayah_u17,0) AS total_mpsii_avlayah_u17,
--     COALESCE(el17.total_mpsii_elaprase_u17,0) AS total_mpsii_elaprase_u17,
--     COALESCE(m17.total_mpsii_avlayah_elaprase_u17,0) AS total_mpsii_avlayah_elaprase_u17
-- FROM hcp_universe u
-- LEFT JOIN hcp_base b ON u.hcp_id = b.hcp_id
-- LEFT JOIN total_mpsii_u17 u17 ON u.hcp_id <=> u17.hcp_id
-- LEFT JOIN total_patients t ON u.hcp_id <=> t.hcp_id
-- LEFT JOIN avlayah_claims a ON u.hcp_id <=> a.hcp_id
-- LEFT JOIN elaprase_claims e ON u.hcp_id <=> e.hcp_id
-- LEFT JOIN avlayah_sp sp ON u.hcp_id <=> sp.hcp_id
-- LEFT JOIN avlayah_hub h ON u.hcp_id <=> h.hcp_id
-- LEFT JOIN avlayah_crm c ON u.hcp_id <=> c.hcp_id
-- LEFT JOIN total_mpsii_avlayah_elaprase m ON u.hcp_id <=> m.hcp_id
-- LEFT JOIN total_mpsii_avlayah av ON u.hcp_id <=> av.hcp_id
-- LEFT JOIN total_mpsii_elaprase el ON u.hcp_id <=> el.hcp_id
-- LEFT JOIN total_mpsii_avlayah_u17 av17 ON u.hcp_id <=> av17.hcp_id
-- LEFT JOIN total_mpsii_elaprase_u17 el17 ON u.hcp_id <=> el17.hcp_id
-- LEFT JOIN total_mpsii_avlayah_elaprase_u17 m17 ON u.hcp_id <=> m17.hcp_id;

-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient_360_hco_table AS

-- WITH reference_dedup AS (
--     SELECT *
--     FROM (
--         SELECT *,
--                ROW_NUMBER() OVER (
--                    PARTITION BY hco_veeva_crm_id
--                    ORDER BY territory_id
--                ) rn
--         FROM cmpa_insights_internal_schema.reference_file
--     )
--     WHERE rn = 1
-- ),

-- hco_base AS (
--     SELECT DISTINCT
--         r.hco_veeva_crm_id,
--         r.hco_name,
--         r.reporting_parent_tier,
--         r.hco_city,
--         r.hco_state,
--         r.territory_id,
--         r.territory AS territory_name,
--         r.region_id,
--         r.region AS region_name
--     FROM reference_dedup r
-- ),

-- patient_hco AS (
--     SELECT
--         patient_id,
--         primary_hcp_hco_name_2yr AS hco_name,
--         patient_age,
--         latest_mpsii_tx_type
--     FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
-- ),

-- hco_universe AS (
--     SELECT DISTINCT hco_name
--     FROM patient_hco
-- ),

-- total_patients AS (
--     SELECT
--         hco_name,
--         COUNT(DISTINCT patient_id) AS total_mps_ii_patient
--     FROM patient_hco
--     GROUP BY hco_name
-- ),

-- avlayah_claims_base AS (
--     SELECT DISTINCT patient_id
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE (
--             NDC11 IN ('8497600101')
--             OR PROCEDURE_CODE IN ('J3490','J3590','J9999')
--           )
--       AND service_date >= '2026-03-01'

--     UNION

--     SELECT DISTINCT patient_id
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE NDC11 IN ('8497600101')
--       AND transaction_result = 'PAID'
--       AND fill_date >= '2026-03-01'
-- ),

-- avlayah_claims AS (
--     SELECT
--         p.hco_name,
--         COUNT(DISTINCT p.patient_id) AS avlayah_claims_pt_ct
--     FROM patient_hco p
--     INNER JOIN avlayah_claims_base a
--         ON p.patient_id = a.patient_id
--     WHERE p.patient_age < 17
--     GROUP BY p.hco_name
-- ),

-- elaprase_claims_base AS (
--     SELECT DISTINCT patient_id
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE procedure_code IN ('J1743')
--       AND service_date BETWEEN '2020-08-01'
--       AND (SELECT MAX(end_date) FROM runtime_parameters)

--     UNION

--     SELECT DISTINCT patient_id
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE NDC11 IN ('54092070001','540920700')
--       AND transaction_result = 'PAID'
--       AND fill_date BETWEEN '2020-08-01'
--       AND (SELECT MAX(end_date) FROM runtime_parameters)
-- ),

-- elaprase_claims AS (
--     SELECT
--         p.hco_name,
--         COUNT(DISTINCT p.patient_id) AS elaprase_claims_pt_ct
--     FROM patient_hco p
--     INNER JOIN elaprase_claims_base e
--         ON p.patient_id = e.patient_id
--     WHERE p.patient_age < 17
--     GROUP BY p.hco_name
-- ),

-- total_mpsii_avlayah_elaprase AS (
--     SELECT
--         hco_name,
--         COUNT(DISTINCT patient_id) AS total_mpsii_avlayah_elaprase
--     FROM patient_hco
--     WHERE UPPER(latest_mpsii_tx_type) IN ('AVLAYAH','ELAPRASE')
--     GROUP BY hco_name
-- ),

-- total_mpsii_avlayah AS (
--     SELECT
--         hco_name,
--         COUNT(DISTINCT patient_id) AS total_mpsii_avlayah
--     FROM patient_hco
--     WHERE UPPER(latest_mpsii_tx_type) = 'AVLAYAH'
--     GROUP BY hco_name
-- ),

-- total_mpsii_elaprase AS (
--     SELECT
--         hco_name,
--         COUNT(DISTINCT patient_id) AS total_mpsii_elaprase
--     FROM patient_hco
--     WHERE UPPER(latest_mpsii_tx_type) = 'ELAPRASE'
--     GROUP BY hco_name
-- ),

-- total_mpsii_avlayah_u17 AS (
--     SELECT
--         hco_name,
--         COUNT(DISTINCT patient_id) AS total_mpsii_avlayah_u17
--     FROM patient_hco
--     WHERE UPPER(latest_mpsii_tx_type) = 'AVLAYAH'
--       AND patient_age < 17
--     GROUP BY hco_name
-- ),

-- total_mpsii_elaprase_u17 AS (
--     SELECT
--         hco_name,
--         COUNT(DISTINCT patient_id) AS total_mpsii_elaprase_u17
--     FROM patient_hco
--     WHERE UPPER(latest_mpsii_tx_type) = 'ELAPRASE'
--       AND patient_age < 17
--     GROUP BY hco_name
-- ),

-- total_mpsii_avlayah_elaprase_u17 AS (
--     SELECT
--         hco_name,
--         COUNT(DISTINCT patient_id) AS total_mpsii_avlayah_elaprase_u17
--     FROM patient_hco
--     WHERE UPPER(latest_mpsii_tx_type) IN ('AVLAYAH','ELAPRASE')
--       AND patient_age < 17
--     GROUP BY hco_name
-- ),

-- avlayah_sp AS (
--     SELECT hco_name, 0 AS avlayah_sp_pt_ct
--     FROM hco_universe
-- ),

-- avlayah_hub AS (
--     SELECT hco_name, 0 AS avlayah_hub_pt_ct
--     FROM hco_universe
-- ),

-- avlayah_crm AS (
--     SELECT hco_name, 0 AS avlayah_crm_pt_ct
--     FROM hco_universe
-- ),

-- hco_ordered AS (
--     SELECT *
--     FROM (
--         SELECT *,
--                ROW_NUMBER() OVER (
--                    PARTITION BY account_facility_name
--                    ORDER BY WTD DESC
--                ) AS rn
--         FROM com_edp_prd.cmpa_insights_internal_schema.HCO_Ordered_Account_Level
--     )
--     WHERE rn = 1
-- ),

-- total_mpsii_u17 AS (

--     SELECT
--         hco_name,
--         COUNT(DISTINCT patient_id) AS total_mpsii_u17
--     FROM patient_hco
--     WHERE patient_age < 17
--       AND latest_mpsii_tx_type IS NOT NULL
--     GROUP BY hco_name

-- )

-- SELECT
--     b.hco_veeva_crm_id,
--     u.hco_name,
--     b.reporting_parent_tier,
--     b.hco_city,
--     b.hco_state,
--     b.territory_id,
--     b.territory_name,
--     b.region_id,
--     b.region_name,
--     COALESCE(t.total_mps_ii_patient,0) AS total_mps_ii_patient,
--     COALESCE(a.avlayah_claims_pt_ct,0) AS avlayah_claims_pt_ct,
--     COALESCE(e.elaprase_claims_pt_ct,0) AS elaprase_claims_pt_ct,
--     COALESCE(u17.total_mpsii_u17,0) AS total_mpsii_u17,
--     COALESCE(sp.avlayah_sp_pt_ct,0) AS avlayah_sp_pt_ct,
--     COALESCE(h.avlayah_hub_pt_ct,0) AS avlayah_hub_pt_ct,
--     COALESCE(c.avlayah_crm_pt_ct,0) AS avlayah_crm_pt_ct,
--     COALESCE(m.total_mpsii_avlayah_elaprase,0) AS total_mpsii_avlayah_elaprase,
--     COALESCE(av.total_mpsii_avlayah,0) AS total_mpsii_avlayah,
--     COALESCE(el.total_mpsii_elaprase,0) AS total_mpsii_elaprase,
--     COALESCE(av17.total_mpsii_avlayah_u17,0) AS total_mpsii_avlayah_u17,
--     COALESCE(el17.total_mpsii_elaprase_u17,0) AS total_mpsii_elaprase_u17,
--     COALESCE(m17.total_mpsii_avlayah_elaprase_u17,0) AS total_mpsii_avlayah_elaprase_u17,
--     ho.crx_account_id,
--     ho.account_type AS ordered_account_type,
--     ho.postal_code AS ordered_postal_code,
--     ho.LTD AS ordered_ltd_vials,
--     ho.MTD AS ordered_mtd_vials,
--     ho.QTD AS ordered_qtd_vials,
--     ho.YTD AS ordered_ytd_vials,
--     ho.WTD_START_DATE,
--     ho.WTD_END_DATE,
--     ho.WTD AS ordered_wtd_vials
-- FROM hco_universe u
-- LEFT JOIN hco_base b
--     ON u.hco_name = b.hco_name
-- LEFT JOIN total_patients t ON u.hco_name <=> t.hco_name
-- LEFT JOIN avlayah_claims a ON u.hco_name <=> a.hco_name
-- LEFT JOIN elaprase_claims e ON u.hco_name <=> e.hco_name
-- LEFT JOIN avlayah_sp sp ON u.hco_name <=> sp.hco_name
-- LEFT JOIN avlayah_hub h ON u.hco_name <=> h.hco_name
-- LEFT JOIN avlayah_crm c ON u.hco_name <=> c.hco_name
-- LEFT JOIN total_mpsii_u17 u17 ON u.hco_name <=> u17.hco_name
-- LEFT JOIN total_mpsii_avlayah_elaprase m ON u.hco_name <=> m.hco_name
-- LEFT JOIN total_mpsii_avlayah av ON u.hco_name <=> av.hco_name
-- LEFT JOIN total_mpsii_elaprase el ON u.hco_name <=> el.hco_name
-- LEFT JOIN total_mpsii_avlayah_u17 av17 ON u.hco_name <=> av17.hco_name
-- LEFT JOIN total_mpsii_elaprase_u17 el17 ON u.hco_name <=> el17.hco_name
-- LEFT JOIN total_mpsii_avlayah_elaprase_u17 m17 ON u.hco_name <=> m17.hco_name
-- LEFT JOIN hco_ordered ho
--     ON b.hco_name = ho.account_facility_name;

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.patient360_master

In [0]:
SELECT *
FROM (
    SELECT 
        t.*,
        ROW_NUMBER() OVER (
            PARTITION BY PATIENT_ID 
            ORDER BY SERVICE_DATE DESC
        ) AS rn
    FROM com_edp_prd.com_raw.kom_medical_events t
    WHERE PATIENT_ID IN (
        '00P0S2C0','261HTDTY','2XPN1XBS','33PEP1TE','39BDW50Z',
        '5HREMYP4','7TH16LJF','7W3FYDPQ','9GQLZ50J','GYR5QB5F',
        'K3VNKN2P','LS4M6FMC','MS7PBQ8Z','P80QX3EB','RBJZ2XC4','SFKC682M'
    )
    -- AND SERVICE_DATE >= '2026-03-01'
    AND PROCEDURE_CODE = 'J1743'
) x
WHERE rn = 1;

In [0]:
select * from com_edp_prd.com_raw.kom_medical_events where NDC11 IN ('54092070001')
and PATIENT_ID IN (
        '00P0S2C0','261HTDTY','2XPN1XBS','33PEP1TE','39BDW50Z',
        '5HREMYP4','7TH16LJF','7W3FYDPQ','9GQLZ50J','GYR5QB5F',
        'K3VNKN2P','LS4M6FMC','MS7PBQ8Z','P80QX3EB','RBJZ2XC4','SFKC682M'
    )
    and PROCEDURE_CODE not in ('J1743')

In [0]:
select * from com_edp_prd.com_raw.kom_medical_events
where PATIENT_ID = '9GQLZ50J'

In [0]:
select * from com_edp_prd.com_raw.kom_medical_events
WHERE PROCEDURE_CODE IN ('J3490','J3590','J9999')
      AND SERVICE_DATE >= '2026-03-01'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

In [0]:
select sum(avlayah_crm_pt_ct)
from com_edp_prd.cmpa_insights_internal_schema.patient_360_hcp_table

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.sp_patients_avlayah

In [0]:
Select * from com_edp_prd.cmpa_insights_internal_schema.patient_360_hcp_table

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.patient_360_hco_table

In [0]:
select sum(total_mps_ii_patient) from com_edp_prd.cmpa_insights_internal_schema.patient_360_hcp_table

In [0]:
SELECT *
FROM com_edp_prd.com_raw.vcrm_account__v

In [0]:
SELECT DISTINCT *
FROM com_edp_prd.com_raw.kom_medical_events
WHERE PATIENT_ID IN (
    '00P0S2C0',
    '261HTDTY',
    '2XPN1XBS',
    '33PEP1TE',
    '39BDW50Z',
    '5HREMYP4',
    '7TH16LJF',
    '7W3FYDPQ',
    '9GQLZ50J',
    'GYR5QB5F',
    'K3VNKN2P',
    'LS4M6FMC',
    'MS7PBQ8Z',
    'P80QX3EB',
    'RBJZ2XC4',
    'SFKC682M'
)
AND (
       NDC11 IN ('8497600101')
    OR (
           PROCEDURE_CODE IN ('J3490','J3590','J9999')
       AND SERVICE_DATE >= '2026-03-01'
       )
);

In [0]:
SELECT DISTINCT *
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE PATIENT_ID IN (
    '00P0S2C0',
    '261HTDTY',
    '2XPN1XBS',
    '33PEP1TE',
    '39BDW50Z',
    '5HREMYP4',
    '7TH16LJF',
    '7W3FYDPQ',
    '9GQLZ50J',
    'GYR5QB5F',
    'K3VNKN2P',
    'LS4M6FMC',
    'MS7PBQ8Z',
    'P80QX3EB',
    'RBJZ2XC4',
    'SFKC682M'
)
AND (
       NDC11 IN ('8497600101')
    
       AND FILL_DATE >= '2026-03-01'
       );